## Data Pipeline (use balanced dataset)


In [ ]:
#!pip install -q "monai[all]" torch torchvision nibabel tqdm matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import os

print("--- Data Loading & Encoding (No Training) ---")

# 1. Load Data
meta_path = os.path.join(base_drive_path, "Brain_Tumor_Subset/balanced_metadata_gradeaware.csv")
df_meta = pd.read_csv(meta_path)
df_meta['ID'] = df_meta['ID'].astype(str)

rad_path = os.path.join(base_drive_path, "Brain_Tumor_Subset/radiomics_selected_20.csv")
df_rad = pd.read_csv(rad_path)
df_rad['Official_ID'] = df_rad['Official_ID'].astype(str)

# 2. Merge
df = pd.merge(df_meta, df_rad, left_on='ID', right_on='Official_ID', how='inner')

# 3. Features & Exclusions
exclude_cols = [
    'ID', 'Official_ID', 'patient_id', 'Merge_ID',
    'Grade', 'Grade_x', 'Grade_y',
    'Diagnosis', 'Diagnosis_x', 'Diagnosis_y',
    'Grade_Label', 'Diagnosis_Label'
]
raw_feature_cols = [c for c in df.columns if c not in exclude_cols]
X_raw = df[raw_feature_cols].copy()

# 4. Missing Values & Encoding
num_cols = X_raw.select_dtypes(include=[np.number]).columns
X_raw[num_cols] = X_raw[num_cols].fillna(X_raw[num_cols].median())

cat_cols = X_raw.select_dtypes(exclude=[np.number]).columns
if len(cat_cols) > 0:
    X_encoded = pd.get_dummies(X_raw, columns=cat_cols, drop_first=True)
else:
    X_encoded = X_raw

# 5. Εκτύπωση της Λίστας!
print(f"✅ Final Feature Count: {X_encoded.shape[1]}")
print("\n--- ΑΝΤΙΣΤΟΙΧΙΑ 115 ΧΑΡΑΚΤΗΡΙΣΤΙΚΩΝ ---")
for i, col in enumerate(X_encoded.columns):
    print(f"Index {i} ---> {col}")

In [ ]:
# %% Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
import os
import shutil
from glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import gc
from IPython.display import display
import ignite
# MONAI Imports
from monai.data import (
    DataLoader as MonaiDataLoader,
    PersistentDataset,
    partition_dataset,
    decollate_batch
)
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    ScaleIntensityRanged, RandCropByPosNegLabeld, RandFlipd, RandAffined,
    EnsureTyped, AsDiscrete, Activations, RandBiasFieldd,
    RandStdShiftIntensityd, RandScaleIntensityd, RandGaussianNoised,
    CenterSpatialCropd
)
from monai.networks.nets import SwinUNETR, UNet
from monai.losses import DiceCELoss
from monai.metrics import ROCAUCMetric # Η MONAI έχει το AUC
from ignite.metrics import Accuracy, ConfusionMatrix, Precision, Recall # Από το Ignite
from ignite.engine import Events # Απαραίτητο
from monai.inferers import sliding_window_inference
from monai.utils import set_determinism

# Sklearn Imports
#from sklearn.preprocessing import StandardScaler, LabelEncoder

# --- 0. Ρυθμίσεις (Configuration) ---
set_determinism(seed=42)

# --- Βασικές Διαδρομές (Paths) ---
base_drive_path = "/content/drive/MyDrive/Διπλωματικη"
# Χρησιμοποιούμε τα αρχεία από το ισορροπημένο subset
image_dir_ft = os.path.join(base_drive_path, "Brain_Tumor_Subset/volumes_subset")
label_dir_ft = os.path.join(base_drive_path, "Brain_Tumor_Subset/segmentations_subset")

cache_base_dir = os.path.join(base_drive_path, "cache_finetune_mtl")
runs_base_dir = os.path.join(base_drive_path, "runs")

# --- ΔΙΑΔΡΟΜΗ ΓΙΑ ΤΟ ΕΚΠΑΙΔΕΥΜΕΝΟ ΜΟΝΤΕΛΟ ΤΜΗΜΑΤΟΠΟΙΗΣΗΣ ---
finetuned_model_path = os.path.join(runs_base_dir, "segmentation_finetune_custom/best_metric_model_custom.pt")

# --- Hyperparameters ---
ROI_SIZE_FT = (96, 96, 96)
BATCH_SIZE_FT = 2
NUM_EPOCHS_MTL = 160
LEARNING_RATE_MTL = 1e-4
WEIGHT_DECAY_MTL = 1e-5
VAL_INTERVAL = 1
NUM_WORKERS = 2

# --- Model Parameters (Από το Stage 1) ---
IN_CHANNELS = 1
OUT_CHANNELS_SEG = 1
FEATURE_SIZE = 48
USE_CHECKPOINTING = True

# --- Data Keys ---
image_key = "image"
label_key = "label"
clinical_key = "clinical"
grade_key = "grade"
radiomics_key = "radiomics"
diag_key = "diagnosis"
spatial_keys = [image_key, label_key]
all_keys = [image_key, label_key, clinical_key, grade_key, diag_key, radiomics_key]

# --- Device ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
os.makedirs(cache_base_dir, exist_ok=True)
os.makedirs(runs_base_dir, exist_ok=True)

## Balanced Data Preprocessing


In [ ]:
# %% Βήμα 2 (Διορθωμένο): Προετοιμασία Metadata και Δημιουργία Data Dicts
import re
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split # <-- ΝΕΟ IMPORT
from monai.data import partition_dataset # (Αυτό δεν θα το χρησιμοποιήσουμε, αλλά το αφήνουμε)
from IPython.display import display
from glob import glob
import torch
metadata_path = os.path.join(base_drive_path, "Brain_Tumor_Subset/balanced_metadata_gradeaware.csv")
# --- 1. Φόρτωση του Ισορροπημένου Metadata ---
try:
    df_balanced = pd.read_csv(metadata_path)
    print(f"Φορτώθηκε το ισορροπημένο metadata. Σύνολο δειγμάτων: {len(df_balanced)}")
except FileNotFoundError:
    print(f"ΣΦΑΛΜΑ: Δεν βρέθηκε το αρχείο {metadata_path}.")
    # exit()

# --- 2. Επιλογή Βασικών Στηλών ---
columns_to_keep = [
    'ID', 'Grade', 'Diagnosis', # IDs & Στόχοι
    'Age', 'Sex', 'MGMT_Status', 'MGMT_Index', # Features που συζητήσαμε
    'Codeletion_1p19q', 'IDH', 'EOR', 'PreImaging_Biopsy'
]
df_processed = df_balanced[[col for col in columns_to_keep if col in df_balanced.columns]].copy()

# --- 3. Επεξεργασία 'MGMT_Index' (Στρατηγική 1) ---
print("\n--- Επεξεργασία 'MGMT_Index' ---")
if 'MGMT_Index' in df_processed.columns:
    df_processed['MGMT_Index_Is_Unknown'] = (df_processed['MGMT_Index'] == 'unknown').astype(int)
    df_processed['MGMT_Index'] = pd.to_numeric(df_processed['MGMT_Index'], errors='coerce')
    mgmt_index_median = df_processed['MGMT_Index'].median()
    if pd.isna(mgmt_index_median): mgmt_index_median = 0
    df_processed['MGMT_Index'].fillna(mgmt_index_median, inplace=True)
    print(f"Η στήλη 'MGMT_Index' επεξεργάστηκε.")
else:
    print("Η στήλη 'MGMT_Index' δεν βρέθηκε.")

# --- 4. Διαχείριση Άλλων Ελλιπών Τιμών (NaN ή 'unknown') ---
print("\n--- Καθαρισμός Κατηγορικών Στηλών ---")
categorical_features_to_clean = ['Sex', 'MGMT_Status', 'Codeletion_1p19q', 'IDH', 'EOR', 'PreImaging_Biopsy']
for col in categorical_features_to_clean:
    if col in df_processed.columns:
        if df_processed[col].isnull().any():
            mode_val = df_processed[col].mode()[0]
            print(f"Συμπλήρωση NaN στη στήλη '{col}' με '{mode_val}'.")
            df_processed[col].fillna(mode_val, inplace=True)
        if (df_processed[col].astype(str) == 'unknown').any():
            try:
                mode_val = df_processed[col][df_processed[col] != 'unknown'].mode()[0]
            except IndexError:
                mode_val = "Missing"
            print(f"Αντικατάσταση 'unknown' στη στήλη '{col}' με '{mode_val}'.")
            df_processed[col].replace('unknown', mode_val, inplace=True)

if 'Age' in df_processed.columns and df_processed['Age'].isnull().any():
    age_median = df_processed['Age'].median()
    print(f"Συμπλήρωση NaN στην 'Age' με {age_median}.")
    df_processed['Age'].fillna(age_median, inplace=True)

# --- 5. Κωδικοποίηση Κατηγορικών Χαρακτηριστικών (One-Hot Encoding) ---
categorical_features_to_encode = ['Sex', 'Codeletion_1p19q', 'IDH', 'MGMT_Status', 'EOR', 'PreImaging_Biopsy']
existing_categorical_features = [col for col in categorical_features_to_encode if col in df_processed.columns]
if existing_categorical_features:
    print(f"Εφαρμογή One-Hot Encoding στις: {existing_categorical_features}")
    df_processed = pd.get_dummies(df_processed, columns=existing_categorical_features, dummy_na=False)
else:
    print("Δεν βρέθηκαν κατηγορικές στήλες για one-hot encoding.")

# --- 6. Κανονικοποίηση Συνεχών Χαρακτηριστικών (Scaling) ---
scaler = StandardScaler()
continuous_features_to_scale = ['Age', 'MGMT_Index']
existing_continuous_features = [col for col in continuous_features_to_scale if col in df_processed.columns]
if existing_continuous_features:
    print(f"Κανονικοποίηση (Standard Scaling) στις: {existing_continuous_features}")
    df_processed[existing_continuous_features] = scaler.fit_transform(df_processed[existing_continuous_features])
else:
    print("Δεν βρέθηκαν συνεχείς στήλες για scaling.")

# --- 7. Κωδικοποίηση Ετικετών (Label Encoding) ---
print("\n--- Κωδικοποίηση Ετικετών (Targets) ---")
grade_encoder = LabelEncoder()
df_processed['Grade_Label'] = grade_encoder.fit_transform(df_processed['Grade'])
GRADE_CLASSES = grade_encoder.classes_
NUM_CLASSES_GRADE = len(GRADE_CLASSES)
print(f"Κλάσεις Βαθμού (Grade): {list(GRADE_CLASSES)} -> {list(range(NUM_CLASSES_GRADE))}")

diag_encoder = LabelEncoder()
df_processed['Diagnosis_Label'] = diag_encoder.fit_transform(df_processed['Diagnosis'])
DIAG_CLASSES = diag_encoder.classes_
NUM_CLASSES_DIAG = len(DIAG_CLASSES)
print(f"Κλάσεις Διάγνωσης (Diagnosis): {list(DIAG_CLASSES)} -> {list(range(NUM_CLASSES_DIAG))}")

# --- 8. ΑΥΤΟΜΑΤΟΣ Ορισμός Τελικής Λίστας Χαρακτηριστικών ---
print("\n--- Ορισμός Τελικής Λίστας Χαρακτηριστικών ---")
non_feature_cols = [
    'ID', 'Grade', 'Diagnosis', 'Grade_Label', 'Diagnosis_Label',
    'OS', 'Survival_Status', 'BraTS21_ID', 'Segmentation_Cohort',
    'MGMT_Cohort', 'MGMT_Label'
]
clinical_feature_columns = [col for col in df_processed.columns if col not in non_feature_cols and col in df_processed.columns]
CLINICAL_FEATURE_SIZE = len(clinical_feature_columns)
print(f"\n--- Έτοιμο για το Μοντέλο ---")
print(f"CLINICAL_FEATURE_SIZE = {CLINICAL_FEATURE_SIZE}")
print(f"NUM_CLASSES_GRADE = {NUM_CLASSES_GRADE}")
print(f"NUM_CLASSES_DIAG = {NUM_CLASSES_DIAG}")
print(f"\n--- Τελικές Στήλες Χαρακτηριστικών ({CLINICAL_FEATURE_SIZE}): ---")
print(clinical_feature_columns)
# print("\n--- Επεξεργασμένο DataFrame (5 πρώτες σειρές) ---")
# display(df_processed.head())

# --- 9. Εύρεση Αρχείων και Δημιουργία Data Dicts ---
image_files_all = sorted(glob(os.path.join(image_dir_ft, "*.nii.gz")))
label_files_all = sorted(glob(os.path.join(label_dir_ft, "*_tumor_segmentation.nii.gz")))
image_path_map = {os.path.basename(f).split('_')[0]: f for f in image_files_all}
label_path_map = {os.path.basename(f).replace("_tumor_segmentation.nii.gz", ""): f for f in label_files_all}
balanced_ids = set(df_processed['ID'].astype(str))
data_dicts_mtl_balanced = []

for pid_csv in balanced_ids:
    match = re.match(r"([A-Za-z-]+-)(\d+)", pid_csv)
    if match:
        prefix, number_part = match.group(1), match.group(2)
        pid_formatted = f"{prefix}{number_part.zfill(4)}"
    else:
        pid_formatted = pid_csv

    if pid_formatted in image_path_map and pid_formatted in label_path_map:
        patient_meta = df_processed[df_processed['ID'] == pid_csv].iloc[0]
        data_dicts_mtl_balanced.append({
            image_key: image_path_map[pid_formatted],
            label_key: label_path_map[pid_formatted],
            clinical_key: torch.tensor(patient_meta[clinical_feature_columns].values.astype(np.float32), dtype=torch.float32),
            grade_key: torch.tensor(patient_meta['Grade_Label'], dtype=torch.long),
            diag_key: torch.tensor(patient_meta['Diagnosis_Label'], dtype=torch.long)
        })
    else:
        print(f"Warning: Δεν βρέθηκαν αρχεία για το ID {pid_csv} (formatted: {pid_formatted}) στους φακέλους subset.")

print(f"Δημιουργήθηκαν {len(data_dicts_mtl_balanced)} data dicts.")

# --- 10. Νέο Split (Train/Val/Test) με STRATIFICATION ---
print("\n--- Δημιουργία Stratified Split ---")

# Εξαγωγή ετικετών για στρωματοποίηση.
# Χρησιμοποιούμε το Grade, καθώς ήταν η βάση για το balancing.
try:
    stratify_labels = [d[grade_key].item() for d in data_dicts_mtl_balanced]
except KeyError:
    print("ΣΦΑΛΜΑ: Το 'grade_key' δεν βρέθηκε στα data dicts. Έλεγχος Βήματος 9.")
    # exit()

# 1ο Split: 80% train, 20% rest (για val+test)
train_files_mtl, rest_files_mtl, y_train_strat, y_rest_strat = train_test_split(
    data_dicts_mtl_balanced,
    stratify_labels,
    test_size=0.3, # 20% για val/test
    random_state=42,
    stratify=stratify_labels
)

# 2ο Split: 50% val, 50% test από το 'rest' (10% του συνόλου το καθένα)
val_files_mtl, test_files_mtl, y_val_strat, y_test_strat = train_test_split(
    rest_files_mtl,
    y_rest_strat, # Χρησιμοποίησε τις ετικέτες του 'rest' για το 2ο split
    test_size=0.5, # 50% του 20% = 10%
    random_state=42,
    stratify=y_rest_strat
)

print(f"Νέο Stratified Split (MTL): {len(train_files_mtl)} train, {len(val_files_mtl)} val, {len(test_files_mtl)} test.")

# (Προαιρετικό) Έλεγχος κατανομής
print("\nTrain Grade Distribution (Normalized):")
print(pd.Series([d[grade_key].item() for d in train_files_mtl]).value_counts(normalize=True).sort_index())
print("\nValidation Grade Distribution (Normalized):")
print(pd.Series([d[grade_key].item() for d in val_files_mtl]).value_counts(normalize=True).sort_index())
print("\nTest Grade Distribution (Normalized):")
print(pd.Series([d[grade_key].item() for d in test_files_mtl]).value_counts(normalize=True).sort_index())

In [ ]:
pip install iterative-stratification

In [ ]:
# %% Βήμα 2 (Διορθωμένο): Προετοιμασία Metadata και Δημιουργία Data Dicts
import re
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split # Θα το χρησιμοποιήσουμε, αλλά με προσοχή
# --- ΝΕΟ IMPORT ---
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
# ---
from IPython.display import display
from glob import glob
import torch

# --- 1. Φόρτωση του Ισορροπημένου Metadata ---
try:
    df_balanced = pd.read_csv(metadata_path)
    print(f"Φορτώθηκε το ισορροπημένο metadata. Σύνολο δειγμάτων: {len(df_balanced)}")
except FileNotFoundError:
    print(f"ΣΦΑΛΜΑ: Δεν βρέθηκε το αρχείο {metadata_path}.")
    # exit()

# --- 2. Επιλογή Βασικών Στηλών ---
columns_to_keep = [
    'ID', 'Grade', 'Diagnosis', 'Age', 'Sex', 'MGMT_Status', 'MGMT_Index',
    'Codeletion_1p19q', 'IDH', 'EOR', 'PreImaging_Biopsy'
]
df_processed = df_balanced[[col for col in columns_to_keep if col in df_balanced.columns]].copy()

# --- 3. Επεξεργασία 'MGMT_Index' ---
print("\n--- Επεξεργασία 'MGMT_Index' ---")
if 'MGMT_Index' in df_processed.columns:
    df_processed['MGMT_Index_Is_Unknown'] = (df_processed['MGMT_Index'] == 'unknown').astype(int)
    df_processed['MGMT_Index'] = pd.to_numeric(df_processed['MGMT_Index'], errors='coerce')
    mgmt_index_median = df_processed['MGMT_Index'].median()
    if pd.isna(mgmt_index_median): mgmt_index_median = 0
    df_processed['MGMT_Index'].fillna(mgmt_index_median, inplace=True)
    print(f"Η στήλη 'MGMT_Index' επεξεργάστηκε.")
else:
    print("Η στήλη 'MGMT_Index' δεν βρέθηκε.")

# --- 4. Διαχείριση Άλλων Ελλιπών Τιμών ---
print("\n--- Καθαρισμός Κατηγορικών Στηλών ---")
categorical_features_to_clean = ['Sex', 'MGMT_Status', 'Codeletion_1p19q', 'IDH', 'EOR', 'PreImaging_Biopsy']
for col in categorical_features_to_clean:
    if col in df_processed.columns:
        if df_processed[col].isnull().any():
            mode_val = df_processed[col].mode()[0]
            df_processed[col].fillna(mode_val, inplace=True)
        if (df_processed[col].astype(str) == 'unknown').any():
            try:
                mode_val = df_processed[col][df_processed[col] != 'unknown'].mode()[0]
            except IndexError:
                mode_val = "Missing"
            df_processed[col].replace('unknown', mode_val, inplace=True)

if 'Age' in df_processed.columns and df_processed['Age'].isnull().any():
    age_median = df_processed['Age'].median()
    df_processed['Age'].fillna(age_median, inplace=True)

# --- 5. Κωδικοποίηση Κατηγορικών Χαρακτηριστικών (One-Hot Encoding) ---
categorical_features_to_encode = ['Sex', 'Codeletion_1p19q', 'IDH', 'MGMT_Status', 'EOR', 'PreImaging_Biopsy']
existing_categorical_features = [col for col in categorical_features_to_encode if col in df_processed.columns]
if existing_categorical_features:
    print(f"Εφαρμογή One-Hot Encoding στις: {existing_categorical_features}")
    df_processed = pd.get_dummies(df_processed, columns=existing_categorical_features, dummy_na=False)
else:
    print("Δεν βρέθηκαν κατηγορικές στήλες για one-hot encoding.")

# --- 6. Κανονικοποίηση Συνεχών Χαρακτηριστικών (Scaling) ---
scaler = StandardScaler()
continuous_features_to_scale = ['Age', 'MGMT_Index']
existing_continuous_features = [col for col in continuous_features_to_scale if col in df_processed.columns]
if existing_continuous_features:
    print(f"Κανονικοποίηση (Standard Scaling) στις: {existing_continuous_features}")
    df_processed[existing_continuous_features] = scaler.fit_transform(df_processed[existing_continuous_features])
else:
    print("Δεν βρέθηκαν συνεχείς στήλες για scaling.")

# --- 7. Κωδικοποίηση Ετικετών (Label Encoding) ---
print("\n--- Κωδικοποίηση Ετικετών (Targets) ---")
grade_encoder = LabelEncoder()
df_processed['Grade_Label'] = grade_encoder.fit_transform(df_processed['Grade'])
GRADE_CLASSES = grade_encoder.classes_
NUM_CLASSES_GRADE = len(GRADE_CLASSES)
print(f"Κλάσεις Βαθμού (Grade): {list(GRADE_CLASSES)} -> {list(range(NUM_CLASSES_GRADE))}")

diag_encoder = LabelEncoder()
df_processed['Diagnosis_Label'] = diag_encoder.fit_transform(df_processed['Diagnosis'])
DIAG_CLASSES = diag_encoder.classes_
NUM_CLASSES_DIAG = len(DIAG_CLASSES)
print(f"Κλάσεις Διάγνωσης (Diagnosis): {list(DIAG_CLASSES)} -> {list(range(NUM_CLASSES_DIAG))}")

# --- 8. ΑΥΤΟΜΑΤΟΣ Ορισμός Τελικής Λίστας Χαρακτηριστικών ---
print("\n--- Ορισμός Τελικής Λίστας Χαρακτηριστικών ---")
non_feature_cols = [
    'ID', 'Grade', 'Diagnosis', 'Grade_Label', 'Diagnosis_Label',
    'OS', 'Survival_Status', 'BraTS21_ID', 'Segmentation_Cohort',
    'MGMT_Cohort', 'MGMT_Label'
]
clinical_feature_columns = [col for col in df_processed.columns if col not in non_feature_cols and col in df_processed.columns]
CLINICAL_FEATURE_SIZE = len(clinical_feature_columns)
print(f"\n--- Έτοιμο για το Μοντέλο ---")
print(f"CLINICAL_FEATURE_SIZE = {CLINICAL_FEATURE_SIZE}")
print(f"NUM_CLASSES_GRADE = {NUM_CLASSES_GRADE}")
print(f"NUM_CLASSES_DIAG = {NUM_CLASSES_DIAG}")
print(f"\n--- Τελικές Στήλες Χαρακτηριστικών ({CLINICAL_FEATURE_SIZE}): ---")
print(clinical_feature_columns)

# --- 9. Εύρεση Αρχείων και Δημιουργία Data Dicts ---
# ... (Αυτή η λογική παραμένει ακριβώς η ίδια με πριν) ...
image_files_all = sorted(glob(os.path.join(image_dir_ft, "*.nii.gz")))
label_files_all = sorted(glob(os.path.join(label_dir_ft, "*_tumor_segmentation.nii.gz")))
image_path_map = {os.path.basename(f).split('_')[0]: f for f in image_files_all}
label_path_map = {os.path.basename(f).replace("_tumor_segmentation.nii.gz", ""): f for f in label_files_all}
balanced_ids = set(df_processed['ID'].astype(str))
data_dicts_mtl_balanced = []
# Χρειαζόμαστε έναν τρόπο να αντιστοιχίσουμε τα data_dicts πίσω στο df_processed για το stratification
df_processed['pid_formatted'] = df_processed['ID'].astype(str).apply(
    lambda pid_csv: f"{re.match(r'([A-Za-z-]+-)(\d+)', pid_csv).group(1)}{re.match(r'([A-Za-z-]+-)(\d+)', pid_csv).group(2).zfill(4)}" if re.match(r"([A-Za-z-]+-)(\d+)", pid_csv) else pid_csv
)
df_processed.set_index('pid_formatted', inplace=True)

for pid_formatted in image_path_map.keys():
    if pid_formatted in label_path_map and pid_formatted in df_processed.index:
        patient_meta = df_processed.loc[pid_formatted]
        data_dicts_mtl_balanced.append({
            "pid_formatted": pid_formatted, # Προσθέτουμε το ID για το split
            image_key: image_path_map[pid_formatted],
            label_key: label_path_map[pid_formatted],
            clinical_key: torch.tensor(patient_meta[clinical_feature_columns].values.astype(np.float32), dtype=torch.float32),
            grade_key: torch.tensor(patient_meta['Grade_Label'], dtype=torch.long),
            diag_key: torch.tensor(patient_meta['Diagnosis_Label'], dtype=torch.long)
        })
    else:
        print(f"Warning: Δεν βρέθηκαν αρχεία ή metadata για το ID {pid_formatted}.")
print(f"Δημιουργήθηκαν {len(data_dicts_mtl_balanced)} data dicts.")

# --- 10. Νέο Split (Train/Val/Test) με MULTILABEL STRATIFICATION ---
print("\n--- Δημιουργία Stratified Split (Βάσει Grade και Diagnosis) ---")

# Δημιουργία του πίνακα Y (labels) για stratification
# Ο πίνακας Y πρέπει να έχει σχήμα [num_samples, num_labels_to_stratify]
Y_stratify = df_processed.loc[
    [d['pid_formatted'] for d in data_dicts_mtl_balanced], # Παίρνουμε τα PIDs με τη σωστή σειρά
    ['Grade_Label', 'Diagnosis_Label'] # Οι δύο στήλες που θέλουμε
].values

# 1ο Split: 80% train, 20% rest (για val+test)
msss_train_rest = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_indices, rest_indices = next(msss_train_rest.split(data_dicts_mtl_balanced, Y_stratify))

train_files_mtl = [data_dicts_mtl_balanced[i] for i in train_indices]
rest_files_mtl = [data_dicts_mtl_balanced[i] for i in rest_indices]
Y_rest_stratify = Y_stratify[rest_indices] # Οι ετικέτες για το 2ο split

# 2ο Split: 50% val, 50% test από το 'rest' (10% του συνόλου το καθένα)
msss_val_test = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_indices, test_indices = next(msss_val_test.split(rest_files_mtl, Y_rest_stratify))

val_files_mtl = [rest_files_mtl[i] for i in val_indices]
test_files_mtl = [rest_files_mtl[i] for i in test_indices]

print(f"Νέο Stratified Split (MTL): {len(train_files_mtl)} train, {len(val_files_mtl)} val, {len(test_files_mtl)} test.")

# (Προαιρετικό) Έλεγχος κατανομής
print("\n--- Κατανομή Diagnosis στο Validation Set (Έλεγχος) ---")
print(pd.Series([d[diag_key].item() for d in val_files_mtl]).value_counts(normalize=True).sort_index())
print("\n--- Κατανομή Diagnosis στο Test Set (Έλεγχος) ---")
print(pd.Series([d[diag_key].item() for d in test_files_mtl]).value_counts(normalize=True).sort_index())

In [ ]:
# %% Βήμα 3 (Διορθωμένο): Transforms & DataLoaders (MTL)

# --- Βεβαιώσου ότι έχει γίνει import η list_data_collate ---
from monai.data import (
    DataLoader as MonaiDataLoader,
    PersistentDataset,
    partition_dataset,
    decollate_batch,
    list_data_collate # <-- ΣΗΜΑΝΤΙΚΟ
)
from monai.transforms import CenterSpatialCropd # (Βεβαιώσου ότι είναι imported)
# ----------------------------------------------------

# Κλειδιά που θέλουμε ως float32 Tensors
float_tensor_keys = [image_key, label_key, clinical_key]
# Κλειδιά που θέλουμε ως long Tensors (για το CrossEntropyLoss)
long_tensor_keys = [grade_key, diag_key]

# --- Transforms Εκπαίδευσης (Training) ---
train_transforms_mtl = Compose([
    LoadImaged(keys=spatial_keys, image_only=False),
    EnsureChannelFirstd(keys=spatial_keys),
    Orientationd(keys=spatial_keys, axcodes="RAS"),
    Spacingd(keys=spatial_keys, pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys=image_key, a_min=0, a_max=3000, b_min=0.0, b_max=1.0, clip=True),
    RandCropByPosNegLabeld(
        keys=spatial_keys,
        label_key=label_key,
        spatial_size=ROI_SIZE_FT,
        pos=1.0, neg=1.0,
        num_samples=2,
        image_key=image_key, image_threshold=0.0
    ),
    # ... (Όλα τα Rand augmentations: Flip, Affine, GaussianNoise, StdShift, Scale, BiasField) ...
    RandFlipd(keys=spatial_keys, prob=0.5, spatial_axis=0),
    RandFlipd(keys=spatial_keys, prob=0.5, spatial_axis=1),
    RandFlipd(keys=spatial_keys, prob=0.5, spatial_axis=2),
    RandAffined(keys=spatial_keys, prob=0.2, rotate_range=(0.05, 0.05, 0.05), scale_range=(0.1, 0.1, 0.1), mode=("bilinear", "nearest"), padding_mode="zeros"),
    RandGaussianNoised(keys=image_key, prob=0.2, mean=0.0, std=0.01),
    RandStdShiftIntensityd(keys=image_key, factors=0.1, prob=0.3),
    RandScaleIntensityd(keys=image_key, factors=0.1, prob=0.3),
    RandBiasFieldd(keys=image_key, prob=0.3),

    # --- ΔΙΟΡΘΩΣΗ 2: Εφάρμοσε το EnsureTyped σε ΟΛΑ τα κλειδιά ---
    EnsureTyped(keys=float_tensor_keys, dtype=torch.float32),
    EnsureTyped(keys=long_tensor_keys, dtype=torch.long),
])

# --- Transforms Επικύρωσης & Δοκιμής (Validation & Test) ---
val_transforms_mtl = Compose([
    LoadImaged(keys=spatial_keys, image_only=False),
    EnsureChannelFirstd(keys=spatial_keys),
    Orientationd(keys=spatial_keys, axcodes="RAS"),
    Spacingd(keys=spatial_keys, pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys=image_key, a_min=0, a_max=3000, b_min=0.0, b_max=1.0, clip=True),
    CenterSpatialCropd(keys=spatial_keys, roi_size=ROI_SIZE_FT),

    # --- ΔΙΟΡΘΩΣΗ 2: Εφάρμοσε το EnsureTyped σε ΟΛΑ τα κλειδιά ---
    EnsureTyped(keys=float_tensor_keys, dtype=torch.float32),
    EnsureTyped(keys=long_tensor_keys, dtype=torch.long),
])

test_transforms_mtl = val_transforms_mtl

# --- DataLoaders ---
mtl_cache_dir = os.path.join(cache_base_dir, "mtl_run_cascaded")
print(f"Δημιουργία cache στο: {mtl_cache_dir}")

# --- ΣΗΜΑΝΤΙΚΟ: Καθάρισε την παλιά "χαλασμένη" cache ---
print("Καθαρισμός παλιάς cache (αν υπάρχει)...")
shutil.rmtree(mtl_cache_dir, ignore_errors=True)
# ----------------------------------------------------

train_ds_mtl = PersistentDataset(
    data=train_files_mtl,
    transform=train_transforms_mtl,
    cache_dir=os.path.join(mtl_cache_dir, "train")
)
val_ds_mtl = PersistentDataset(
    data=val_files_mtl,
    transform=val_transforms_mtl,
    cache_dir=os.path.join(mtl_cache_dir, "val")
)
test_ds_mtl = PersistentDataset(
    data=test_files_mtl,
    transform=test_transforms_mtl,
    cache_dir=os.path.join(mtl_cache_dir, "test")
)

train_loader_mtl = MonaiDataLoader(
    train_ds_mtl,
    batch_size=BATCH_SIZE_FT,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=list_data_collate # <-- *** ΔΙΟΡΘΩΣΗ 1 ***
)
val_loader_mtl = MonaiDataLoader(
    val_ds_mtl,
    batch_size=BATCH_SIZE_FT * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
    # (Δεν χρειάζεται collate_fn εδώ, το CenterSpatialCrop βγάζει 1 dict)
)
test_loader_mtl = MonaiDataLoader(
    test_ds_mtl,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("\nDataLoaders για Cascaded MTL δημιουργήθηκαν.")
print(f"Training batches: {len(train_loader_mtl)}")
print(f"Validation batches: {len(val_loader_mtl)}")
print(f"Test batches: {len(test_loader_mtl)}")

## Cascade

In [ ]:
# %% Βήμα 4 (ΔΙΟΡΘΩΜΕΝΟ): Ορισμός του Μοντέλου (Stage 2 - Cascaded με Normalization)
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.networks.nets import SwinUNETR
import os

print("Ορισμός Μοντέλου CascadedClassifier (με Pre-Fusion Normalization)...")

class CascadedClassifier(nn.Module):
    def __init__(self,
                 in_channels_img, out_channels_seg, feature_size, use_checkpoint,
                 num_classes_grade, num_classes_diag, clinical_feature_size,
                 proj_dim_clinical=64, finetuned_seg_model_path=None):

        super().__init__()

        # --- 1. Φόρτωση και Πάγωμα του SwinUNETR ---
        self.swin_unetr_feature_extractor = SwinUNETR(
            in_channels=in_channels_img, out_channels=out_channels_seg,
            feature_size=feature_size, use_checkpoint=use_checkpoint
        )
        if finetuned_seg_model_path and os.path.exists(finetuned_seg_model_path):
            print(f"Φόρτωση checkpoint τμηματοποίησης από: {finetuned_seg_model_path}")
            checkpoint = torch.load(finetuned_seg_model_path, map_location=torch.device('cpu'))
            try:
                self.swin_unetr_feature_extractor.load_state_dict(checkpoint['model_state'], strict=False)
                print(f"Επιτυχής φόρτωση βαρών μοντέλου τμηματοποίησης.")
            except Exception as e: print(f"Σφάλμα φόρτωσης state_dict: {e}")
        else: print("ΠΡΟΣΟΧΗ: Δεν βρέθηκε εκπαιδευμένο μοντέλο.")

        print("--- ΠΑΓΩΜΑ (FREEZING) ολόκληρου του SwinUNETR extractor ---")
        for param in self.swin_unetr_feature_extractor.parameters():
            param.requires_grad = False
        # -----------------------------------------------------------

        # --- 2. Encoder Κλινικών Δεδομένων (Εκπαιδεύσιμος) ---
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_feature_size, proj_dim_clinical),
            nn.LayerNorm(proj_dim_clinical),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # --- 3. Fusion & Classification Heads (Εκπαιδεύσιμα) ---
        encoder_feature_dim = feature_size * 16 # (768)

        # *** ΝΕΟ: Layers Κανονικοποίησης (Normalization) ***
        self.encoder_norm = nn.LayerNorm(encoder_feature_dim)
        self.clinical_norm = nn.LayerNorm(proj_dim_clinical)
        # ***

        fused_dim = encoder_feature_dim + proj_dim_clinical # (768 + 64)
        print(f"Διαστάσεις Fusion (μετά από Norm): Encoder({encoder_feature_dim}) + Clinical({proj_dim_clinical}) = {fused_dim}")

        # Απλοποιούμε το MLP για να ξεκινήσει πιο εύκολα
        self.fusion_mlp = nn.Sequential(
            nn.Linear(fused_dim, 256), # Πιο μικρό layer
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.3) # Μικρότερο dropout
        )

        self.grade_head = nn.Linear(256, num_classes_grade)
        self.diagnosis_head = nn.Linear(256, num_classes_diag)

    def forward(self, x_img, x_clinical):

        # 1. Εξαγωγή Features (ΜΟΝΟ ENCODER)
        with torch.no_grad():
            encoder_features_list = self.swin_unetr_feature_extractor.swinViT(x_img)
            encoder_features = encoder_features_list[-1]

        # 2. Pooling
        encoder_pooled = F.adaptive_avg_pool3d(encoder_features, 1).flatten(1) # [B, 768]

        # 3. Clinical Features
        clinical_features = self.clinical_encoder(x_clinical) # [B, 64]

        # *** ΝΕΟ: Κανονικοποίηση πριν τη συνένωση ***
        encoder_normed = self.encoder_norm(encoder_pooled)
        clinical_normed = self.clinical_norm(clinical_features)
        # ***

        # 4. Fusion
        fused_features = torch.cat((encoder_normed, clinical_normed), dim=1) # [B, 768 + 64]

        # 5. Classification Heads
        fused_output = self.fusion_mlp(fused_features)
        logits_grade = self.grade_head(fused_output)
        logits_diag = self.diagnosis_head(fused_output)

        return logits_grade, logits_diag

print("Ορισμός ΑΠΛΟΠΟΙΗΜΕΝΟΥ Μοντέλου CascadedClassifier (v4 - με Pre-Fusion Norm) ολοκληρώθηκε.")

In [ ]:
# %% Βήμα 4 (ΔΙΟΡΘΩΜΕΝΟ): Ορισμός του Μοντέλου Ταξινόμησης (Stage 2 Classifier - Cascaded)
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.networks.nets import SwinUNETR
import os

print("Ορισμός ΑΠΛΟΠΟΙΗΜΕΝΟΥ Μοντέλου CascadedClassifier (Μόνο Encoder)...")

class CascadedClassifierSimple(nn.Module):
    def __init__(self,
                 in_channels_img, out_channels_seg, feature_size, use_checkpoint,
                 num_classes_grade, num_classes_diag, clinical_feature_size,
                 proj_dim_clinical=64, finetuned_seg_model_path=None):

        super().__init__()

        # --- 1. Φόρτωσε το ΕΚΠΑΙΔΕΥΜΕΝΟ Μοντέλο Τμηματοποίησης (SwinUNETR) ---
        # *** ΟΡΙΣΜΟΣ ΜΕ ΤΟ ΣΩΣΤΟ ΟΝΟΜΑ ***
        self.swin_unetr_feature_extractor = SwinUNETR(
            in_channels=in_channels_img,
            out_channels=out_channels_seg,
            feature_size=feature_size,
            use_checkpoint=use_checkpoint
        )

        if finetuned_seg_model_path and os.path.exists(finetuned_seg_model_path):
            print(f"Φόρτωση checkpoint τμηματοποίησης από: {finetuned_seg_model_path}")
            checkpoint = torch.load(finetuned_seg_model_path, map_location=torch.device('cpu'))
            try:
                # Χρήση του σωστού ονόματος
                self.swin_unetr_feature_extractor.load_state_dict(checkpoint['model_state'], strict=False)
                print(f"Επιτυχής φόρτωση βαρών μοντέλου τμηματοποίησης.")
            except Exception as e:
                print(f"Σφάλμα φόρτωσης state_dict: {e}")
        else:
            print("ΠΡΟΣΟΧΗ: Δεν βρέθηκε εκπαιδευμένο μοντέλο.")

        # --- 2. ΠΑΓΩΣΕ (FREEZE) ΟΛΟ ΤΟ ΜΟΝΤΕΛΟ ΤΜΗΜΑΤΟΠΟΙΗΣΗΣ ---
        print("--- ΠΑΓΩΜΑ (FREEZING) ολόκληρου του SwinUNETR extractor ---")
        for param in self.swin_unetr_feature_extractor.parameters(): # Χρήση του σωστού ονόματος
            param.requires_grad = False
        # -----------------------------------------------------------

        # --- 3. Encoder Κλινικών Δεδομένων (Αυτό θα εκπαιδευτεί) ---
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_feature_size, proj_dim_clinical),
            nn.LayerNorm(proj_dim_clinical),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # --- 4. Fusion & Classification Heads (Αυτά θα εκπαιδευτούν) ---
        encoder_feature_dim = feature_size * 16 # (768)
        fused_dim = encoder_feature_dim + proj_dim_clinical # (768 + 64)
        print(f"Διαστάσεις Fusion: Encoder({encoder_feature_dim}) + Clinical({proj_dim_clinical}) = {fused_dim}")

        self.fusion_mlp = nn.Sequential(
            nn.Linear(fused_dim, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        self.grade_head = nn.Linear(512, num_classes_grade)
        self.diagnosis_head = nn.Linear(512, num_classes_diag)

    def forward(self, x_img, x_clinical):

        # 1. Εξαγωγή Features (ΜΟΝΟ ENCODER)
        with torch.no_grad():
            encoder_features_list = self.swin_unetr_feature_extractor.swinViT(x_img) # Χρήση του σωστού ονόματος
            encoder_features = encoder_features_list[-1]

        # 2. Pooling
        encoder_pooled = F.adaptive_avg_pool3d(encoder_features, 1).flatten(1)

        # 3. Clinical Features
        clinical_features = self.clinical_encoder(x_clinical)

        # 4. Fusion
        fused_features = torch.cat((encoder_pooled, clinical_features), dim=1)

        # 5. Classification Heads
        fused_output = self.fusion_mlp(fused_features)
        logits_grade = self.grade_head(fused_output)
        logits_diag = self.diagnosis_head(fused_output)

        return logits_grade, logits_diag

print("Ορισμός ΑΠΛΟΠΟΙΗΜΕΝΟΥ Μοντέλου CascadedClassifier ολοκληρώθηκε.")

In [ ]:
# %% Βήμα 4 (ΔΙΟΡΘΩΜΕΝΟ): Ορισμός Μοντέλου (Stage 2 - Ξεχωριστά MLPs, Χωρίς Clinical Encoder)
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.networks.nets import SwinUNETR
import os

print("Ορισμός Μοντέλου CascadedClassifier (Ξεχωριστά MLPs, Χωρίς Clinical Encoder)...")

class CascadedClassifierSeparateHeads(nn.Module):
    def __init__(self,
                 in_channels_img, out_channels_seg, feature_size, use_checkpoint,
                 num_classes_grade, num_classes_diag, clinical_feature_size,
                 finetuned_seg_model_path=None):

        super().__init__()

        # --- 1. Φόρτωση και Πάγωμα του SwinUNETR ---
        self.swin_unetr_feature_extractor = SwinUNETR(
            in_channels=in_channels_img,
            out_channels=out_channels_seg,
            feature_size=feature_size,
            use_checkpoint=use_checkpoint
        )

        if finetuned_seg_model_path and os.path.exists(finetuned_seg_model_path):
            print(f"Φόρτωση checkpoint τμηματοποίησης από: {finetuned_seg_model_path}")
            checkpoint = torch.load(finetuned_seg_model_path, map_location=torch.device('cpu'))
            try:
                self.swin_unetr_feature_extractor.load_state_dict(checkpoint['model_state'], strict=False)
                print(f"Επιτυχής φόρτωση βαρών μοντέλου τμηματοποίησης.")
            except Exception as e: print(f"Σφάλμα φόρτωσης state_dict: {e}")
        else: print("ΠΡΟΣΟΧΗ: Δεν βρέθηκε εκπαιδευμένο μοντέλο.")

        print("--- ΠΑΓΩΜΑ (FREEZING) ολόκληρου του SwinUNETR extractor ---")
        for param in self.swin_unetr_feature_extractor.parameters():
            param.requires_grad = False
        # -----------------------------------------------------------

        # --- 2. Hooks για Εξαγωγή Features ---
        self.encoder_features_out = None
        self.decoder_features_out = None
        self.swin_unetr_feature_extractor.swinViT.register_forward_hook(self.hook_encoder_features)
        self.swin_unetr_feature_extractor.decoder1.register_forward_hook(self.hook_decoder_features)

        # --- 3. Encoder Κλινικών Δεδομένων (ΑΦΑΙΡΕΘΗΚΕ) ---

        # --- 4. Fusion & Classification Heads (Εκπαιδεύσιμα) ---
        encoder_feature_dim = feature_size * 16 # (768)
        decoder_feature_dim = feature_size # (48)

        # Η διάσταση εισόδου για τα MLPs (ΧΩΡΙΣ proj_dim_clinical)
        fused_dim = encoder_feature_dim + decoder_feature_dim + clinical_feature_size
        print(f"Διαστάσεις Fusion: Encoder({encoder_feature_dim}) + Decoder({decoder_feature_dim}) + Clinical({clinical_feature_size}) = {fused_dim}")

        # --- ΞΕΧΩΡΙΣΤΑ MLPs ---
        self.grade_mlp = nn.Sequential(
            nn.Linear(fused_dim, 256), # Είσοδος_ fuser_dim
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.grade_head = nn.Linear(256, num_classes_grade)

        self.diag_mlp = nn.Sequential(
            nn.Linear(fused_dim, 256), # Είσοδος_ fuser_dim
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.diagnosis_head = nn.Linear(256, num_classes_diag)
        # ---

    # --- Hook functions ---
    def hook_encoder_features(self, module, input, output):
        self.encoder_features_out = output[-1]
    def hook_decoder_features(self, module, input, output):
        self.decoder_features_out = output

    def forward(self, x_img, x_clinical):

        # 1. Εξαγωγή Features (με Hooks)
        with torch.no_grad():
            _ = self.swin_unetr_feature_extractor(x_img)

        # 2. Pooling
        encoder_pooled = F.adaptive_avg_pool3d(self.encoder_features_out, 1).flatten(1)
        decoder_pooled = F.adaptive_avg_pool3d(self.decoder_features_out, 1).flatten(1)

        # 3. Clinical Features (ΑΚΑΤΕΡΓΑΣΤΑ - ΧΩΡΙΣ ENCODER)
        # x_clinical είναι ήδη [B, 25]

        # 4. Fusion
        fused_features = torch.cat((encoder_pooled, decoder_pooled, x_clinical), dim=1)

        # 5. Classification Heads (ΞΕΧΩΡΙΣΤΟΙ ΔΡΟΜΟΙ)
        logits_grade = self.grade_head(self.grade_mlp(fused_features))
        logits_diag = self.diagnosis_head(self.diag_mlp(fused_features))

        return logits_grade, logits_diag

print("Ορισμός Μοντέλου CascadedClassifier (v5 - Ξεχωριστά MLPs, Χωρίς Clinical Encoder) ολοκληρώθηκε.")

In [ ]:
# %% Βήμα 5: Ορισμός Loss & Εκπαίδευση (Stage 2 - Cascaded, ΧΕΙΡΟΚΙΝΗΤΕΣ ΜΕΤΡΙΚΕΣ)
import torch.nn as nn
from tqdm import tqdm
import pandas as pd
import numpy as np
import os
import gc
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

# --- Συναρτήσεις Απώλειας (Loss Functions) ---
loss_grade = nn.CrossEntropyLoss()
loss_diag = nn.CrossEntropyLoss()
w_grade = 0.5
w_diag = 0.5
def classification_loss_fn(pred_grade, true_grade, pred_diag, true_diag):
    l_g = loss_grade(pred_grade, true_grade)
    l_d = loss_diag(pred_diag, true_diag)
    total_loss = (w_grade * l_g) + (w_diag * l_d)
    return total_loss, l_g, l_d

# --- Ρυθμίσεις για το Stage 2 run ---
output_dir_stage2 = os.path.join(runs_base_dir, "classification_cascaded_v1")
history_csv_path_stage2 = os.path.join(output_dir_stage2, "training_history_cascaded.csv")
latest_checkpoint_path_stage2 = os.path.join(output_dir_stage2, "latest_model_cascaded.pt")
best_checkpoint_path_stage2 = os.path.join(output_dir_stage2, "best_metric_model_cascaded.pt")
os.makedirs(output_dir_stage2, exist_ok=True)


# --- Αρχικοποίηση Μοντέλου Stage 2 ---
try:
    # *** ΑΛΛΑΓΗ ΕΔΩ: Κάλεσε το CascadedClassifierSeparateHeads ***
    model_stage2 = CascadedClassifierSeparateHeads(
        in_channels_img=IN_CHANNELS, out_channels_seg=OUT_CHANNELS_SEG,
        feature_size=FEATURE_SIZE, use_checkpoint=USE_CHECKPOINTING,
        num_classes_grade=NUM_CLASSES_GRADE, num_classes_diag=NUM_CLASSES_DIAG,
        clinical_feature_size=CLINICAL_FEATURE_SIZE, # Αυτό είναι τώρα 25
        finetuned_seg_model_path=finetuned_model_path
    ).to(DEVICE)
except NameError:
    print("ΣΦΑΛΜΑ: Η κλάση 'CascadedClassifierSeparateHeads' δεν έχει οριστεί. Εκτέλεσε ξανά το Κελί 4.")
    # exit()

# Ο Optimizer βλέπει ΜΟΝΟ τις παραμέτρους που εκπαιδεύονται
params_to_train = [
    {'params': model_stage2.grade_mlp.parameters()},
    {'params': model_stage2.grade_head.parameters()},
    {'params': model_stage2.diag_mlp.parameters()},
    {'params': model_stage2.diagnosis_head.parameters()}
]
print(f"Σύνολο παραμέτρων προς εκπαίδευση: {sum(p.numel() for group in params_to_train for p in group['params'])}")
optimizer_stage2 = torch.optim.AdamW(params_to_train, lr=LEARNING_RATE_MTL, weight_decay=WEIGHT_DECAY_MTL)
scheduler_stage2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_stage2, T_max=NUM_EPOCHS_MTL)

# --- ΧΩΡΙΣ GradScaler ---
best_val_metric = -1.0
start_epoch = 0
history = {
    "epoch": [], "train_loss_total": [], "train_loss_grade": [], "train_loss_diag": [],
    "val_loss_total": [], "val_loss_grade": [], "val_loss_diag": [],
    "val_acc_grade": [], "val_auc_grade": [],
    "val_acc_diag": [], "val_auc_diag": [],
    "val_metric_combined": []
}

# --- Resuming Logic (Χωρίς Scaler State) ---
if os.path.exists(latest_checkpoint_path_stage2):
    print(f"Φόρτωση checkpoint: {latest_checkpoint_path_stage2}")
    checkpoint = torch.load(latest_checkpoint_path_stage2, map_location=DEVICE)
    try:
        model_stage2.clinical_encoder.load_state_dict(checkpoint['clinical_encoder_state'])
        model_stage2.fusion_mlp.load_state_dict(checkpoint['fusion_mlp_state'])
        model_stage2.grade_head.load_state_dict(checkpoint['grade_head_state'])
        model_stage2.diagnosis_head.load_state_dict(checkpoint['diagnosis_head_state'])
        #optimizer_stage2.load_state_dict(checkpoint['optimizer_state'])
        if checkpoint.get('scheduler_state'): scheduler_stage2.load_state_dict(checkpoint['scheduler_state'])
        start_epoch = checkpoint.get('epoch', 0)
        best_val_metric = checkpoint.get('best_val_metric', -1.0)
        print(f"Συνέχιση Stage 2 από εποχή {start_epoch + 1}")
        if os.path.exists(history_csv_path_stage2):
             df_history_old = pd.read_csv(history_csv_path_stage2)
             df_history_old = df_history_old[df_history_old['epoch'] <= start_epoch]
             history = df_history_old.to_dict('list')
             for key in history.keys():
                 if not isinstance(history[key], list): history[key] = list(history[key])
    except Exception as e:
        print(f"Σφάλμα φόρτωσης checkpoint: {e}. Έναρξη από την αρχή.")
        start_epoch = 0; best_val_metric = -1.0; history = {k: [] for k in history.keys()}
else:
    print("Έναρξη Stage 2 εκπαίδευσης από την αρχή.")


print(f"\n--- Έναρξη Stage 2 Classification Loop από εποχή {start_epoch + 1} (ΧΩΡΙΣ AMP) ---")

try:
    _ = train_loader_mtl
except NameError:
    print("\n!!! ΣΦΑΛΜΑ: Ο 'train_loader_mtl' δεν έχει οριστεί. Εκτέλεσε το Κελί 3.")
    # exit()

for epoch in range(start_epoch, NUM_EPOCHS_MTL):
    current_epoch = epoch + 1
    print(f"\n===== Epoch {current_epoch}/{NUM_EPOCHS_MTL} =====")
    model_stage2.train()
    model_stage2.swin_unetr_feature_extractor.eval()

    epoch_loss_total, epoch_loss_grade, epoch_loss_diag = 0.0, 0.0, 0.0
    step = 0
    pbar_train = tqdm(train_loader_mtl, desc=f"Training Stage 2 Epoch {current_epoch}")

    for batch_data in pbar_train:
        step += 1
        inputs = batch_data[image_key].to(DEVICE)
        data_clinical = batch_data[clinical_key].to(DEVICE)
        labels_grade = batch_data[grade_key].to(DEVICE)
        labels_diag = batch_data[diag_key].to(DEVICE)

        optimizer_stage2.zero_grad(set_to_none=True)

        logits_grade, logits_diag = model_stage2(inputs, data_clinical)
        total_loss, l_g, l_d = classification_loss_fn(
            logits_grade, labels_grade, logits_diag, labels_diag
        )

        if not torch.isfinite(total_loss):
            print(f"!!! WARNING: Non-finite loss ({total_loss.item() if total_loss.numel() == 1 else 'NaN/Inf'}) detected. Skipping step.")
            continue

        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(
            filter(lambda p: p.requires_grad, model_stage2.parameters()), 1.0
        )
        optimizer_stage2.step()

        epoch_loss_total += total_loss.item(); epoch_loss_grade += l_g.item(); epoch_loss_diag += l_d.item()
        pbar_train.set_postfix(total_loss=f"{total_loss.item():.4f}")
        del inputs, data_clinical, labels_grade, labels_diag, logits_grade, logits_diag, total_loss, l_g, l_d
        if DEVICE == 'cuda': torch.cuda.empty_cache()

    avg_epoch_loss_total = epoch_loss_total / step if step > 0 else np.nan
    avg_epoch_loss_grade = epoch_loss_grade / step if step > 0 else np.nan
    avg_epoch_loss_diag = epoch_loss_diag / step if step > 0 else np.nan
    current_lr = optimizer_stage2.param_groups[0]['lr']
    print(f"Epoch {current_epoch} Avg Train Loss: {avg_epoch_loss_total:.4f} (Grade: {avg_epoch_loss_grade:.4f}, Diag: {avg_epoch_loss_diag:.4f}), LR: {current_lr:.6f}")

    history["epoch"].append(current_epoch)
    history["train_loss_total"].append(avg_epoch_loss_total); history["train_loss_grade"].append(avg_epoch_loss_grade); history["train_loss_diag"].append(avg_epoch_loss_diag)

    # --- Validation ---
    if current_epoch % VAL_INTERVAL == 0:
        model_stage2.eval()

        # --- ΛΙΣΤΕΣ ΓΙΑ ΧΕΙΡΟΚΙΝΗΤΗ ΣΥΛΛΟΓΗ ---
        all_labels_grade = []
        all_preds_grade_indices = []
        all_preds_grade_probs = []

        all_labels_diag = []
        all_preds_diag_indices = []
        all_preds_diag_probs = []
        # ---

        epoch_val_loss_total, epoch_val_loss_grade, epoch_val_loss_diag = 0.0, 0.0, 0.0
        val_step = 0
        pbar_val = tqdm(val_loader_mtl, desc=f"Validation Stage 2 Epoch {current_epoch}")

        with torch.no_grad():
            for val_data in pbar_val:
                val_step += 1
                val_inputs = val_data[image_key].to(DEVICE); val_data_clinical = val_data[clinical_key].to(DEVICE)
                val_labels_grade = val_data[grade_key].to(DEVICE); val_labels_diag = val_data[diag_key].to(DEVICE)
                val_logits_grade, val_logits_diag = model_stage2(val_inputs, val_data_clinical)
                total_loss, l_g, l_d = classification_loss_fn(val_logits_grade, val_labels_grade, val_logits_diag, val_labels_diag)
                if torch.isfinite(total_loss): epoch_val_loss_total += total_loss.item(); epoch_val_loss_grade += l_g.item(); epoch_val_loss_diag += l_d.item()

                # --- ΧΕΙΡΟΚΙΝΗΤΗ ΣΥΛΛΟΓΗ ΜΕΤΡΙΚΩΝ ---

                # Υπολογισμός Probabilities (για AUC)
                probs_grade = F.softmax(val_logits_grade, dim=1)
                probs_diag = F.softmax(val_logits_diag, dim=1)

                # Υπολογισμός Indices (για Accuracy/CM)
                indices_grade = torch.argmax(val_logits_grade, dim=1)
                indices_diag = torch.argmax(val_logits_diag, dim=1)

                # Προσθήκη στις λίστες (στο CPU)
                all_preds_grade_indices.append(indices_grade.cpu())
                all_labels_grade.append(val_labels_grade.cpu())
                all_preds_grade_probs.append(probs_grade.cpu())

                all_preds_diag_indices.append(indices_diag.cpu())
                all_labels_diag.append(val_labels_diag.cpu())
                all_preds_diag_probs.append(probs_diag.cpu())
                # --- ΤΕΛΟΣ ΧΕΙΡΟΚΙΝΗΤΗΣ ΣΥΛΛΟΓΗΣ ---

                del val_inputs, val_data_clinical, val_labels_grade, val_labels_diag, val_logits_grade, val_logits_diag, total_loss, l_g, l_d
                torch.cuda.empty_cache()

        # --- Aggregate (Συγκέντρωση) όλων των μετρικών ---

        # Συνένωση όλων των λιστών σε ένα μεγάλο tensor/array
        all_preds_grade_indices = torch.cat(all_preds_grade_indices).numpy()
        all_labels_grade = torch.cat(all_labels_grade).numpy()
        all_preds_grade_probs = torch.cat(all_preds_grade_probs).numpy()

        all_preds_diag_indices = torch.cat(all_preds_diag_indices).numpy()
        all_labels_diag = torch.cat(all_labels_diag).numpy()
        all_preds_diag_probs = torch.cat(all_preds_diag_probs).numpy()

        # Υπολογισμός με sklearn
        mean_acc_grade = accuracy_score(all_labels_grade, all_preds_grade_indices)
        mean_acc_diag = accuracy_score(all_labels_diag, all_preds_diag_indices)

        # (Χειροκίνητος έλεγχος για binary/multiclass AUC)
        try:
            mean_auc_grade = roc_auc_score(all_labels_grade, all_preds_grade_probs, multi_class='ovr', average='macro')
        except ValueError as e:
            print(f"Σφάλμα υπολογισμού AUC (Grade): {e}")
            mean_auc_grade = np.nan

        try:
            mean_auc_diag = roc_auc_score(all_labels_diag, all_preds_diag_probs, multi_class='ovr', average='macro')
        except ValueError as e:
            print(f"Σφάλμα υπολογισμού AUC (Diagnosis): {e}")
            mean_auc_diag = np.nan

        cm_grade = confusion_matrix(all_labels_grade, all_preds_grade_indices, labels=list(range(NUM_CLASSES_GRADE)))
        cm_diag = confusion_matrix(all_labels_diag, all_preds_diag_indices, labels=list(range(NUM_CLASSES_DIAG)))

        report_grade_str = classification_report(all_labels_grade, all_preds_grade_indices, target_names=[str(c) for c in GRADE_CLASSES], zero_division=0)
        report_diag_str = classification_report(all_labels_diag, all_preds_diag_indices, target_names=[str(c) for c in DIAG_CLASSES], zero_division=0)

        mean_val_loss_total = epoch_val_loss_total / val_step if val_step > 0 else np.nan
        mean_val_loss_grade = epoch_val_loss_grade / val_step if val_step > 0 else np.nan
        mean_val_loss_diag = epoch_val_loss_diag / val_step if val_step > 0 else np.nan

        combined_metric = (mean_acc_grade + mean_acc_diag) / 2.0

        print(f"Epoch {current_epoch} Avg Val Loss: {mean_val_loss_total:.4f}, Val Acc Grade: {mean_acc_grade:.4f} (AUC: {mean_auc_grade:.4f}), Val Acc Diag: {mean_acc_diag:.4f} (AUC: {mean_auc_diag:.4f})")
        print(f"Validation Classification Report (Grade):\n{report_grade_str}")
        print(f"Validation Confusion Matrix (Grade):\n{cm_grade}")
        print(f"Validation Classification Report (Diagnosis):\n{report_diag_str}")
        print(f"Validation Confusion Matrix (Diagnosis):\n{cm_diag}")

        history["val_loss_total"].append(mean_val_loss_total); history["val_loss_grade"].append(mean_val_loss_grade); history["val_loss_diag"].append(mean_val_loss_diag)
        history["val_acc_grade"].append(mean_acc_grade); history["val_auc_grade"].append(mean_auc_grade)
        history["val_acc_diag"].append(mean_acc_diag); history["val_auc_diag"].append(mean_auc_diag)
        history["val_metric_combined"].append(combined_metric)

        current_metric = combined_metric

        if current_metric > best_val_metric:
            best_val_metric = current_metric
            checkpoint = {
                'epoch': current_epoch,

                'grade_mlp_state': model_stage2.grade_mlp.state_dict(),
                'grade_head_state': model_stage2.grade_head.state_dict(),
                'diag_mlp_state': model_stage2.diag_mlp.state_dict(),
                'diagnosis_head_state': model_stage2.diagnosis_head.state_dict(),
                'optimizer_state': optimizer_stage2.state_dict(),
                'best_val_metric': best_val_metric,
                'scheduler_state': scheduler_stage2.state_dict() if scheduler_stage2 else None,
                'val_loss': mean_val_loss_total
            }
            torch.save(checkpoint, best_checkpoint_path_stage2)
            print(f"✅ Saved new best model (Combined Acc: {best_val_metric:.4f}) to {best_checkpoint_path_stage2}")

        torch.save(checkpoint, latest_checkpoint_path_stage2)

    try:
        df_history = pd.DataFrame(history)
        df_history.to_csv(history_csv_path_stage2, index=False)
    except Exception as e:
        print(f"Error saving history CSV: {e}")

    if scheduler_stage2:
        scheduler_stage2.step()

print(f"🏁 Stage 2 Training finished. Best Validation Metric (Combined Acc): {best_val_metric:.4f}")

In [ ]:
!pip install iterative-stratification

In [ ]:
import shutil
import os

# Η διαδρομή cache που ορίστηκε στο Κελί 1
mtl_cache_dir = os.path.join(cache_base_dir, "mtl_run_cascaded")

print(f"--- Καθαρισμός Cache ---")
print(f"Προσπάθεια διαγραφής του φακέλου: {mtl_cache_dir}")

# Διαγραφή ολόκληρου του φακέλου cache του mtl_run_cascaded
shutil.rmtree(mtl_cache_dir, ignore_errors=True)

# Έλεγχος
if not os.path.exists(mtl_cache_dir):
    print("Η cache διαγράφηκε επιτυχώς.")
else:
    print("Η cache δεν διαγράφηκε (ίσως δεν υπήρχε ή υπάρχει πρόβλημα δικαιωμάτων).")

print("Ο καθαρισμός ολοκληρώθηκε. Προχώρησε στο Κελί 3.")

In [ ]:
# Ο φάκελος εξόδου του Stage 2
output_dir_stage2 = os.path.join(runs_base_dir, "classification_cascaded_v1")

# Οι διαδρομές των αρχείων που προκαλούν το σφάλμα
latest_checkpoint_path_stage2 = os.path.join(output_dir_stage2, "latest_model_cascaded.pt")
best_checkpoint_path_stage2 = os.path.join(output_dir_stage2, "best_metric_model_cascaded.pt")

print(f"Διαγραφή παλιών checkpoints από: {output_dir_stage2}")
if os.path.exists(latest_checkpoint_path_stage2):
    os.remove(latest_checkpoint_path_stage2)
    print("Διαγράφηκε το 'latest_model_cascaded.pt'")
if os.path.exists(best_checkpoint_path_stage2):
    os.remove(best_checkpoint_path_stage2)
    print("Διαγράφηκε το 'best_metric_model_cascaded.pt'")

print("Ο καθαρισμός ολοκληρώθηκε.")

# MTL

In [ ]:
# %% Βήμα 4 (ΔΙΟΡΘΩΜΕΝΟ): Ορισμός του Μοντέλου MTL (ΧΩΡΙΣ FREEZE)
import torch.nn as nn
import torch.nn.functional as F
from monai.networks.nets import SwinUNETR
import os

print("Ορισμός Μοντέλου MTL (Encoder+Clinical)...")

class MTLClassifier(nn.Module):
    def __init__(self,
                 in_channels_img, out_channels_seg, feature_size, use_checkpoint,
                 num_classes_grade, num_classes_diag, clinical_feature_size,
                 proj_dim_clinical=64, finetuned_seg_model_path=None): # Χρησιμοποιούμε το finetuned encoder

        super().__init__()

        # --- 1. Φόρτωσε το SwinUNETR (Encoder + Decoder) ---
        self.swin_unetr = SwinUNETR(
            in_channels=in_channels_img,
            out_channels=out_channels_seg,
            feature_size=feature_size,
            use_checkpoint=use_checkpoint
        )

        if finetuned_seg_model_path and os.path.exists(finetuned_seg_model_path):
            print(f"Φόρτωση checkpoint τμηματοποίησης από: {finetuned_seg_model_path}")
            checkpoint = torch.load(finetuned_seg_model_path, map_location=torch.device('cpu'))
            try:
                # Χρήση του σωστού ονόματος
                self.swin_unetr_feature_extractor.load_state_dict(checkpoint['model_state'], strict=False)
                print(f"Επιτυχής φόρτωση βαρών μοντέλου τμηματοποίησης.")
            except Exception as e:
                print(f"Σφάλμα φόρτωσης state_dict: {e}")
        else:
            print("ΠΡΟΣΟΧΗ: Δεν βρέθηκε εκπαιδευμένο μοντέλο.")

        # --- 2. Ο ENCODER ΕΙΝΑΙ ΕΚΠΑΙΔΕΥΣΙΜΟΣ (requires_grad = True) ---
        print("--- ΟΛΟ το SwinUNETR (Encoder & Decoder) είναι ΕΚΠΑΙΔΕΥΣΙΜΟ (fine-tuning) ---")
        # Δεν παγώνουμε τίποτα

        # --- 3. Encoder Κλινικών Δεδομένων (Εκπαιδεύσιμος) ---
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_feature_size, proj_dim_clinical),
            nn.LayerNorm(proj_dim_clinical), nn.ReLU(), nn.Dropout(0.3)
        )

        # --- 4. Fusion & Classification Heads (Εκπαιδεύσιμα) ---
        encoder_feature_dim = feature_size * 16 # (768)
        fused_dim = encoder_feature_dim + proj_dim_clinical # (768 + 64)
        print(f"Διαστάσεις Fusion: Encoder({encoder_feature_dim}) + Clinical({proj_dim_clinical}) = {fused_dim}")

        self.fusion_mlp = nn.Sequential(
            nn.Linear(fused_dim, 512),
            nn.LayerNorm(512), nn.ReLU(), nn.Dropout(0.5)
        )
        self.grade_head = nn.Linear(512, num_classes_grade)
        self.diagnosis_head = nn.Linear(512, num_classes_diag)

    def forward(self, x_img, x_clinical):

        # --- Task 1: Segmentation Output ---
        # Τρέχει τον (εκπαιδεύσιμο) encoder ΚΑΙ τον (εκπαιδεύσιμο) decoder
        logits_seg = self.swin_unetr(x_img)

        # --- Task 2 & 3: Classification Path ---

        # 1. Image Features (από τον εκπαιδεύσιμο encoder)
        # (ΠΡΟΣΟΧΗ: Το SwinUNETR αποθηκεύει τα features από το .swinViT)


        encoder_features_list = self.swin_unetr_feature_extractor.swinViT(x_img) # Χρήση του σωστού ονόματος
        encoder_features = encoder_features_list[-1]


        # 2. Pooling
        encoder_pooled = F.adaptive_avg_pool3d(encoder_features, 1).flatten(1)

        # 3. Clinical Features
        clinical_features = self.clinical_encoder(x_clinical)

        # 4. Fusion
        fused_features = torch.cat((encoder_pooled, clinical_features), dim=1)

        # 5. Classification Heads
        fused_output = self.fusion_mlp(fused_features)
        logits_grade = self.grade_head(fused_output)
        logits_diag = self.diagnosis_head(fused_output)

        return logits_seg, logits_diag

print("Ορισμός Μοντέλου MTL (Χωρίς Freeze) ολοκληρώθηκε.")

In [ ]:
# %% Βήμα 5: Ορισμός Loss & Εκπαίδευση (MTL - ΧΩΡΙΣ FREEZE)
import torch.nn as nn
from monai.metrics import ROCAUCMetric, DiceMetric
from ignite.metrics import Accuracy, ConfusionMatrix
from ignite.engine import Events
from monai.transforms import AsDiscrete, Activations
from tqdm import tqdm
import pandas as pd
import numpy as np
import os
import gc
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# --- Συναρτήσεις Απώλειας (Loss Functions) ---
loss_seg = DiceCELoss(include_background=False, sigmoid=True) # <-- ΤΩΡΑ ΧΡΕΙΑΖΕΤΑΙ
loss_grade = nn.CrossEntropyLoss()
loss_diag = nn.CrossEntropyLoss()

W_SEG = 0.6 # 60%
W_GRADE = 0.2 # 20%
W_DIAG = 0.2 # 20%
def multi_task_loss_fn(pred_seg, true_seg, pred_grade, true_grade, pred_diag, true_diag):
    l_s = loss_seg(pred_seg, true_seg) # <--- Υπολογίζουμε ξανά την απώλεια τμηματοποίησης
    l_g = loss_grade(pred_grade, true_grade)
    l_d = loss_diag(pred_diag, true_diag)
    total_loss = (W_SEG * l_s) + (W_GRADE * l_g) + (W_DIAG * l_d)
    return total_loss, l_s, l_g, l_d

# --- Post-processing ---
post_pred_seg = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
post_label_seg = Compose([AsDiscrete(to_onehot=None)])
post_pred_auc = Activations(softmax=True)
post_label_onehot_grade = AsDiscrete(to_onehot=NUM_CLASSES_GRADE)
post_label_onehot_diag = AsDiscrete(to_onehot=NUM_CLASSES_DIAG)

# --- Ρυθμίσεις ---
output_dir_mtl = os.path.join(runs_base_dir, "classification_mtl_v1") # Νέος φάκελος
history_csv_path_mtl = os.path.join(output_dir_mtl, "training_history_mtl.csv")
latest_checkpoint_path_mtl = os.path.join(output_dir_mtl, "latest_model_mtl.pt")
best_checkpoint_path_mtl = os.path.join(output_dir_mtl, "best_metric_model_mtl.pt")
os.makedirs(output_dir_mtl, exist_ok=True)

# --- Αρχικοποίηση Μοντέλου MTL ---
try:
    model_mtl = MTLClassifier( # <-- Κάλεσε το σωστό, νέο μοντέλο
        in_channels_img=IN_CHANNELS, out_channels_seg=OUT_CHANNELS_SEG,
        feature_size=FEATURE_SIZE, use_checkpoint=USE_CHECKPOINTING,
        num_classes_grade=NUM_CLASSES_GRADE, num_classes_diag=NUM_CLASSES_DIAG,
        clinical_feature_size=CLINICAL_FEATURE_SIZE,
        finetuned_seg_model_path=finetuned_model_path # Χρησιμοποίησε τα βάρη του encoder
    ).to(DEVICE)
except NameError:
    print("ΣΦΑΛΜΑ: Η κλάση 'MTLClassifier' δεν έχει οριστεί. Εκτέλεσε ξανά το Κελί 4.")
    # exit()

# --- Optimizer ---
# Χρησιμοποιούμε Differential Learning Rate
# Μικρό LR για τον ήδη εκπαιδευμένο encoder
# Μεγάλο LR για τα νέα, τυχαία layers
LR_ENCODER = 1e-5
LR_HEADS = 1e-4

params_to_train = [
    {'params': model_mtl.swin_unetr.parameters(), 'lr': LR_ENCODER}, # ΟΛΟ το SwinUNETR
    {'params': model_mtl.clinical_encoder.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.fusion_mlp.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.grade_head.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.diagnosis_head.parameters(), 'lr': LR_HEADS}
]
print(f"Σύνολο παραμέτρων προς εκπαίδευση: {sum(p.numel() for p in model_mtl.parameters() if p.requires_grad)}")

optimizer_mtl = torch.optim.AdamW(params_to_train, lr=LR_HEADS, weight_decay=WEIGHT_DECAY_MTL) # Το default LR είναι το LR_HEADS
scheduler_mtl = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_mtl, T_max=NUM_EPOCHS_MTL)

# --- ΧΩΡΙΣ GradScaler ---
best_val_metric = -1.0 # Τώρα θα βασιστούμε στο (Dice + Acc) / 2
start_epoch = 0
history = {
    "epoch": [], "train_loss_total": [], "train_loss_seg": [], "train_loss_grade": [], "train_loss_diag": [],
    "val_loss_total": [], "val_loss_seg": [], "val_loss_grade": [], "val_loss_diag": [],
    "val_dice": [], "val_acc_grade": [], "val_auc_grade": [],
    "val_acc_diag": [], "val_auc_diag": [],
    "val_metric_combined": []
}

# --- Resuming Logic (Προαιρετικό) ---
if os.path.exists(latest_checkpoint_path_mtl):
    print(f"Φόρτωση checkpoint: {latest_checkpoint_path_mtl}")
    checkpoint = torch.load(latest_checkpoint_path_mtl, map_location=DEVICE)
    try:
        model_mtl.load_state_dict(checkpoint['model_state']) # Φόρτωσε όλο το μοντέλο
        optimizer_mtl.load_state_dict(checkpoint['optimizer_state'])
        if checkpoint.get('scheduler_state'): scheduler_mtl.load_state_dict(checkpoint['scheduler_state'])
        start_epoch = checkpoint.get('epoch', 0)
        best_val_metric = checkpoint.get('best_val_metric', -1.0)
        print(f"Συνέχιση MTL από εποχή {start_epoch + 1}")
        # ... (Λογική φόρτωσης ιστορικού) ...
    except Exception as e:
        print(f"Σφάλμα φόρτωσης checkpoint: {e}. Έναρξη από την αρχή.")
        start_epoch = 0; best_val_metric = -1.0; history = {k: [] for k in history.keys()}
else:
    print("Έναρξη MTL εκπαίδευσης από την αρχή.")


print(f"\n--- Έναρξη MTL Loop από εποχή {start_epoch + 1} (ΧΩΡΙΣ AMP, ΜΕ UNFREEZE) ---")

try: _ = train_loader_mtl
except NameError: print("\n!!! ΣΦΑΛΜΑ: Ο 'train_loader_mtl' δεν έχει οριστεί. Εκτέλεσε το Κελί 3."); # exit()

for epoch in range(start_epoch, NUM_EPOCHS_MTL):
    current_epoch = epoch + 1
    print(f"\n===== Epoch {current_epoch}/{NUM_EPOCHS_MTL} =====")
    model_mtl.train() # ΟΛΟ το μοντέλο σε train mode

    epoch_loss_total, epoch_loss_seg, epoch_loss_grade, epoch_loss_diag = 0.0, 0.0, 0.0, 0.0
    step = 0
    pbar_train = tqdm(train_loader_mtl, desc=f"Training MTL Epoch {current_epoch}")

    for batch_data in pbar_train:
        step += 1
        inputs = batch_data[image_key].to(DEVICE)
        labels_seg = batch_data[label_key].to(DEVICE) # <-- ΤΩΡΑ ΤΟ ΧΡΕΙΑΖΟΜΑΣΤΕ
        data_clinical = batch_data[clinical_key].to(DEVICE)
        labels_grade = batch_data[grade_key].to(DEVICE)
        labels_diag = batch_data[diag_key].to(DEVICE)

        optimizer_mtl.zero_grad(set_to_none=True)

        logits_seg, logits_grade, logits_diag = model_mtl(inputs, data_clinical)
        total_loss, l_s, l_d, l_g = multi_task_loss_fn( # <-- Διορθωμένη σειρά
            logits_seg, labels_seg, logits_grade, labels_grade, logits_diag, labels_diag
        )

        if not torch.isfinite(total_loss): print(f"WARNING: Non-finite loss. Skipping."); continue

        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model_mtl.parameters(), 1.0) # Clip gradients σε *όλο* το μοντέλο
        optimizer_mtl.step()

        epoch_loss_total += total_loss.item(); epoch_loss_seg += l_s.item();
        epoch_loss_grade += l_g.item(); epoch_loss_diag += l_d.item()
        pbar_train.set_postfix(total_loss=f"{total_loss.item():.4f}", seg_loss=f"{l_s.item():.4f}")
        del inputs, labels_seg, data_clinical, labels_grade, labels_diag, logits_seg, logits_grade, logits_diag, total_loss, l_s, l_g, l_d
        if DEVICE == 'cuda': torch.cuda.empty_cache()

    avg_epoch_loss_total = epoch_loss_total / step if step > 0 else np.nan
    avg_epoch_loss_seg = epoch_loss_seg / step if step > 0 else np.nan
    avg_epoch_loss_grade = epoch_loss_grade / step if step > 0 else np.nan
    avg_epoch_loss_diag = epoch_loss_diag / step if step > 0 else np.nan
    current_lr_encoder = optimizer_mtl.param_groups[0]['lr']
    current_lr_heads = optimizer_mtl.param_groups[1]['lr']
    print(f"Epoch {current_epoch} Avg Train Loss: {avg_epoch_loss_total:.4f} (Seg: {avg_epoch_loss_seg:.4f}, Grade: {avg_epoch_loss_grade:.4f}, Diag: {avg_epoch_loss_diag:.4f})")
    print(f"LRs: Encoder={current_lr_encoder:.2e}, Heads={current_lr_heads:.2e}")

    history["epoch"].append(current_epoch)
    history["train_loss_total"].append(avg_epoch_loss_total); history["train_loss_seg"].append(avg_epoch_loss_seg)
    history["train_loss_grade"].append(avg_epoch_loss_grade); history["train_loss_diag"].append(avg_epoch_loss_diag)

    # --- Validation ---
    if current_epoch % VAL_INTERVAL == 0:
        model_mtl.eval()

        dice_metric = DiceMetric(include_background=False, reduction="mean", get_not_nans=False) # <--- ΝΕΟ: Αρχικοποίηση DiceMetric
        acc_metric_grade_ign = Accuracy(device=DEVICE); acc_metric_diag_ign = Accuracy(device=DEVICE)
        cm_metric_grade = ConfusionMatrix(num_classes=NUM_CLASSES_GRADE, device=DEVICE); cm_metric_diag = ConfusionMatrix(num_classes=NUM_CLASSES_DIAG, device=DEVICE)
        auc_metric_grade = ROCAUCMetric(); auc_metric_diag = ROCAUCMetric()

        all_labels_grade = []; all_preds_grade_indices = []
        all_labels_diag = []; all_preds_diag_indices = []

        epoch_val_loss_total, epoch_val_loss_seg, epoch_val_loss_grade, epoch_val_loss_diag = 0.0, 0.0, 0.0, 0.0
        val_step = 0
        pbar_val = tqdm(val_loader_mtl, desc=f"Validation MTL Epoch {current_epoch}")

        with torch.no_grad():
            for val_data in pbar_val:
                val_step += 1
                val_inputs = val_data[image_key].to(DEVICE); val_data_clinical = val_data[clinical_key].to(DEVICE)
                val_labels_seg = val_data[label_key].to(DEVICE) # <-- ΤΩΡΑ ΤΟ ΧΡΕΙΑΖΟΜΑΣΤΕ
                val_labels_grade = val_data[grade_key].to(DEVICE); val_labels_diag = val_data[diag_key].to(DEVICE)

                val_logits_seg, val_logits_grade, val_logits_diag = model_mtl(val_inputs, val_data_clinical)

                total_loss, l_s, l_g, l_d = classification_loss_fn(val_logits_seg, val_labels_seg, val_logits_grade, val_labels_grade, val_logits_diag, val_labels_diag)
                if torch.isfinite(total_loss): epoch_val_loss_total += total_loss.item(); epoch_val_loss_seg += l_s.item(); epoch_val_loss_grade += l_g.item(); epoch_val_loss_diag += l_d.item()

                # --- Μετρικές Τμηματοποίησης ---
                val_outputs_seg_post = [post_pred_seg(i) for i in decollate_batch(val_logits_seg)]
                val_labels_seg_post = [post_label_seg(i) for i in decollate_batch(val_labels_seg)]
                dice_metric(y_pred=val_outputs_seg_post, y=val_labels_seg_post)

                # --- Μετρικές Ταξινόμησης ---
                val_outputs_grade_indices = torch.argmax(val_logits_grade, dim=1)
                val_outputs_diag_indices = torch.argmax(val_logits_diag, dim=1)
                acc_metric_grade_ign.update((val_logits_grade, val_labels_grade)); acc_metric_diag_ign.update((val_logits_diag, val_labels_diag))
                cm_metric_grade.update((val_logits_grade, val_labels_grade)); cm_metric_diag.update((val_logits_diag, val_labels_diag))
                val_probs_grade = F.softmax(val_logits_grade, dim=1); val_labels_grade_onehot = F.one_hot(val_labels_grade, num_classes=NUM_CLASSES_GRADE)
                val_probs_diag = F.softmax(val_logits_diag, dim=1); val_labels_diag_onehot = F.one_hot(val_labels_diag, num_classes=NUM_CLASSES_DIAG)
                auc_metric_grade(y_pred=val_probs_grade, y=val_labels_grade_onehot); auc_metric_diag(y_pred=val_probs_diag, y=val_labels_diag_onehot)
                all_preds_grade_indices.append(val_outputs_grade_indices.cpu()); all_labels_grade.append(val_labels_grade.cpu())
                all_preds_diag_indices.append(val_outputs_diag_indices.cpu()); all_labels_diag.append(val_labels_diag.cpu())

                del val_inputs, val_labels_seg, val_data_clinical, val_labels_grade, val_labels_diag, val_logits_seg, val_logits_grade, val_logits_diag, total_loss, l_s, l_g, l_d
                torch.cuda.empty_cache()

        # --- Aggregate (Συγκέντρωση) όλων των μετρικών ---
        mean_dice = dice_metric.aggregate().item(); dice_metric.reset()
        mean_acc_grade = acc_metric_grade_ign.aggregate(); acc_metric_grade_ign.reset()
        mean_acc_diag = acc_metric_diag_ign.aggregate(); acc_metric_diag_ign.reset()
        mean_auc_grade = auc_metric_grade.aggregate().item(); auc_metric_grade.reset()
        mean_auc_diag = auc_metric_diag.aggregate().item(); auc_metric_diag.reset()
        cm_grade = cm_metric_grade.aggregate().cpu().numpy(); cm_metric_grade.reset()
        cm_diag = cm_metric_diag.aggregate().cpu().numpy(); cm_metric_diag.reset()
        mean_val_loss_total = epoch_val_loss_total / val_step if val_step > 0 else np.nan
        mean_val_loss_seg = epoch_val_loss_seg / val_step if val_step > 0 else np.nan
        mean_val_loss_grade = epoch_val_loss_grade / val_step if val_step > 0 else np.nan
        mean_val_loss_diag = epoch_val_loss_diag / val_step if val_step > 0 else np.nan

        combined_metric = (mean_dice + mean_acc_grade + mean_acc_diag) / 3.0 # Συνδυασμός και των 3

        print(f"Epoch {current_epoch} Avg Val Loss: {mean_val_loss_total:.4f} (Seg: {mean_val_loss_seg:.4f})")
        print(f"--- Grade Metrics ---")
        print(f"  Val Acc Grade: {mean_acc_grade:.4f} (AUC: {mean_auc_grade:.4f})")
        print(f"  CM (Grade):\n{cm_grade}")
        print(f"--- Diagnosis Metrics ---")
        print(f"  Val Acc Diag: {mean_acc_diag:.4f} (AUC: {mean_auc_diag:.4f})")
        print(f"  CM (Diagnosis):\n{cm_diag}")
        print(f"--- Segmentation Metric ---")
        print(f"  Val Dice: {mean_dice:.4f}")

        # ... (Λογική αποθήκευσης History) ...

        current_metric = combined_metric # Αποθήκευση με βάση το συνδυασμό

        if current_metric > best_val_metric:
            best_val_metric = current_metric
            checkpoint = {
                'epoch': current_epoch,
                'model_state': model_mtl.state_dict(), # Αποθηκεύει ΟΛΟ το μοντέλο
                'optimizer_state': optimizer_mtl.state_dict(),
                'best_val_metric': best_val_metric,
                'scheduler_state': scheduler_mtl.state_dict() if scheduler_mtl else None,
                'val_loss': mean_val_loss_total
            }
            torch.save(checkpoint, best_checkpoint_path_mtl)
            print(f"✅ Saved new best model (Combined Metric: {best_val_metric:.4f}) to {best_checkpoint_path_mtl}")

        torch.save(checkpoint, latest_checkpoint_path_mtl)

    # ... (Λογική αποθήκευσης CSV) ...

    if scheduler_mtl:
        scheduler_mtl.step()

print(f"🏁 MTL Training finished. Best Validation Metric (Combined): {best_val_metric:.4f}")

# Cross Attentions

In [ ]:
# %% Βήμα 4 (Στρατηγική 2): Ορισμός του Μοντέλου (Cross-Attention)
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.networks.nets import SwinUNETR
import os

print("Ορισμός Μοντέλου CascadedClassifier (Cross-Attention Fusion)...")

class CrossAttentionClassifier(nn.Module):
    def __init__(self,
                 in_channels_img, out_channels_seg, feature_size, use_checkpoint,
                 num_classes_grade, num_classes_diag, clinical_feature_size,
                 proj_dim_clinical=64, # Διάσταση του embedding των κλινικών
                 finetuned_seg_model_path=None):

        super().__init__()

        # --- 1. Φόρτωση και Πάγωμα του SwinUNETR ---
        self.swin_unetr_extractor = SwinUNETR(
            in_channels=in_channels_img,
            out_channels=out_channels_seg,
            feature_size=feature_size,
            use_checkpoint=use_checkpoint
        )
        if finetuned_seg_model_path and os.path.exists(finetuned_seg_model_path):
            print(f"Φόρτωση checkpoint τμηματοποίησης από: {finetuned_seg_model_path}")
            checkpoint = torch.load(finetuned_seg_model_path, map_location=torch.device('cpu'))
            try:
                self.swin_unetr_extractor.load_state_dict(checkpoint['model_state'], strict=False)
                print(f"Επιτυχής φόρτωση βαρών μοντέλου τμηματοποίησης.")
            except Exception as e: print(f"Σφάλμα φόρτωσης state_dict: {e}")
        else: print("ΠΡΟΣΟΧΗ: Δεν βρέθηκε εκπαιδευμένο μοντέλο.")

        print("--- ΠΑΓΩΜΑ (FREEZING) ολόκληρου του SwinUNETR extractor ---")
        for param in self.swin_unetr_extractor.parameters():
            param.requires_grad = False
        # -----------------------------------------------------------

        encoder_feature_dim = feature_size * 16 # (768)

        # --- 2. Encoder Κλινικών Δεδομένων (Εκπαιδεύσιμος) ---
        # Μετατρέπει τα 25 clinical features σε 768 (για να ταιριάζει με τον encoder)
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_feature_size, 256),
            nn.LayerNorm(256), nn.ReLU(),
            nn.Linear(256, encoder_feature_dim) # Έξοδος: 768
        )

        # --- 3. Cross-Attention Layer (Εκπαιδεύσιμο) ---
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=encoder_feature_dim, # (768)
            num_heads=8,
            dropout=0.2,
            batch_first=True # ΣΗΜΑΝΤΙΚΟ: (Batch, Seq, Features)
        )

        # --- 4. MLP & Heads (Εκπαιδεύσιμα) ---
        self.fusion_mlp = nn.Sequential(
            nn.LayerNorm(encoder_feature_dim), # LayerNorm στην έξοδο του Attention
            nn.Linear(encoder_feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.5)
        )
        self.grade_head = nn.Linear(512, num_classes_grade)
        self.diagnosis_head = nn.Linear(512, num_classes_diag)

        print(f"Αρχιτεκτονική Cross-Attention: Clinical({clinical_feature_size}) -> {encoder_feature_dim} Q, Image -> K, V")

    def forward(self, x_img, x_clinical):

        # 1. Εξαγωγή Features Εικόνας (Παγωμένο)
        with torch.no_grad():
            encoder_features_list = self.swin_unetr_extractor.swinViT(x_img)
            encoder_features = encoder_features_list[-1] # [B, 768, D, H, W]

        # Μετατροπή σε Ακολουθία (Sequence) για το Attention
        # [B, 768, D*H*W] -> [B, N_Patches, 768]
        B, C, D, H, W = encoder_features.shape
        img_seq = encoder_features.flatten(2).permute(0, 2, 1) # [B, N, C]
        # Αυτό είναι τα Key (K) και Value (V)

        # 2. Εξαγωγή Features Κλινικών (Εκπαιδεύσιμο)
        clinical_embed = self.clinical_encoder(x_clinical) # [B, 768]

        # Μετατροπή σε "Query" (Q)
        clinical_query = clinical_embed.unsqueeze(1) # [B, 1, 768]

        # 3. Fusion (Cross-Attention)
        # Το Query (κλινικά) "ρωτάει" τα Key/Value (εικόνα)
        fused_features, _ = self.cross_attention(
            query=clinical_query, # [B, 1, 768]
            key=img_seq,          # [B, N, 768]
            value=img_seq         # [B, N, 768]
        ) # Έξοδος: [B, 1, 768]

        fused_features = fused_features.squeeze(1) # [B, 768]

        # 4. Classification Heads
        fused_output = self.fusion_mlp(fused_features) # [B, 512]
        logits_grade = self.grade_head(fused_output)
        logits_diag = self.diagnosis_head(fused_output)

        return logits_grade, logits_diag

print("Ορισμός Μοντέλου CrossAttentionClassifier (v6) ολοκληρώθηκε.")

In [ ]:
# %% Βήμα 5: Ορισμός Loss & Εκπαίδευση (Stage 2 - Cross-Attention)
import torch.nn as nn
from tqdm import tqdm
import pandas as pd
import numpy as np
import os
import gc
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
from monai.metrics import ROCAUCMetric
from monai.transforms import AsDiscrete, Activations
from ignite.metrics import Accuracy, ConfusionMatrix
from ignite.engine import Events

# --- Συναρτήσεις Απώλειας (Loss Functions) ---
loss_grade = nn.CrossEntropyLoss()
loss_diag = nn.CrossEntropyLoss()
w_grade = 0.5; w_diag = 0.5
def classification_loss_fn(pred_grade, true_grade, pred_diag, true_diag):
    l_g = loss_grade(pred_grade, true_grade)
    l_d = loss_diag(pred_diag, true_diag)
    total_loss = (w_grade * l_g) + (w_diag * l_d)
    return total_loss, l_g, l_d

# --- Post-processing για Μετρικές ---
try:
    post_pred_auc = Activations(softmax=True)
    post_label_onehot_grade = AsDiscrete(to_onehot=NUM_CLASSES_GRADE)
    post_label_onehot_diag = AsDiscrete(to_onehot=NUM_CLASSES_DIAG)
except NameError:
    print("ΣΦΑΛΜΑ: Οι μεταβλητές NUM_CLASSES_GRADE/DIAG δεν έχουν οριστεί. Εκτέλεσε ξανά το Κελί 2.")
    # exit()

# --- Ρυθμίσεις για το Stage 2 run ---
output_dir_stage2 = os.path.join(runs_base_dir, "classification_cascaded_v2_cross_attn") # Νέος φάκελος
history_csv_path_stage2 = os.path.join(output_dir_stage2, "training_history_cascaded_attn.csv")
latest_checkpoint_path_stage2 = os.path.join(output_dir_stage2, "latest_model_cascaded_attn.pt")
best_checkpoint_path_stage2 = os.path.join(output_dir_stage2, "best_metric_model_cascaded_attn.pt")
os.makedirs(output_dir_stage2, exist_ok=True)
print(f"Τα αποτελέσματα θα αποθηκευτούν στο: {output_dir_stage2}")

# --- Αρχικοποίηση Μοντέλου Stage 2 ---
try:
    # *** ΑΛΛΑΓΗ ΕΔΩ: Κάλεσε το CrossAttentionClassifier ***
    model_stage2 = CrossAttentionClassifier(
        in_channels_img=IN_CHANNELS, out_channels_seg=OUT_CHANNELS_SEG,
        feature_size=FEATURE_SIZE, use_checkpoint=USE_CHECKPOINTING,
        num_classes_grade=NUM_CLASSES_GRADE, num_classes_diag=NUM_CLASSES_DIAG,
        clinical_feature_size=CLINICAL_FEATURE_SIZE,
        finetuned_seg_model_path=finetuned_model_path
    ).to(DEVICE)
except NameError:
    print("ΣΦΑΛΜΑ: Η κλάση 'CrossAttentionClassifier' δεν έχει οριστεί. Εκτέλεσε ξανά το Κελί 4.")
    # exit()

# *** ΑΛΛΑΓΗ ΕΔΩ: Νέα params_to_train ***
params_to_train = [
    {'params': model_stage2.clinical_encoder.parameters()},
    {'params': model_stage2.cross_attention.parameters()},
    {'params': model_stage2.fusion_mlp.parameters()},
    {'params': model_stage2.grade_head.parameters()},
    {'params': model_stage2.diagnosis_head.parameters()}
]
print(f"Σύνολο παραμέτρων προς εκπαίδευση: {sum(p.numel() for group in params_to_train for p in group['params'])}")

optimizer_stage2 = torch.optim.AdamW(params_to_train, lr=LEARNING_RATE_MTL, weight_decay=WEIGHT_DECAY_MTL)
scheduler_stage2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_stage2, T_max=NUM_EPOCHS_MTL)

# --- ΧΩΡΙΣ GradScaler ---
best_val_metric = -1.0
start_epoch = 0
history = {
    "epoch": [], "train_loss_total": [], "train_loss_grade": [], "train_loss_diag": [],
    "val_loss_total": [], "val_loss_grade": [], "val_loss_diag": [],
    "val_acc_grade": [], "val_auc_grade": [],
    "val_acc_diag": [], "val_auc_diag": [],
    "val_metric_combined": []
}

# --- Resuming Logic (Διαγραφή Checkpoints συνιστάται) ---
if os.path.exists(latest_checkpoint_path_stage2):
    print(f"ΠΡΟΣΟΧΗ: Βρέθηκε παλιό checkpoint. Διαγράφεται για να ξεκινήσει η νέα αρχιτεκτονική.")
    os.remove(latest_checkpoint_path_stage2)
    if os.path.exists(best_checkpoint_path_stage2):
        os.remove(best_checkpoint_path_stage2)
print("Έναρξη Stage 2 εκπαίδευσης από την αρχή (Cross-Attention).")
start_epoch = 0; best_val_metric = -1.0; history = {k: [] for k in history.keys()}


print(f"\n--- Έναρξη Stage 2 Classification Loop από εποχή {start_epoch + 1} (ΧΩΡΙΣ AMP) ---")

try: _ = train_loader_mtl
except NameError: print("\n!!! ΣΦΑΛΜΑ: Ο 'train_loader_mtl' δεν έχει οριστεί. Εκτέλεσε το Κελί 3."); # exit()

for epoch in range(start_epoch, NUM_EPOCHS_MTL):
    current_epoch = epoch + 1
    print(f"\n===== Epoch {current_epoch}/{NUM_EPOCHS_MTL} =====")
    model_stage2.train()
    model_stage2.swin_unetr_extractor.eval()

    epoch_loss_total, epoch_loss_grade, epoch_loss_diag = 0.0, 0.0, 0.0
    step = 0
    pbar_train = tqdm(train_loader_mtl, desc=f"Training Stage 2 Epoch {current_epoch}")

    for batch_data in pbar_train:
        step += 1
        inputs = batch_data[image_key].to(DEVICE)
        data_clinical = batch_data[clinical_key].to(DEVICE)
        labels_grade = batch_data[grade_key].to(DEVICE)
        labels_diag = batch_data[diag_key].to(DEVICE)

        optimizer_stage2.zero_grad(set_to_none=True)

        logits_grade, logits_diag = model_stage2(inputs, data_clinical)
        total_loss, l_g, l_d = classification_loss_fn(
            logits_grade, labels_grade, logits_diag, labels_diag
        )

        if not torch.isfinite(total_loss):
            print(f"!!! WARNING: Non-finite loss. Skipping step.")
            continue

        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(
            filter(lambda p: p.requires_grad, model_stage2.parameters()), 1.0
        )
        optimizer_stage2.step()

        epoch_loss_total += total_loss.item(); epoch_loss_grade += l_g.item(); epoch_loss_diag += l_d.item()
        pbar_train.set_postfix(total_loss=f"{total_loss.item():.4f}")
        del inputs, data_clinical, labels_grade, labels_diag, logits_grade, logits_diag, total_loss, l_g, l_d
        if DEVICE == 'cuda': torch.cuda.empty_cache()

    # ... (Ο υπόλοιπος κώδικας (logging, validation, checkpointing, history) παραμένει ΑΚΡΙΒΩΣ Ο ΙΔΙΟΣ
    #      με τον "Χειροκίνητες Μετρικές" κώδικα που δούλεψε) ...

    # --- Logging ( ίδιο με πριν) ---
    avg_epoch_loss_total = epoch_loss_total / step if step > 0 else np.nan
    avg_epoch_loss_grade = epoch_loss_grade / step if step > 0 else np.nan
    avg_epoch_loss_diag = epoch_loss_diag / step if step > 0 else np.nan
    current_lr = optimizer_stage2.param_groups[0]['lr']
    print(f"Epoch {current_epoch} Avg Train Loss: {avg_epoch_loss_total:.4f} (Grade: {avg_epoch_loss_grade:.4f}, Diag: {avg_epoch_loss_diag:.4f}), LR: {current_lr:.6f}")
    history["epoch"].append(current_epoch)
    history["train_loss_total"].append(avg_epoch_loss_total); history["train_loss_grade"].append(avg_epoch_loss_grade); history["train_loss_diag"].append(avg_epoch_loss_diag)

    # --- Validation ( ίδιο με πριν) ---
    if current_epoch % VAL_INTERVAL == 0:
        model_stage2.eval()
        auc_metric_grade = ROCAUCMetric(); auc_metric_diag = ROCAUCMetric()
        all_labels_grade = []; all_preds_grade_indices = []; all_preds_grade_probs = []
        all_labels_diag = []; all_preds_diag_indices = []; all_preds_diag_probs = []
        epoch_val_loss_total, epoch_val_loss_grade, epoch_val_loss_diag = 0.0, 0.0, 0.0
        val_step = 0
        pbar_val = tqdm(val_loader_mtl, desc=f"Validation Stage 2 Epoch {current_epoch}")
        with torch.no_grad():
            for val_data in pbar_val:
                val_step += 1
                val_inputs = val_data[image_key].to(DEVICE); val_data_clinical = val_data[clinical_key].to(DEVICE)
                val_labels_grade = val_data[grade_key].to(DEVICE); val_labels_diag = val_data[diag_key].to(DEVICE)
                val_logits_grade, val_logits_diag = model_stage2(val_inputs, val_data_clinical)
                total_loss, l_g, l_d = classification_loss_fn(val_logits_grade, val_labels_grade, val_logits_diag, val_labels_diag)
                if torch.isfinite(total_loss): epoch_val_loss_total += total_loss.item(); epoch_val_loss_grade += l_g.item(); epoch_val_loss_diag += l_d.item()
                val_outputs_grade_indices = torch.argmax(val_logits_grade, dim=1)
                val_outputs_diag_indices = torch.argmax(val_logits_diag, dim=1)
                val_probs_grade = F.softmax(val_logits_grade, dim=1)
                val_probs_diag = F.softmax(val_logits_diag, dim=1)
                val_labels_grade_onehot = F.one_hot(val_labels_grade, num_classes=NUM_CLASSES_GRADE).float()
                val_labels_diag_onehot = F.one_hot(val_labels_diag, num_classes=NUM_CLASSES_DIAG).float()
                auc_metric_grade(y_pred=val_probs_grade, y=val_labels_grade_onehot)
                auc_metric_diag(y_pred=val_probs_diag, y=val_labels_diag_onehot)
                all_preds_grade_indices.append(val_outputs_grade_indices.cpu())
                all_labels_grade.append(val_labels_grade.cpu())
                all_preds_grade_probs.append(val_probs_grade.cpu())
                all_preds_diag_indices.append(val_outputs_diag_indices.cpu())
                all_labels_diag.append(val_labels_diag.cpu())
                all_preds_diag_probs.append(val_probs_diag.cpu())
                del val_inputs, val_data_clinical, val_labels_grade, val_labels_diag, val_logits_grade, val_logits_diag, total_loss, l_g, l_d
                torch.cuda.empty_cache()

        # --- Aggregate ( ίδιο με πριν) ---
        all_preds_grade_indices = torch.cat(all_preds_grade_indices).numpy()
        all_labels_grade = torch.cat(all_labels_grade).numpy()
        all_preds_grade_probs = torch.cat(all_preds_grade_probs).numpy()
        all_preds_diag_indices = torch.cat(all_preds_diag_indices).numpy()
        all_labels_diag = torch.cat(all_labels_diag).numpy()
        all_preds_diag_probs = torch.cat(all_preds_diag_probs).numpy()
        grade_label_indices = list(range(NUM_CLASSES_GRADE)); diag_label_indices = list(range(NUM_CLASSES_DIAG))
        mean_acc_grade = accuracy_score(all_labels_grade, all_preds_grade_indices)
        mean_acc_diag = accuracy_score(all_labels_diag, all_preds_diag_indices)
        cm_grade = confusion_matrix(all_labels_grade, all_preds_grade_indices, labels=grade_label_indices)
        cm_diag = confusion_matrix(all_labels_diag, all_preds_diag_indices, labels=diag_label_indices)
        report_grade_str = classification_report(all_labels_grade, all_preds_grade_indices, labels=grade_label_indices, target_names=[str(c) for c in GRADE_CLASSES], zero_division=0)
        report_diag_str = classification_report(all_labels_diag, all_preds_diag_indices, labels=diag_label_indices, target_names=[str(c) for c in DIAG_CLASSES], zero_division=0)
        try: mean_auc_grade = roc_auc_score(all_labels_grade, all_preds_grade_probs, multi_class='ovr', average='macro', labels=grade_label_indices)
        except Exception: mean_auc_grade = np.nan
        auc_metric_grade.reset()
        try: mean_auc_diag = roc_auc_score(all_labels_diag, all_preds_diag_probs, multi_class='ovr', average='macro', labels=diag_label_indices)
        except Exception: mean_auc_diag = np.nan
        auc_metric_diag.reset()
        mean_val_loss_total = epoch_val_loss_total / val_step if val_step > 0 else np.nan
        mean_val_loss_grade = epoch_val_loss_grade / val_step if val_step > 0 else np.nan
        mean_val_loss_diag = epoch_val_loss_diag / val_step if val_step > 0 else np.nan
        combined_metric = (mean_acc_grade + mean_acc_diag) / 2.0

        # --- Print & History ( ίδιο με πριν) ---
        print(f"Epoch {current_epoch} Avg Val Loss: {mean_val_loss_total:.4f}, Val Acc Grade: {mean_acc_grade:.4f} (AUC: {mean_auc_grade:.4f}), Val Acc Diag: {mean_acc_diag:.4f} (AUC: {mean_auc_diag:.4f})")
        print(f"Validation Classification Report (Grade):\n{report_grade_str}")
        print(f"Validation Confusion Matrix (Grade):\n{cm_grade}")
        print(f"Validation Classification Report (Diagnosis):\n{report_diag_str}")
        print(f"Validation Confusion Matrix (Diagnosis):\n{cm_diag}")
        history["val_loss_total"].append(mean_val_loss_total); history["val_loss_grade"].append(mean_val_loss_grade); history["val_loss_diag"].append(mean_val_loss_diag)
        history["val_acc_grade"].append(mean_acc_grade); history["val_auc_grade"].append(mean_auc_grade)
        history["val_acc_diag"].append(mean_acc_diag); history["val_auc_diag"].append(mean_auc_diag)
        history["val_metric_combined"].append(combined_metric)

        # --- Checkpointing ( ίδιο με πριν, με τα σωστά keys) ---
        current_metric = combined_metric
        if current_metric > best_val_metric:
            best_val_metric = current_metric
            checkpoint = {
                'epoch': current_epoch,
                'clinical_encoder_state': model_stage2.clinical_encoder.state_dict(),
                'cross_attention_state': model_stage2.cross_attention.state_dict(), # ΑΛΛΑΓΗ
                'fusion_mlp_state': model_stage2.fusion_mlp.state_dict(),
                'grade_head_state': model_stage2.grade_head.state_dict(),
                'diagnosis_head_state': model_stage2.diagnosis_head.state_dict(),
                'optimizer_state': optimizer_stage2.state_dict(),
                'best_val_metric': best_val_metric,
                'scheduler_state': scheduler_stage2.state_dict() if scheduler_stage2 else None,
                'val_loss': mean_val_loss_total
            }
            torch.save(checkpoint, best_checkpoint_path_stage2)
            print(f"✅ Saved new best model (Combined Acc: {best_val_metric:.4f}) to {best_checkpoint_path_stage2}")

        torch.save(checkpoint, latest_checkpoint_path_stage2)

    try:
        df_history = pd.DataFrame(history)
        df_history.to_csv(history_csv_path_stage2, index=False)
    except Exception as e:
        print(f"Error saving history CSV: {e}")

    if scheduler_stage2:
        scheduler_stage2.step()

print(f"🏁 Stage 2 Training finished. Best Validation Metric (Combined Acc): {best_val_metric:.4f}")

# Image Only

In [ ]:
# %% Βήμα 4 (ΔΙΟΡΘΩΜΕΝΟ): Ορισμός Μοντέλου (Stage 2 - v5 - ΜΟΝΟ Εικόνα)
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.networks.nets import SwinUNETR
import os

print("Ορισμός Μοντέλου CascadedClassifier (v5 - ΜΟΝΟ Image Features)...")

class CascadedImageOnlyClassifier(nn.Module):
    def __init__(self,
                 in_channels_img, out_channels_seg, feature_size, use_checkpoint,
                 num_classes_grade, num_classes_diag,
                 finetuned_seg_model_path=None):

        super().__init__()

        # --- 1. Φόρτωση και Πάγωμα του SwinUNETR ---
        self.swin_unetr_extractor = SwinUNETR(
            in_channels=in_channels_img,
            out_channels=out_channels_seg,
            feature_size=feature_size,
            use_checkpoint=use_checkpoint
        )
        if finetuned_seg_model_path and os.path.exists(finetuned_seg_model_path):
            print(f"Φόρτωση checkpoint τμηματοποίησης από: {finetuned_seg_model_path}")
            checkpoint = torch.load(finetuned_seg_model_path, map_location=torch.device('cpu'))
            try:
                self.swin_unetr_extractor.load_state_dict(checkpoint['model_state'], strict=False)
                print(f"Επιτυχής φόρτωση βαρών μοντέλου τμηματοποίησης.")
            except Exception as e: print(f"Σφάλμα φόρτωσης state_dict: {e}")
        else: print("ΠΡΟΣΟΧΗ: Δεν βρέθηκε εκπαιδευμένο μοντέλο.")

        print("--- ΠΑΓΩΜΑ (FREEZING) ολόκληρου του SwinUNETR extractor ---")
        for param in self.swin_unetr_extractor.parameters():
            param.requires_grad = False
        # -----------------------------------------------------------

        # --- 2. Hooks για Εξαγωγή Features ---
        self.encoder_features_out = None
        self.decoder_features_out = None
        self.swin_unetr_extractor.swinViT.register_forward_hook(self.hook_encoder_features)
        self.swin_unetr_extractor.decoder1.register_forward_hook(self.hook_decoder_features)

        # --- 3. Encoder Κλινικών Δεδομένων (ΑΦΑΙΡΕΘΗΚΕ) ---

        # --- 4. Fusion & Classification Heads (Εκπαιδεύσιμα) ---
        encoder_feature_dim = feature_size * 16 # (768)
        decoder_feature_dim = feature_size # (48)

        # Η διάσταση εισόδου για τα MLPs (ΜΟΝΟ ΕΙΚΟΝΑ)
        fused_dim = encoder_feature_dim + decoder_feature_dim
        print(f"Διαστάσεις Fusion: Encoder({encoder_feature_dim}) + Decoder({decoder_feature_dim}) = {fused_dim}")

        # Ένα κοινό MLP (απλοποιημένο)
        self.fusion_mlp = nn.Sequential(
            nn.Linear(fused_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.grade_head = nn.Linear(256, num_classes_grade)
        self.diagnosis_head = nn.Linear(256, num_classes_diag)

    # --- Hook functions ---
    def hook_encoder_features(self, module, input, output):
        self.encoder_features_out = output[-1]
    def hook_decoder_features(self, module, input, output):
        self.decoder_features_out = output

    def forward(self, x_img): # <-- ΔΕΧΕΤΑΙ ΜΟΝΟ x_img

        # 1. Εξαγωγή Features (με Hooks)
        with torch.no_grad():
            _ = self.swin_unetr_extractor(x_img)

        # 2. Pooling
        encoder_pooled = F.adaptive_avg_pool3d(self.encoder_features_out, 1).flatten(1)
        decoder_pooled = F.adaptive_avg_pool3d(self.decoder_features_out, 1).flatten(1)

        # 3. Clinical Features (ΑΦΑΙΡΕΘΗΚΕ)

        # 4. Fusion
        fused_features = torch.cat((encoder_pooled, decoder_pooled), dim=1) # [B, 816]

        # 5. Classification Heads
        fused_output = self.fusion_mlp(fused_features)
        logits_grade = self.grade_head(fused_output)
        logits_diag = self.diagnosis_head(fused_output)

        return logits_grade, logits_diag

print("Ορισμός Μοντέλου CascadedImageOnlyClassifier (v5 - ΜΟΝΟ Εικόνα) ολοκληρώθηκε.")

In [ ]:
# %% Βήμα 5: Ορισμός Loss & Εκπαίδευση (Stage 2 - v5 - ΜΟΝΟ Εικόνα)
import torch.nn as nn
from tqdm import tqdm
import pandas as pd
import numpy as np
import os
import gc
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
from monai.metrics import ROCAUCMetric
from monai.transforms import AsDiscrete, Activations
from ignite.metrics import Accuracy, ConfusionMatrix
from ignite.engine import Events

# --- Συναρτήσεις Απώλειας (Loss Functions) ---
loss_grade = nn.CrossEntropyLoss()
loss_diag = nn.CrossEntropyLoss()
w_grade = 0.5; w_diag = 0.5
def classification_loss_fn(pred_grade, true_grade, pred_diag, true_diag):
    l_g = loss_grade(pred_grade, true_grade)
    l_d = loss_diag(pred_diag, true_diag)
    total_loss = (w_grade * l_g) + (w_diag * l_d)
    return total_loss, l_g, l_d

# --- Post-processing για Μετρικές ---
try:
    post_pred_auc = Activations(softmax=True)
    post_label_onehot_grade = AsDiscrete(to_onehot=NUM_CLASSES_GRADE)
    post_label_onehot_diag = AsDiscrete(to_onehot=NUM_CLASSES_DIAG)
except NameError:
    print("ΣΦΑΛΜΑ: Οι μεταβλητές NUM_CLASSES_GRADE/DIAG δεν έχουν οριστεί. Εκτέλεσε ξανά το Κελί 2.")
    # exit()

# --- Ρυθμίσεις για το Stage 2 run ---
output_dir_stage2 = os.path.join(runs_base_dir, "classification_cascaded_v3_image_only") # Νέος φάκελος
history_csv_path_stage2 = os.path.join(output_dir_stage2, "training_history_cascaded.csv")
latest_checkpoint_path_stage2 = os.path.join(output_dir_stage2, "latest_model_cascaded.pt")
best_checkpoint_path_stage2 = os.path.join(output_dir_stage2, "best_metric_model_cascaded.pt")
os.makedirs(output_dir_stage2, exist_ok=True)
print(f"Τα αποτελέσματα θα αποθηκευτούν στο: {output_dir_stage2}")

# --- Αρχικοποίηση Μοντέλου Stage 2 ---
try:
    # *** ΑΛΛΑΓΗ ΕΔΩ: Κάλεσε το CascadedImageOnlyClassifier ***
    model_stage2 = CascadedImageOnlyClassifier(
        in_channels_img=IN_CHANNELS, out_channels_seg=OUT_CHANNELS_SEG,
        feature_size=FEATURE_SIZE, use_checkpoint=USE_CHECKPOINTING,
        num_classes_grade=NUM_CLASSES_GRADE, num_classes_diag=NUM_CLASSES_DIAG,
        finetuned_seg_model_path=finetuned_model_path
    ).to(DEVICE)
except NameError:
    print("ΣΦΑΛΜΑ: Η κλάση 'CascadedImageOnlyClassifier' δεν έχει οριστεί. Εκτέλεσε ξανά το Κελί 4.")
    # exit()

# *** ΑΛΛΑΓΗ ΕΔΩ: Νέα params_to_train ***
params_to_train = [
    {'params': model_stage2.fusion_mlp.parameters()},
    {'params': model_stage2.grade_head.parameters()},
    {'params': model_stage2.diagnosis_head.parameters()}
]
print(f"Σύνολο παραμέτρων προς εκπαίδευση: {sum(p.numel() for group in params_to_train for p in group['params'])}")

optimizer_stage2 = torch.optim.AdamW(params_to_train, lr=LEARNING_RATE_MTL, weight_decay=WEIGHT_DECAY_MTL)
scheduler_stage2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_stage2, T_max=NUM_EPOCHS_MTL)

# --- ΧΩΡΙΣ GradScaler ---
best_val_metric = -1.0
start_epoch = 0
history = {
    "epoch": [], "train_loss_total": [], "train_loss_grade": [], "train_loss_diag": [],
    "val_loss_total": [], "val_loss_grade": [], "val_loss_diag": [],
    "val_acc_grade": [], "val_auc_grade": [],
    "val_acc_diag": [], "val_auc_diag": [],
    "val_metric_combined": []
}

# --- Resuming Logic (ΔΙΑΓΡΑΦΗ CHECKPOINTS ΣΥΝΙΣΤΑΤΑΙ) ---
if os.path.exists(latest_checkpoint_path_stage2):
    print(f"ΠΡΟΣΟΧΗ: Βρέθηκε παλιό checkpoint. Διαγράφεται για να ξεκινήσει η νέα αρχιτεκτονική.")
    os.remove(latest_checkpoint_path_stage2)
    if os.path.exists(best_checkpoint_path_stage2):
        os.remove(best_checkpoint_path_stage2)
print("Έναρξη Stage 2 εκπαίδευσης από την αρχή (v5 - ΜΟΝΟ Εικόνα).")
start_epoch = 0; best_val_metric = -1.0; history = {k: [] for k in history.keys()}


print(f"\n--- Έναρξη Stage 2 Classification Loop από εποχή {start_epoch + 1} (ΧΩΡΙΣ AMP) ---")

try: _ = train_loader_mtl
except NameError: print("\n!!! ΣΦΑΛΜΑ: Ο 'train_loader_mtl' δεν έχει οριστεί. Εκτέλεσε το Κελί 3."); # exit()

for epoch in range(start_epoch, NUM_EPOCHS_MTL):
    current_epoch = epoch + 1
    print(f"\n===== Epoch {current_epoch}/{NUM_EPOCHS_MTL} =====")
    model_stage2.train()
    model_stage2.swin_unetr_extractor.eval()

    epoch_loss_total, epoch_loss_grade, epoch_loss_diag = 0.0, 0.0, 0.0
    step = 0
    pbar_train = tqdm(train_loader_mtl, desc=f"Training Stage 2 Epoch {current_epoch}")

    for batch_data in pbar_train:
        step += 1
        inputs = batch_data[image_key].to(DEVICE)
        # data_clinical = batch_data[clinical_key].to(DEVICE) # <-- ΔΕΝ ΤΟ ΧΡΕΙΑΖΟΜΑΣΤΕ
        labels_grade = batch_data[grade_key].to(DEVICE)
        labels_diag = batch_data[diag_key].to(DEVICE)

        optimizer_stage2.zero_grad(set_to_none=True)

        # *** ΑΛΛΑΓΗ: Κλήση ΜΟΝΟ με 'inputs' ***
        logits_grade, logits_diag = model_stage2(inputs)

        total_loss, l_g, l_d = classification_loss_fn(
            logits_grade, labels_grade, logits_diag, labels_diag
        )

        if not torch.isfinite(total_loss):
            print(f"!!! WARNING: Non-finite loss. Skipping step.")
            continue

        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(
            filter(lambda p: p.requires_grad, model_stage2.parameters()), 1.0
        )
        optimizer_stage2.step()

        epoch_loss_total += total_loss.item(); epoch_loss_grade += l_g.item(); epoch_loss_diag += l_d.item()
        pbar_train.set_postfix(total_loss=f"{total_loss.item():.4f}")
        del inputs, labels_grade, labels_diag, logits_grade, logits_diag, total_loss, l_g, l_d # data_clinical αφαιρέθηκε
        if DEVICE == 'cuda': torch.cuda.empty_cache()

    # ... (Logging και history append) ...
    avg_epoch_loss_total = epoch_loss_total / step if step > 0 else np.nan
    # ... (print, history append) ...

    # --- Validation ---
    if current_epoch % VAL_INTERVAL == 0:
        model_stage2.eval()

        auc_metric_grade = ROCAUCMetric(); auc_metric_diag = ROCAUCMetric()
        all_labels_grade = []; all_preds_grade_indices = []; all_preds_grade_probs = []
        all_labels_diag = []; all_preds_diag_indices = []; all_preds_diag_probs = []

        epoch_val_loss_total, epoch_val_loss_grade, epoch_val_loss_diag = 0.0, 0.0, 0.0
        val_step = 0
        pbar_val = tqdm(val_loader_mtl, desc=f"Validation Stage 2 Epoch {current_epoch}")

        with torch.no_grad():
            for val_data in pbar_val:
                val_step += 1
                val_inputs = val_data[image_key].to(DEVICE)
                # val_data_clinical = val_data[clinical_key].to(DEVICE) # <-- ΔΕΝ ΧΡΕΙΑΖΕΤΑΙ
                val_labels_grade = val_data[grade_key].to(DEVICE); val_labels_diag = val_data[diag_key].to(DEVICE)

                # *** ΑΛΛΑΓΗ: Κλήση ΜΟΝΟ με 'val_inputs' ***
                val_logits_grade, val_logits_diag = model_stage2(val_inputs)

                total_loss, l_g, l_d = classification_loss_fn(val_logits_grade, val_labels_grade, val_logits_diag, val_labels_diag)
                if torch.isfinite(total_loss): epoch_val_loss_total += total_loss.item(); epoch_val_loss_grade += l_g.item(); epoch_val_loss_diag += l_d.item()

                # --- Χειροκίνητη Συλλογή Μετρικών (Ίδιο με πριν) ---
                val_outputs_grade_indices = torch.argmax(val_logits_grade, dim=1)
                val_outputs_diag_indices = torch.argmax(val_logits_diag, dim=1)
                val_probs_grade = F.softmax(val_logits_grade, dim=1)
                val_probs_diag = F.softmax(val_logits_diag, dim=1)
                val_labels_grade_onehot = F.one_hot(val_labels_grade, num_classes=NUM_CLASSES_GRADE).float()
                val_labels_diag_onehot = F.one_hot(val_labels_diag, num_classes=NUM_CLASSES_DIAG).float()
                auc_metric_grade(y_pred=val_probs_grade, y=val_labels_grade_onehot)
                auc_metric_diag(y_pred=val_probs_diag, y=val_labels_diag_onehot)
                all_preds_grade_indices.append(val_outputs_grade_indices.cpu())
                all_labels_grade.append(val_labels_grade.cpu())
                all_preds_grade_probs.append(val_probs_grade.cpu())
                all_preds_diag_indices.append(val_outputs_diag_indices.cpu())
                all_labels_diag.append(val_labels_diag.cpu())
                all_preds_diag_probs.append(val_probs_diag.cpu())

                del val_inputs, val_labels_grade, val_labels_diag, val_logits_grade, val_logits_diag, total_loss, l_g, l_d
                torch.cuda.empty_cache()

        # --- Aggregate (Συγκέντρωση) όλων των μετρικών (Ίδιο με πριν) ---
        # ... (Υπολογισμός sklearn metrics: accuracy_score, confusion_matrix, classification_report) ...
        # ... (Aggregate MONAI AUC) ...
        # ... (Print reports) ...

        # --- Checkpointing (Ίδιο με πριν, αλλά με λιγότερα keys) ---
        current_metric = (accuracy_score(all_labels_grade, all_preds_grade_indices) + accuracy_score(all_labels_diag, all_preds_diag_indices)) / 2.0

        if current_metric > best_val_metric:
            best_val_metric = current_metric
            checkpoint = {
                'epoch': current_epoch,
                # *** ΑΛΛΑΓΗ: Αφαίρεση clinical_encoder_state ***
                'fusion_mlp_state': model_stage2.fusion_mlp.state_dict(),
                'grade_head_state': model_stage2.grade_head.state_dict(),
                'diagnosis_head_state': model_stage2.diagnosis_head.state_dict(),
                'optimizer_state': optimizer_stage2.state_dict(),
                'best_val_metric': best_val_metric,
                'scheduler_state': scheduler_stage2.state_dict() if scheduler_stage2 else None,
                'val_loss': mean_val_loss_total
            }
            torch.save(checkpoint, best_checkpoint_path_stage2)
            print(f"✅ Saved new best model (Combined Acc: {best_val_metric:.4f}) to {best_checkpoint_path_stage2}")

        torch.save(checkpoint, latest_checkpoint_path_stage2)

    # ... (Save history CSV) ...
    # ... (Scheduler step) ...

print(f"🏁 Stage 2 Training finished. Best Validation Metric (Combined Acc): {best_val_metric:.4f}")

# Transformer fash

In [ ]:
# %% Βήμα 4 (Στρατηγική 3): Ορισμός του Μοντέλου (Transformer Fusion)
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.networks.nets import SwinUNETR
import os

print("Ορισμός Μοντέλου CascadedTransformerClassifier (με Fusion Transformer)...")

class CascadedTransformerClassifier(nn.Module):
    def __init__(self,
                 in_channels_img, out_channels_seg, feature_size, use_checkpoint,
                 num_classes_grade, num_classes_diag, clinical_feature_size,
                 finetuned_seg_model_path=None,
                 d_model=256, # Η κοινή διάσταση για όλα τα tokens
                 n_heads=4,   # Αριθμός κεφαλών (32 channels/head αν d_model=128)
                 num_layers=2 # Πόσα transformer layers
                 ):

        super().__init__()

        # --- 1. Φόρτωση και Πάγωμα του SwinUNETR ---
        self.swin_unetr_extractor = SwinUNETR(
            in_channels=in_channels_img,
            out_channels=out_channels_seg,
            feature_size=feature_size,
            use_checkpoint=use_checkpoint
        )
        if finetuned_seg_model_path and os.path.exists(finetuned_seg_model_path):
            print(f"Φόρτωση checkpoint τμηματοποίησης από: {finetuned_seg_model_path}")
            checkpoint = torch.load(finetuned_seg_model_path, map_location=torch.device('cpu'))
            try:
                self.swin_unetr_extractor.load_state_dict(checkpoint['model_state'], strict=False)
                print(f"Επιτυχής φόρτωση βαρών μοντέλου τμηματοποίησης.")
            except Exception as e: print(f"Σφάλμα φόρτωσης state_dict: {e}")
        else: print("ΠΡΟΣΟΧΗ: Δεν βρέθηκε εκπαιδευμένο μοντέλο.")

        print("--- ΠΑΓΩΜΑ (FREEZING) ολόκληρου του SwinUNETR extractor ---")
        for param in self.swin_unetr_extractor.parameters():
            param.requires_grad = False

        # --- 2. Hooks για Εξαγωγή Features ---
        self.encoder_features_out = None
        self.decoder_features_out = None
        self.swin_unetr_extractor.swinViT.register_forward_hook(self.hook_encoder_features)
        self.swin_unetr_extractor.decoder1.register_forward_hook(self.hook_decoder_features)

        # --- 3. Layers Προβολής (Projectors) ---
        # Μετατρέπουν τα διαφορετικά inputs στο ίδιο μέγεθος (d_model)
        encoder_dim = feature_size * 16 # 768
        decoder_dim = feature_size      # 48

        self.encoder_proj = nn.Sequential(
            nn.LayerNorm(encoder_dim),
            nn.Linear(encoder_dim, d_model)
        )
        self.decoder_proj = nn.Sequential(
            nn.LayerNorm(decoder_dim),
            nn.Linear(decoder_dim, d_model)
        )
        self.clinical_proj = nn.Sequential(
            nn.LayerNorm(clinical_feature_size),
            nn.Linear(clinical_feature_size, d_model)
        )

        # --- 4. Task Tokens (Learnable) ---
        # Αυτά τα tokens θα "μαζέψουν" την πληροφορία
        self.grade_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.diag_token = nn.Parameter(torch.randn(1, 1, d_model))

        # --- 5. Fusion Transformer ---
        transformer_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 2,
            dropout=0.2,
            batch_first=True,
            norm_first=True # Καλύτερη σταθερότητα
        )
        self.fusion_transformer = nn.TransformerEncoder(transformer_layer, num_layers=num_layers)

       # --- 6. Classification Heads (ΙΣΧΥΡΑ MLPs - Η ΑΛΛΑΓΗ) ---
        # Αντί για ένα Linear layer, βάζουμε ένα μικρό δίκτυο 2 επιπέδων
        # για να μιμηθούμε την ισχύ του Ablation MLP

        self.grade_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes_grade)
        )

        self.diagnosis_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes_diag)
        )

        print(f"Transformer: d_model={d_model}. Heads: MLP (256->128->64->Classes)")

    # --- Hooks ---
    def hook_encoder_features(self, module, input, output):
        self.encoder_features_out = output[-1]
    def hook_decoder_features(self, module, input, output):
        self.decoder_features_out = output

    def forward(self, x_img, x_clinical):
        B = x_img.shape[0]

        with torch.no_grad():
            _ = self.swin_unetr_extractor(x_img)

        enc_feat = F.adaptive_avg_pool3d(self.encoder_features_out, 1).flatten(1)
        enc_token = self.encoder_proj(enc_feat).unsqueeze(1)

        dec_feat = F.adaptive_avg_pool3d(self.decoder_features_out, 1).flatten(1)
        dec_token = self.decoder_proj(dec_feat).unsqueeze(1)

        clin_token = self.clinical_proj(x_clinical).unsqueeze(1)

        grade_token = self.grade_token.expand(B, -1, -1)
        diag_token = self.diag_token.expand(B, -1, -1)

        tokens = torch.cat([grade_token, diag_token, enc_token, dec_token, clin_token], dim=1)
        fused_tokens = self.fusion_transformer(tokens)

        out_grade = fused_tokens[:, 0, :]
        out_diag = fused_tokens[:, 1, :]

        logits_grade = self.grade_head(out_grade)
        logits_diag = self.diagnosis_head(out_diag)

        return logits_grade, logits_diag

print("Ορισμός Μοντέλου CascadedTransformerClassifier (Deep Heads) ολοκληρώθηκε.")

In [ ]:
# %% Βήμα 5: Εκπαίδευση Stage 2 (Transformer Fusion - Optimized)
import torch.nn as nn
from tqdm import tqdm
import pandas as pd
import numpy as np
import os
import gc
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from monai.metrics import ROCAUCMetric
from monai.transforms import AsDiscrete, Activations

# --- 1. Υπολογισμός Class Weights (ΓΙΑ ΝΑ ΞΕΚΟΛΛΗΣΕΙ ΤΟ ΜΟΝΤΕΛΟ) ---
print("Υπολογισμός Class Weights...")
# Παίρνουμε τα labels από το df_processed (που φτιάξαμε στο Κελί 2)
# Βεβαιώσου ότι το df_processed υπάρχει. Αν όχι, τρέξε το Κελί 2.
try:
    y_grade_all = df_processed['Grade_Label'].values
    y_diag_all = df_processed['Diagnosis_Label'].values

    # Υπολογισμός βαρών: "balanced"
    weights_grade = compute_class_weight(class_weight='balanced', classes=np.unique(y_grade_all), y=y_grade_all)
    weights_diag = compute_class_weight(class_weight='balanced', classes=np.unique(y_diag_all), y=y_diag_all)

    # Μετατροπή σε Tensor
    weights_grade_tensor = torch.tensor(weights_grade, dtype=torch.float).to(DEVICE)
    weights_diag_tensor = torch.tensor(weights_diag, dtype=torch.float).to(DEVICE)

    print(f"Weights Grade: {weights_grade_tensor}")
    print(f"Weights Diagnosis: {weights_diag_tensor}")
except NameError:
    print("ΠΡΟΣΟΧΗ: Το df_processed δεν βρέθηκε. Χρήση default βαρών (1.0).")
    weights_grade_tensor = None
    weights_diag_tensor = None

# --- 2. Συναρτήσεις Απώλειας (με ΒΑΡΗ) ---
loss_grade = nn.CrossEntropyLoss(weight=weights_grade_tensor)
loss_diag = nn.CrossEntropyLoss(weight=weights_diag_tensor)
w_grade = 0.5
w_diag = 0.5

def classification_loss_fn(pred_grade, true_grade, pred_diag, true_diag):
    l_g = loss_grade(pred_grade, true_grade)
    l_d = loss_diag(pred_diag, true_diag)
    total_loss = (w_grade * l_g) + (w_diag * l_d)
    return total_loss, l_g, l_d

# --- Post-processing (Μόνο για AUC) ---
post_pred_auc = Activations(softmax=True)
try:
    post_label_onehot_grade = AsDiscrete(to_onehot=NUM_CLASSES_GRADE)
    post_label_onehot_diag = AsDiscrete(to_onehot=NUM_CLASSES_DIAG)
except NameError:
    print("ΣΦΑΛΜΑ: NUM_CLASSES_GRADE/DIAG λείπουν. Τρέξε Κελί 2.")

# --- Ρυθμίσεις ---
output_dir_stage2 = os.path.join(runs_base_dir, "classification_cascaded_v4_transformer_weighted") # Νέος φάκελος
history_csv_path_stage2 = os.path.join(output_dir_stage2, "training_history.csv")
latest_checkpoint_path_stage2 = os.path.join(output_dir_stage2, "latest_model.pt")
best_checkpoint_path_stage2 = os.path.join(output_dir_stage2, "best_model.pt")
os.makedirs(output_dir_stage2, exist_ok=True)

# --- Αρχικοποίηση Μοντέλου ---
try:
    model_stage2 = CascadedTransformerClassifier( # Χρήση Transformer
        in_channels_img=IN_CHANNELS, out_channels_seg=OUT_CHANNELS_SEG,
        feature_size=FEATURE_SIZE, use_checkpoint=USE_CHECKPOINTING,
        num_classes_grade=NUM_CLASSES_GRADE, num_classes_diag=NUM_CLASSES_DIAG,
        clinical_feature_size=CLINICAL_FEATURE_SIZE,
        finetuned_seg_model_path=finetuned_model_path,
        d_model=64, n_heads=4, num_layers=2
    ).to(DEVICE)
except NameError:
    print("ΣΦΑΛΜΑ: 'CascadedTransformerClassifier' δεν ορίστηκε. Τρέξε Κελί 4.")

# --- Optimizer (HIGHER LR) ---
NEW_LR = 1e-3 # Υψηλότερο LR για να ξεκολλήσει
print(f"Learning Rate: {NEW_LR}")

params_to_train = [
    {'params': model_stage2.encoder_proj.parameters()},
    {'params': model_stage2.decoder_proj.parameters()},
    {'params': model_stage2.clinical_proj.parameters()},
    {'params': model_stage2.grade_token},
    {'params': model_stage2.diag_token},
    {'params': model_stage2.fusion_transformer.parameters()},
    {'params': model_stage2.grade_head.parameters()},
    {'params': model_stage2.diagnosis_head.parameters()}
]

optimizer_stage2 = torch.optim.AdamW(params_to_train, lr=NEW_LR, weight_decay=1e-4)
scheduler_stage2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_stage2, T_max=NUM_EPOCHS_MTL)

# --- Μεταβλητές ---
best_val_metric = -1.0
start_epoch = 0
history = {
    "epoch": [], "train_loss_total": [],
    "val_loss_total": [], "val_loss_grade": [], "val_loss_diag": [],
    "val_acc_grade": [], "val_f1_grade": [], "val_auc_grade": [],
    "val_acc_diag": [], "val_f1_diag": [], "val_auc_diag": [],
    "val_metric_combined": []
}

# --- Resuming Logic (ΔΙΑΓΡΑΦΗ ΠΑΛΙΩΝ) ---
if os.path.exists(latest_checkpoint_path_stage2):
    print("Διαγραφή παλιών checkpoints για καθαρή εκκίνηση...")
    try: os.remove(latest_checkpoint_path_stage2)
    except: pass
    try: os.remove(best_checkpoint_path_stage2)
    except: pass

print("\n--- Έναρξη Training (Weighted Loss + Grad Accum + High LR) ---")

# --- Gradient Accumulation Steps ---
GRAD_ACCUM_STEPS = 4 # Effective Batch Size = 2 * 4 = 8
print(f"Gradient Accumulation Steps: {GRAD_ACCUM_STEPS}")

for epoch in range(start_epoch, NUM_EPOCHS_MTL):
    current_epoch = epoch + 1
    model_stage2.train()
    model_stage2.swin_unetr_extractor.eval()

    epoch_loss_total = 0.0
    step = 0
    optimizer_stage2.zero_grad() # Zero grad ΠΡΙΝ το loop

    pbar_train = tqdm(train_loader_mtl, desc=f"Epoch {current_epoch} Train")

    for i, batch_data in enumerate(pbar_train):
        step += 1
        inputs = batch_data[image_key].to(DEVICE)
        data_clinical = batch_data[clinical_key].to(DEVICE)
        labels_grade = batch_data[grade_key].to(DEVICE)
        labels_diag = batch_data[diag_key].to(DEVICE)

        logits_grade, logits_diag = model_stage2(inputs, data_clinical)

        total_loss, _, _ = classification_loss_fn(logits_grade, labels_grade, logits_diag, labels_diag)

        if not torch.isfinite(total_loss):
            print("Warning: Non-finite loss"); continue

        # Διαίρεση loss για accumulation
        (total_loss / GRAD_ACCUM_STEPS).backward()

        if (i + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(params_to_train[0]['params'], 1.0)
            optimizer_stage2.step()
            optimizer_stage2.zero_grad()

        epoch_loss_total += total_loss.item()
        pbar_train.set_postfix(loss=f"{total_loss.item():.4f}")
        del inputs, data_clinical, labels_grade, labels_diag, logits_grade, logits_diag, total_loss
        if DEVICE == 'cuda': torch.cuda.empty_cache()

    avg_loss = epoch_loss_total / step if step > 0 else 0
    print(f"Epoch {current_epoch} Train Loss: {avg_loss:.4f}, LR: {optimizer_stage2.param_groups[0]['lr']:.6f}")
    history["epoch"].append(current_epoch); history["train_loss_total"].append(avg_loss)

    # --- Validation ---
    if current_epoch % VAL_INTERVAL == 0:
        model_stage2.eval()
        auc_metric_grade = ROCAUCMetric(); auc_metric_diag = ROCAUCMetric()
        all_labels_grade = []; all_preds_grade_indices = []; all_preds_grade_probs = []
        all_labels_diag = []; all_preds_diag_indices = []; all_preds_diag_probs = []

        epoch_val_loss_total, epoch_val_loss_grade, epoch_val_loss_diag = 0.0, 0.0, 0.0
        val_step = 0

        with torch.no_grad():
            for val_data in tqdm(val_loader_mtl, desc="Validating"):
                val_step += 1
                inputs = val_data[image_key].to(DEVICE); data_clinical = val_data[clinical_key].to(DEVICE)
                lbl_g = val_data[grade_key].to(DEVICE); lbl_d = val_data[diag_key].to(DEVICE)

                lg, ld = model_stage2(inputs, data_clinical)

                v_loss, vl_g, vl_d = classification_loss_fn(lg, lbl_g, ld, lbl_d)
                if torch.isfinite(v_loss):
                    epoch_val_loss_total += v_loss.item()
                    epoch_val_loss_grade += vl_g.item()
                    epoch_val_loss_diag += vl_d.item()

                # Metrics prep
                all_preds_grade_indices.append(torch.argmax(lg, 1).cpu())
                all_labels_grade.append(lbl_g.cpu())
                all_preds_grade_probs.append(F.softmax(lg, 1).cpu())

                all_preds_diag_indices.append(torch.argmax(ld, 1).cpu())
                all_labels_diag.append(lbl_d.cpu())
                all_preds_diag_probs.append(F.softmax(ld, 1).cpu())

                # AUC (MONAI)
                auc_metric_grade(y_pred=F.softmax(lg, 1), y=F.one_hot(lbl_g, NUM_CLASSES_GRADE).float())
                auc_metric_diag(y_pred=F.softmax(ld, 1), y=F.one_hot(lbl_d, NUM_CLASSES_DIAG).float())

                del inputs, data_clinical, lbl_g, lbl_d, lg, ld, v_loss
                torch.cuda.empty_cache()

        # --- Metrics Calculation ---
        all_preds_grade_indices = torch.cat(all_preds_grade_indices).numpy()
        all_labels_grade = torch.cat(all_labels_grade).numpy()
        all_preds_grade_probs = torch.cat(all_preds_grade_probs).numpy()
        all_preds_diag_indices = torch.cat(all_preds_diag_indices).numpy()
        all_labels_diag = torch.cat(all_labels_diag).numpy()
        all_preds_diag_probs = torch.cat(all_preds_diag_probs).numpy()

        grade_labels = list(range(NUM_CLASSES_GRADE)); diag_labels = list(range(NUM_CLASSES_DIAG))

        acc_g = accuracy_score(all_labels_grade, all_preds_grade_indices)
        f1_g = f1_score(all_labels_grade, all_preds_grade_indices, average='macro')
        acc_d = accuracy_score(all_labels_diag, all_preds_diag_indices)
        f1_d = f1_score(all_labels_diag, all_preds_diag_indices, average='macro')

        try: auc_g = auc_metric_grade.aggregate().item()
        except: auc_g = 0.5
        try: auc_d = auc_metric_diag.aggregate().item()
        except: auc_d = 0.5

        mean_val_loss_total = epoch_val_loss_total / val_step if val_step > 0 else np.nan
        mean_val_loss_grade = epoch_val_loss_grade / val_step if val_step > 0 else np.nan
        mean_val_loss_diag = epoch_val_loss_diag / val_step if val_step > 0 else np.nan

        combined = (f1_g + f1_d) / 2.0

        print(f"--- Validation Report Epoch {current_epoch} ---")
        print(f"Loss: {mean_val_loss_total:.4f} (G: {mean_val_loss_grade:.4f}, D: {mean_val_loss_diag:.4f})")
        print(f"[GRADE] Acc: {acc_g:.4f}, F1: {f1_g:.4f}, AUC: {auc_g:.4f}")
        print(classification_report(all_labels_grade, all_preds_grade_indices, labels=grade_labels, target_names=[str(c) for c in GRADE_CLASSES], zero_division=0))
        print(confusion_matrix(all_labels_grade, all_preds_grade_indices, labels=grade_labels))

        print(f"[DIAGNOSIS] Acc: {acc_d:.4f}, F1: {f1_d:.4f}, AUC: {auc_d:.4f}")
        print(classification_report(all_labels_diag, all_preds_diag_indices, labels=diag_labels, target_names=[str(c) for c in DIAG_CLASSES], zero_division=0))
        print(confusion_matrix(all_labels_diag, all_preds_diag_indices, labels=diag_labels))

        # History
        history["val_loss_total"].append(mean_val_loss_total); history["val_loss_grade"].append(mean_val_loss_grade); history["val_loss_diag"].append(mean_val_loss_diag)
        history["val_acc_grade"].append(acc_g); history["val_f1_grade"].append(f1_g); history["val_auc_grade"].append(auc_g)
        history["val_acc_diag"].append(acc_d); history["val_f1_diag"].append(f1_d); history["val_auc_diag"].append(auc_d)
        history["val_metric_combined"].append(combined)

        if combined > best_val_metric:
            best_val_metric = combined
            torch.save(model_stage2.state_dict(), best_checkpoint_path_stage2)
            print(f"✅ New Best! ({combined:.4f})")

    if scheduler_stage2: scheduler_stage2.step()


    # --- TESTING PHASE (ΝΕΟ) ---
print("\n=== Έναρξη Τελικής Αξιολόγησης (Test Set) ===")

if os.path.exists(best_checkpoint_path_stage2):
    print(f"Φόρτωση καλύτερου μοντέλου από: {best_checkpoint_path_stage2}")
    # Φορτώνουμε τα βάρη στο μοντέλο (το οποίο έχει ήδη τον παγωμένο encoder)
    best_weights = torch.load(best_checkpoint_path_stage2, map_location=DEVICE)
    model_stage2.load_state_dict(best_weights, strict=False) # strict=False γιατί λείπει ο extractor από το save
    model_stage2.eval()

    test_preds_grade, test_labels_grade = [], []
    test_preds_diag, test_labels_diag = [], []

    with torch.no_grad():
        for test_data in tqdm(test_loader_mtl, desc="Testing"):
            inputs = test_data[image_key].to(DEVICE); data_clinical = test_data[clinical_key].to(DEVICE)
            lbl_g = test_data[grade_key].to(DEVICE); lbl_d = test_data[diag_key].to(DEVICE)

            lg, ld = model_stage2(inputs, data_clinical)

            test_preds_grade.extend(torch.argmax(lg, 1).cpu().numpy())
            test_labels_grade.extend(lbl_g.cpu().numpy())
            test_preds_diag.extend(torch.argmax(ld, 1).cpu().numpy())
            test_labels_diag.extend(lbl_d.cpu().numpy())

    # --- Final Metrics ---
    print("\n" + "="*30)
    print("   FINAL TEST RESULTS   ")
    print("="*30)

    print("\n[GRADE - TEST]")
    print(classification_report(test_labels_grade, test_preds_grade, labels=list(range(NUM_CLASSES_GRADE)), target_names=[str(c) for c in GRADE_CLASSES], zero_division=0))
    print("Confusion Matrix:\n", confusion_matrix(test_labels_grade, test_preds_grade, labels=list(range(NUM_CLASSES_GRADE))))

    print("\n[DIAGNOSIS - TEST]")
    print(classification_report(test_labels_diag, test_preds_diag, labels=list(range(NUM_CLASSES_DIAG)), target_names=[str(c) for c in DIAG_CLASSES], zero_division=0))
    print("Confusion Matrix:\n", confusion_matrix(test_labels_diag, test_preds_diag, labels=list(range(NUM_CLASSES_DIAG))))

else:
    print("ΣΦΑΛΜΑ: Δεν βρέθηκε το αρχείο του καλύτερου μοντέλου για testing.")

In [ ]:
df_history = pd.DataFrame(history)
df_history.to_csv(history_csv_path_stage2, index=False)


In [ ]:
# %% Βήμα 6 (Διορθωμένο): Οπτικοποίηση Αποτελεσμάτων (Robust Plotting)
import pandas as pd
import matplotlib.pyplot as plt
import os

# --- Ρυθμίσεις Διαδρομής (Βεβαιώσου ότι είναι ο σωστός φάκελος) ---
# Χρησιμοποιούμε τον φάκελο που ορίσαμε στο Κελί 5
target_dir = os.path.join(runs_base_dir, "classification_cascaded_v4_transformer_weighted")
csv_filename = "training_history.csv" # Στο τελευταίο κελί 5 το ονομάσαμε έτσι
csv_path = os.path.join(target_dir, csv_filename)

print(f"Προσπάθεια φόρτωσης από: {csv_path}")

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    epochs = df['epoch']

    print(f"Επιτυχής φόρτωση. Βρέθηκαν {len(df)} εποχές.")
    print("Διαθέσιμες στήλες:", df.columns.tolist()) # Για να δούμε τι υπάρχει μέσα

    plt.style.use('seaborn-v0_8-darkgrid')

    # Δυναμικός καθορισμός του grid (αν λείπουν δεδομένα, φτιάχνουμε λιγότερα plots)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Training & Validation Metrics', fontsize=16)

    # --- PLOT 1: LOSSES ---
    if 'train_loss_total' in df.columns:
        axes[0].plot(epochs, df['train_loss_total'], label='Train Total', color='blue', linestyle='--')
    if 'val_loss_total' in df.columns:
        axes[0].plot(epochs, df['val_loss_total'], label='Val Total', color='blue')

    # Grade Loss (Αν υπάρχει)
    if 'train_loss_grade' in df.columns:
        axes[0].plot(epochs, df['train_loss_grade'], label='Train Grade', color='green', linestyle='--')
    if 'val_loss_grade' in df.columns:
        axes[0].plot(epochs, df['val_loss_grade'], label='Val Grade', color='green')

    # Diagnosis Loss (Αν υπάρχει)
    if 'train_loss_diag' in df.columns:
        axes[0].plot(epochs, df['train_loss_diag'], label='Train Diag', color='purple', linestyle='--')
    if 'val_loss_diag' in df.columns:
        axes[0].plot(epochs, df['val_loss_diag'], label='Val Diag', color='purple')

    axes[0].set_title('Losses')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()

    # --- PLOT 2: GRADE METRICS ---
    if 'val_acc_grade' in df.columns:
        axes[1].plot(epochs, df['val_acc_grade'], label='Acc Grade', color='green')
    if 'val_f1_grade' in df.columns:
        axes[1].plot(epochs, df['val_f1_grade'], label='F1 Grade', color='lightgreen', linestyle='--')
    if 'val_auc_grade' in df.columns:
        axes[1].plot(epochs, df['val_auc_grade'], label='AUC Grade', color='olive', linestyle=':')

    axes[1].set_title('Grade Performance')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylim(0, 1.0)
    axes[1].legend()

    # --- PLOT 3: DIAGNOSIS METRICS ---
    if 'val_acc_diag' in df.columns:
        axes[2].plot(epochs, df['val_acc_diag'], label='Acc Diag', color='purple')
    if 'val_f1_diag' in df.columns:
        axes[2].plot(epochs, df['val_f1_diag'], label='F1 Diag', color='violet', linestyle='--')
    if 'val_auc_diag' in df.columns:
        axes[2].plot(epochs, df['val_auc_diag'], label='AUC Diag', color='magenta', linestyle=':')

    axes[2].set_title('Diagnosis Performance')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylim(0, 1.0)
    axes[2].legend()

    plt.tight_layout()
    plt.show()

    if 'val_metric_combined' in df.columns:
        print(f"Καλύτερο Combined Metric: {df['val_metric_combined'].max():.4f}")

else:
    print(f"Δεν βρέθηκε το αρχείο: {csv_path}")
    print("Έλεγξε αν η εκπαίδευση ολοκλήρωσε τουλάχιστον μία εποχή.")

In [ ]:
# %% Βήμα 4: Ορισμός Μοντέλου Transformer Fusion (Anti-Overfitting Version)
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.networks.nets import SwinUNETR
import os

print("Ορισμός Μοντέλου CascadedTransformerClassifier (Small & Regularized)...")

class CascadedTransformerClassifier(nn.Module):
    def __init__(self,
                 in_channels_img, out_channels_seg, feature_size, use_checkpoint,
                 num_classes_grade, num_classes_diag, clinical_feature_size,
                 finetuned_seg_model_path=None,
                 # --- ΑΛΛΑΓΕΣ ΓΙΑ OVERFITTING ---
                 d_model=64,        # Μειώθηκε από 256 -> 64
                 n_heads=4,         # 4 κεφαλές των 16 διαστάσεων
                 num_layers=2,      # Κρατάμε 2 layers
                 dropout_rate=0.5   # Αυξήθηκε από default 0.1 -> 0.5
                 # -----------------------------
                 ):

        super().__init__()

        # 1. SwinUNETR (Frozen)
        self.swin_unetr_extractor = SwinUNETR(
            in_channels=in_channels_img, out_channels=out_channels_seg,
            feature_size=feature_size, use_checkpoint=use_checkpoint
        )
        if finetuned_seg_model_path and os.path.exists(finetuned_seg_model_path):
            checkpoint = torch.load(finetuned_seg_model_path, map_location=torch.device('cpu'))
            try: self.swin_unetr_extractor.load_state_dict(checkpoint['model_state'], strict=False)
            except: pass

        for param in self.swin_unetr_extractor.parameters():
            param.requires_grad = False

        # 2. Hooks
        self.encoder_features_out = None
        self.decoder_features_out = None
        self.swin_unetr_extractor.swinViT.register_forward_hook(self.hook_encoder_features)
        self.swin_unetr_extractor.decoder1.register_forward_hook(self.hook_decoder_features)

        # 3. Projectors (με Dropout)
        encoder_dim = feature_size * 16
        decoder_dim = feature_size

        self.encoder_proj = nn.Sequential(
            nn.LayerNorm(encoder_dim),
            nn.Linear(encoder_dim, d_model),
            nn.Dropout(dropout_rate) # Προσθήκη Dropout
        )
        self.decoder_proj = nn.Sequential(
            nn.LayerNorm(decoder_dim),
            nn.Linear(decoder_dim, d_model),
            nn.Dropout(dropout_rate) # Προσθήκη Dropout
        )
        self.clinical_proj = nn.Sequential(
            nn.LayerNorm(clinical_feature_size),
            nn.Linear(clinical_feature_size, d_model),
            nn.Dropout(dropout_rate) # Προσθήκη Dropout
        )

        # 4. Tokens
        self.grade_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.diag_token = nn.Parameter(torch.randn(1, 1, d_model))

        # 5. Transformer (με υψηλό Dropout)
        transformer_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 2,
            dropout=dropout_rate, # <-- ΥΨΗΛΟ DROPOUT
            batch_first=True,
            norm_first=True
        )
        self.fusion_transformer = nn.TransformerEncoder(transformer_layer, num_layers=num_layers)


       # --- 6. Classification Heads (ΙΣΧΥΡΑ MLPs - Η ΑΛΛΑΓΗ) ---
        # Αντί για ένα Linear layer, βάζουμε ένα μικρό δίκτυο 2 επιπέδων
        # για να μιμηθούμε την ισχύ του Ablation MLP

        self.grade_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes_grade)
        )

        self.diagnosis_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes_diag)
        )

        print(f"Transformer: d_model={d_model}. Heads: MLP (256->128->64->Classes)")

    # --- Hooks ---
    def hook_encoder_features(self, module, input, output):
        self.encoder_features_out = output[-1]
    def hook_decoder_features(self, module, input, output):
        self.decoder_features_out = output

    def forward(self, x_img, x_clinical):
        B = x_img.shape[0]

        with torch.no_grad():
            _ = self.swin_unetr_extractor(x_img)

        enc_feat = F.adaptive_avg_pool3d(self.encoder_features_out, 1).flatten(1)
        enc_token = self.encoder_proj(enc_feat).unsqueeze(1)

        dec_feat = F.adaptive_avg_pool3d(self.decoder_features_out, 1).flatten(1)
        dec_token = self.decoder_proj(dec_feat).unsqueeze(1)

        clin_token = self.clinical_proj(x_clinical).unsqueeze(1)

        grade_token = self.grade_token.expand(B, -1, -1)
        diag_token = self.diag_token.expand(B, -1, -1)

        tokens = torch.cat([grade_token, diag_token, enc_token, dec_token, clin_token], dim=1)
        fused_tokens = self.fusion_transformer(tokens)

        out_grade = fused_tokens[:, 0, :]
        out_diag = fused_tokens[:, 1, :]

        logits_grade = self.grade_head(out_grade)
        logits_diag = self.diagnosis_head(out_diag)

        return logits_grade, logits_diag

print("Ορισμός Μοντέλου CascadedTransformerClassifier (Deep Heads) ολοκληρώθηκε.")

In [ ]:
# %% Βήμα 6 (Διορθωμένο): Οπτικοποίηση Αποτελεσμάτων (Robust Plotting)
import pandas as pd
import matplotlib.pyplot as plt
import os

# --- Ρυθμίσεις Διαδρομής (Βεβαιώσου ότι είναι ο σωστός φάκελος) ---
# Χρησιμοποιούμε τον φάκελο που ορίσαμε στο Κελί 5
target_dir = os.path.join(runs_base_dir, "classification_cascaded_v4_transformer_weighted")
csv_filename = "training_history.csv" # Στο τελευταίο κελί 5 το ονομάσαμε έτσι
csv_path = os.path.join(target_dir, csv_filename)

print(f"Προσπάθεια φόρτωσης από: {csv_path}")

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    epochs = df['epoch']

    print(f"Επιτυχής φόρτωση. Βρέθηκαν {len(df)} εποχές.")
    print("Διαθέσιμες στήλες:", df.columns.tolist()) # Για να δούμε τι υπάρχει μέσα

    plt.style.use('seaborn-v0_8-darkgrid')

    # Δυναμικός καθορισμός του grid (αν λείπουν δεδομένα, φτιάχνουμε λιγότερα plots)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Training & Validation Metrics', fontsize=16)

    # --- PLOT 1: LOSSES ---
    if 'train_loss_total' in df.columns:
        axes[0].plot(epochs, df['train_loss_total'], label='Train Total', color='blue', linestyle='--')
    if 'val_loss_total' in df.columns:
        axes[0].plot(epochs, df['val_loss_total'], label='Val Total', color='blue')

    # Grade Loss (Αν υπάρχει)
    if 'train_loss_grade' in df.columns:
        axes[0].plot(epochs, df['train_loss_grade'], label='Train Grade', color='green', linestyle='--')
    if 'val_loss_grade' in df.columns:
        axes[0].plot(epochs, df['val_loss_grade'], label='Val Grade', color='green')

    # Diagnosis Loss (Αν υπάρχει)
    if 'train_loss_diag' in df.columns:
        axes[0].plot(epochs, df['train_loss_diag'], label='Train Diag', color='purple', linestyle='--')
    if 'val_loss_diag' in df.columns:
        axes[0].plot(epochs, df['val_loss_diag'], label='Val Diag', color='purple')

    axes[0].set_title('Losses')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()

    # --- PLOT 2: GRADE METRICS ---
    if 'val_acc_grade' in df.columns:
        axes[1].plot(epochs, df['val_acc_grade'], label='Acc Grade', color='green')
    if 'val_f1_grade' in df.columns:
        axes[1].plot(epochs, df['val_f1_grade'], label='F1 Grade', color='lightgreen', linestyle='--')
    if 'val_auc_grade' in df.columns:
        axes[1].plot(epochs, df['val_auc_grade'], label='AUC Grade', color='olive', linestyle=':')

    axes[1].set_title('Grade Performance')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylim(0, 1.0)
    axes[1].legend()

    # --- PLOT 3: DIAGNOSIS METRICS ---
    if 'val_acc_diag' in df.columns:
        axes[2].plot(epochs, df['val_acc_diag'], label='Acc Diag', color='purple')
    if 'val_f1_diag' in df.columns:
        axes[2].plot(epochs, df['val_f1_diag'], label='F1 Diag', color='violet', linestyle='--')
    if 'val_auc_diag' in df.columns:
        axes[2].plot(epochs, df['val_auc_diag'], label='AUC Diag', color='magenta', linestyle=':')

    axes[2].set_title('Diagnosis Performance')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylim(0, 1.0)
    axes[2].legend()

    plt.tight_layout()
    plt.show()

    if 'val_metric_combined' in df.columns:
        print(f"Καλύτερο Combined Metric: {df['val_metric_combined'].max():.4f}")

else:
    print(f"Δεν βρέθηκε το αρχείο: {csv_path}")
    print("Έλεγξε αν η εκπαίδευση ολοκλήρωσε τουλάχιστον μία εποχή.")

# ΜΤΛ

In [ ]:
# %% Βήμα 4: Ορισμός Μοντέλου MTLTransformerModel (Full Unfrozen + MLP Heads)
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.networks.nets import SwinUNETR
import os

print("Ορισμός Μοντέλου MTLTransformerModel (Unfrozen + Transformer + MLP Heads)...")

class MTLTransformerModel(nn.Module):
    def __init__(self,
                 in_channels_img, out_channels_seg, feature_size, use_checkpoint,
                 num_classes_grade, num_classes_diag, clinical_feature_size,
                 finetuned_seg_model_path=None,
                 d_model=128,        # Μέγεθος διανύσματος (Tokens)
                 n_heads=4,          # Κεφαλές Attention
                 num_layers=2,       # Βάθος Transformer
                 dropout_rate=0.4    # Dropout (λίγο αυξημένο για το Unfrozen)
                 ):

        super().__init__()

        # --- 1. SwinUNETR Backbone (UNFROZEN) ---
        self.swin_unetr = SwinUNETR(
            in_channels=in_channels_img,
            out_channels=out_channels_seg,
            feature_size=feature_size,
            use_checkpoint=use_checkpoint
        )

        if finetuned_seg_model_path and os.path.exists(finetuned_seg_model_path):
            print(f"Φόρτωση αρχικών βαρών από: {finetuned_seg_model_path}")
            checkpoint = torch.load(finetuned_seg_model_path, map_location=torch.device('cpu'))
            try:
                self.swin_unetr.load_state_dict(checkpoint['model_state'], strict=False)
                print("✅ Επιτυχής φόρτωση βαρών SwinUNETR.")
            except Exception as e: print(f"Warning loading weights: {e}")

        print("ℹ️ Ο Encoder/Decoder είναι UNFROZEN.")

        # --- 2. Hooks ---
        self.encoder_features_out = None
        self.decoder_features_out = None
        self.swin_unetr.swinViT.register_forward_hook(self.hook_encoder_features)
        self.swin_unetr.decoder1.register_forward_hook(self.hook_decoder_features)

        # --- 3. Projectors ---
        encoder_dim = feature_size * 16
        decoder_dim = feature_size

        self.encoder_proj = nn.Sequential(
            nn.LayerNorm(encoder_dim), nn.Linear(encoder_dim, d_model), nn.Dropout(dropout_rate)
        )
        self.decoder_proj = nn.Sequential(
            nn.LayerNorm(decoder_dim), nn.Linear(decoder_dim, d_model), nn.Dropout(dropout_rate)
        )
        self.clinical_proj = nn.Sequential(
            nn.LayerNorm(clinical_feature_size), nn.Linear(clinical_feature_size, d_model), nn.Dropout(dropout_rate)
        )

        # --- 4. Transformer ---
        self.grade_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.diag_token = nn.Parameter(torch.randn(1, 1, d_model))

        transformer_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model*2,
            dropout=dropout_rate, batch_first=True, norm_first=True
        )
        self.fusion_transformer = nn.TransformerEncoder(transformer_layer, num_layers=num_layers)

        # --- 5. Classification Heads (MLPs - ΒΕΛΤΙΩΣΗ) ---
        # Αντί για απλό Linear, βάζουμε MLP για μεγαλύτερη ισχύ απόφασης

        self.grade_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, num_classes_grade)
        )

        self.diagnosis_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, num_classes_diag)
        )
        print("✅ Heads architecture: MLP (d_model -> 64 -> ReLU -> Dropout -> Classes)")

    def hook_encoder_features(self, module, input, output): self.encoder_features_out = output[-1]
    def hook_decoder_features(self, module, input, output): self.decoder_features_out = output

    def forward(self, x_img, x_clinical):
        B = x_img.shape[0]

        # Task 1: Segmentation
        logits_seg = self.swin_unetr(x_img)

        # Task 2 & 3: Classification
        enc_feat = F.adaptive_avg_pool3d(self.encoder_features_out, 1).flatten(1)
        dec_feat = F.adaptive_avg_pool3d(self.decoder_features_out, 1).flatten(1)

        enc_token = self.encoder_proj(enc_feat).unsqueeze(1)
        dec_token = self.decoder_proj(dec_feat).unsqueeze(1)
        clin_token = self.clinical_proj(x_clinical).unsqueeze(1)

        grade_token = self.grade_token.expand(B, -1, -1)
        diag_token = self.diag_token.expand(B, -1, -1)

        tokens = torch.cat([grade_token, diag_token, enc_token, dec_token, clin_token], dim=1)
        fused_tokens = self.fusion_transformer(tokens)

        # Περνάμε τα tokens από τα MLP Heads
        logits_grade = self.grade_head(fused_tokens[:, 0, :])
        logits_diag = self.diagnosis_head(fused_tokens[:, 1, :])

        return logits_seg, logits_grade, logits_diag

In [ ]:
# %% Βήμα 5: Εκπαίδευση MTL (Unfrozen) - Full Metrics (Dice included)
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import pandas as pd
import numpy as np
import os
import gc
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
from monai.metrics import DiceMetric, ROCAUCMetric
from monai.transforms import AsDiscrete, Activations, Compose
from monai.losses import DiceCELoss
from monai.data import decollate_batch

# --- 1. Class Weights ---
print("Υπολογισμός Class Weights...")
try:
    y_grade_all = df_processed['Grade_Label'].values
    y_diag_all = df_processed['Diagnosis_Label'].values

    weights_grade = compute_class_weight(class_weight='balanced', classes=np.unique(y_grade_all), y=y_grade_all)
    weights_diag = compute_class_weight(class_weight='balanced', classes=np.unique(y_diag_all), y=y_diag_all)

    weights_grade_tensor = torch.tensor(weights_grade, dtype=torch.float).to(DEVICE)
    weights_diag_tensor = torch.tensor(weights_diag, dtype=torch.float).to(DEVICE)
    print(f"Weights Grade: {weights_grade_tensor}")
except:
    print("Default weights (1.0).")
    weights_grade_tensor = None; weights_diag_tensor = None

# --- 2. Loss Functions ---
loss_seg = DiceCELoss(include_background=False, sigmoid=True)
loss_grade = nn.CrossEntropyLoss(weight=weights_grade_tensor)
loss_diag = nn.CrossEntropyLoss(weight=weights_diag_tensor)

W_SEG = 1.0; W_GRADE = 0.5; W_DIAG = 0.5

def mtl_loss_fn(pred_seg, true_seg, pred_grade, true_grade, pred_diag, true_diag):
    l_s = loss_seg(pred_seg, true_seg)
    l_g = loss_grade(pred_grade, true_grade)
    l_d = loss_diag(pred_diag, true_diag)
    total = (W_SEG * l_s) + (W_GRADE * l_g) + (W_DIAG * l_d)
    return total, l_s, l_g, l_d

# --- 3. Setup ---
output_dir_mtl = os.path.join(runs_base_dir, "mtl_unfrozen_transformer_final")
history_csv_path = os.path.join(output_dir_mtl, "training_history_full.csv")
best_model_path = os.path.join(output_dir_mtl, "best_mtl_model.pt")
os.makedirs(output_dir_mtl, exist_ok=True)

# --- 4. Model ---
try:
    model_mtl = MTLTransformerModel(
        in_channels_img=IN_CHANNELS, out_channels_seg=OUT_CHANNELS_SEG,
        feature_size=FEATURE_SIZE, use_checkpoint=USE_CHECKPOINTING,
        num_classes_grade=NUM_CLASSES_GRADE, num_classes_diag=NUM_CLASSES_DIAG,
        clinical_feature_size=CLINICAL_FEATURE_SIZE,
        finetuned_seg_model_path=finetuned_model_path,
        d_model=128, n_heads=4, num_layers=2, dropout_rate=0.4
    ).to(DEVICE)
except NameError: print("ΣΦΑΛΜΑ: MTLTransformerModel not defined.")

# --- 5. Optimizer ---
LR_BACKBONE = 1e-5; LR_HEADS = 1e-3
params = [
    {'params': model_mtl.swin_unetr.parameters(), 'lr': LR_BACKBONE},
    {'params': model_mtl.encoder_proj.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.decoder_proj.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.clinical_proj.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.grade_token, 'lr': LR_HEADS},
    {'params': model_mtl.diag_token, 'lr': LR_HEADS},
    {'params': model_mtl.fusion_transformer.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.grade_head.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.diagnosis_head.parameters(), 'lr': LR_HEADS},
]
optimizer = torch.optim.AdamW(params, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS_MTL)

# --- 6. Metrics Helpers ---
# Δύο ξεχωριστά metrics για να μην μπερδεύονται
dice_metric_train = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)
dice_metric_val = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)

auc_metric_grade = ROCAUCMetric(); auc_metric_diag = ROCAUCMetric()
post_pred_seg = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
post_label_seg = Compose([AsDiscrete(to_onehot=None)])

# --- 7. Training Loop ---
best_val_metric = -1.0; start_epoch = 0; GRAD_ACCUM_STEPS = 4

history = {
    "epoch": [],
    "train_loss": [], "train_dice": [],
    "train_acc_g": [], "train_f1_g": [], "train_acc_d": [], "train_f1_d": [],
    "val_loss": [], "val_dice": [],
    "val_acc_g": [], "val_f1_g": [], "val_auc_g": [],
    "val_acc_d": [], "val_f1_d": [], "val_auc_d": [],
    "combined_score": []
}

if os.path.exists(best_model_path):
    try: os.remove(best_model_path)
    except: pass

print(f"\n--- Έναρξη MTL Training ({NUM_EPOCHS_MTL} Epochs) ---")

for epoch in range(start_epoch, NUM_EPOCHS_MTL):
    current_epoch = epoch + 1

    # === TRAIN ===
    model_mtl.train()
    dice_metric_train.reset()
    ep_loss = 0; step = 0
    tr_preds_g, tr_lbls_g = [], []; tr_preds_d, tr_lbls_d = [] , []

    optimizer.zero_grad()
    pbar = tqdm(train_loader_mtl, desc=f"Epoch {current_epoch} Train")

    for i, batch in enumerate(pbar):
        step += 1
        img = batch[image_key].to(DEVICE); seg = batch[label_key].to(DEVICE)
        clin = batch[clinical_key].to(DEVICE)
        lbl_g = batch[grade_key].to(DEVICE); lbl_d = batch[diag_key].to(DEVICE)

        p_seg, p_g, p_d = model_mtl(img, clin)
        loss, l_s, l_g, l_d = mtl_loss_fn(p_seg, seg, p_g, lbl_g, p_d, lbl_d)

        if not torch.isfinite(loss): continue
        (loss / GRAD_ACCUM_STEPS).backward()

        if (i + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model_mtl.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

        ep_loss += loss.item()

        # Train Metrics Accumulation
        # Dice
        val_outputs = [post_pred_seg(i) for i in decollate_batch(p_seg)]
        val_labels = [post_label_seg(i) for i in decollate_batch(seg)]
        dice_metric_train(y_pred=val_outputs, y=val_labels)

        # Class
        tr_preds_g.extend(torch.argmax(p_g, 1).cpu().numpy()); tr_lbls_g.extend(lbl_g.cpu().numpy())
        tr_preds_d.extend(torch.argmax(p_d, 1).cpu().numpy()); tr_lbls_d.extend(lbl_d.cpu().numpy())

        pbar.set_postfix(loss=f"{loss.item():.2f}")

    # Train Epoch Metrics
    tr_dice = dice_metric_train.aggregate().item()
    tr_acc_g = accuracy_score(tr_lbls_g, tr_preds_g); tr_f1_g = f1_score(tr_lbls_g, tr_preds_g, average='macro')
    tr_acc_d = accuracy_score(tr_lbls_d, tr_preds_d); tr_f1_d = f1_score(tr_lbls_d, tr_preds_d, average='macro')

    history["epoch"].append(current_epoch)
    history["train_loss"].append(ep_loss/step); history["train_dice"].append(tr_dice)
    history["train_acc_g"].append(tr_acc_g); history["train_f1_g"].append(tr_f1_g)
    history["train_acc_d"].append(tr_acc_d); history["train_f1_d"].append(tr_f1_d)

    print(f"Train | Loss: {ep_loss/step:.3f} | Dice: {tr_dice:.3f} | G_F1: {tr_f1_g:.3f} | D_F1: {tr_f1_d:.3f}")

    # === VALIDATION ===
    if current_epoch % VAL_INTERVAL == 0:
        model_mtl.eval()
        dice_metric_val.reset(); auc_metric_grade.reset(); auc_metric_diag.reset()
        preds_g, lbls_g = [], []; preds_d, lbls_d = [], []
        val_loss_sum = 0; val_step = 0

        with torch.no_grad():
            for batch in tqdm(val_loader_mtl, desc="Validating"):
                val_step += 1
                img = batch[image_key].to(DEVICE); seg = batch[label_key].to(DEVICE)
                clin = batch[clinical_key].to(DEVICE)
                lbl_g = batch[grade_key].to(DEVICE); lbl_d = batch[diag_key].to(DEVICE)

                p_seg, p_g, p_d = model_mtl(img, clin)

                vl, _, _, _ = mtl_loss_fn(p_seg, seg, p_g, lbl_g, p_d, lbl_d)
                val_loss_sum += vl.item()

                # Dice
                val_outputs = [post_pred_seg(i) for i in decollate_batch(p_seg)]
                val_labels = [post_label_seg(i) for i in decollate_batch(seg)]
                dice_metric_val(y_pred=val_outputs, y=val_labels)

                # Class
                preds_g.extend(torch.argmax(p_g, 1).cpu().numpy()); lbls_g.extend(lbl_g.cpu().numpy())
                preds_d.extend(torch.argmax(p_d, 1).cpu().numpy()); lbls_d.extend(lbl_d.cpu().numpy())

                # AUC
                auc_metric_grade(y_pred=F.softmax(p_g, 1), y=F.one_hot(lbl_g, NUM_CLASSES_GRADE).float())
                auc_metric_diag(y_pred=F.softmax(p_d, 1), y=F.one_hot(lbl_d, NUM_CLASSES_DIAG).float())

        # Val Metrics
        val_dice = dice_metric_val.aggregate().item()
        val_acc_g = accuracy_score(lbls_g, preds_g); val_f1_g = f1_score(lbls_g, preds_g, average='macro')
        val_acc_d = accuracy_score(lbls_d, preds_d); val_f1_d = f1_score(lbls_d, preds_d, average='macro')
        try: val_auc_g = auc_metric_grade.aggregate().item()
        except: val_auc_g = 0.5
        try: val_auc_d = auc_metric_diag.aggregate().item()
        except: val_auc_d = 0.5

        # Store
        history["val_loss"].append(val_loss_sum/val_step); history["val_dice"].append(val_dice)
        history["val_acc_g"].append(val_acc_g); history["val_f1_g"].append(val_f1_g); history["val_auc_g"].append(val_auc_g)
        history["val_acc_d"].append(val_acc_d); history["val_f1_d"].append(val_f1_d); history["val_auc_d"].append(val_auc_d)

        # Combined Score (Prioritize Grade & Dice)
        combined = (val_dice * 0.4) + (val_f1_g * 0.4) + (val_f1_d * 0.2)
        history["combined_score"].append(combined)

        print(f"--- Epoch {current_epoch} Validation ---")
        print(f"Combined: {combined:.4f} | Loss: {val_loss_sum/val_step:.4f} | Dice: {val_dice:.4f}")
        print(f"[GRADE] F1: {val_f1_g:.3f} | AUC: {val_auc_g:.3f}")
        print(confusion_matrix(lbls_g, preds_g, labels=list(range(NUM_CLASSES_GRADE))))
        print(f"[DIAGNOSIS] F1: {val_f1_d:.3f} | AUC: {val_auc_d:.3f}")
        print(confusion_matrix(lbls_d, preds_d, labels=list(range(NUM_CLASSES_DIAG))))

        if combined > best_val_metric:
            best_val_metric = combined
            # Αποθήκευση ΟΛΟΥ του μοντέλου (Unfrozen)
            torch.save(model_mtl.state_dict(), best_model_path)
            print("✅ New Best Model Saved!")

        pd.DataFrame(history).to_csv(history_csv_path, index=False)

    if scheduler: scheduler.step()

print(f"🏁 Training Finished. Best Combined: {best_val_metric:.4f}")

# --- 8. TEST PHASE ---
print("\n=== FINAL TEST (MTL Unfrozen) ===")
if os.path.exists(best_model_path):
    model_mtl.load_state_dict(torch.load(best_model_path))
    model_mtl.eval()

    t_pg, t_lg, t_pd, t_ld = [], [], [], []
    dice_metric_val.reset()

    with torch.no_grad():
        for batch in tqdm(test_loader_mtl, desc="Testing"):
            img = batch[image_key].to(DEVICE); seg = batch[label_key].to(DEVICE)
            clin = batch[clinical_key].to(DEVICE)
            lbl_g = batch[grade_key].to(DEVICE); lbl_d = batch[diag_key].to(DEVICE)

            p_seg, p_g, p_d = model_mtl(img, clin)

            # Seg
            val_outputs = [post_pred_seg(i) for i in decollate_batch(p_seg)]
            val_labels = [post_label_seg(i) for i in decollate_batch(seg)]
            dice_metric_val(y_pred=val_outputs, y=val_labels)

            # Class
            t_pg.extend(torch.argmax(p_g, 1).cpu().numpy()); t_lg.extend(lbl_g.cpu().numpy())
            t_pd.extend(torch.argmax(p_d, 1).cpu().numpy()); t_ld.extend(lbl_d.cpu().numpy())

    print(f"\n[TEST RESULTS]")
    print(f"Test Dice: {dice_metric_val.aggregate().item():.4f}")
    print("\nGrade Report:")
    print(classification_report(t_lg, t_pg, labels=list(range(NUM_CLASSES_GRADE)), target_names=[str(c) for c in GRADE_CLASSES], zero_division=0))
    print(confusion_matrix(t_lg, t_pg))
    print("\nDiagnosis Report:")
    print(classification_report(t_ld, t_pd, labels=list(range(NUM_CLASSES_DIAG)), target_names=[str(c) for c in DIAG_CLASSES], zero_division=0))
    print(confusion_matrix(t_ld, t_pd))
else: print("Model file not found.")

In [ ]:
torch.cuda.empty_cache()

In [ ]:
del(model_mtl)

In [ ]:
# %% Βήμα 6: Οπτικοποίηση Αποτελεσμάτων (MTL Unfrozen Plots)
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np

# --- Ρυθμίσεις ---
# Βεβαιώσου ότι το path είναι ίδιο με το output_dir_mtl του Κελιού 5
target_dir = os.path.join(runs_base_dir, "mtl_unfrozen_transformer_final")
csv_path = os.path.join(target_dir, "training_history_full.csv")

print(f"Ανάγνωση αρχείου: {csv_path}")

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)

    # Φιλτράρισμα: Οι γραμμές validation έχουν τιμές, οι άλλες NaN.
    # Τις κρατάμε όλες για το Train, αλλά για το Val φτιάχνουμε subset χωρίς NaN για να μην σπάνε οι γραμμές.
    df_val = df.dropna(subset=['val_loss'])

    epochs = df['epoch']
    val_epochs = df_val['epoch']

    # Στυλ γραφημάτων
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(2, 2, figsize=(20, 12))
    fig.suptitle('MTL Unfrozen Training Analysis', fontsize=20, weight='bold')

    # ---------------------------------------------------------
    # PLOT 1: LOSSES & COMBINED SCORE
    # ---------------------------------------------------------
    ax1 = axes[0, 0]
    ax1.plot(epochs, df['train_loss'], label='Train Loss', color='blue', alpha=0.6)
    ax1.plot(val_epochs, df_val['val_loss'], label='Val Loss', color='navy', linewidth=2, marker='o', markersize=4)

    # Secondary axis για το Combined Score
    ax1b = ax1.twinx()
    ax1b.plot(val_epochs, df_val['combined_score'], label='Combined Score (Optimization Target)', color='gold', linestyle='--', linewidth=2)

    ax1.set_title('Overall Loss & Optimization Score', fontsize=14)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1b.set_ylabel('Combined Score')

    lines, labels = ax1.get_legend_handles_labels()
    lines2, labels2 = ax1b.get_legend_handles_labels()
    ax1.legend(lines + lines2, labels + labels2, loc='upper left')
    ax1.grid(True, linestyle='--', alpha=0.7)

    # ---------------------------------------------------------
    # PLOT 2: SEGMENTATION (DICE)
    # ---------------------------------------------------------
    ax2 = axes[0, 1]
    ax2.plot(epochs, df['train_dice'], label='Train Dice', color='gray', linestyle=':', alpha=0.8)
    ax2.plot(val_epochs, df_val['val_dice'], label='Val Dice', color='black', linewidth=2, marker='s', markersize=4)

    ax2.set_title('Task 1: Segmentation Performance (Dice)', fontsize=14)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Dice Score')
    ax2.set_ylim(0, 1.0)
    ax2.legend(loc='lower right')
    ax2.grid(True)

    # ---------------------------------------------------------
    # PLOT 3: GRADE CLASSIFICATION
    # ---------------------------------------------------------
    ax3 = axes[1, 0]
    # Train
    ax3.plot(epochs, df['train_f1_g'], label='Train F1', color='lightgreen', linestyle='--')

    # Val
    ax3.plot(val_epochs, df_val['val_acc_g'], label='Val Accuracy', color='darkgreen', alpha=0.5)
    ax3.plot(val_epochs, df_val['val_f1_g'], label='Val F1 (Macro)', color='forestgreen', linewidth=2.5)
    ax3.plot(val_epochs, df_val['val_auc_g'], label='Val AUC', color='lime', linestyle=':', linewidth=2)

    ax3.set_title('Task 2: Tumor Grade Classification', fontsize=14)
    ax3.set_xlabel('Epoch')
    ax3.set_ylim(0, 1.0)
    ax3.legend(loc='lower right')
    ax3.grid(True)

    # ---------------------------------------------------------
    # PLOT 4: DIAGNOSIS CLASSIFICATION
    # ---------------------------------------------------------
    ax4 = axes[1, 1]
    # Train
    ax4.plot(epochs, df['train_f1_d'], label='Train F1', color='plum', linestyle='--')

    # Val
    ax4.plot(val_epochs, df_val['val_acc_d'], label='Val Accuracy', color='indigo', alpha=0.5)
    ax4.plot(val_epochs, df_val['val_f1_d'], label='Val F1 (Macro)', color='darkmagenta', linewidth=2.5)
    ax4.plot(val_epochs, df_val['val_auc_d'], label='Val AUC', color='magenta', linestyle=':', linewidth=2)

    ax4.set_title('Task 3: Diagnosis Classification', fontsize=14)
    ax4.set_xlabel('Epoch')
    ax4.set_ylim(0, 1.0)
    ax4.legend(loc='lower right')
    ax4.grid(True)

    plt.tight_layout()
    plt.show()

    # --- Print Stats ---
    best_epoch_idx = df_val['combined_score'].idxmax()
    best_row = df.loc[best_epoch_idx]
    print("\n" + "="*40)
    print(f"🏆 BEST EPOCH: {int(best_row['epoch'])}")
    print("="*40)
    print(f"Combined Score: {best_row['combined_score']:.4f}")
    print(f"Seg Dice:       {best_row['val_dice']:.4f}")
    print(f"Grade F1:       {best_row['val_f1_g']:.4f} (Acc: {best_row['val_acc_g']:.4f})")
    print(f"Diag F1:        {best_row['val_f1_d']:.4f} (Acc: {best_row['val_acc_d']:.4f})")
    print(f"Loss Total:     {best_row['val_loss']:.4f}")
    print("="*40)

else:
    print("❌ Δεν βρέθηκε το CSV. Βεβαιώσου ότι το Κελί 5 έτρεξε τουλάχιστον για 1 εποχή.")

Sequential MTL

In [ ]:
# %% Βήμα 4: Ορισμός Μοντέλου Sequential MTL (Unfrozen)
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.networks.nets import SwinUNETR
import os

print("Ορισμός Μοντέλου SequentialMTLModel (Hierarchical: Diagnosis -> Grade)...")

class SequentialMTLModel(nn.Module):
    def __init__(self,
                 in_channels_img, out_channels_seg, feature_size, use_checkpoint,
                 num_classes_grade, num_classes_diag, clinical_feature_size,
                 finetuned_seg_model_path=None,
                 d_model=128, n_heads=4, num_layers=2, dropout_rate=0.4
                 ):

        super().__init__()

        # --- 1. SwinUNETR Backbone (UNFROZEN) ---
        self.swin_unetr = SwinUNETR(
            in_channels=in_channels_img,
            out_channels=out_channels_seg,
            feature_size=feature_size,
            use_checkpoint=use_checkpoint
        )

        if finetuned_seg_model_path and os.path.exists(finetuned_seg_model_path):
            print(f"Φόρτωση βαρών από: {finetuned_seg_model_path}")
            checkpoint = torch.load(finetuned_seg_model_path, map_location=torch.device('cpu'))
            try: self.swin_unetr.load_state_dict(checkpoint['model_state'], strict=False)
            except: pass

        print("ℹ️ Encoder/Decoder: UNFROZEN")

        # Hooks
        self.encoder_features_out = None
        self.decoder_features_out = None
        self.swin_unetr.swinViT.register_forward_hook(self.hook_encoder_features)
        self.swin_unetr.decoder1.register_forward_hook(self.hook_decoder_features)

        # --- 2. Transformer Fusion Components ---
        encoder_dim = feature_size * 16
        decoder_dim = feature_size

        self.encoder_proj = nn.Sequential(nn.LayerNorm(encoder_dim), nn.Linear(encoder_dim, d_model), nn.Dropout(dropout_rate))
        self.decoder_proj = nn.Sequential(nn.LayerNorm(decoder_dim), nn.Linear(decoder_dim, d_model), nn.Dropout(dropout_rate))
        self.clinical_proj = nn.Sequential(nn.LayerNorm(clinical_feature_size), nn.Linear(clinical_feature_size, d_model), nn.Dropout(dropout_rate))

        # Ένα κοινό Token που θα μαζέψει την πληροφορία
        self.fusion_token = nn.Parameter(torch.randn(1, 1, d_model))

        transformer_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=d_model*2, dropout=dropout_rate, batch_first=True, norm_first=True)
        self.fusion_transformer = nn.TransformerEncoder(transformer_layer, num_layers=num_layers)

        # --- 3. Sequential Heads (Η ΚΑΙΝΟΤΟΜΙΑ) ---

        # A. Diagnosis Head (Πρώτη)
        # Παίρνει το Fused Vector (d_model) -> Βγάζει Diagnosis
        self.diag_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, num_classes_diag)
        )

        # B. Grade Head (Δεύτερη - Εξαρτώμενη)
        # Είσοδος: Fused Vector (d_model) + Diagnosis Probabilities (num_classes_diag)
        grade_input_dim = d_model + num_classes_diag

        self.grade_head = nn.Sequential(
            nn.LayerNorm(grade_input_dim),
            nn.Linear(grade_input_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, num_classes_grade)
        )

        print(f"Sequential Architecture: Fusion -> Diag -> (Fusion+DiagProbs) -> Grade")

    def hook_encoder_features(self, module, input, output): self.encoder_features_out = output[-1]
    def hook_decoder_features(self, module, input, output): self.decoder_features_out = output

    def forward(self, x_img, x_clinical):
        B = x_img.shape[0]

        # 1. Segmentation
        logits_seg = self.swin_unetr(x_img)

        # 2. Features -> Tokens
        enc_feat = F.adaptive_avg_pool3d(self.encoder_features_out, 1).flatten(1)
        dec_feat = F.adaptive_avg_pool3d(self.decoder_features_out, 1).flatten(1)

        enc_token = self.encoder_proj(enc_feat).unsqueeze(1)
        dec_token = self.decoder_proj(dec_feat).unsqueeze(1)
        clin_token = self.clinical_proj(x_clinical).unsqueeze(1)
        fusion_token = self.fusion_token.expand(B, -1, -1)

        # 3. Transformer Fusion
        tokens = torch.cat([fusion_token, enc_token, dec_token, clin_token], dim=1)
        fused_sequence = self.fusion_transformer(tokens)
        fused_vec = fused_sequence[:, 0, :] # [B, d_model]

        # 4. ΒΗΜΑ 1: Πρόβλεψη Διάγνωσης
        logits_diag = self.diag_head(fused_vec)

        # 5. ΒΗΜΑ 2: Πρόβλεψη Βαθμού (Χρησιμοποιώντας τη γνώση της Διάγνωσης)
        # Παίρνουμε τις πιθανότητες (Softmax) ως "hint"
        diag_probs = F.softmax(logits_diag, dim=1)

        # Συνένωση: [Features, Diagnosis_Hint]
        grade_input = torch.cat([fused_vec, diag_probs], dim=1)
        logits_grade = self.grade_head(grade_input)

        return logits_seg, logits_grade, logits_diag

In [ ]:
# %% Βήμα 5: Εκπαίδευση Sequential MTL (Full Pipeline)
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import pandas as pd
import numpy as np
import os
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight
from monai.metrics import DiceMetric, ROCAUCMetric
from monai.transforms import AsDiscrete, Activations, Compose
from monai.losses import DiceCELoss
from monai.data import decollate_batch

# --- 1. Class Weights ---
print("Υπολογισμός Class Weights...")
try:
    y_grade_all = df_processed['Grade_Label'].values
    y_diag_all = df_processed['Diagnosis_Label'].values
    weights_grade = compute_class_weight('balanced', classes=np.unique(y_grade_all), y=y_grade_all)
    weights_diag = compute_class_weight('balanced', classes=np.unique(y_diag_all), y=y_diag_all)
    weights_grade_tensor = torch.tensor(weights_grade, dtype=torch.float).to(DEVICE)
    weights_diag_tensor = torch.tensor(weights_diag, dtype=torch.float).to(DEVICE)
    print(f"Weights Grade: {weights_grade_tensor}\nWeights Diag: {weights_diag_tensor}")
except:
    weights_grade_tensor = None; weights_diag_tensor = None

# --- 2. Loss Functions & Task Weights ---
loss_seg = DiceCELoss(include_background=False, sigmoid=True)
loss_grade = nn.CrossEntropyLoss(weight=weights_grade_tensor)
loss_diag = nn.CrossEntropyLoss(weight=weights_diag_tensor)

# Sequential Logic: Η Διάγνωση είναι κρίσιμη για το Grade, άρα ανεβάζουμε το βάρος της
W_SEG = 1.0
W_DIAG = 0.8  # Αυξημένο (ήταν 0.5)
W_GRADE = 0.5

def mtl_loss_fn(pred_seg, true_seg, pred_grade, true_grade, pred_diag, true_diag):
    l_s = loss_seg(pred_seg, true_seg)
    l_d = loss_diag(pred_diag, true_diag) # Diagnosis first
    l_g = loss_grade(pred_grade, true_grade)
    total = (W_SEG * l_s) + (W_DIAG * l_d) + (W_GRADE * l_g)
    return total, l_s, l_g, l_d

# --- 3. Ρυθμίσεις ---
output_dir_mtl = os.path.join(runs_base_dir, "mtl_sequential_final")
history_csv_path = os.path.join(output_dir_mtl, "history_seq.csv")
best_model_path = os.path.join(output_dir_mtl, "best_seq_model.pt")
os.makedirs(output_dir_mtl, exist_ok=True)

# --- 4. Model ---
try:
    model_mtl = SequentialMTLModel( # <-- Η νέα κλάση
        in_channels_img=IN_CHANNELS, out_channels_seg=OUT_CHANNELS_SEG,
        feature_size=FEATURE_SIZE, use_checkpoint=USE_CHECKPOINTING,
        num_classes_grade=NUM_CLASSES_GRADE, num_classes_diag=NUM_CLASSES_DIAG,
        clinical_feature_size=CLINICAL_FEATURE_SIZE,
        finetuned_seg_model_path=finetuned_model_path,
        d_model=128, n_heads=4, num_layers=2, dropout_rate=0.4
    ).to(DEVICE)
except NameError: print("ΣΦΑΛΜΑ: SequentialMTLModel not defined.")

# --- 5. Optimizer ---
LR_BACKBONE = 1e-5; LR_HEADS = 1e-3
params = [
    {'params': model_mtl.swin_unetr.parameters(), 'lr': LR_BACKBONE},
    {'params': model_mtl.encoder_proj.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.decoder_proj.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.clinical_proj.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.fusion_token, 'lr': LR_HEADS},
    {'params': model_mtl.fusion_transformer.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.diag_head.parameters(), 'lr': LR_HEADS},  # <-- Sequential Heads
    {'params': model_mtl.grade_head.parameters(), 'lr': LR_HEADS}, # <-- Sequential Heads
]
optimizer = torch.optim.AdamW(params, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS_MTL)

# --- 6. Metrics & Loop Vars ---
dice_metric = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)
post_pred_seg = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
post_label_seg = Compose([AsDiscrete(to_onehot=None)])
best_val_metric = -1.0; start_epoch = 0; GRAD_ACCUM_STEPS = 4

history = {
    "epoch": [], "train_loss": [],
    "val_loss": [], "val_dice": [],
    "val_acc_g": [], "val_f1_g": [],
    "val_acc_d": [], "val_f1_d": [],
    "combined_score": []
}

if os.path.exists(best_model_path): os.remove(best_model_path)

print(f"\n--- Έναρξη Sequential MTL ({NUM_EPOCHS_MTL} Epochs) ---")

for epoch in range(start_epoch, NUM_EPOCHS_MTL):
    current_epoch = epoch + 1

    # === TRAIN ===
    model_mtl.train()
    ep_loss = 0; step = 0
    optimizer.zero_grad()

    pbar = tqdm(train_loader_mtl, desc=f"Epoch {current_epoch} Train")
    for i, batch in enumerate(pbar):
        step += 1
        img = batch[image_key].to(DEVICE); seg = batch[label_key].to(DEVICE)
        clin = batch[clinical_key].to(DEVICE)
        lbl_g = batch[grade_key].to(DEVICE); lbl_d = batch[diag_key].to(DEVICE)

        p_seg, p_g, p_d = model_mtl(img, clin)
        loss, l_s, l_g, l_d = mtl_loss_fn(p_seg, seg, p_g, lbl_g, p_d, lbl_d)

        if not torch.isfinite(loss): continue
        (loss / GRAD_ACCUM_STEPS).backward()

        if (i + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model_mtl.parameters(), 1.0)
            optimizer.step(); optimizer.zero_grad()

        ep_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.2f}")

    avg_train_loss = ep_loss / step
    history["epoch"].append(current_epoch); history["train_loss"].append(avg_train_loss)
    print(f"Train Loss: {avg_train_loss:.4f}")

    # === VALIDATION ===
    if current_epoch % VAL_INTERVAL == 0:
        model_mtl.eval()
        dice_metric.reset()
        preds_g, lbls_g = [], []; preds_d, lbls_d = [], []
        val_loss_sum = 0; val_steps = 0

        with torch.no_grad():
            for batch in tqdm(val_loader_mtl, desc="Validating"):
                val_steps += 1
                img = batch[image_key].to(DEVICE); seg = batch[label_key].to(DEVICE)
                clin = batch[clinical_key].to(DEVICE)
                lbl_g = batch[grade_key].to(DEVICE); lbl_d = batch[diag_key].to(DEVICE)

                p_seg, p_g, p_d = model_mtl(img, clin)

                vl, _, _, _ = mtl_loss_fn(p_seg, seg, p_g, lbl_g, p_d, lbl_d)
                val_loss_sum += vl.item()

                # Metrics
                val_outputs = [post_pred_seg(i) for i in decollate_batch(p_seg)]
                val_labels = [post_label_seg(i) for i in decollate_batch(seg)]
                dice_metric(y_pred=val_outputs, y=val_labels)

                preds_g.extend(torch.argmax(p_g, 1).cpu().numpy()); lbls_g.extend(lbl_g.cpu().numpy())
                preds_d.extend(torch.argmax(p_d, 1).cpu().numpy()); lbls_d.extend(lbl_d.cpu().numpy())

        # Calc Stats
        val_dice = dice_metric.aggregate().item()
        val_f1_g = f1_score(lbls_g, preds_g, average='macro')
        val_f1_d = f1_score(lbls_d, preds_d, average='macro')
        val_acc_g = accuracy_score(lbls_g, preds_g)
        val_acc_d = accuracy_score(lbls_d, preds_d)

        combined = (val_dice * 0.4) + (val_f1_g * 0.3) + (val_f1_d * 0.3)

        # Update History
        history["val_loss"].append(val_loss_sum/val_steps); history["val_dice"].append(val_dice)
        history["val_acc_g"].append(val_acc_g); history["val_f1_g"].append(val_f1_g)
        history["val_acc_d"].append(val_acc_d); history["val_f1_d"].append(val_f1_d)
        history["combined_score"].append(combined)

        print(f"--- Epoch {current_epoch} Val ---")
        print(f"Combined: {combined:.4f} | Loss: {val_loss_sum/val_steps:.4f} | Dice: {val_dice:.4f}")
        print(f"[GRADE] F1: {val_f1_g:.3f} (Acc: {val_acc_g:.3f})")
        print(confusion_matrix(lbls_g, preds_g))
        print(f"[DIAGNOSIS] F1: {val_f1_d:.3f} (Acc: {val_acc_d:.3f})")
        print(confusion_matrix(lbls_d, preds_d))

        if combined > best_val_metric:
            best_val_metric = combined
            torch.save(model_mtl.state_dict(), best_model_path)
            print("✅ New Best Model Saved!")

        try: pd.DataFrame(history).to_csv(history_csv_path, index=False)
        except: pass

    if scheduler: scheduler.step()

print(f"🏁 Finished. Best Combined: {best_val_metric:.4f}")

# --- 7. FINAL TEST ---
print("\n=== FINAL TEST (Sequential MTL) ===")
if os.path.exists(best_model_path):
    model_mtl.load_state_dict(torch.load(best_model_path))
    model_mtl.eval()
    t_pg, t_lg, t_pd, t_ld = [], [], [], []
    dice_metric.reset()

    with torch.no_grad():
        for batch in tqdm(test_loader_mtl, desc="Testing"):
            img = batch[image_key].to(DEVICE); seg = batch[label_key].to(DEVICE)
            clin = batch[clinical_key].to(DEVICE)
            lbl_g = batch[grade_key].to(DEVICE); lbl_d = batch[diag_key].to(DEVICE)

            p_seg, p_g, p_d = model_mtl(img, clin)

            val_outputs = [post_pred_seg(i) for i in decollate_batch(p_seg)]
            val_labels = [post_label_seg(i) for i in decollate_batch(seg)]
            dice_metric(y_pred=val_outputs, y=val_labels)

            t_pg.extend(torch.argmax(p_g, 1).cpu().numpy()); t_lg.extend(lbl_g.cpu().numpy())
            t_pd.extend(torch.argmax(p_d, 1).cpu().numpy()); t_ld.extend(lbl_d.cpu().numpy())

    print(f"\nTest Dice: {dice_metric.aggregate().item():.4f}")
    print("\n[GRADE TEST]\n", classification_report(t_lg, t_pg, labels=list(range(NUM_CLASSES_GRADE)), target_names=[str(c) for c in GRADE_CLASSES], zero_division=0))
    print(confusion_matrix(t_lg, t_pg))
    print("\n[DIAGNOSIS TEST]\n", classification_report(t_ld, t_pd, labels=list(range(NUM_CLASSES_DIAG)), target_names=[str(c) for c in DIAG_CLASSES], zero_division=0))
    print(confusion_matrix(t_ld, t_pd))

In [ ]:
# %% Βήμα 6: Οπτικοποίηση Αποτελεσμάτων (Sequential MTL Analysis)
import pandas as pd
import matplotlib.pyplot as plt
import os

# --- Ρυθμίσεις Διαδρομής ---
# ΠΡΟΣΟΧΗ: Βεβαιώσου ότι το όνομα του φακέλου είναι ίδιο με το output_dir_mtl στο Κελί 5
target_dir = os.path.join(runs_base_dir, "mtl_sequential_final")
csv_path = os.path.join(target_dir, "history_seq.csv")

print(f"Ανάγνωση αρχείου metrics από: {csv_path}")

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    epochs = df['epoch']

    # Ρύθμιση στυλ
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    fig.suptitle('Sequential MTL Training Analysis (Diagnosis -> Grade)', fontsize=18, weight='bold')

    # ---------------------------------------------------------
    # 1. OPTIMIZATION: LOSSES & COMBINED SCORE
    # ---------------------------------------------------------
    ax1 = axes[0, 0]
    ax1.plot(epochs, df['train_loss'], label='Train Loss (Total)', color='blue', alpha=0.6)
    ax1.plot(epochs, df['val_loss'], label='Val Loss (Total)', color='navy', linewidth=2)

    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Optimization Landscape', fontsize=14)
    ax1.legend(loc='upper left')

    # Secondary Axis για το Combined Score
    ax1b = ax1.twinx()
    ax1b.plot(epochs, df['combined_score'], label='Combined Optimization Score', color='gold', linestyle='--', linewidth=2)
    ax1b.set_ylabel('Combined Score (Higher is Better)')
    ax1b.legend(loc='upper right')

    # ---------------------------------------------------------
    # 2. TASK 1: SEGMENTATION (DICE)
    # ---------------------------------------------------------
    ax2 = axes[0, 1]
    ax2.plot(epochs, df['val_dice'], label='Validation Dice', color='black', linewidth=2, marker='s', markersize=4)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Dice Score')
    ax2.set_title('Task 1: Segmentation Performance', fontsize=14)
    ax2.set_ylim(0, 1.0)
    ax2.grid(True, linestyle='--', alpha=0.7)
    ax2.legend()

    # ---------------------------------------------------------
    # 3. TASK 2: DIAGNOSIS (The "Parent" Task)
    # ---------------------------------------------------------
    ax3 = axes[1, 0]
    ax3.plot(epochs, df['val_acc_d'], label='Accuracy', color='indigo', alpha=0.6)
    ax3.plot(epochs, df['val_f1_d'], label='F1-Macro', color='darkmagenta', linewidth=2.5)

    ax3.set_xlabel('Epoch')
    ax3.set_title('Task 2: Diagnosis (Prioritized)', fontsize=14)
    ax3.set_ylim(0, 1.0)
    ax3.legend()
    ax3.grid(True, linestyle='--', alpha=0.7)

    # ---------------------------------------------------------
    # 4. TASK 3: GRADE (The "Child" Task)
    # ---------------------------------------------------------
    ax4 = axes[1, 1]
    ax4.plot(epochs, df['val_acc_g'], label='Accuracy', color='darkgreen', alpha=0.6)
    ax4.plot(epochs, df['val_f1_g'], label='F1-Macro', color='forestgreen', linewidth=2.5)

    ax4.set_xlabel('Epoch')
    ax4.set_title('Task 3: Tumor Grade (Dependent)', fontsize=14)
    ax4.set_ylim(0, 1.0)
    ax4.legend()
    ax4.grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.show()

    # --- Best Model Summary ---
    # Βρίσκουμε την εποχή με το καλύτερο combined score
    best_idx = df['combined_score'].idxmax()
    best_row = df.iloc[best_idx]

    print("\n" + "="*40)
    print(f"🏆 BEST MODEL FOUND AT EPOCH: {int(best_row['epoch'])}")
    print("="*40)
    print(f"Combined Score: {best_row['combined_score']:.4f}")
    print(f"-"*20)
    print(f"Segmentation Dice: {best_row['val_dice']:.4f}")
    print(f"Diagnosis F1:      {best_row['val_f1_d']:.4f} (Acc: {best_row['val_acc_d']:.4f})")
    print(f"Grade F1:          {best_row['val_f1_g']:.4f} (Acc: {best_row['val_acc_g']:.4f})")
    print("="*40)

else:
    print(f"❌ Το αρχείο {csv_path} δεν βρέθηκε.")
    print("Βεβαιώσου ότι ολοκληρώθηκε τουλάχιστον 1 εποχή εκπαίδευσης στο Κελί 5.")

## Mask Guided

In [ ]:
# %% Βήμα 4: Ορισμός Μοντέλου Mask-Guided MTL (Hard Attention)
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.networks.nets import SwinUNETR
import os

print("Ορισμός Μοντέλου MaskGuidedMTLModel (Attention on Tumor Area)...")

class MaskGuidedMTLModel(nn.Module):
    def __init__(self,
                 in_channels_img, out_channels_seg, feature_size, use_checkpoint,
                 num_classes_grade, num_classes_diag, clinical_feature_size,
                 finetuned_seg_model_path=None,
                 d_model=128, n_heads=4, num_layers=2, dropout_rate=0.4
                 ):

        super().__init__()

        # --- 1. SwinUNETR Backbone (Unfrozen) ---
        self.swin_unetr = SwinUNETR(
            in_channels=in_channels_img,
            out_channels=out_channels_seg,
            feature_size=feature_size,
            use_checkpoint=use_checkpoint
        )

        if finetuned_seg_model_path and os.path.exists(finetuned_seg_model_path):
            print(f"Φόρτωση βαρών από: {finetuned_seg_model_path}")
            checkpoint = torch.load(finetuned_seg_model_path, map_location=torch.device('cpu'))
            try: self.swin_unetr.load_state_dict(checkpoint['model_state'], strict=False)
            except: pass

        # Hooks για να πάρουμε τα features πριν το Pooling
        self.encoder_features_out = None
        self.swin_unetr.swinViT.register_forward_hook(self.hook_encoder_features)

        # --- 2. Transformer & Projectors ---
        encoder_dim = feature_size * 16

        self.encoder_proj = nn.Sequential(nn.LayerNorm(encoder_dim), nn.Linear(encoder_dim, d_model), nn.Dropout(dropout_rate))
        self.clinical_proj = nn.Sequential(nn.LayerNorm(clinical_feature_size), nn.Linear(clinical_feature_size, d_model), nn.Dropout(dropout_rate))

        self.grade_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.diag_token = nn.Parameter(torch.randn(1, 1, d_model))

        transformer_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=d_model*2, dropout=dropout_rate, batch_first=True, norm_first=True)
        self.fusion_transformer = nn.TransformerEncoder(transformer_layer, num_layers=num_layers)

        # --- 3. MLP Heads ---
        self.grade_head = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, 64), nn.ReLU(), nn.Dropout(dropout_rate), nn.Linear(64, num_classes_grade)
        )
        self.diagnosis_head = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, 64), nn.ReLU(), nn.Dropout(dropout_rate), nn.Linear(64, num_classes_diag)
        )

        print("✅ Mask-Guided Architecture: Features * Segmentation_Mask")

    def hook_encoder_features(self, module, input, output):
        # Το output του swinViT είναι συνήθως tuple, παίρνουμε το τελευταίο feature map
        self.encoder_features_out = output[-1]

    def forward(self, x_img, x_clinical):
        B = x_img.shape[0]

        # 1. Segmentation Output & Features Extraction
        logits_seg = self.swin_unetr(x_img) # [B, 3, H, W, D]

        # 2. Δημιουργία Μάσκας Προσοχής (Soft Attention)
        # Χρησιμοποιούμε Sigmoid για να είναι διαφορίσιμο (differentiable)
        # Παίρνουμε το max από τα 3 κανάλια (αν υπάρχει όγκος σε οποιοδήποτε κανάλι)
        seg_prob = torch.sigmoid(logits_seg)
        attention_mask = torch.max(seg_prob, dim=1, keepdim=True)[0] # [B, 1, H, W, D]

        # 3. Feature Processing with HARD ATTENTION
        raw_feats = self.encoder_features_out # [B, 768, H', W', D'] (bottleneck size)

        # Downsample της μάσκας στο μέγεθος των features
        mask_resized = F.interpolate(attention_mask, size=raw_feats.shape[2:], mode='nearest')

        # *** ΠΟΛΛΑΠΛΑΣΙΑΣΜΟΣ (MASKING) ***
        # Εδώ μηδενίζουμε το background. Προσθέτουμε ένα μικρό epsilon για σταθερότητα; Όχι, το 0 είναι οκ.
        masked_feats = raw_feats * mask_resized

        # 4. Pooling & Projection
        # Τώρα το Average Pooling θα αθροίσει μόνο τα tumor pixels (τα υπόλοιπα είναι 0)
        enc_feat_pooled = F.adaptive_avg_pool3d(masked_feats, 1).flatten(1)

        # Αν η μάσκα είναι όλη μηδέν (δεν βρήκε όγκο), το feature θα είναι 0.
        # Αυτό είναι σωστό: αν δεν υπάρχει όγκος, δεν υπάρχει Grade.

        enc_token = self.encoder_proj(enc_feat_pooled).unsqueeze(1)
        clin_token = self.clinical_proj(x_clinical).unsqueeze(1)

        grade_token = self.grade_token.expand(B, -1, -1)
        diag_token = self.diag_token.expand(B, -1, -1)

        # 5. Fusion
        # Σημείωση: Δεν χρησιμοποιώ Decoder features εδώ για απλότητα και focus στο tumor core
        tokens = torch.cat([grade_token, diag_token, enc_token, clin_token], dim=1)
        fused_tokens = self.fusion_transformer(tokens)

        logits_grade = self.grade_head(fused_tokens[:, 0, :])
        logits_diag = self.diagnosis_head(fused_tokens[:, 1, :])

        return logits_seg, logits_grade, logits_diag

In [ ]:
# %% Βήμα 5: Εκπαίδευση Mask-Guided MTL (Full Metrics History)
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import pandas as pd
import numpy as np
import os
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
from monai.metrics import DiceMetric, ROCAUCMetric
from monai.transforms import AsDiscrete, Activations, Compose
from monai.losses import DiceCELoss
from monai.data import decollate_batch

# --- 1. Class Weights ---
print("Υπολογισμός Class Weights...")
try:
    y_grade_all = df_processed['Grade_Label'].values
    y_diag_all = df_processed['Diagnosis_Label'].values
    weights_grade = compute_class_weight('balanced', classes=np.unique(y_grade_all), y=y_grade_all)
    weights_diag = compute_class_weight('balanced', classes=np.unique(y_diag_all), y=y_diag_all)
    weights_grade_tensor = torch.tensor(weights_grade, dtype=torch.float).to(DEVICE)
    weights_diag_tensor = torch.tensor(weights_diag, dtype=torch.float).to(DEVICE)
except:
    weights_grade_tensor = None; weights_diag_tensor = None

# --- 2. Loss Functions ---
loss_seg = DiceCELoss(include_background=False, sigmoid=True)
loss_grade = nn.CrossEntropyLoss(weight=weights_grade_tensor)
loss_diag = nn.CrossEntropyLoss(weight=weights_diag_tensor)

W_SEG = 1.0; W_GRADE = 0.5; W_DIAG = 0.5

def mtl_loss_fn(pred_seg, true_seg, pred_grade, true_grade, pred_diag, true_diag):
    l_s = loss_seg(pred_seg, true_seg)
    l_g = loss_grade(pred_grade, true_grade)
    l_d = loss_diag(pred_diag, true_diag)
    total = (W_SEG * l_s) + (W_GRADE * l_g) + (W_DIAG * l_d)
    return total, l_s, l_g, l_d

# --- 3. Ρυθμίσεις ---
output_dir_mtl = os.path.join(runs_base_dir, "mtl_mask_guided_final")
history_csv_path = os.path.join(output_dir_mtl, "history_mask_full.csv") # Νέο όνομα CSV
best_model_path = os.path.join(output_dir_mtl, "best_mask_model.pt")
os.makedirs(output_dir_mtl, exist_ok=True)

# --- 4. Model (MaskGuided) ---
try:
    model_mtl = MaskGuidedMTLModel(
        in_channels_img=IN_CHANNELS, out_channels_seg=OUT_CHANNELS_SEG,
        feature_size=FEATURE_SIZE, use_checkpoint=USE_CHECKPOINTING,
        num_classes_grade=NUM_CLASSES_GRADE, num_classes_diag=NUM_CLASSES_DIAG,
        clinical_feature_size=CLINICAL_FEATURE_SIZE,
        finetuned_seg_model_path=finetuned_model_path,
        d_model=128, n_heads=4, num_layers=2, dropout_rate=0.4
    ).to(DEVICE)
except NameError: print("ΣΦΑΛΜΑ: MaskGuidedMTLModel not defined.")

# --- 5. Optimizer ---
LR_BACKBONE = 1e-5; LR_HEADS = 1e-3
params = [
    {'params': model_mtl.swin_unetr.parameters(), 'lr': LR_BACKBONE},
    {'params': model_mtl.encoder_proj.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.clinical_proj.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.grade_token, 'lr': LR_HEADS},
    {'params': model_mtl.diag_token, 'lr': LR_HEADS},
    {'params': model_mtl.fusion_transformer.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.grade_head.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.diagnosis_head.parameters(), 'lr': LR_HEADS},
]
optimizer = torch.optim.AdamW(params, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS_MTL)

# --- 6. Metrics & Loop Vars ---
dice_metric_train = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)
dice_metric_val = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)
post_pred_seg = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
post_label_seg = Compose([AsDiscrete(to_onehot=None)])

best_val_metric = -1.0; start_epoch = 0; GRAD_ACCUM_STEPS = 4

# Εμπλουτισμένο History
history = {
    "epoch": [],
    "train_loss": [], "train_dice": [],
    "train_f1_g": [], "train_acc_g": [],
    "train_f1_d": [], "train_acc_d": [],
    "val_loss": [], "val_dice": [],
    "val_f1_g": [], "val_acc_g": [],
    "val_f1_d": [], "val_acc_d": [],
    "combined_score": []
}

if os.path.exists(best_model_path): os.remove(best_model_path)

print(f"\n--- Έναρξη Mask-Guided MTL ({NUM_EPOCHS_MTL} Epochs) ---")

for epoch in range(start_epoch, NUM_EPOCHS_MTL):
    current_epoch = epoch + 1

    # === TRAIN ===
    model_mtl.train()
    dice_metric_train.reset()
    ep_loss = 0; step = 0
    tr_pred_g, tr_lbl_g = [], []; tr_pred_d, tr_lbl_d = [], []

    optimizer.zero_grad()
    pbar = tqdm(train_loader_mtl, desc=f"Epoch {current_epoch} Train")

    for i, batch in enumerate(pbar):
        step += 1
        img = batch[image_key].to(DEVICE); seg = batch[label_key].to(DEVICE)
        clin = batch[clinical_key].to(DEVICE)
        lbl_g = batch[grade_key].to(DEVICE); lbl_d = batch[diag_key].to(DEVICE)

        p_seg, p_g, p_d = model_mtl(img, clin)
        loss, l_s, l_g, l_d = mtl_loss_fn(p_seg, seg, p_g, lbl_g, p_d, lbl_d)

        if not torch.isfinite(loss): continue
        (loss / GRAD_ACCUM_STEPS).backward()

        if (i + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model_mtl.parameters(), 1.0)
            optimizer.step(); optimizer.zero_grad()

        ep_loss += loss.item()

        # Train Metrics
        val_outputs = [post_pred_seg(i) for i in decollate_batch(p_seg)]
        val_labels = [post_label_seg(i) for i in decollate_batch(seg)]
        dice_metric_train(y_pred=val_outputs, y=val_labels)

        tr_pred_g.extend(torch.argmax(p_g, 1).cpu().numpy()); tr_lbl_g.extend(lbl_g.cpu().numpy())
        tr_pred_d.extend(torch.argmax(p_d, 1).cpu().numpy()); tr_lbl_d.extend(lbl_d.cpu().numpy())

        pbar.set_postfix(loss=f"{loss.item():.2f}")

    # Calc Train Stats
    train_loss_avg = ep_loss / step
    train_dice = dice_metric_train.aggregate().item()
    train_f1_g = f1_score(tr_lbl_g, tr_pred_g, average='macro'); train_acc_g = accuracy_score(tr_lbl_g, tr_pred_g)
    train_f1_d = f1_score(tr_lbl_d, tr_pred_d, average='macro'); train_acc_d = accuracy_score(tr_lbl_d, tr_pred_d)

    # Store Train History
    history["epoch"].append(current_epoch)
    history["train_loss"].append(train_loss_avg); history["train_dice"].append(train_dice)
    history["train_f1_g"].append(train_f1_g); history["train_acc_g"].append(train_acc_g)
    history["train_f1_d"].append(train_f1_d); history["train_acc_d"].append(train_acc_d)

    print(f"Train | Loss: {train_loss_avg:.3f} | Dice: {train_dice:.3f} | G_F1: {train_f1_g:.3f} | D_F1: {train_f1_d:.3f}")

    # === VALIDATION ===
    if current_epoch % VAL_INTERVAL == 0:
        model_mtl.eval()
        dice_metric_val.reset()
        preds_g, lbls_g, preds_d, lbls_d = [], [], [], []
        val_loss_sum = 0; val_steps = 0

        with torch.no_grad():
            for batch in tqdm(val_loader_mtl, desc="Validating"):
                val_steps += 1
                img = batch[image_key].to(DEVICE); seg = batch[label_key].to(DEVICE)
                clin = batch[clinical_key].to(DEVICE)
                lbl_g = batch[grade_key].to(DEVICE); lbl_d = batch[diag_key].to(DEVICE)

                p_seg, p_g, p_d = model_mtl(img, clin)

                vl, _, _, _ = mtl_loss_fn(p_seg, seg, p_g, lbl_g, p_d, lbl_d)
                val_loss_sum += vl.item()

                val_outputs = [post_pred_seg(i) for i in decollate_batch(p_seg)]
                val_labels = [post_label_seg(i) for i in decollate_batch(seg)]
                dice_metric_val(y_pred=val_outputs, y=val_labels)

                preds_g.extend(torch.argmax(p_g, 1).cpu().numpy()); lbls_g.extend(lbl_g.cpu().numpy())
                preds_d.extend(torch.argmax(p_d, 1).cpu().numpy()); lbls_d.extend(lbl_d.cpu().numpy())

        # Calc Val Stats
        val_dice = dice_metric_val.aggregate().item()
        val_f1_g = f1_score(lbls_g, preds_g, average='macro'); val_acc_g = accuracy_score(lbls_g, preds_g)
        val_f1_d = f1_score(lbls_d, preds_d, average='macro'); val_acc_d = accuracy_score(lbls_d, preds_d)

        combined = (val_dice * 0.4) + (val_f1_g * 0.3) + (val_f1_d * 0.3)

        # Store Val History
        history["val_loss"].append(val_loss_sum/val_steps); history["val_dice"].append(val_dice)
        history["val_f1_g"].append(val_f1_g); history["val_acc_g"].append(val_acc_g)
        history["val_f1_d"].append(val_f1_d); history["val_acc_d"].append(val_acc_d)
        history["combined_score"].append(combined)

        print(f"--- Epoch {current_epoch} Val ---")
        print(f"Combined: {combined:.4f} | Loss: {val_loss_sum/val_steps:.4f} | Dice: {val_dice:.4f}")
        print(f"[GRADE] F1: {val_f1_g:.3f} (Acc: {val_acc_g:.3f})")
        print(confusion_matrix(lbls_g, preds_g))
        print(f"[DIAGNOSIS] F1: {val_f1_d:.3f} (Acc: {val_acc_d:.3f})")
        print(confusion_matrix(lbls_d, preds_d))

        if combined > best_val_metric:
            best_val_metric = combined
            torch.save(model_mtl.state_dict(), best_model_path)
            print("✅ New Best Model Saved!")

        # Save CSV every epoch
        try: pd.DataFrame(history).to_csv(history_csv_path, index=False)
        except: pass

    else:
        # Fill NaN for non-validation epochs to keep CSV aligned
        for k in ["val_loss", "val_dice", "val_f1_g", "val_acc_g", "val_f1_d", "val_acc_d", "combined_score"]:
            history[k].append(np.nan)
        pd.DataFrame(history).to_csv(history_csv_path, index=False)

    if scheduler: scheduler.step()

print(f"🏁 Finished. Best: {best_val_metric:.4f}")

# --- Test ---
print("\n=== FINAL TEST (Mask-Guided) ===")
if os.path.exists(best_model_path):
    model_mtl.load_state_dict(torch.load(best_model_path))
    model_mtl.eval()
    t_pg, t_lg, t_pd, t_ld = [], [], [], []
    dice_metric.reset()
    with torch.no_grad():
        for batch in tqdm(test_loader_mtl, desc="Testing"):
            img = batch[image_key].to(DEVICE); seg = batch[label_key].to(DEVICE)
            clin = batch[clinical_key].to(DEVICE)
            lbl_g = batch[grade_key].to(DEVICE); lbl_d = batch[diag_key].to(DEVICE)
            p_seg, p_g, p_d = model_mtl(img, clin)

            val_outputs = [post_pred_seg(i) for i in decollate_batch(p_seg)]
            val_labels = [post_label_seg(i) for i in decollate_batch(seg)]
            dice_metric(y_pred=val_outputs, y=val_labels)

            t_pg.extend(torch.argmax(p_g, 1).cpu().numpy()); t_lg.extend(lbl_g.cpu().numpy())
            t_pd.extend(torch.argmax(p_d, 1).cpu().numpy()); t_ld.extend(lbl_d.cpu().numpy())
    print(f"\nTest Dice: {dice_metric.aggregate().item():.4f}")
    print("\n[GRADE]\n", classification_report(t_lg, t_pg, labels=list(range(NUM_CLASSES_GRADE)), zero_division=0))
    print(confusion_matrix(t_lg, t_pg))
    print("\n[DIAGNOSIS]\n", classification_report(t_ld, t_pd, labels=list(range(NUM_CLASSES_DIAG)), zero_division=0))
else: print("Model not found.")

In [ ]:
# %% Βήμα 6: Οπτικοποίηση Αποτελεσμάτων (Full History Analysis)
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np

# --- Ρυθμίσεις ---
target_dir = os.path.join(runs_base_dir, "mtl_mask_guided_final")
csv_path = os.path.join(target_dir, "history_mask_full.csv")

print(f"Ανάγνωση αρχείου: {csv_path}")

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)

    # Διαχωρισμός γραμμών που έχουν validation data
    df_val = df.dropna(subset=['val_loss'])
    epochs = df['epoch']
    val_epochs = df_val['epoch']

    # Στυλ
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(2, 2, figsize=(20, 12))
    fig.suptitle('Mask-Guided MTL Training Analysis (Train vs Val)', fontsize=20, weight='bold')

    # 1. LOSSES & COMBINED SCORE
    ax1 = axes[0, 0]
    ax1.plot(epochs, df['train_loss'], label='Train Loss', color='blue', alpha=0.6)
    ax1.plot(val_epochs, df_val['val_loss'], label='Val Loss', color='navy', linewidth=2, marker='o')

    ax1b = ax1.twinx()
    ax1b.plot(val_epochs, df_val['combined_score'], label='Combined Score (Optimization Target)', color='gold', linestyle='--', linewidth=2)

    ax1.set_title('Optimization Landscape', fontsize=14)
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.legend(loc='upper left'); ax1b.legend(loc='upper right')
    ax1.grid(True, linestyle='--', alpha=0.7)

    # 2. SEGMENTATION (DICE)
    ax2 = axes[0, 1]
    ax2.plot(epochs, df['train_dice'], label='Train Dice', color='gray', linestyle=':', alpha=0.8)
    ax2.plot(val_epochs, df_val['val_dice'], label='Val Dice', color='black', linewidth=2, marker='s')

    ax2.set_title('Task 1: Segmentation (Dice Score)', fontsize=14)
    ax2.set_xlabel('Epoch'); ax2.set_ylim(0, 1.0)
    ax2.legend(loc='lower right'); ax2.grid(True)

    # 3. GRADE CLASSIFICATION
    ax3 = axes[1, 0]
    ax3.plot(epochs, df['train_f1_g'], label='Train F1', color='lightgreen', linestyle='--')
    ax3.plot(epochs, df['train_acc_g'], label='Train Acc', color='palegreen', linestyle=':')
    ax3.plot(val_epochs, df_val['val_f1_g'], label='Val F1', color='forestgreen', linewidth=2.5)
    ax3.plot(val_epochs, df_val['val_acc_g'], label='Val Acc', color='darkgreen', alpha=0.6)

    ax3.set_title('Task 2: Tumor Grade Classification', fontsize=14)
    ax3.set_xlabel('Epoch'); ax3.set_ylim(0, 1.0)
    ax3.legend(loc='lower right'); ax3.grid(True)

    # 4. DIAGNOSIS CLASSIFICATION
    ax4 = axes[1, 1]
    ax4.plot(epochs, df['train_f1_d'], label='Train F1', color='plum', linestyle='--')
    ax4.plot(epochs, df['train_acc_d'], label='Train Acc', color='thistle', linestyle=':')
    ax4.plot(val_epochs, df_val['val_f1_d'], label='Val F1', color='darkmagenta', linewidth=2.5)
    ax4.plot(val_epochs, df_val['val_acc_d'], label='Val Acc', color='indigo', alpha=0.6)

    ax4.set_title('Task 3: Diagnosis Classification', fontsize=14)
    ax4.set_xlabel('Epoch'); ax4.set_ylim(0, 1.0)
    ax4.legend(loc='lower right'); ax4.grid(True)

    plt.tight_layout()
    plt.show()

    # Stats Printout
    best_idx = df_val['combined_score'].idxmax()
    best_row = df.loc[best_idx]
    print(f"\n🏆 BEST MODEL at Epoch {int(best_row['epoch'])}")
    print(f"Combined Score: {best_row['combined_score']:.4f}")
    print(f"Seg Dice: {best_row['val_dice']:.4f}")
    print(f"Grade F1: {best_row['val_f1_g']:.4f} (Acc: {best_row['val_acc_g']:.4f})")
    print(f"Diag F1:  {best_row['val_f1_d']:.4f} (Acc: {best_row['val_acc_d']:.4f})")
else:
    print("❌ CSV File not found. Run training first.")

# Interpretations

In [ ]:

torch.cuda.empty_cache()
gc.collect()

In [ ]:
# %% Βήμα 7 (ULTIMATE VISUALIZATION): 5 Samples με Segmentation Prediction
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from monai.visualize import GradCAM
from monai.metrics import compute_dice
import os
import gc

# --- 1. Helper: Βρες το όνομα του layer ---
def get_last_swin_block_name_from_root(model_instance):
    last_block_name = None
    for name, module in model_instance.named_modules():
        if "SwinTransformerBlock" in str(type(module)):
            last_block_name = name
    if last_block_name is None:
        for name, module in model_instance.named_modules():
            if isinstance(module, nn.LayerNorm) and "swinViT" in name:
                last_block_name = name
    return last_block_name

# --- 2. Wrapper για Grad-CAM ---
class ModelWrapperForGradCAM(nn.Module):
    def __init__(self, model, clinical_data):
        super().__init__()
        self.model = model
        self.clinical_data = clinical_data

    def forward(self, x_img):
        outputs = self.model(x_img, self.clinical_data)
        # Επιστροφή Grade Logits
        if isinstance(outputs, tuple):
            if len(outputs) == 3: return outputs[1] # MTL (Seg, Grade, Diag) -> Grade
            elif len(outputs) == 2: return outputs[0] # Cascaded (Grade, Diag) -> Grade
        return outputs

# --- 3. Πειράματα ---
experiments = [
    {
        "name": "1. Cascaded Transformer (Frozen)",
        "model_class": CascadedTransformerClassifier,
        "path": os.path.join(runs_base_dir, "classification_cascaded_v4_transformer_weighted", "best_model.pt"),
        "kwargs": {"d_model": 64, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.5},
        "has_seg_output": False # Το Cascaded δεν επιστρέφει seg στο forward του
    },
    {
        "name": "2. Parallel MTL (Unfrozen)",
        "model_class": MTLTransformerModel,
        "path": os.path.join(runs_base_dir, "mtl_unfrozen_transformer_final", "best_mtl_model.pt"),
        "kwargs": {"d_model": 128, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.4},
        "has_seg_output": True
    },
    {
        "name": "3. Sequential MTL (Hierarchical)",
        "model_class": SequentialMTLModel,
        "path": os.path.join(runs_base_dir, "mtl_sequential_final", "best_seq_model.pt"),
        "kwargs": {"d_model": 128, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.4},
        "has_seg_output": True
    },
    {
        "name": "4. Sequential MTL (Masked)",
        "model_class": MaskGuidedMTLModel,
        "path": os.path.join(runs_base_dir, "mtl_mask_guided_final", "best_mask_model.pt"),
        "kwargs": {"d_model": 128, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.4},
        "has_seg_output": True
    }

]

# --- 4. Επιλογή Δειγμάτων (Samples) ---
print("🔍 Αναζήτηση 5 δειγμάτων με όγκο...")
samples_to_viz = []
samples_needed = 5

for batch in test_loader_mtl:
    lbl_b = batch[label_key]
    for i in range(len(lbl_b)):
        if lbl_b[i].sum() > 1000: # Να έχει αρκετό όγκο
            samples_to_viz.append({
                image_key: batch[image_key][i:i+1],
                clinical_key: batch[clinical_key][i:i+1],
                label_key: batch[label_key][i:i+1],
                grade_key: batch[grade_key][i:i+1]
            })
            if len(samples_to_viz) >= samples_needed: break
    if len(samples_to_viz) >= samples_needed: break

print(f"✅ Βρέθηκαν {len(samples_to_viz)} δείγματα.")

# --- 5. Main Loop ---
for exp in experiments:
    print(f"\n{'='*60}")
    print(f"🔄 Ανάλυση Μοντέλου: {exp['name']}")
    print(f"{'='*60}")

    if not os.path.exists(exp['path']):
        print(f"❌ Το αρχείο δεν βρέθηκε: {exp['path']}"); continue

    # A. Load Model
    try:
        model_interp = exp['model_class'](
            in_channels_img=IN_CHANNELS, out_channels_seg=OUT_CHANNELS_SEG,
            feature_size=FEATURE_SIZE, use_checkpoint=False,
            num_classes_grade=NUM_CLASSES_GRADE, num_classes_diag=NUM_CLASSES_DIAG,
            clinical_feature_size=CLINICAL_FEATURE_SIZE,
            finetuned_seg_model_path=None,
            **exp['kwargs']
        ).to(DEVICE)

        state_dict = torch.load(exp['path'], map_location=DEVICE)
        model_interp.load_state_dict(state_dict, strict=False)
        # Force Gradients for Grad-CAM
        for param in model_interp.parameters(): param.requires_grad = True
        model_interp.eval()
    except Exception as e:
        print(f"❌ Σφάλμα φόρτωσης: {e}"); continue

    # B. Setup Grad-CAM
    layer_name = get_last_swin_block_name_from_root(model_interp)
    if layer_name is None: print("⚠️ Layer not found."); continue
    target_layer_string = f"model.{layer_name}"

    # C. Loop Samples
    for s_idx, sample in enumerate(samples_to_viz):
        img_tensor = sample[image_key].to(DEVICE)
        clin_tensor = sample[clinical_key].to(DEVICE)
        true_grade = sample[grade_key].item()
        true_mask = sample[label_key].to(DEVICE)

        # --- 1. Get Predictions (Classification & Segmentation) ---
        pred_seg_mask = None

        # Ανάλογα με το μοντέλο, παίρνουμε το segmentation διαφορετικά
        if exp['has_seg_output']:
            # MTL / Sequential: Επιστρέφουν (seg, grade, diag)
            logits_seg, logits_grade, _ = model_interp(img_tensor, clin_tensor)
            pred_seg_mask = torch.sigmoid(logits_seg) > 0.5
        else:
            # Cascaded: Επιστρέφει (grade, diag). Πρέπει να καλέσουμε τον extractor ρητά.
            logits_grade, _ = model_interp(img_tensor, clin_tensor)
            # Χειροκίνητο πέρασμα για Seg
            logits_seg = model_interp.swin_unetr_extractor(img_tensor)
            pred_seg_mask = torch.sigmoid(logits_seg) > 0.5

        pred_grade_idx = torch.argmax(logits_grade, dim=1).item()

        # Υπολογισμός Dice για αυτό το δείγμα
        dice_score = compute_dice(pred_seg_mask, true_mask, include_background=False).item()

        # --- 2. Run Grad-CAM ---
        wrapped_model = ModelWrapperForGradCAM(model_interp, clin_tensor)
        cam = GradCAM(nn_module=wrapped_model, target_layers=target_layer_string)
        try:
            heatmap_res = cam(x=img_tensor, class_idx=pred_grade_idx)
            heatmap = heatmap_res[0, 0].cpu().detach().numpy()
        except: heatmap = np.zeros(img_tensor.shape[2:])

        # --- 3. Visualization ---
        vol = img_tensor[0, 0].cpu().numpy()
        mask_gt = true_mask[0, 0].cpu().numpy()
        mask_pred = pred_seg_mask[0, 0].cpu().numpy()

        # Find best slice (based on GT)
        if mask_gt.sum() > 0: slice_idx = np.argmax(np.sum(mask_gt, axis=(0, 1)))
        else: slice_idx = vol.shape[2] // 2

        # Plotting (4 Columns)
        plt.figure(figsize=(16, 4))

        # Col 1: MRI
        plt.subplot(1, 4, 1)
        plt.imshow(np.rot90(vol[:, :, slice_idx]), cmap='gray')
        plt.title(f"Sample {s_idx+1} (Grade: {GRADE_CLASSES[true_grade]})")
        plt.axis('off')

        # Col 2: Ground Truth
        plt.subplot(1, 4, 2)
        plt.imshow(np.rot90(vol[:, :, slice_idx]), cmap='gray')
        plt.imshow(np.rot90(mask_gt[:, :, slice_idx]), cmap='jet', alpha=0.5)
        plt.title("Ground Truth Mask")
        plt.axis('off')

        # Col 3: Predicted Seg
        plt.subplot(1, 4, 3)
        plt.imshow(np.rot90(vol[:, :, slice_idx]), cmap='gray')
        plt.imshow(np.rot90(mask_pred[:, :, slice_idx]), cmap='autumn', alpha=0.5) # Autumn = Red/Yellow
        plt.title(f"Pred Mask (Dice: {dice_score:.2f})")
        plt.axis('off')

        # Col 4: Grad-CAM
        plt.subplot(1, 4, 4)
        plt.imshow(np.rot90(vol[:, :, slice_idx]), cmap='gray')
        plt.imshow(np.rot90(heatmap[:, :, slice_idx]), cmap='hot', alpha=0.5)

        color = 'green' if pred_grade_idx == true_grade else 'red'
        plt.title(f"Attention (Pred: {GRADE_CLASSES[pred_grade_idx]})", color=color, fontweight='bold')
        plt.axis('off')

        plt.tight_layout()
        plt.show()

    # Clean
    del model_interp, wrapped_model, cam
    torch.cuda.empty_cache()
    gc.collect()

print("\n✅ Ολοκληρώθηκε.")

In [ ]:
# %% VRAM INSPECTOR & CLEANER
import torch
import gc
import sys

def check_vram_usage():
    print(f"\n{'='*40}")
    print(f"{'ΜΕΤΑΒΛΗΤΗ':<25} | {'ΤΥΠΟΣ':<10} | {'ΜΕΓΕΘΟΣ (MB)':<10}")
    print(f"{'='*40}")

    total_usage = 0
    # Σαρώνουμε όλες τις global μεταβλητές
    for name, obj in globals().items():
        if not name.startswith('_'): # Αγνοούμε εσωτερικές μεταβλητές
            size = 0
            obj_type = "Unknown"
            is_cuda = False

            # Έλεγχος αν είναι Tensor
            if torch.is_tensor(obj):
                obj_type = "Tensor"
                if obj.is_cuda:
                    is_cuda = True
                    size = obj.element_size() * obj.nelement() / (1024 ** 2)

            # Έλεγχος αν είναι Μοντέλο (nn.Module)
            elif isinstance(obj, torch.nn.Module):
                obj_type = "Model"
                # Υπολογισμός παραμέτρων μοντέλου
                try:
                    params_size = sum(p.numel() * p.element_size() for p in obj.parameters())
                    bufs_size = sum(b.numel() * b.element_size() for b in obj.buffers())
                    size = (params_size + bufs_size) / (1024 ** 2)
                    # Έλεγχος αν έστω και μία παράμετρος είναι στην GPU
                    if next(obj.parameters()).is_cuda: is_cuda = True
                except: pass

            if is_cuda and size > 0.1: # Δείχνουμε μόνο ό,τι πιάνει > 0.1 MB
                print(f"{name:<25} | {obj_type:<10} | {size:.2f} MB")
                total_usage += size

    print(f"{'='*40}")
    print(f"Σύνολο από μεταβλητές: {total_usage:.2f} MB")

    # Πραγματική δέσμευση PyTorch (περιλαμβάνει και cached memory που δεν φαίνεται παραπάνω)
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / (1024 ** 2)
        reserved = torch.cuda.memory_reserved() / (1024 ** 2)
        print(f"\n[PyTorch Internal Report]")
        print(f"Allocated (Ενεργή Μνήμη): {allocated:.2f} MB")
        print(f"Reserved  (Κατειλημμένη Cache): {reserved:.2f} MB")
    else:
        print("\n❌ Δεν βρέθηκε GPU.")

def clean_vram():
    print("\n🧹 Καθαρισμός VRAM...")

    # Λίστα με τα 'βαριά' αντικείμενα που χρησιμοποιούμε συχνά
    targets = [
        'model', 'model_mtl', 'model_interp', 'model_full',
        'wrapped_model', 'cam', 'optimizer', 'loss',
        'img_tensor', 'clin_tensor', 'heatmap_res',
        'logits', 'logits_seg', 'logits_grade',
        'full_loader', 'train_loader_mtl', 'test_loader_mtl'
    ]

    deleted_count = 0
    for t in targets:
        if t in globals():
            del globals()[t]
            deleted_count += 1

    # Γενικός καθαρισμός Python Garbage Collector
    gc.collect()

    # Άδειασμα PyTorch Cache
    torch.cuda.empty_cache()

    print(f"✅ Διαγράφηκαν {deleted_count} μεταβλητές.")
    print("✅ Εκτελέστηκε garbage collection & empty_cache.")

    # Τελικός έλεγχος
    if torch.cuda.is_available():
        print(f"--> Νέα Δεσμευμένη Μνήμη: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

# --- ΕΚΤΕΛΕΣΗ ---
print("1️⃣ ΠΡΙΝ ΤΟΝ ΚΑΘΑΡΙΣΜΟ:")
check_vram_usage()

# Αν θες να τα σβήσεις όλα, βγάλε το σχόλιο από την παρακάτω γραμμή:
clean_vram()

In [ ]:
# %% Βήμα 7 (FINAL PATCH): Visualization με ξεκλείδωμα του Cascaded Model
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F # Χρειάζεται για το patch
from monai.visualize import GradCAM
from monai.metrics import compute_dice
import os
import gc
import types # Για το monkey patching

# --- 1. Helper: Find Layer ---
def get_target_layer_high_res(model_instance, depth_index=-2):
    target_name = None
    backbone = None
    if hasattr(model_instance, 'swin_unetr'): backbone = model_instance.swin_unetr
    elif hasattr(model_instance, 'swin_unetr_extractor'): backbone = model_instance.swin_unetr_extractor

    if backbone:
        try:
            target_block = backbone.swinViT.layers[depth_index].blocks[-1]
            for name, module in model_instance.named_modules():
                if module is target_block: target_name = name; break
        except: pass

    if target_name is None:
        for name, module in model_instance.named_modules():
            if "SwinTransformerBlock" in str(type(module)): target_name = name
    return target_name

def get_tumor_center(mask_3d):
    coords = np.argwhere(mask_3d > 0)
    if len(coords) > 0:
        center = coords.mean(axis=0).astype(int)
        return center[0], center[1], center[2]
    return mask_3d.shape[0]//2, mask_3d.shape[1]//2, mask_3d.shape[2]//2

# --- 2. ΤΟ FIX ΓΙΑ ΤΟ CASCADED MODEL (The Patch) ---
def cascaded_forward_patched(self, x_img, x_clinical):
    """
    Αυτή η συνάρτηση αντικαθιστά την original forward του Cascaded.
    Η διαφορά: ΔΕΝ έχει 'with torch.no_grad()'.
    """
    B = x_img.shape[0]
    # Αλλαγή: Τρέχουμε τον extractor ΧΩΡΙΣ no_grad
    _ = self.swin_unetr_extractor(x_img)

    enc_feat = F.adaptive_avg_pool3d(self.encoder_features_out, 1).flatten(1)
    dec_feat = F.adaptive_avg_pool3d(self.decoder_features_out, 1).flatten(1)

    enc_token = self.encoder_proj(enc_feat).unsqueeze(1)
    dec_token = self.decoder_proj(dec_feat).unsqueeze(1)
    clin_token = self.clinical_proj(x_clinical).unsqueeze(1)

    grade_token = self.grade_token.expand(B, -1, -1)
    diag_token = self.diag_token.expand(B, -1, -1)

    tokens = torch.cat([grade_token, diag_token, enc_token, dec_token, clin_token], dim=1)
    fused_tokens = self.fusion_transformer(tokens)

    return self.grade_head(fused_tokens[:, 0, :]), self.diagnosis_head(fused_tokens[:, 1, :])

# --- 3. Wrapper ---
class ModelWrapperForGradCAM(nn.Module):
    def __init__(self, model, clinical_data):
        super().__init__()
        self.model = model
        self.clinical_data = clinical_data
    def forward(self, x_img):
        outputs = self.model(x_img, self.clinical_data)
        if isinstance(outputs, tuple):
            if len(outputs) == 3: return outputs[1]
            elif len(outputs) == 2: return outputs[0]
        return outputs

# --- 4. Experiments ---
experiments = [
    {
        "name": "1. Cascaded Transformer (Frozen)",
        "model_class": CascadedTransformerClassifier,
        "path": os.path.join(runs_base_dir, "classification_cascaded_v4_transformer_weighted", "best_model.pt"),
        "kwargs": {"d_model": 64, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.5},
        "has_seg_output": False,
        "needs_patch": True # <-- Σημαία για το patch
    },
    {
        "name": "2. Parallel MTL (Unfrozen)",
        "model_class": MTLTransformerModel,
        "path": os.path.join(runs_base_dir, "mtl_unfrozen_transformer_final", "best_mtl_model.pt"),
        "kwargs": {"d_model": 128, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.4},
        "has_seg_output": True,
        "needs_patch": False
    },
    {
        "name": "3. Sequential MTL (Hierarchical)",
        "model_class": SequentialMTLModel,
        "path": os.path.join(runs_base_dir, "mtl_sequential_final", "best_seq_model.pt"),
        "kwargs": {"d_model": 128, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.4},
        "has_seg_output": True,
        "needs_patch": False
    },
    {
        "name": "4. Sequential MTL (Masked)",
        "model_class": MaskGuidedMTLModel,
        "path": os.path.join(runs_base_dir, "mtl_mask_guided_final", "best_mask_model.pt"),
        "kwargs": {"d_model": 128, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.4},
        "has_seg_output": True,
        "needs_patch": False
    }
]

# --- 5. Find Samples ---
print("🔍 Αναζήτηση 5 δειγμάτων...")
samples_to_viz = []
samples_needed = 5
for batch in test_loader_mtl:
    lbl_b = batch[label_key]
    for i in range(len(lbl_b)):
        if lbl_b[i].sum() > 2000:
            samples_to_viz.append({
                image_key: batch[image_key][i:i+1],
                clinical_key: batch[clinical_key][i:i+1],
                label_key: batch[label_key][i:i+1],
                grade_key: batch[grade_key][i:i+1]
            })
            if len(samples_to_viz) >= samples_needed: break
    if len(samples_to_viz) >= samples_needed: break

# --- 6. Main Loop ---
for exp in experiments:
    print(f"\n{'='*60}")
    print(f"🔄 Ανάλυση: {exp['name']}")
    print(f"{'='*60}")

    if not os.path.exists(exp['path']): print("Model not found."); continue

    try:
        model_interp = exp['model_class'](
            in_channels_img=IN_CHANNELS, out_channels_seg=OUT_CHANNELS_SEG,
            feature_size=FEATURE_SIZE, use_checkpoint=False,
            num_classes_grade=NUM_CLASSES_GRADE, num_classes_diag=NUM_CLASSES_DIAG,
            clinical_feature_size=CLINICAL_FEATURE_SIZE,
            finetuned_seg_model_path=None, **exp['kwargs']
        ).to(DEVICE)

        state_dict = torch.load(exp['path'], map_location=DEVICE)
        model_interp.load_state_dict(state_dict, strict=False)

        # --- THE FIX: MONKEY PATCHING ---
        if exp.get('needs_patch', False):
            print("🔧 Εφαρμογή Patch στο 'forward' για αφαίρεση του no_grad()...")
            # Αντικαθιστούμε τη μέθοδο forward του αντικειμένου με τη δική μας
            model_interp.forward = types.MethodType(cascaded_forward_patched, model_interp)

        # Ενεργοποίηση Gradients παντού
        for param in model_interp.parameters(): param.requires_grad = True
        model_interp.eval()

    except Exception as e: print(f"Load Error: {e}"); continue

    # Grad-CAM Setup
    layer_name = get_target_layer_high_res(model_interp, depth_index=-2)
    target_layer_string = f"model.{layer_name}"

    # Loop Samples
    for s_idx, sample in enumerate(samples_to_viz):
        img_tensor = sample[image_key].to(DEVICE)
        clin_tensor = sample[clinical_key].to(DEVICE)
        true_grade = sample[grade_key].item()
        true_mask = sample[label_key].to(DEVICE)

        # Predictions
        if exp['has_seg_output']:
            logits_seg, logits_grade, _ = model_interp(img_tensor, clin_tensor)
            pred_seg_mask = torch.sigmoid(logits_seg) > 0.5
        else:
            logits_grade, _ = model_interp(img_tensor, clin_tensor)
            if hasattr(model_interp, 'swin_unetr_extractor'):
                logits_seg = model_interp.swin_unetr_extractor(img_tensor)
            else: logits_seg = model_interp.swin_unetr(img_tensor)
            pred_seg_mask = torch.sigmoid(logits_seg) > 0.5

        pred_grade_idx = torch.argmax(logits_grade, dim=1).item()
        dice_score = compute_dice(pred_seg_mask, true_mask, include_background=False).item()

        # Grad-CAM Run
        wrapped_model = ModelWrapperForGradCAM(model_interp, clin_tensor)
        try:
            cam = GradCAM(nn_module=wrapped_model, target_layers=target_layer_string)
            heatmap_res = cam(x=img_tensor, class_idx=pred_grade_idx)
            heatmap = heatmap_res[0, 0].cpu().detach().numpy()
        except Exception as e:
            print(f"GradCAM Error: {e}")
            heatmap = np.zeros(img_tensor.shape[2:])

        # Visualization
        vol = img_tensor[0, 0].cpu().detach().numpy()
        mask_gt = true_mask[0, 0].cpu().detach().numpy()
        mask_pred = pred_seg_mask[0, 0].cpu().detach().numpy()
        brain_area = vol > 0.01

        # 2. Εφαρμογή στο Heatmap:
        # Μηδενίζουμε (κάνουμε μαύρο) το heatmap όπου δεν υπάρχει εγκέφαλος
        heatmap = heatmap * brain_area
        cx, cy, cz = get_tumor_center(mask_gt)
        views = [
            ("Axial (Z)", np.rot90(vol[:, :, cz]), np.rot90(mask_gt[:, :, cz]), np.rot90(mask_pred[:, :, cz]), np.rot90(heatmap[:, :, cz])),
            ("Coronal (Y)", np.rot90(vol[:, cy, :]), np.rot90(mask_gt[:, cy, :]), np.rot90(mask_pred[:, cy, :]), np.rot90(heatmap[:, cy, :])),
            ("Sagittal (X)", np.rot90(vol[cx, :, :]), np.rot90(mask_gt[cx, :, :]), np.rot90(mask_pred[cx, :, :]), np.rot90(heatmap[cx, :, :]))
        ]

        fig, axes = plt.subplots(3, 4, figsize=(16, 10))
        grade_color = 'green' if pred_grade_idx == true_grade else 'red'
        fig.suptitle(f"Sample {s_idx+1} | Grade: {GRADE_CLASSES[true_grade]} vs Pred: {GRADE_CLASSES[pred_grade_idx]}", fontsize=16, color=grade_color, fontweight='bold')

        for row_idx, (name, v_img, v_gt, v_pred, v_cam) in enumerate(views):
            axes[row_idx, 0].imshow(v_img, cmap='gray'); axes[row_idx, 0].set_ylabel(name, fontsize=12, fontweight='bold')
            axes[row_idx, 1].imshow(v_img, cmap='gray'); axes[row_idx, 1].imshow(v_gt, cmap='jet', alpha=0.5 if v_gt.max()>0 else 0)
            axes[row_idx, 2].imshow(v_img, cmap='gray'); axes[row_idx, 2].imshow(v_pred, cmap='autumn', alpha=0.5 if v_pred.max()>0 else 0)
            axes[row_idx, 3].imshow(v_img, cmap='gray'); axes[row_idx, 3].imshow(v_cam, cmap='hot', alpha=0.5)
            if row_idx == 0:
                axes[0, 0].set_title("MRI"); axes[0, 1].set_title("Ground Truth"); axes[0, 2].set_title("Pred Seg"); axes[0, 3].set_title("Attention")

        for ax in axes.flatten(): ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout(); plt.show()

    del model_interp, wrapped_model, cam
    gc.collect(); torch.cuda.empty_cache()

print("\n✅ Visualization Completed.")

In [ ]:
# %% DEBUG: Εκτύπωση ΟΛΩΝ των Layers για να βρούμε το σωστό όνομα
import torch
import os

# Φόρτωση (π.χ. Parallel MTL)
model_debug = MTLTransformerModel(
    in_channels_img=IN_CHANNELS, out_channels_seg=OUT_CHANNELS_SEG,
    feature_size=FEATURE_SIZE, use_checkpoint=False,
    num_classes_grade=NUM_CLASSES_GRADE, num_classes_diag=NUM_CLASSES_DIAG,
    clinical_feature_size=CLINICAL_FEATURE_SIZE,
    d_model=128, n_heads=4, num_layers=2
).to(DEVICE)

print("🔍 Αναζήτηση 'SwinTransformerBlock' μέσα στο μοντέλο...")
print(f"{'='*60}")
print(f"{'FULL LAYER NAME':<50} | {'TYPE'}")
print(f"{'='*60}")

found_blocks = []

# Σκανάρουμε αναδρομικά ΟΛΟ το μοντέλο
for name, module in model_debug.named_modules():
    # Ψάχνουμε οτιδήποτε περιέχει τη λέξη "Block" ή "Swin" στον τύπο του
    if "SwinTransformerBlock" in str(type(module)):
        print(f"{name:<50} | Block ✅")
        found_blocks.append(name)
    elif "PatchMerging" in str(type(module)):
        print(f"{name:<50} | ⬇️ Downsample (Patch Merging)")

print(f"{'='*60}")
print(f"Βρέθηκαν συνολικά {len(found_blocks)} Blocks.")

if len(found_blocks) > 0:
    print("\n--- ΠΡΟΤΑΣΕΙΣ ΓΙΑ GRAD-CAM ---")
    print("1. Bottleneck (Low Res - Αυτό που έχεις τώρα):")
    print(f"   target_layer_string = 'model.{found_blocks[-1]}'")

    # Βρίσκουμε το προ-τελευταίο στάδιο (συνήθως 2 blocks πριν το τέλος)
    if len(found_blocks) > 2:
        mid_index = len(found_blocks) - 3 # Πάμε λίγο πίσω
        print("\n2. High Res (Texture/Details - Αυτό που θέλουμε):")
        print(f"   target_layer_string = 'model.{found_blocks[mid_index]}'")

In [ ]:
# %% Βήμα 7 (PRECISION VISUALIZATION): High-Res 3D Grad-CAM
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.visualize import GradCAM
from monai.metrics import compute_dice
import os
import gc
import types

# --- 1. Helper: Tumor Center ---
def get_tumor_center(mask_3d):
    coords = np.argwhere(mask_3d > 0)
    if len(coords) > 0:
        center = coords.mean(axis=0).astype(int)
        return center[0], center[1], center[2]
    return mask_3d.shape[0]//2, mask_3d.shape[1]//2, mask_3d.shape[2]//2

# --- 2. Patch για το Cascaded Model (Unlocking Gradients) ---
def cascaded_forward_patched(self, x_img, x_clinical):
    # Τρέχουμε τον extractor ΧΩΡΙΣ no_grad
    _ = self.swin_unetr_extractor(x_img)
    B = x_img.shape[0]

    enc_feat = F.adaptive_avg_pool3d(self.encoder_features_out, 1).flatten(1)
    dec_feat = F.adaptive_avg_pool3d(self.decoder_features_out, 1).flatten(1)

    enc_token = self.encoder_proj(enc_feat).unsqueeze(1)
    dec_token = self.decoder_proj(dec_feat).unsqueeze(1)
    clin_token = self.clinical_proj(x_clinical).unsqueeze(1)

    grade_token = self.grade_token.expand(B, -1, -1)
    diag_token = self.diag_token.expand(B, -1, -1)

    tokens = torch.cat([grade_token, diag_token, enc_token, dec_token, clin_token], dim=1)
    fused_tokens = self.fusion_transformer(tokens)

    return self.grade_head(fused_tokens[:, 0, :]), self.diagnosis_head(fused_tokens[:, 1, :])

# --- 3. Wrapper ---
class ModelWrapperForGradCAM(nn.Module):
    def __init__(self, model, clinical_data):
        super().__init__()
        self.model = model
        self.clinical_data = clinical_data
    def forward(self, x_img):
        outputs = self.model(x_img, self.clinical_data)
        if isinstance(outputs, tuple):
            if len(outputs) == 3: return outputs[1]
            elif len(outputs) == 2: return outputs[0]
        return outputs

# --- 4. Πειράματα ---
experiments = [
    {
        "name": "1. Cascaded Transformer (Frozen)",
        "model_class": CascadedTransformerClassifier,
        "path": os.path.join(runs_base_dir, "classification_cascaded_v4_transformer_weighted", "best_model.pt"),
        "kwargs": {"d_model": 64, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.5},
        "has_seg_output": False,
        "needs_patch": True,
        # Χρησιμοποιούμε swin_unetr_extractor + layers3 (High Res)
        "target_layer": "model.swin_unetr_extractor.swinViT.layers3.0.blocks.1"
    },
    {
        "name": "2. Parallel MTL (Unfrozen)",
        "model_class": MTLTransformerModel,
        "path": os.path.join(runs_base_dir, "mtl_unfrozen_transformer_final", "best_mtl_model.pt"),
        "kwargs": {"d_model": 128, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.4},
        "has_seg_output": True,
        "needs_patch": False,
        # Χρησιμοποιούμε swin_unetr + layers3 (High Res)
        "target_layer": "model.swin_unetr.swinViT.layers3.0.blocks.1"
    },
    {
        "name": "3. Sequential MTL (Hierarchical)",
        "model_class": SequentialMTLModel,
        "path": os.path.join(runs_base_dir, "mtl_sequential_final", "best_seq_model.pt"),
        "kwargs": {"d_model": 128, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.4},
        "has_seg_output": True,
        "needs_patch": False,
        # Χρησιμοποιούμε swin_unetr + layers3 (High Res)
        "target_layer": "model.swin_unetr.swinViT.layers3.0.blocks.1"
    },
    {
        "name": "4. Sequential MTL (Masked)",
        "model_class": MaskGuidedMTLModel,
        "path": os.path.join(runs_base_dir, "mtl_mask_guided_final", "best_mask_model.pt"),
        "kwargs": {"d_model": 128, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.4},
        "has_seg_output": True,
        "needs_patch": False,
        "target_layer": "model.swin_unetr.swinViT.layers3.0.blocks.1"
    }
]

# --- 5. Εύρεση Δειγμάτων ---
print("🔍 Αναζήτηση 5 δειγμάτων...")
samples_to_viz = []
samples_needed = 3
for batch in test_loader_mtl:
    lbl_b = batch[label_key]
    for i in range(len(lbl_b)):
        if lbl_b[i].sum() > 2000:
            samples_to_viz.append({
                image_key: batch[image_key][i:i+1],
                clinical_key: batch[clinical_key][i:i+1],
                label_key: batch[label_key][i:i+1],
                grade_key: batch[grade_key][i:i+1]
            })
            if len(samples_to_viz) >= samples_needed: break
    if len(samples_to_viz) >= samples_needed: break

# --- 6. Main Loop ---
for exp in experiments:
    print(f"\n{'='*60}")
    print(f"🔄 Μοντέλο: {exp['name']}")
    print(f"🎯 Target Layer: {exp['target_layer']}")
    print(f"{'='*60}")

    if not os.path.exists(exp['path']): print("Model not found."); continue

    try:
        model_interp = exp['model_class'](
            in_channels_img=IN_CHANNELS, out_channels_seg=OUT_CHANNELS_SEG,
            feature_size=FEATURE_SIZE, use_checkpoint=False,
            num_classes_grade=NUM_CLASSES_GRADE, num_classes_diag=NUM_CLASSES_DIAG,
            clinical_feature_size=CLINICAL_FEATURE_SIZE,
            finetuned_seg_model_path=None, **exp['kwargs']
        ).to(DEVICE)

        state_dict = torch.load(exp['path'], map_location=DEVICE)
        model_interp.load_state_dict(state_dict, strict=False)

        # Patching & Gradients
        if exp.get('needs_patch', False):
            model_interp.forward = types.MethodType(cascaded_forward_patched, model_interp)
        for param in model_interp.parameters(): param.requires_grad = True
        model_interp.eval()

    except Exception as e: print(f"Load Error: {e}"); continue

    # Loop Samples
    for s_idx, sample in enumerate(samples_to_viz):
        img_tensor = sample[image_key].to(DEVICE)
        clin_tensor = sample[clinical_key].to(DEVICE)
        true_grade = sample[grade_key].item()
        true_mask = sample[label_key].to(DEVICE)

        # Preds
        if exp['has_seg_output']:
            logits_seg, logits_grade, _ = model_interp(img_tensor, clin_tensor)
            pred_seg_mask = torch.sigmoid(logits_seg) > 0.5
        else:
            logits_grade, _ = model_interp(img_tensor, clin_tensor)
            if hasattr(model_interp, 'swin_unetr_extractor'):
                logits_seg = model_interp.swin_unetr_extractor(img_tensor)
            else: logits_seg = model_interp.swin_unetr(img_tensor)
            pred_seg_mask = torch.sigmoid(logits_seg) > 0.5

        pred_grade_idx = torch.argmax(logits_grade, dim=1).item()
        dice_score = compute_dice(pred_seg_mask, true_mask, include_background=False).item()

        # Grad-CAM Run
        wrapped_model = ModelWrapperForGradCAM(model_interp, clin_tensor)
        try:
            cam = GradCAM(nn_module=wrapped_model, target_layers=exp['target_layer'])
            heatmap_res = cam(x=img_tensor, class_idx=pred_grade_idx)
            heatmap = heatmap_res[0, 0].cpu().detach().numpy()
        except Exception as e:
            print(f"GradCAM Error: {e}")
            heatmap = np.zeros(img_tensor.shape[2:])

        # Visualization Logic
        vol = img_tensor[0, 0].cpu().detach().numpy()
        mask_gt = true_mask[0, 0].cpu().detach().numpy()
        mask_pred = pred_seg_mask[0, 0].cpu().detach().numpy()

        # Background Masking (Καθαρισμός)
        brain_area = vol > 0.01
        heatmap = heatmap * brain_area

        cx, cy, cz = get_tumor_center(mask_gt)
        views = [
            ("Axial (Z)", np.rot90(vol[:, :, cz]), np.rot90(mask_gt[:, :, cz]), np.rot90(mask_pred[:, :, cz]), np.rot90(heatmap[:, :, cz])),
            ("Coronal (Y)", np.rot90(vol[:, cy, :]), np.rot90(mask_gt[:, cy, :]), np.rot90(mask_pred[:, cy, :]), np.rot90(heatmap[:, cy, :])),
            ("Sagittal (X)", np.rot90(vol[cx, :, :]), np.rot90(mask_gt[cx, :, :]), np.rot90(mask_pred[cx, :, :]), np.rot90(heatmap[cx, :, :]))
        ]

        fig, axes = plt.subplots(3, 4, figsize=(16, 10))
        col = 'green' if pred_grade_idx == true_grade else 'red'
        fig.suptitle(f"Sample {s_idx+1} | Grade: {GRADE_CLASSES[true_grade]} vs Pred: {GRADE_CLASSES[pred_grade_idx]}", fontsize=16, color=col, fontweight='bold')

        for r, (name, v_img, v_gt, v_pred, v_cam) in enumerate(views):
            axes[r, 0].imshow(v_img, cmap='gray'); axes[r, 0].set_ylabel(name, fontsize=12, fontweight='bold')
            axes[r, 1].imshow(v_img, cmap='gray'); axes[r, 1].imshow(v_gt, cmap='jet', alpha=0.5 if v_gt.max() else 0)
            axes[r, 2].imshow(v_img, cmap='gray'); axes[r, 2].imshow(v_pred, cmap='autumn', alpha=0.5 if v_pred.max() else 0)
            # Χρησιμοποιούμε bicubic interpolation για να φαίνονται πιο ομαλά τα High-Res features
            axes[r, 3].imshow(v_img, cmap='gray'); axes[r, 3].imshow(v_cam, cmap='hot', alpha=0.5, interpolation='bicubic')
            if r==0:
                axes[0, 0].set_title("MRI"); axes[0, 1].set_title("Ground Truth"); axes[0, 2].set_title("Pred Seg"); axes[0, 3].set_title("High-Res Attention")

        for ax in axes.flatten(): ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout(); plt.show()

    del model_interp, wrapped_model, cam
    gc.collect(); torch.cuda.empty_cache()

print("\n✅ Visualization Completed.")

# Αρχιτεκτονικα Κολπα


In [ ]:
# %% Βήμα 8 (LAYER EVOLUTION - ROBUST FIX): Σύγκριση 4 Μοντέλων σε 4 Βάθη
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.visualize import GradCAM
import os
import gc
import types

# --- 1. Helper: Robust Layer Finder (The Fix) ---
def get_layer_name_by_stage(model_instance, stage_idx):
    """
    Βρίσκει το τελευταίο block για το stage_idx (0-3).
    Διαχειρίζεται τη δομή 'layers' (list) και 'layers1', 'layers2'... (attributes).
    """
    target_name = None
    backbone = None

    # Εντοπισμός Backbone
    if hasattr(model_instance, 'swin_unetr'): backbone = model_instance.swin_unetr
    elif hasattr(model_instance, 'swin_unetr_extractor'): backbone = model_instance.swin_unetr_extractor

    if backbone:
        target_block = None

        # Προσπάθεια 1: Λίστα 'layers' (Standard implementation)
        try:
            target_block = backbone.swinViT.layers[stage_idx].blocks[-1]
        except:
            pass

        # Προσπάθεια 2: Attributes 'layers1', 'layers2'... (MONAI specific)
        if target_block is None:
            try:
                # Τα ονόματα είναι 1-based: layers1, layers2, layers3, layers4
                attr_name = f"layers{stage_idx + 1}"
                layer_module = getattr(backbone.swinViT, attr_name)
                # Συνήθως είναι ModuleList, παίρνουμε το 0 -> BasicLayer -> blocks -> last
                target_block = layer_module[0].blocks[-1]
            except:
                pass

        # Αν βρήκαμε το block αντικείμενο, ψάχνουμε το όνομά του
        if target_block is not None:
            for name, module in model_instance.named_modules():
                if module is target_block:
                    target_name = name
                    break

    return f"model.{target_name}" if target_name else None

# --- 2. Patch για το Cascaded Model ---
def cascaded_forward_patched(self, x_img, x_clinical):
    _ = self.swin_unetr_extractor(x_img) # Run without no_grad
    B = x_img.shape[0]
    enc_feat = F.adaptive_avg_pool3d(self.encoder_features_out, 1).flatten(1)
    dec_feat = F.adaptive_avg_pool3d(self.decoder_features_out, 1).flatten(1)
    enc_token = self.encoder_proj(enc_feat).unsqueeze(1)
    dec_token = self.decoder_proj(dec_feat).unsqueeze(1)
    clin_token = self.clinical_proj(x_clinical).unsqueeze(1)
    grade_token = self.grade_token.expand(B, -1, -1)
    diag_token = self.diag_token.expand(B, -1, -1)
    tokens = torch.cat([grade_token, diag_token, enc_token, dec_token, clin_token], dim=1)
    fused_tokens = self.fusion_transformer(tokens)
    return self.grade_head(fused_tokens[:, 0, :]), self.diagnosis_head(fused_tokens[:, 1, :])

# --- 3. Wrapper ---
class ModelWrapperForGradCAM(nn.Module):
    def __init__(self, model, clinical_data):
        super().__init__()
        self.model = model
        self.clinical_data = clinical_data
    def forward(self, x_img):
        outputs = self.model(x_img, self.clinical_data)
        if isinstance(outputs, tuple):
            if len(outputs) == 3: return outputs[1]
            elif len(outputs) == 2: return outputs[0]
        return outputs

# --- 4. Ρυθμίσεις Πειραμάτων ---
experiments = [
    {
        "name": "1. Cascaded Transformer (Frozen)",
        "model_class": CascadedTransformerClassifier,
        "path": os.path.join(runs_base_dir, "classification_cascaded_v4_transformer_weighted", "best_model.pt"),
        "kwargs": {"d_model": 64, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.5},
        "needs_patch": True
    },
    {
        "name": "2. Parallel MTL (Unfrozen)",
        "model_class": MTLTransformerModel,
        "path": os.path.join(runs_base_dir, "mtl_unfrozen_transformer_final", "best_mtl_model.pt"),
        "kwargs": {"d_model": 128, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.4},
        "needs_patch": False
    },
    {
        "name": "3. Sequential MTL (Hierarchical)",
        "model_class": SequentialMTLModel,
        "path": os.path.join(runs_base_dir, "mtl_sequential_final", "best_seq_model.pt"),
        "kwargs": {"d_model": 128, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.4},
        "needs_patch": False
    },
    {
        "name": "4. Sequential MTL (Mask-Guided)",
        "model_class": MaskGuidedMTLModel,
        "path": os.path.join(runs_base_dir, "mtl_mask_guided_final", "best_mask_model.pt"),
        "kwargs": {"d_model": 128, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.4},
        "needs_patch": False
    }
]

# --- 5. Εύρεση Δείγματος ---
print("🔍 Αναζήτηση δείγματος αναφοράς...")
sample_ref = None
for batch in test_loader_mtl:
    lbl = batch[label_key]
    for i in range(len(lbl)):
        if lbl[i].sum() > 2000: # Big tumor
            sample_ref = {
                'img': batch[image_key][i:i+1].to(DEVICE),
                'clin': batch[clinical_key][i:i+1].to(DEVICE),
                'mask': batch[label_key][i:i+1].numpy(),
                'grade': batch[grade_key][i].item()
            }
            break
    if sample_ref: break

if not sample_ref:
    print("⚠️ Δεν βρέθηκε κατάλληλο δείγμα.")
else:
    vol = sample_ref['img'][0, 0].cpu().numpy()
    mask_gt = sample_ref['mask'][0, 0]
    if mask_gt.sum() > 0: slice_idx = np.argmax(np.sum(mask_gt, axis=(0, 1)))
    else: slice_idx = vol.shape[2] // 2

    print(f"✅ Δείγμα: Grade {GRADE_CLASSES[sample_ref['grade']]}")

    # --- 6. Main Loop ---
    for exp in experiments:
        print(f"\n{'='*60}")
        print(f"🔄 Μοντέλο: {exp['name']}")
        print(f"{'='*60}")

        if not os.path.exists(exp['path']):
            print(f"❌ Path not found: {exp['path']}"); continue

        try:
            model_interp = exp['model_class'](
                in_channels_img=IN_CHANNELS, out_channels_seg=OUT_CHANNELS_SEG,
                feature_size=FEATURE_SIZE, use_checkpoint=False,
                num_classes_grade=NUM_CLASSES_GRADE, num_classes_diag=NUM_CLASSES_DIAG,
                clinical_feature_size=CLINICAL_FEATURE_SIZE,
                finetuned_seg_model_path=None, **exp['kwargs']
            ).to(DEVICE)

            state_dict = torch.load(exp['path'], map_location=DEVICE)
            model_interp.load_state_dict(state_dict, strict=False)

            if exp.get('needs_patch', False):
                model_interp.forward = types.MethodType(cascaded_forward_patched, model_interp)
            for param in model_interp.parameters(): param.requires_grad = True
            model_interp.eval()

        except Exception as e: print(f"Load Error: {e}"); continue

        # B. Plot Setup
        plt.figure(figsize=(20, 4))
        plt.suptitle(f"{exp['name']} - Attention Evolution", fontsize=14)

        # MRI & GT
        plt.subplot(1, 6, 1); plt.imshow(np.rot90(vol[:, :, slice_idx]), cmap='gray'); plt.title("MRI"); plt.axis('off')
        plt.subplot(1, 6, 2); plt.imshow(np.rot90(vol[:, :, slice_idx]), cmap='gray'); plt.imshow(np.rot90(mask_gt[:, :, slice_idx]), cmap='jet', alpha=0.5); plt.title("GT"); plt.axis('off')

        # Stages Loop
        wrapped_model = ModelWrapperForGradCAM(model_interp, sample_ref['clin'])
        cam = None # Init to avoid NameError

        for i, stage in enumerate([0, 1, 2, 3]):
            target_layer = get_layer_name_by_stage(model_interp, stage)

            if target_layer is None:
                print(f"  ⚠️ Stage {stage} not found in model structure.")
                continue

            # print(f"  Computing Stage {stage} ({target_layer})...")

            try:
                cam = GradCAM(nn_module=wrapped_model, target_layers=target_layer)
                heatmap_res = cam(x=sample_ref['img'], class_idx=sample_ref['grade'])
                heatmap = heatmap_res[0, 0].cpu().detach().numpy()

                # Cleanup
                heatmap = heatmap * (vol > 0.01)

                plt.subplot(1, 6, i + 3)
                plt.imshow(np.rot90(vol[:, :, slice_idx]), cmap='gray')
                plt.imshow(np.rot90(heatmap[:, :, slice_idx]), cmap='hot', alpha=0.6, interpolation='bicubic')
                plt.title(f"Stage {stage}")
                plt.axis('off')

            except Exception as e:
                print(f"  ❌ Error at Stage {stage}: {e}")

        plt.tight_layout()
        plt.show()

        # Clean
        del model_interp, wrapped_model, cam
        gc.collect(); torch.cuda.empty_cache()

print("\n✅ Ολοκληρώθηκε.")

In [ ]:
# %% Βήμα 8 (MULTI-SAMPLE EVOLUTION): Εξέλιξη Προσοχής σε 3 Δείγματα
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.visualize import GradCAM
import os
import gc
import types

# --- 1. Helpers & Wrappers ---
def get_layer_name_by_stage(model_instance, stage_idx):
    target_name = None
    backbone = None
    if hasattr(model_instance, 'swin_unetr'): backbone = model_instance.swin_unetr
    elif hasattr(model_instance, 'swin_unetr_extractor'): backbone = model_instance.swin_unetr_extractor

    if backbone:
        target_block = None
        try:
            # Λίστα 'layers'
            target_block = backbone.swinViT.layers[stage_idx].blocks[-1]
        except: pass

        if target_block is None:
            try:
                # Attributes 'layers1' etc.
                attr_name = f"layers{stage_idx + 1}"
                layer_module = getattr(backbone.swinViT, attr_name)
                target_block = layer_module[0].blocks[-1]
            except: pass

        if target_block is not None:
            for name, module in model_instance.named_modules():
                if module is target_block:
                    target_name = name; break

    return f"model.{target_name}" if target_name else None

def cascaded_forward_patched(self, x_img, x_clinical):
    _ = self.swin_unetr_extractor(x_img)
    B = x_img.shape[0]
    enc_feat = F.adaptive_avg_pool3d(self.encoder_features_out, 1).flatten(1)
    dec_feat = F.adaptive_avg_pool3d(self.decoder_features_out, 1).flatten(1)
    enc_token = self.encoder_proj(enc_feat).unsqueeze(1)
    dec_token = self.decoder_proj(dec_feat).unsqueeze(1)
    clin_token = self.clinical_proj(x_clinical).unsqueeze(1)
    grade_token = self.grade_token.expand(B, -1, -1)
    diag_token = self.diag_token.expand(B, -1, -1)
    tokens = torch.cat([grade_token, diag_token, enc_token, dec_token, clin_token], dim=1)
    fused_tokens = self.fusion_transformer(tokens)
    return self.grade_head(fused_tokens[:, 0, :]), self.diagnosis_head(fused_tokens[:, 1, :])

class ModelWrapperForGradCAM(nn.Module):
    def __init__(self, model, clinical_data):
        super().__init__()
        self.model = model
        self.clinical_data = clinical_data
    def forward(self, x_img):
        outputs = self.model(x_img, self.clinical_data)
        if isinstance(outputs, tuple):
            if len(outputs) == 3: return outputs[1]
            elif len(outputs) == 2: return outputs[0]
        return outputs

# --- 2. Ρυθμίσεις Πειραμάτων ---
experiments = [
    {
        "name": "1. Cascaded Transformer (Frozen)",
        "model_class": CascadedTransformerClassifier,
        "path": os.path.join(runs_base_dir, "classification_cascaded_v4_transformer_weighted", "best_model.pt"),
        "kwargs": {"d_model": 64, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.5},
        "needs_patch": True
    },
    {
        "name": "2. Parallel MTL (Unfrozen)",
        "model_class": MTLTransformerModel,
        "path": os.path.join(runs_base_dir, "mtl_unfrozen_transformer_final", "best_mtl_model.pt"),
        "kwargs": {"d_model": 128, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.4},
        "needs_patch": False
    },
    {
        "name": "3. Sequential MTL (Hierarchical)",
        "model_class": SequentialMTLModel,
        "path": os.path.join(runs_base_dir, "mtl_sequential_final", "best_seq_model.pt"),
        "kwargs": {"d_model": 128, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.4},
        "needs_patch": False
    },
    {
        "name": "4. Sequential MTL (Mask-Guided)",
        "model_class": MaskGuidedMTLModel,
        "path": os.path.join(runs_base_dir, "mtl_mask_guided_final", "best_mask_model.pt"),
        "kwargs": {"d_model": 128, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.4},
        "needs_patch": False
    }
]

# --- 3. Εύρεση 3 Δειγμάτων (SAMPLES) ---
print("🔍 Αναζήτηση 3 δειγμάτων με όγκο...")
samples_list = []
samples_needed = 3

for batch in test_loader_mtl:
    lbl = batch[label_key]
    for i in range(len(lbl)):
        if lbl[i].sum() > 2000: # Big tumor check
            samples_list.append({
                'img': batch[image_key][i:i+1].to(DEVICE),
                'clin': batch[clinical_key][i:i+1].to(DEVICE),
                'mask': batch[label_key][i:i+1].numpy(),
                'grade': batch[grade_key][i].item(),
                'id': i
            })
            if len(samples_list) >= samples_needed: break
    if len(samples_list) >= samples_needed: break

print(f"✅ Βρέθηκαν {len(samples_list)} δείγματα.")

# --- 4. Main Loop (Per Model) ---
for exp in experiments:
    print(f"\n{'='*60}")
    print(f"🔄 Μοντέλο: {exp['name']}")
    print(f"{'='*60}")

    if not os.path.exists(exp['path']): print(f"❌ Path not found: {exp['path']}"); continue

    try:
        model_interp = exp['model_class'](
            in_channels_img=IN_CHANNELS, out_channels_seg=OUT_CHANNELS_SEG,
            feature_size=FEATURE_SIZE, use_checkpoint=False,
            num_classes_grade=NUM_CLASSES_GRADE, num_classes_diag=NUM_CLASSES_DIAG,
            clinical_feature_size=CLINICAL_FEATURE_SIZE,
            finetuned_seg_model_path=None, **exp['kwargs']
        ).to(DEVICE)

        state_dict = torch.load(exp['path'], map_location=DEVICE)
        model_interp.load_state_dict(state_dict, strict=False)

        if exp.get('needs_patch', False):
            model_interp.forward = types.MethodType(cascaded_forward_patched, model_interp)
        for param in model_interp.parameters(): param.requires_grad = True
        model_interp.eval()

    except Exception as e: print(f"Load Error: {e}"); continue

    # --- Plot Setup: 3 Rows (Samples) x 6 Cols (MRI+GT+4Stages) ---
    fig, axes = plt.subplots(3, 6, figsize=(20, 12))
    fig.suptitle(f"{exp['name']} - Layer Evolution across Samples", fontsize=16, fontweight='bold')

    # Loop over Samples (Rows)
    for s_idx, sample in enumerate(samples_list):
        vol = sample['img'][0, 0].cpu().detach().numpy()
        mask_gt = sample['mask'][0, 0]

        # Best Slice for this sample
        if mask_gt.sum() > 0: slice_idx = np.argmax(np.sum(mask_gt, axis=(0, 1)))
        else: slice_idx = vol.shape[2] // 2

        # Col 1: MRI
        axes[s_idx, 0].imshow(np.rot90(vol[:, :, slice_idx]), cmap='gray')
        axes[s_idx, 0].set_ylabel(f"Sample {s_idx+1}\nGrade {GRADE_CLASSES[sample['grade']]}", fontsize=12, fontweight='bold')
        if s_idx == 0: axes[s_idx, 0].set_title("MRI Input")
        axes[s_idx, 0].set_xticks([]); axes[s_idx, 0].set_yticks([])

        # Col 2: GT
        axes[s_idx, 1].imshow(np.rot90(vol[:, :, slice_idx]), cmap='gray')
        axes[s_idx, 1].imshow(np.rot90(mask_gt[:, :, slice_idx]), cmap='jet', alpha=0.5)
        if s_idx == 0: axes[s_idx, 1].set_title("Ground Truth")
        axes[s_idx, 1].axis('off')

        # Col 3-6: Stages 0, 1, 2, 3
        wrapped_model = ModelWrapperForGradCAM(model_interp, sample['clin'])

        for i, stage in enumerate([0, 1, 2, 3]):
            target_layer = get_layer_name_by_stage(model_interp, stage)
            if target_layer is None: continue

            try:
                cam = GradCAM(nn_module=wrapped_model, target_layers=target_layer)
                heatmap_res = cam(x=sample['img'], class_idx=sample['grade'])
                heatmap = heatmap_res[0, 0].cpu().detach().numpy()

                # Cleanup Background
                heatmap = heatmap * (vol > 0.01)

                ax = axes[s_idx, i + 2]
                ax.imshow(np.rot90(vol[:, :, slice_idx]), cmap='gray')
                ax.imshow(np.rot90(heatmap[:, :, slice_idx]), cmap='hot', alpha=0.6, interpolation='bicubic')

                if s_idx == 0:
                    titles = ["Stage 0 (High)", "Stage 1 (Mid)", "Stage 2 (Texture)", "Stage 3 (Low)"]
                    ax.set_title(titles[i])
                ax.axis('off')

            except Exception as e:
                print(f"Err S{s_idx} Stage{stage}: {e}")

    plt.tight_layout()
    plt.show()

    # Clean
    del model_interp, wrapped_model, cam
    gc.collect(); torch.cuda.empty_cache()

print("\n✅ Ολοκληρώθηκε η ανάλυση για όλα τα δείγματα.")

In [ ]:
# %% Βήμα 9 (FINAL): Attention Matrix για τα 3 Δείγματα x 4 Μοντέλα
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
import numpy as np
import os

# --- 1. Helper: Extract Attention ---
def extract_attention_weights(model, x_img, x_clinical, layer_idx=0):
    """
    Υπολογίζει τα attention weights του Fusion Transformer.
    """
    model.eval()
    B = x_img.shape[0]

    with torch.no_grad():
        # Feature Extraction Logic
        if hasattr(model, 'swin_unetr_extractor'): # Cascaded
            # Τρέχουμε τον extractor (με patch αν χρειάζεται, εδώ το κάνουμε manual)
            _ = model.swin_unetr_extractor(x_img)
            enc_feat = F.adaptive_avg_pool3d(model.encoder_features_out, 1).flatten(1)
            dec_feat = F.adaptive_avg_pool3d(model.decoder_features_out, 1).flatten(1)

            enc_token = model.encoder_proj(enc_feat).unsqueeze(1)
            dec_token = model.decoder_proj(dec_feat).unsqueeze(1)
            clin_token = model.clinical_proj(x_clinical).unsqueeze(1)

            grade_token = model.grade_token.expand(B, -1, -1)
            diag_token = model.diag_token.expand(B, -1, -1)

            # Σειρά: [Grade, Diag, Enc, Dec, Clin]
            tokens = torch.cat([grade_token, diag_token, enc_token, dec_token, clin_token], dim=1)
            labels = ["Grade", "Diag", "Img(Enc)", "Img(Dec)", "Clin"]

        else: # MTL / Sequential / Masked
            # Για Masked, πρέπει να αναπαράγουμε τη λογική μασκαρίσματος
            if "MaskGuided" in str(type(model)):
                logits_seg = model.swin_unetr(x_img)
                seg_prob = torch.sigmoid(logits_seg)
                mask = torch.max(seg_prob, dim=1, keepdim=True)[0]
                mask_resized = F.interpolate(mask, size=model.encoder_features_out.shape[2:], mode='nearest')
                masked_feats = model.encoder_features_out * mask_resized
                enc_feat = F.adaptive_avg_pool3d(masked_feats, 1).flatten(1)
            else:
                _ = model.swin_unetr(x_img) # Trigger hooks
                enc_feat = F.adaptive_avg_pool3d(model.encoder_features_out, 1).flatten(1)

            enc_token = model.encoder_proj(enc_feat).unsqueeze(1)
            clin_token = model.clinical_proj(x_clinical).unsqueeze(1)

            grade_token = model.grade_token.expand(B, -1, -1)
            diag_token = model.diag_token.expand(B, -1, -1)

            # Σειρά: [Grade, Diag, Image, Clinical]
            tokens = torch.cat([grade_token, diag_token, enc_token, clin_token], dim=1)
            labels = ["Grade", "Diag", "Image", "Clin"]

        # Access Transformer Layer
        attn_layer = model.fusion_transformer.layers[layer_idx].self_attn

        # Manual Attention
        d_model = tokens.shape[2]
        qkv = F.linear(tokens, attn_layer.in_proj_weight, attn_layer.in_proj_bias)
        qkv = qkv.reshape(B, tokens.shape[1], 3, d_model).permute(2, 0, 1, 3)
        q, k, v = qkv[0], qkv[1], qkv[2]

        # Q * K^T / sqrt(d)
        scores = torch.matmul(q, k.transpose(-2, -1)) / (d_model ** 0.5)
        attn_map = F.softmax(scores, dim=-1)

        return attn_map[0].cpu().numpy(), labels

# --- 2. Πειράματα ---
experiments = [
    {
        "name": "1. Cascaded (Frozen)",
        "model_class": CascadedTransformerClassifier,
        "path": os.path.join(runs_base_dir, "classification_cascaded_v4_transformer_weighted", "best_model.pt"),
        "kwargs": {"d_model": 64, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.5},
        "needs_patch": True
    },
    {
        "name": "2. Parallel MTL (Unfrozen)",
        "model_class": MTLTransformerModel,
        "path": os.path.join(runs_base_dir, "mtl_unfrozen_transformer_final", "best_mtl_model.pt"),
        "kwargs": {"d_model": 128, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.4},
        "needs_patch": False
    },
    {
        "name": "3. Sequential MTL",
        "model_class": SequentialMTLModel,
        "path": os.path.join(runs_base_dir, "mtl_sequential_final", "best_seq_model.pt"),
        "kwargs": {"d_model": 128, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.4},
        "needs_patch": False
    },
    {
        "name": "4. Mask-Guided MTL",
        "model_class": MaskGuidedMTLModel,
        "path": os.path.join(runs_base_dir, "mtl_mask_guided_final", "best_mask_model.pt"),
        "kwargs": {"d_model": 128, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.4},
        "needs_patch": False
    }
]

# --- 3. Main Plot Loop ---
# Χρησιμοποιούμε τη λίστα 'samples_list' από το προηγούμενο κελί (Κελί 8)
if 'samples_list' not in globals() or len(samples_list) == 0:
    print("⚠️ Δεν βρέθηκε η λίστα 'samples_list'. Τρέξτε πρώτα το Κελί 8.")
else:
    print(f"📊 Visualization Attention για {len(samples_list)} δείγματα...")

    # 3 Samples x 4 Models Grid
    fig, axes = plt.subplots(len(samples_list), 4, figsize=(24, 16))
    plt.suptitle("Fusion Transformer Attention Maps (Who listens to whom?)", fontsize=20, fontweight='bold', y=0.95)

    # Load Models Once (Optimization)
    loaded_models = []
    for exp in experiments:
        if os.path.exists(exp['path']):
            try:
                m = exp['model_class'](
                    in_channels_img=IN_CHANNELS, out_channels_seg=OUT_CHANNELS_SEG,
                    feature_size=FEATURE_SIZE, use_checkpoint=False,
                    num_classes_grade=NUM_CLASSES_GRADE, num_classes_diag=NUM_CLASSES_DIAG,
                    clinical_feature_size=CLINICAL_FEATURE_SIZE,
                    finetuned_seg_model_path=None, **exp['kwargs']
                ).to(DEVICE)
                m.load_state_dict(torch.load(exp['path'], map_location=DEVICE), strict=False)

                # Manual Patch for Cascaded Feature Extraction hook
                if exp.get('needs_patch', False):
                    # We need to register the hook manually since we bypass forward
                    def hook_enc(module, input, output): m.encoder_features_out = output
                    def hook_dec(module, input, output): m.decoder_features_out = output
                    # Find correct layers to hook (usually bottlenecks)
                    m.swin_unetr_extractor.swinViT.register_forward_hook(hook_enc)
                    m.swin_unetr_extractor.encoder10.register_forward_hook(hook_dec)

                m.eval()
                loaded_models.append((exp['name'], m))
            except Exception as e:
                print(f"Failed to load {exp['name']}: {e}")
                loaded_models.append((exp['name'], None))
        else:
            loaded_models.append((exp['name'], None))

    # Loop Rows (Samples)
    for s_idx, sample in enumerate(samples_list):
        img_tensor = sample['img']
        clin_tensor = sample['clin']
        true_grade = sample['grade']

        # Loop Cols (Models)
        for m_idx, (model_name, model) in enumerate(loaded_models):
            ax = axes[s_idx, m_idx]

            if model is None:
                ax.text(0.5, 0.5, "Model Not Found", ha='center'); ax.axis('off'); continue

            try:
                # Get Attention
                attn_map, labels = extract_attention_weights(model, img_tensor, clin_tensor, layer_idx=0)

                # Plot Heatmap
                sns.heatmap(attn_map, annot=True, fmt=".2f",
                            xticklabels=labels, yticklabels=labels,
                            cmap="Blues", cbar=False, ax=ax, annot_kws={"size": 10})

                # Formatting
                if s_idx == 0: ax.set_title(model_name, fontsize=14, fontweight='bold')
                if m_idx == 0: ax.set_ylabel(f"Sample {s_idx+1}\n(True: {GRADE_CLASSES[true_grade]})", fontsize=14, fontweight='bold')

                # Highlight "Grade" row
                # Βάζουμε ένα κόκκινο περίγραμμα στη γραμμή του Grade για έμφαση
                for i in range(len(labels)):
                    rect = plt.Rectangle((i, 0), 1, 1, fill=False, edgecolor='red', lw=2)
                    ax.add_patch(rect)

            except Exception as e:
                print(f"Error S{s_idx} M{m_idx}: {e}")
                ax.text(0.5, 0.5, "Error", ha='center')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

    # Clean
    del loaded_models
    gc.collect(); torch.cuda.empty_cache()

# Νέα ενότητα

In [ ]:
# %% Βήμα 10 (GLOBAL FULL DATASET - ROBUST): Μέσος Όρος Attention (Layer 0 & 1)
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
import numpy as np
import os
from tqdm import tqdm
from monai.data import Dataset, DataLoader

# --- 1. Helper: Extract Batch Attention (ίδιο με πριν) ---
def get_batch_attention_all_layers(model, x_img, x_clinical):
    model.eval()
    with torch.no_grad():
        B = x_img.shape[0]
        # Feature Extraction Logic
        if hasattr(model, 'swin_unetr_extractor'): # Cascaded
            _ = model.swin_unetr_extractor(x_img)
            enc_feat = F.adaptive_avg_pool3d(model.encoder_features_out, 1).flatten(1)
            dec_feat = F.adaptive_avg_pool3d(model.decoder_features_out, 1).flatten(1)
            enc_token = model.encoder_proj(enc_feat).unsqueeze(1)
            dec_token = model.decoder_proj(dec_feat).unsqueeze(1)
            clin_token = model.clinical_proj(x_clinical).unsqueeze(1)
            grade_token = model.grade_token.expand(B, -1, -1)
            diag_token = model.diag_token.expand(B, -1, -1)
            tokens = torch.cat([grade_token, diag_token, enc_token, dec_token, clin_token], dim=1)
            labels = ["Grade", "Diag", "Enc", "Dec", "Clin"]
        else: # MTL / Masked
            if "MaskGuided" in str(type(model)):
                logits_seg = model.swin_unetr(x_img)
                seg_prob = torch.sigmoid(logits_seg)
                mask = torch.max(seg_prob, dim=1, keepdim=True)[0]
                mask_resized = F.interpolate(mask, size=model.encoder_features_out.shape[2:], mode='nearest')
                masked_feats = model.encoder_features_out * mask_resized
                enc_feat = F.adaptive_avg_pool3d(masked_feats, 1).flatten(1)
            else:
                _ = model.swin_unetr(x_img)
                enc_feat = F.adaptive_avg_pool3d(model.encoder_features_out, 1).flatten(1)

            enc_token = model.encoder_proj(enc_feat).unsqueeze(1)
            clin_token = model.clinical_proj(x_clinical).unsqueeze(1)
            grade_token = model.grade_token.expand(B, -1, -1)
            diag_token = model.diag_token.expand(B, -1, -1)
            tokens = torch.cat([grade_token, diag_token, enc_token, clin_token], dim=1)
            labels = ["Grade", "Diag", "Image", "Clin"]

        # Loop Layers
        attn_maps = []
        for layer in model.fusion_transformer.layers:
            attn_layer = layer.self_attn
            d_model = tokens.shape[2]
            qkv = F.linear(tokens, attn_layer.in_proj_weight, attn_layer.in_proj_bias)
            qkv = qkv.reshape(B, tokens.shape[1], 3, d_model).permute(2, 0, 1, 3)
            q, k, v = qkv[0], qkv[1], qkv[2]
            scores = torch.matmul(q, k.transpose(-2, -1)) / (d_model ** 0.5)
            attn_map = F.softmax(scores, dim=-1)
            attn_maps.append(attn_map)
            tokens = layer(tokens)

        return attn_maps, labels

# --- 2. Setup Full Dataset (Χρησιμοποιώντας τα MTL Data) ---
print("📊 Ενοποίηση δεδομένων (Train + Val + Test)...")

# Ενώνουμε τις λίστες που υπάρχουν ήδη από το Βήμα 3
try:
    all_mtl_files = train_files_mtl + val_files_mtl + test_files_mtl
    print(f"✅ Επιτυχής ενοποίηση {len(all_mtl_files)} αρχείων.")
except NameError:
    print("❌ Δεν βρέθηκαν οι μεταβλητές *_files_mtl. Βεβαιώσου ότι έτρεξες το Βήμα 3.")
    # Stop execution here if variables are missing
    raise

# Δημιουργούμε Dataset με VAL transforms (για να μην έχουμε random cropping/noise)
full_ds = Dataset(data=all_mtl_files, transform=val_transforms_mtl)
full_loader = DataLoader(full_ds, batch_size=4, shuffle=False, num_workers=0)

# --- 3. Experiments ---
experiments = [
    {
        "name": "1. Cascaded (Frozen)",
        "model_class": CascadedTransformerClassifier,
        "path": os.path.join(runs_base_dir, "classification_cascaded_v4_transformer_weighted", "best_model.pt"),
        "kwargs": {"d_model": 64, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.5},
        "needs_patch": True
    },
    {
        "name": "2. Parallel MTL (Unfrozen)",
        "model_class": MTLTransformerModel,
        "path": os.path.join(runs_base_dir, "mtl_unfrozen_transformer_final", "best_mtl_model.pt"),
        "kwargs": {"d_model": 128, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.4},
        "needs_patch": False
    },
    {
        "name": "3. Sequential MTL",
        "model_class": SequentialMTLModel,
        "path": os.path.join(runs_base_dir, "mtl_sequential_final", "best_seq_model.pt"),
        "kwargs": {"d_model": 128, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.4},
        "needs_patch": False
    },
    {
        "name": "4. Mask-Guided MTL",
        "model_class": MaskGuidedMTLModel,
        "path": os.path.join(runs_base_dir, "mtl_mask_guided_final", "best_mask_model.pt"),
        "kwargs": {"d_model": 128, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.4},
        "needs_patch": False
    }
]

# --- 4. Global Analysis Loop ---
print("\n🔄 Υπολογισμός Global Attention...")
fig, axes = plt.subplots(2, 4, figsize=(24, 12)) # 2 Layers x 4 Models

for col_idx, exp in enumerate(experiments):
    if not os.path.exists(exp['path']): continue
    print(f"  > Processing {exp['name']}...")

    try:
        model = exp['model_class'](
            in_channels_img=IN_CHANNELS, out_channels_seg=OUT_CHANNELS_SEG,
            feature_size=FEATURE_SIZE, use_checkpoint=False,
            num_classes_grade=NUM_CLASSES_GRADE, num_classes_diag=NUM_CLASSES_DIAG,
            clinical_feature_size=CLINICAL_FEATURE_SIZE,
            finetuned_seg_model_path=None, **exp['kwargs']
        ).to(DEVICE)
        model.load_state_dict(torch.load(exp['path'], map_location=DEVICE), strict=False)

        if exp.get('needs_patch', False):
            def hook_enc(module, input, output): model.encoder_features_out = output
            def hook_dec(module, input, output): model.decoder_features_out = output
            model.swin_unetr_extractor.swinViT.register_forward_hook(hook_enc)
            model.swin_unetr_extractor.encoder10.register_forward_hook(hook_dec)

        model.eval()
    except Exception as e:
        print(f"Error loading model: {e}")
        continue

    total_attn_L0, total_attn_L1 = None, None
    count = 0
    labels = []

    for batch in tqdm(full_loader, leave=False):
        img = batch[image_key].to(DEVICE)
        clin = batch[clinical_key].to(DEVICE)

        try:
            attn_maps, lbls = get_batch_attention_all_layers(model, img, clin)
            labels = lbls

            curr_L0 = attn_maps[0].sum(dim=0).cpu()
            if total_attn_L0 is None: total_attn_L0 = curr_L0
            else: total_attn_L0 += curr_L0

            if len(attn_maps) > 1:
                curr_L1 = attn_maps[1].sum(dim=0).cpu()
                if total_attn_L1 is None: total_attn_L1 = curr_L1
                else: total_attn_L1 += curr_L1

            count += img.shape[0]
        except: pass

    if count > 0:
        # Layer 0
        mean_L0 = total_attn_L0 / count
        ax0 = axes[0, col_idx]
        sns.heatmap(mean_L0.numpy(), annot=True, fmt=".2f", xticklabels=labels, yticklabels=labels, cmap="Blues", cbar=False, ax=ax0, vmin=0, vmax=1)
        ax0.set_title(f"{exp['name']}\nLayer 0 (Mixing)", fontsize=12, fontweight='bold')
        for i in range(len(labels)): ax0.add_patch(plt.Rectangle((i, 0), 1, 1, fill=False, edgecolor='red', lw=2))

        # Layer 1
        if total_attn_L1 is not None:
            mean_L1 = total_attn_L1 / count
            ax1 = axes[1, col_idx]
            sns.heatmap(mean_L1.numpy(), annot=True, fmt=".2f", xticklabels=labels, yticklabels=labels, cmap="Purples", cbar=False, ax=ax1, vmin=0, vmax=1)
            ax1.set_title("Layer 1 (Decision)", fontsize=12, fontweight='bold')
            for i in range(len(labels)): ax1.add_patch(plt.Rectangle((i, 0), 1, 1, fill=False, edgecolor='red', lw=2))

plt.tight_layout()
plt.show()
print("✅ Ολοκληρώθηκε.")

In [ ]:
# %% Βήμα 11 : Global Attention με Intercept Hook
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
import numpy as np
import os
from tqdm import tqdm
from monai.data import Dataset, DataLoader
import types

# --- 1. THE INTERCEPTOR: Πιάνει τα tokens στην πόρτα του Transformer ---
class TransformerInputCatcher:
    def __init__(self, model):
        self.tokens = None
        # Βάζουμε hook ακριβώς πριν μπει η πληροφορία στον Transformer
        self.hook = model.fusion_transformer.register_forward_pre_hook(self.save_input)

    def save_input(self, module, inputs):
        # Τα inputs είναι tuple (tokens,), παίρνουμε το πρώτο στοιχείο
        self.tokens = inputs[0]

    def remove(self):
        self.hook.remove()

# --- 2. Patch για το Cascaded (ώστε να τρέχει σωστά το forward) ---
def cascaded_forward_patched(self, x_img, x_clinical):
    # Παρακάμπτουμε το no_grad και τρέχουμε κανονικά
    if hasattr(self, 'swin_unetr_extractor'):
        _ = self.swin_unetr_extractor(x_img)

    # Αναπαραγωγή της λογικής του Cascaded για να φτάσουμε στον Transformer
    B = x_img.shape[0]
    enc_feat = F.adaptive_avg_pool3d(self.encoder_features_out, 1).flatten(1)
    dec_feat = F.adaptive_avg_pool3d(self.decoder_features_out, 1).flatten(1)

    enc_token = self.encoder_proj(enc_feat).unsqueeze(1)
    dec_token = self.decoder_proj(dec_feat).unsqueeze(1)
    clin_token = self.clinical_proj(x_clinical).unsqueeze(1)
    grade_token = self.grade_token.expand(B, -1, -1)
    diag_token = self.diag_token.expand(B, -1, -1)

    # Αυτό καλεί το fusion_transformer, όπου θα πιάσουμε τα tokens!
    tokens = torch.cat([grade_token, diag_token, enc_token, dec_token, clin_token], dim=1)
    fused_tokens = self.fusion_transformer(tokens)
    return self.grade_head(fused_tokens[:, 0, :]), self.diagnosis_head(fused_tokens[:, 1, :])

# --- 3. Compute Attention from Captured Tokens ---
def get_attention_from_catcher(model, catcher):
    # Παίρνουμε τα tokens που έπιασε το hook
    tokens = catcher.tokens # Shape: (B, N, D)
    B, N, D = tokens.shape

    # Καθορισμός Labels με βάση τον αριθμό των tokens
    if N == 5: labels = ["Grade", "Diag", "Enc", "Dec", "Clin"] # Cascaded
    elif N == 4: labels = ["Grade", "Diag", "Image", "Clin"]    # MTL / Masked
    else: labels = [str(i) for i in range(N)]

    attn_maps = []

    # Loop στα Layers του Transformer
    with torch.no_grad():
        curr_tokens = tokens.clone()

        for layer in model.fusion_transformer.layers:
            attn_layer = layer.self_attn

            # Manual Attention Math
            qkv = F.linear(curr_tokens, attn_layer.in_proj_weight, attn_layer.in_proj_bias)
            qkv = qkv.reshape(B, N, 3, D).permute(2, 0, 1, 3)
            q, k, v = qkv[0], qkv[1], qkv[2]

            scores = torch.matmul(q, k.transpose(-2, -1)) / (D ** 0.5)
            attn_map = F.softmax(scores, dim=-1) # (B, N, N)

            attn_maps.append(attn_map)

            # Προχωράμε τα tokens στο επόμενο layer (απλοποιημένα)
            curr_tokens = layer(curr_tokens)

    return attn_maps, labels

# --- 4. Setup Data ---
try:
    all_files = train_files_mtl + val_files_mtl + test_files_mtl
    # Χρησιμοποιούμε μικρό dataset για να τρέξει γρήγορα το demo (π.χ. 50 δείγματα)
    # Αφαίρεσε το [:50] για να τρέξει σε ΟΛΑ
    full_ds = Dataset(data=all_files, transform=val_transforms_mtl)
    full_loader = DataLoader(full_ds, batch_size=4, shuffle=False, num_workers=0)
    print(f"✅ Dataset: {len(full_ds)} samples ready.")
except:
    print("⚠️ Δεν βρέθηκαν τα file lists. Τρέξτε το Βήμα 3.")

# --- 5. Experiments ---
experiments = [
    {
        "name": "1. Cascaded (Frozen)",
        "model_class": CascadedTransformerClassifier,
        "path": os.path.join(runs_base_dir, "classification_cascaded_v4_transformer_weighted", "best_model.pt"),
        "kwargs": {"d_model": 64, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.5},
        "needs_patch": True
    },
    {
        "name": "2. Parallel MTL (Unfrozen)",
        "model_class": MTLTransformerModel,
        "path": os.path.join(runs_base_dir, "mtl_unfrozen_transformer_final", "best_mtl_model.pt"),
        "kwargs": {"d_model": 128, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.4},
        "needs_patch": False
    },
    {
        "name": "3. Sequential MTL",
        "model_class": SequentialMTLModel,
        "path": os.path.join(runs_base_dir, "mtl_sequential_final", "best_seq_model.pt"),
        "kwargs": {"d_model": 128, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.4},
        "needs_patch": False
    },
    {
        "name": "4. Mask-Guided MTL",
        "model_class": MaskGuidedMTLModel,
        "path": os.path.join(runs_base_dir, "mtl_mask_guided_final", "best_mask_model.pt"),
        "kwargs": {"d_model": 128, "n_heads": 4, "num_layers": 2, "dropout_rate": 0.4},
        "needs_patch": False
    }
]

# --- 6. Execution Loop ---
print("\n🔄 Υπολογισμός Global Attention (Method: Transformer Intercept)...")
fig, axes = plt.subplots(2, 4, figsize=(24, 12)) # Layer 0 & 1 for 4 Models

for col_idx, exp in enumerate(experiments):
    if not os.path.exists(exp['path']): continue
    print(f"  > Processing {exp['name']}...")

    try:
        # Load Model
        model = exp['model_class'](
            in_channels_img=IN_CHANNELS, out_channels_seg=OUT_CHANNELS_SEG,
            feature_size=FEATURE_SIZE, use_checkpoint=False,
            num_classes_grade=NUM_CLASSES_GRADE, num_classes_diag=NUM_CLASSES_DIAG,
            clinical_feature_size=CLINICAL_FEATURE_SIZE,
            finetuned_seg_model_path=None, **exp['kwargs']
        ).to(DEVICE)
        state_dict = torch.load(exp['path'], map_location=DEVICE)
        model.load_state_dict(state_dict, strict=False)

        # Patch Cascaded features hooks needed for forward pass reconstruction
        if exp['needs_patch']:
            def h_enc(m, i, o): model.encoder_features_out = o
            def h_dec(m, i, o): model.decoder_features_out = o
            # Βεβαιωνόμαστε ότι τα hooks μπαίνουν εκεί που πρέπει
            model.swin_unetr_extractor.swinViT.register_forward_hook(h_enc)
            # Δοκιμάζουμε να πιάσουμε το decoder output από το encoder10 (bottleneck projection)
            # Αν δεν υπάρχει, θα φτιαχτεί dummy στο patched_forward
            if hasattr(model.swin_unetr_extractor, 'encoder10'):
                model.swin_unetr_extractor.encoder10.register_forward_hook(h_dec)

            # Αντικατάσταση forward method
            model.forward = types.MethodType(cascaded_forward_patched, model)

        model.eval()

        # Ενεργοποίηση Interceptor Hook
        catcher = TransformerInputCatcher(model)

    except Exception as e:
        print(f"Init Error {exp['name']}: {e}"); continue

    # Accumulate
    total_L0, total_L1 = None, None
    count = 0
    labels = []

    with torch.no_grad():
        for batch in tqdm(full_loader, leave=False):
            img = batch[image_key].to(DEVICE)
            clin = batch[clinical_key].to(DEVICE)

            try:
                # 1. Run Forward (This triggers the Catcher Hook)
                _ = model(img, clin)

                # 2. Get Attention from Caught Tokens
                if catcher.tokens is not None:
                    maps, lbls = get_attention_from_catcher(model, catcher)
                    labels = lbls

                    # Accumulate Layer 0
                    curr_L0 = maps[0].sum(dim=0).cpu()
                    if total_L0 is None: total_L0 = curr_L0
                    else: total_L0 += curr_L0

                    # Accumulate Layer 1
                    if len(maps) > 1:
                        curr_L1 = maps[1].sum(dim=0).cpu()
                        if total_L1 is None: total_L1 = curr_L1
                        else: total_L1 += curr_L1

                    count += img.shape[0]
            except Exception as e:
                # print(f"Batch Error: {e}")
                pass

    catcher.remove() # Cleanup

    # Plot
    if count > 0:
        # Layer 0
        mean_L0 = total_L0 / count
        ax0 = axes[0, col_idx]
        sns.heatmap(mean_L0.numpy(), annot=True, fmt=".2f", xticklabels=labels, yticklabels=labels, cmap="Blues", cbar=False, ax=ax0, vmin=0, vmax=1)
        ax0.set_title(f"{exp['name']}\nLayer 0 (Mixing)", fontsize=11, fontweight='bold')
        for i in range(len(labels)): ax0.add_patch(plt.Rectangle((i, 0), 1, 1, fill=False, edgecolor='red', lw=2))

        # Layer 1
        if total_L1 is not None:
            mean_L1 = total_L1 / count
            ax1 = axes[1, col_idx]
            sns.heatmap(mean_L1.numpy(), annot=True, fmt=".2f", xticklabels=labels, yticklabels=labels, cmap="Purples", cbar=False, ax=ax1, vmin=0, vmax=1)
            ax1.set_title("Layer 1 (Decision)", fontsize=11, fontweight='bold')
            for i in range(len(labels)): ax1.add_patch(plt.Rectangle((i, 0), 1, 1, fill=False, edgecolor='red', lw=2))

plt.tight_layout()
plt.show()
print("✅ Η διαδικασία ολοκληρώθηκε επιτυχώς για ΟΛΑ τα μοντέλα.")

# Throwing shit at the wall

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.networks.nets import SwinUNETR

class AdvancedMaskGuidedModel(nn.Module):
    def __init__(self,
                 in_channels_img, out_channels_seg, feature_size,
                 num_classes_grade, num_classes_diag, clinical_feature_size,
                 d_model=128, n_heads=8, num_layers=2, dropout_rate=0.4,
                 use_checkpoint=False, finetuned_seg_model_path=None):
        super().__init__()

        # --- 1. SwinUNETR Backbone (UNFROZEN) ---
        self.swin_unetr = SwinUNETR(
            in_channels=in_channels_img,
            out_channels=out_channels_seg,
            feature_size=feature_size,
            use_checkpoint=use_checkpoint
        )

        if finetuned_seg_model_path and os.path.exists(finetuned_seg_model_path):
            print(f"Φόρτωση αρχικών βαρών από: {finetuned_seg_model_path}")
            checkpoint = torch.load(finetuned_seg_model_path, map_location=torch.device('cpu'))
            try:
                self.swin_unetr.load_state_dict(checkpoint['model_state'], strict=False)
                print("✅ Επιτυχής φόρτωση βαρών SwinUNETR.")
            except Exception as e: print(f"Warning loading weights: {e}")

        print("ℹ️ Ο Encoder/Decoder είναι UNFROZEN.")

        # 2. Hooks Setup (Το μυστικό για να μην τρέχουμε 2 φορές)
        self.enc_feat_captured = None
        self.dec_feat_captured = None

        # Hook στον Encoder (SwinViT) -> Πιάνει το τελευταίο output
        self.swin_unetr.swinViT.register_forward_hook(self._hook_encoder)

        # Hook στον Decoder -> Πιάνει το input στο τελικό layer
        self.swin_unetr.out.register_forward_hook(self._hook_decoder)

        # 3. Projections
        # Encoder (Bottleneck): 768 κανάλια (τυπικό για Swin-UNETR base)
        self.encoder_proj = nn.Linear(768, d_model)

        # Decoder: Συνήθως έχει 'feature_size' κανάλια (π.χ. 48)
        self.decoder_proj = nn.Linear(feature_size, d_model)

        # Clinical
        self.clinical_proj = nn.Linear(clinical_feature_size, d_model)

        # 4. Learnable Query Tokens
        # Τώρα έχουμε Grade και Diag tokens που θα ρωτάνε τα υπόλοιπα
        self.grade_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.diag_token = nn.Parameter(torch.zeros(1, 1, d_model))

        # 5. Fusion Transformer
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dropout=dropout_rate, batch_first=True)
        self.fusion_transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 6. Classification Heads
        self.grade_head = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(64, num_classes_grade)
        )
        self.diagnosis_head = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(64, num_classes_diag)
        )

        # 7. Auxiliary Heads (Deep Supervision)
        self.aux_head_img_enc = nn.Linear(d_model, num_classes_grade) # Από Encoder
        self.aux_head_img_dec = nn.Linear(d_model, num_classes_grade) # Από Decoder (ΝΕΟ)
        self.aux_head_clin = nn.Linear(d_model, num_classes_grade)    # Από Clinical

    def _hook_encoder(self, module, input, output):
        # Το SwinViT επιστρέφει λίστα features. Το τελευταίο είναι το bottleneck.
        if isinstance(output, (list, tuple)):
            self.enc_feat_captured = output[-1]
        else:
            self.enc_feat_captured = output

    def _hook_decoder(self, module, input, output):
        # Το input στο τελικό 1x1 conv είναι τα features του decoder
        self.dec_feat_captured = input[0]

    def forward(self, x_img, x_clinical):
        # A. Run Backbone (Segmentation)
        logits_seg = self.swin_unetr(x_img)

        # --- B. Mask Extraction ---
        seg_prob = torch.sigmoid(logits_seg)
        mask = torch.max(seg_prob, dim=1, keepdim=True)[0]

        # --- C. ENCODER TOKEN GENERATION ---
        # 1. Get raw features (Bottleneck)
        raw_enc_feats = self.swin_unetr.swinViT(x_img)[-1] # (B, 768, 3, 3, 3)
        self.encoder_features = raw_enc_feats # Save for reference

        # 2. Resize mask to Encoder size & Apply
        mask_enc = F.interpolate(mask, size=raw_enc_feats.shape[2:], mode='nearest')
        masked_enc = raw_enc_feats * mask_enc

        # 3. Pool & Project
        enc_vec = F.adaptive_avg_pool3d(masked_enc, 1).flatten(1)
        enc_token = self.encoder_proj(enc_vec).unsqueeze(1) # (B, 1, d_model)

        # --- D. DECODER TOKEN GENERATION (NEW) ---
        # 1. Get raw features (Captured by Hook)
        raw_dec_feats = self.decoder_features # (B, 48, 96, 96, 96) - High Res

        # 2. Resize mask to Decoder size (συνήθως είναι ίδια, αλλά για σιγουριά) & Apply
        # Σημείωση: Εδώ η ανάλυση είναι υψηλή, άρα το masking είναι πολύ ακριβές
        mask_dec = F.interpolate(mask, size=raw_dec_feats.shape[2:], mode='nearest')
        masked_dec = raw_dec_feats * mask_dec

        # 3. Pool & Project
        dec_vec = F.adaptive_avg_pool3d(masked_dec, 1).flatten(1)
        dec_token = self.decoder_proj(dec_vec).unsqueeze(1) # (B, 1, d_model)

        # --- E. Clinical Token ---
        clin_token = self.clinical_proj(x_clinical).unsqueeze(1)

        # --- F. Modality Dropout (Training Only) ---
        if self.training:
            # 40% πιθανότητα να σβήσουμε τα κλινικά
            if torch.rand(1).item() < 0.4:
                clin_token = torch.zeros_like(clin_token)

        # G. Fusion
        B = x_img.shape[0]
        grade_t = self.grade_token.expand(B, -1, -1)
        diag_t = self.diag_token.expand(B, -1, -1)

        # Ενώνουμε ΟΛΑ τα tokens: [Grade, Diag, Enc, Dec, Clin]
        tokens = torch.cat([grade_t, diag_t, enc_token, dec_token, clin_token], dim=1)

        fused = self.fusion_transformer(tokens)

        # H. Predictions
        pred_grade = self.grade_head(fused[:, 0, :])
        pred_diag = self.diagnosis_head(fused[:, 1, :])

        # I. Aux Predictions
        aux_enc = self.aux_head_img_enc(enc_token.squeeze(1))
        aux_dec = self.aux_head_img_dec(dec_token.squeeze(1)) # Νέο Aux Head
        aux_clin = self.aux_head_clin(clin_token.squeeze(1))

        return logits_seg, pred_grade, pred_diag, aux_enc, aux_dec, aux_clin

In [ ]:
# %% Ορισμός Advanced Mask-Guided Model (FINAL FIXED & MEMORY OPTIMIZED)
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.networks.nets import SwinUNETR

class AdvancedMaskGuidedModel(nn.Module):
    def __init__(self,
                 in_channels_img, out_channels_seg, feature_size,
                 num_classes_grade, num_classes_diag, clinical_feature_size,
                 d_model=128, n_heads=8, num_layers=2, dropout_rate=0.4,
                 use_checkpoint=True, finetuned_seg_model_path=None):
        super().__init__()

        # --- 1. SwinUNETR Backbone (UNFROZEN) ---
        self.swin_unetr = SwinUNETR(
            in_channels=in_channels_img,
            out_channels=out_channels_seg,
            feature_size=feature_size,
            use_checkpoint=use_checkpoint
        )

        if finetuned_seg_model_path and os.path.exists(finetuned_seg_model_path):
            print(f"Φόρτωση αρχικών βαρών από: {finetuned_seg_model_path}")
            checkpoint = torch.load(finetuned_seg_model_path, map_location=torch.device('cpu'))
            try:
                self.swin_unetr.load_state_dict(checkpoint['model_state'], strict=False)
                print("✅ Επιτυχής φόρτωση βαρών SwinUNETR.")
            except Exception as e: print(f"Warning loading weights: {e}")

        print("ℹ️ Ο Encoder/Decoder είναι UNFROZEN.")

        # 2. Hooks Setup (Capture Features on the fly)
        # Ορίζουμε τις μεταβλητές που έλειπαν και προκαλούσαν το AttributeError
        self.enc_feat_captured = None
        self.dec_feat_captured = None

        # Hook στον Encoder (SwinViT) -> Πιάνει το τελευταίο output
        self.swin_unetr.swinViT.register_forward_hook(self._hook_encoder)

        # Hook στον Decoder -> Πιάνει το input στο τελικό layer
        self.swin_unetr.out.register_forward_hook(self._hook_decoder)

        # 3. Projections
        # Encoder (Bottleneck): 768 κανάλια
        self.encoder_proj = nn.Linear(768, d_model)
        # Decoder: 'feature_size' κανάλια (π.χ. 48)
        self.decoder_proj = nn.Linear(feature_size, d_model)
        # Clinical
        self.clinical_proj = nn.Linear(clinical_feature_size, d_model)

        # 4. Tokens & Transformer
        self.grade_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.diag_token = nn.Parameter(torch.zeros(1, 1, d_model))

        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dropout=dropout_rate, batch_first=True)
        self.fusion_transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 5. Heads
        self.grade_head = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Dropout(dropout_rate), nn.Linear(64, num_classes_grade))
        self.diagnosis_head = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Dropout(dropout_rate), nn.Linear(64, num_classes_diag))

        # 6. Aux Heads
        self.aux_head_img_enc = nn.Linear(d_model, num_classes_grade)
        self.aux_head_img_dec = nn.Linear(d_model, num_classes_grade)
        self.aux_head_clin = nn.Linear(d_model, num_classes_grade)

    def _hook_encoder(self, module, input, output):
        # Το SwinViT επιστρέφει λίστα features. Το τελευταίο είναι το bottleneck.
        if isinstance(output, (list, tuple)):
            self.enc_feat_captured = output[-1]
        else:
            self.enc_feat_captured = output

    def _hook_decoder(self, module, input, output):
        # Το input στο τελικό 1x1 conv είναι τα features του decoder
        self.dec_feat_captured = input[0]

    def forward(self, x_img, x_clinical):
        # A. Run Backbone ONCE (Hooks will capture features automatically)
        logits_seg = self.swin_unetr(x_img)

        # Ανάκτηση features από τα hooks
        # Χρησιμοποιούμε τα captured variables
        raw_enc_feats = self.enc_feat_captured
        raw_dec_feats = self.dec_feat_captured

        # --- B. Mask Extraction ---
        seg_prob = torch.sigmoid(logits_seg)
        mask = torch.max(seg_prob, dim=1, keepdim=True)[0]

        # --- C. Encoder Processing ---
        # Resize mask to 3x3x3 (Encoder Size)
        mask_enc = F.interpolate(mask, size=raw_enc_feats.shape[2:], mode='nearest')
        masked_enc = raw_enc_feats * mask_enc
        enc_vec = F.adaptive_avg_pool3d(masked_enc, 1).flatten(1)
        enc_token = self.encoder_proj(enc_vec).unsqueeze(1)

        # --- D. Decoder Processing ---
        # Resize mask to 96x96x96 (Decoder Size)
        mask_dec = F.interpolate(mask, size=raw_dec_feats.shape[2:], mode='nearest')
        masked_dec = raw_dec_feats * mask_dec
        dec_vec = F.adaptive_avg_pool3d(masked_dec, 1).flatten(1)
        dec_token = self.decoder_proj(dec_vec).unsqueeze(1)

        # --- E. Clinical & Fusion ---
        clin_token = self.clinical_proj(x_clinical).unsqueeze(1)

        if self.training:
            # Modality Dropout (40%)
            if torch.rand(1).item() < 0.4:
                clin_token = torch.zeros_like(clin_token)

        B = x_img.shape[0]
        grade_t = self.grade_token.expand(B, -1, -1)
        diag_t = self.diag_token.expand(B, -1, -1)

        # Ενώνουμε ΟΛΑ τα tokens
        tokens = torch.cat([grade_t, diag_t, enc_token, dec_token, clin_token], dim=1)
        fused = self.fusion_transformer(tokens)

        # F. Predictions
        pred_grade = self.grade_head(fused[:, 0, :])
        pred_diag = self.diagnosis_head(fused[:, 1, :])

        aux_enc = self.aux_head_img_enc(enc_token.squeeze(1))
        aux_dec = self.aux_head_img_dec(dec_token.squeeze(1))
        aux_clin = self.aux_head_clin(clin_token.squeeze(1))

        return logits_seg, pred_grade, pred_diag, aux_enc, aux_dec, aux_clin

In [ ]:
# %% Step 5: Training Advanced Mask-Guided MTL (Accuracy, F1, AUC, Reports)
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import pandas as pd
import numpy as np
import os
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete, Activations, Compose
from monai.losses import DiceCELoss
from monai.losses import FocalLoss
from monai.data import decollate_batch
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
# --- 1. Class Weights ---
print("Computing Class Weights...")
try:
    y_grade_all = df_processed['Grade_Label'].values
    y_diag_all = df_processed['Diagnosis_Label'].values
    weights_grade = compute_class_weight('balanced', classes=np.unique(y_grade_all), y=y_grade_all)
    weights_diag = compute_class_weight('balanced', classes=np.unique(y_diag_all), y=y_diag_all)
    weights_grade_tensor = torch.tensor(weights_grade, dtype=torch.float).to(DEVICE)
    weights_diag_tensor = torch.tensor(weights_diag, dtype=torch.float).to(DEVICE)
    print("Weights loaded.")
except:
    weights_grade_tensor = None; weights_diag_tensor = None
    print("Weights could not be computed (using uniform).")

# --- 2. Loss Functions ---
loss_seg_fn = DiceCELoss(include_background=False, sigmoid=True)
#loss_cls_fn = nn.CrossEntropyLoss(weight=weights_grade_tensor)
#loss_diag_fn = nn.CrossEntropyLoss(weight=weights_diag_tensor)


loss_cls_fn = FocalLoss(include_background=True, to_onehot_y=True, gamma=2.0, weight=weights_grade_tensor)
loss_diag_fn = FocalLoss(include_background=True, to_onehot_y=True, gamma=2.0, weight=weights_diag_tensor)

def advanced_loss_fn(outputs, labels_seg, labels_grade, labels_diag):
    logits_seg, p_grade, p_diag, aux_enc, aux_dec, aux_clin = outputs

    # Main Losses
    l_seg = loss_seg_fn(logits_seg, labels_seg)
    l_grade = loss_cls_fn(p_grade, labels_grade)
    l_diag = loss_diag_fn(p_diag, labels_diag)

    # Aux Losses
    l_aux_enc = loss_cls_fn(aux_enc, labels_grade)
    l_aux_dec = loss_cls_fn(aux_dec, labels_grade)
    l_aux_clin = loss_cls_fn(aux_clin, labels_grade)

    total_loss = (
        1.0 * l_grade +
        0.5 * l_diag +
        1.0 * l_seg +
        0.3 * l_aux_enc +
        0.3 * l_aux_dec +
        0.2 * l_aux_clin
    )
    return total_loss

# --- 3. Settings ---
output_dir_mtl = os.path.join(runs_base_dir, "mtl_mask_guided_final")
history_csv_path = os.path.join(output_dir_mtl, "history_advanced_mask_detailed.csv")
best_model_path = os.path.join(output_dir_mtl, "best_mask_model.pt")
os.makedirs(output_dir_mtl, exist_ok=True)

# --- 4. Initialize Advanced Model ---
try:
    model_mtl = AdvancedMaskGuidedModel(
        in_channels_img=IN_CHANNELS, out_channels_seg=OUT_CHANNELS_SEG,
        feature_size=FEATURE_SIZE, use_checkpoint=False,
        num_classes_grade=NUM_CLASSES_GRADE, num_classes_diag=NUM_CLASSES_DIAG,
        clinical_feature_size=CLINICAL_FEATURE_SIZE,
        finetuned_seg_model_path=finetuned_model_path,
        d_model=128, n_heads=4, num_layers=2, dropout_rate=0.4
    ).to(DEVICE)
    print("✅ AdvancedMaskGuidedModel initialized.")
except NameError:
    print("❌ ERROR: AdvancedMaskGuidedModel class not found.")

# --- 5. Optimizer (Low LR for stability) ---
LR_BACKBONE = 1e-5
LR_HEADS = 2e-4

params = [
    {'params': model_mtl.swin_unetr.parameters(), 'lr': LR_BACKBONE},
    {'params': model_mtl.encoder_proj.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.decoder_proj.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.clinical_proj.parameters(), 'lr': LR_HEADS},
    {'params': [model_mtl.grade_token, model_mtl.diag_token], 'lr': LR_HEADS},
    {'params': model_mtl.fusion_transformer.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.grade_head.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.diagnosis_head.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.aux_head_img_enc.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.aux_head_img_dec.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.aux_head_clin.parameters(), 'lr': LR_HEADS},
]

optimizer = torch.optim.AdamW(params, weight_decay=0.05)


# --- RESTART STRATEGY ---
# T_0 = 20: Το 1ο restart θα γίνει στο Epoch 20.
# T_mult = 2: Το 2ο restart θα γίνει στο Epoch 20 + 40 = 60.
# Το 3ο restart θα γινόταν στο 60 + 80 = 140 (αλλά θα έχουμε τελειώσει στα 100).
# Αυτό ταιριάζει τέλεια με τα 100 Epochs μας.
scheduler = CosineAnnealingWarmRestarts(
    optimizer,
    T_0=20,
    T_mult=2,
    eta_min=1e-6 # Το ελάχιστο LR (για να μην μηδενίσει τελείως)
)

# --- 6. Metrics & Loop Setup ---
dice_metric_train = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)
dice_metric_val = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)
post_pred_seg = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
post_label_seg = Compose([AsDiscrete(to_onehot=None)])

best_val_metric = -1.0
start_epoch = 0
GRAD_ACCUM_STEPS = 8

# --- UPDATED HISTORY (Added AUC) ---
history = {
    "epoch": [],
    "train_loss": [], "train_dice": [],
    "train_f1_g": [], "train_acc_g": [], "train_auc_g": [], # AUC Added
    "train_f1_d": [], "train_acc_d": [], "train_auc_d": [], # AUC Added
    "val_loss": [], "val_dice": [],
    "val_f1_g": [], "val_acc_g": [], "val_auc_g": [],       # AUC Added
    "val_f1_d": [], "val_acc_d": [], "val_auc_d": [],       # AUC Added
    "combined_score": []
}

if os.path.exists(best_model_path): os.remove(best_model_path)

print(f"\n--- Starting Advanced Training ({NUM_EPOCHS_MTL} Epochs) ---")

for epoch in range(start_epoch, NUM_EPOCHS_MTL):
    current_epoch = epoch + 1

    # === TRAIN ===
    model_mtl.train()
    dice_metric_train.reset()
    ep_loss = 0
    step = 0

    # Storage for Epoch Metrics
    tr_pred_g, tr_probs_g, tr_lbl_g = [], [], []
    tr_pred_d, tr_probs_d, tr_lbl_d = [], [], []

    optimizer.zero_grad()
    pbar = tqdm(train_loader_mtl, desc=f"Epoch {current_epoch} Train")

    for i, batch in enumerate(pbar):
        step += 1
        img = batch[image_key].to(DEVICE)
        seg = batch[label_key].to(DEVICE)
        clin = batch[clinical_key].to(DEVICE)
        lbl_g = batch[grade_key].to(DEVICE)
        lbl_d = batch[diag_key].to(DEVICE)

        outputs = model_mtl(img, clin)
        loss = advanced_loss_fn(outputs, seg, lbl_g, lbl_d)

        if not torch.isfinite(loss): continue

        (loss / GRAD_ACCUM_STEPS).backward()

        if (i + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model_mtl.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

        ep_loss += loss.item()

        # Metrics Collection
        logits_seg, p_grade, p_diag, _, _, _ = outputs
        val_outputs = [post_pred_seg(i) for i in decollate_batch(logits_seg)]
        val_labels = [post_label_seg(i) for i in decollate_batch(seg)]
        dice_metric_train(y_pred=val_outputs, y=val_labels)

        # Collect Predictions (Argmax) and Probabilities (Softmax)
        tr_pred_g.extend(torch.argmax(p_grade, 1).cpu().numpy())
        tr_probs_g.extend(torch.softmax(p_grade, 1).detach().cpu().numpy()) # For AUC
        tr_lbl_g.extend(lbl_g.cpu().numpy())

        tr_pred_d.extend(torch.argmax(p_diag, 1).cpu().numpy())
        tr_probs_d.extend(torch.softmax(p_diag, 1).detach().cpu().numpy()) # For AUC
        tr_lbl_d.extend(lbl_d.cpu().numpy())

        pbar.set_postfix(loss=f"{loss.item():.2f}")

    # --- Train Stats Calculation ---
    train_loss_avg = ep_loss / step
    train_dice = dice_metric_train.aggregate().item()

    # Grade Metrics
    train_f1_g = f1_score(tr_lbl_g, tr_pred_g, average='macro')
    train_acc_g = accuracy_score(tr_lbl_g, tr_pred_g)
    try:
        train_auc_g = roc_auc_score(tr_lbl_g, tr_probs_g, multi_class='ovr', average='macro')
    except: train_auc_g = 0.0 # Handle case with missing classes in batch

    # Diagnosis Metrics
    train_f1_d = f1_score(tr_lbl_d, tr_pred_d, average='macro')
    train_acc_d = accuracy_score(tr_lbl_d, tr_pred_d)
    try:
        train_auc_d = roc_auc_score(tr_lbl_d, tr_probs_d, multi_class='ovr', average='macro')
    except: train_auc_d = 0.0

    # History Update
    history["epoch"].append(current_epoch)
    history["train_loss"].append(train_loss_avg)
    history["train_dice"].append(train_dice)
    history["train_f1_g"].append(train_f1_g); history["train_acc_g"].append(train_acc_g); history["train_auc_g"].append(train_auc_g)
    history["train_f1_d"].append(train_f1_d); history["train_acc_d"].append(train_acc_d); history["train_auc_d"].append(train_auc_d)

    print(f"Train | Loss: {train_loss_avg:.3f} | Dice: {train_dice:.3f} | G_F1: {train_f1_g:.3f} | G_AUC: {train_auc_g:.3f}")

    # === VALIDATION ===
    if current_epoch % VAL_INTERVAL == 0:
        model_mtl.eval()
        dice_metric_val.reset()
        val_loss_sum = 0
        val_steps = 0

        # Lists for Validation
        preds_g, probs_g, lbls_g = [], [], []
        preds_d, probs_d, lbls_d = [], [], []

        with torch.no_grad():
            for batch in tqdm(val_loader_mtl, desc="Validating"):
                val_steps += 1
                img = batch[image_key].to(DEVICE)
                seg = batch[label_key].to(DEVICE)
                clin = batch[clinical_key].to(DEVICE)
                lbl_g = batch[grade_key].to(DEVICE)
                lbl_d = batch[diag_key].to(DEVICE)

                outputs = model_mtl(img, clin)
                loss = advanced_loss_fn(outputs, seg, lbl_g, lbl_d)
                val_loss_sum += loss.item()

                logits_seg, p_grade, p_diag, aux_enc, aux_dec, aux_clin = outputs

                val_outputs = [post_pred_seg(i) for i in decollate_batch(logits_seg)]
                val_labels = [post_label_seg(i) for i in decollate_batch(seg)]
                dice_metric_val(y_pred=val_outputs, y=val_labels)

                # Grade
                preds_g.extend(torch.argmax(p_grade, 1).cpu().numpy())
                probs_g.extend(torch.softmax(p_grade, 1).cpu().numpy()) # For AUC
                lbls_g.extend(lbl_g.cpu().numpy())

                # Diagnosis
                preds_d.extend(torch.argmax(p_diag, 1).cpu().numpy())
                probs_d.extend(torch.softmax(p_diag, 1).cpu().numpy()) # For AUC
                lbls_d.extend(lbl_d.cpu().numpy())

        # --- Val Stats Calculation ---
        val_dice = dice_metric_val.aggregate().item()

        # Grade Metrics
        val_f1_g = f1_score(lbls_g, preds_g, average='macro')
        val_acc_g = accuracy_score(lbls_g, preds_g)
        try:
            val_auc_g = roc_auc_score(lbls_g, probs_g, multi_class='ovr', average='macro')
        except: val_auc_g = 0.0

        # Diag Metrics
        val_f1_d = f1_score(lbls_d, preds_d, average='macro')
        val_acc_d = accuracy_score(lbls_d, preds_d)
        try:
            val_auc_d = roc_auc_score(lbls_d, probs_d, multi_class='ovr', average='macro')
        except: val_auc_d = 0.0

        # Combined Score
        combined = (val_dice * 0.4) + (val_f1_g * 0.3) + (val_f1_d * 0.3)

        history["val_loss"].append(val_loss_sum/val_steps)
        history["val_dice"].append(val_dice)
        history["val_f1_g"].append(val_f1_g); history["val_acc_g"].append(val_acc_g); history["val_auc_g"].append(val_auc_g)
        history["val_f1_d"].append(val_f1_d); history["val_acc_d"].append(val_acc_d); history["val_auc_d"].append(val_auc_d)
        history["combined_score"].append(combined)

        print(f"--- Val Epoch {current_epoch} ---")
        print(f"Combined: {combined:.4f} | Loss: {val_loss_sum/val_steps:.4f} | Dice: {val_dice:.4f}")
        print(f"Grade -> F1: {val_f1_g:.3f} | Acc: {val_acc_g:.3f} | AUC: {val_auc_g:.3f}")
        print(f"Diag  -> F1: {val_f1_d:.3f} | Acc: {val_acc_d:.3f} | AUC: {val_auc_d:.3f}")

        # --- 📊 DETAILED REPORTING ---
        print("\n=== 🧠 TUMOR GRADE REPORT ===")
        print(classification_report(lbls_g, preds_g, target_names=[str(c) for c in GRADE_CLASSES], zero_division=0))

        print("\n=== 🧬 DIAGNOSIS REPORT ===")
        print(classification_report(lbls_d, preds_d, target_names=[str(c) for c in DIAG_CLASSES], zero_division=0))

        if combined > best_val_metric:
            best_val_metric = combined
            torch.save(model_mtl.state_dict(), best_model_path)
            print("🏆 New Best Model Saved!")

        pd.DataFrame(history).to_csv(history_csv_path, index=False)

    else:
        # Pad history
        for k in ["val_loss", "val_dice", "val_f1_g", "val_acc_g", "val_auc_g", "val_f1_d", "val_acc_d", "val_auc_d", "combined_score"]:
            history[k].append(np.nan)

    if scheduler: scheduler.step()

print(f"Done. Best Combined Score: {best_val_metric:.4f}")

### CE

In [ ]:
# %% Step 6: Visualization of Results (Advanced Full History Analysis)
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np

# --- Settings ---
# Ensure this matches the directory from Step 5
target_dir = os.path.join(runs_base_dir, "mtl_mask_guided_final")
# Ensure the CSV name matches what was saved in Step 5 (history_advanced_mask_detailed.csv)
csv_path = os.path.join(target_dir, "history_advanced_mask_detailed.csv")

print(f"🔍 Reading history file: {csv_path}")

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)

    # Separate rows that have validation data (not NaN in 'val_loss')
    # Using 'val_loss' as the anchor to find validation epochs
    df_val = df.dropna(subset=['val_loss'])

    epochs = df['epoch']
    val_epochs = df_val['epoch']

    # Plot Style
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(2, 2, figsize=(20, 12))
    fig.suptitle('Advanced Mask-Guided MTL Training Analysis', fontsize=20, weight='bold')

    # 1. LOSSES & COMBINED SCORE
    ax1 = axes[0, 0]
    if 'train_loss' in df.columns:
        ax1.plot(epochs, df['train_loss'], label='Train Loss', color='blue', alpha=0.6)
    if 'val_loss' in df_val.columns:
        ax1.plot(val_epochs, df_val['val_loss'], label='Val Loss', color='navy', linewidth=2, marker='o')

    # Combined Score (Target)
    if 'combined_score' in df_val.columns:
        ax1b = ax1.twinx()
        ax1b.plot(val_epochs, df_val['combined_score'], label='Combined Score', color='gold', linestyle='--', linewidth=2)
        ax1b.legend(loc='upper right')
        ax1b.set_ylabel('Score')

    ax1.set_title('Optimization Landscape', fontsize=14)
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.legend(loc='upper left')
    ax1.grid(True, linestyle='--', alpha=0.7)

    # 2. SEGMENTATION (DICE)
    ax2 = axes[0, 1]
    if 'train_dice' in df.columns:
        ax2.plot(epochs, df['train_dice'], label='Train Dice', color='gray', linestyle=':', alpha=0.8)
    if 'val_dice' in df_val.columns:
        ax2.plot(val_epochs, df_val['val_dice'], label='Val Dice', color='black', linewidth=2, marker='s')

    ax2.set_title('Task 1: Segmentation (Dice Score)', fontsize=14)
    ax2.set_xlabel('Epoch'); ax2.set_ylim(0, 1.0)
    ax2.legend(loc='lower right'); ax2.grid(True)

    # 3. GRADE CLASSIFICATION
    ax3 = axes[1, 0]
    if 'train_f1_g' in df.columns:
        ax3.plot(epochs, df['train_f1_g'], label='Train F1', color='lightgreen', linestyle='--')
    if 'val_f1_g' in df_val.columns:
        ax3.plot(val_epochs, df_val['val_f1_g'], label='Val F1', color='forestgreen', linewidth=2.5)

    # Accuracy (if available)
    if 'val_acc_g' in df_val.columns:
        ax3.plot(val_epochs, df_val['val_acc_g'], label='Val Acc', color='darkgreen', alpha=0.3, linestyle=':')

    ax3.set_title('Task 2: Tumor Grade Classification', fontsize=14)
    ax3.set_xlabel('Epoch'); ax3.set_ylim(0, 1.0)
    ax3.legend(loc='lower right'); ax3.grid(True)

    # 4. DIAGNOSIS CLASSIFICATION
    ax4 = axes[1, 1]
    has_diag = False

    # Check if Diagnosis columns exist
    if 'train_f1_d' in df.columns and not df['train_f1_d'].isnull().all():
        ax4.plot(epochs, df['train_f1_d'], label='Train F1', color='plum', linestyle='--')
        has_diag = True

    if 'val_f1_d' in df_val.columns and not df_val['val_f1_d'].isnull().all():
        ax4.plot(val_epochs, df_val['val_f1_d'], label='Val F1', color='darkmagenta', linewidth=2.5)
        has_diag = True

    # Accuracy (if available)
    if 'val_acc_d' in df_val.columns and not df_val['val_acc_d'].isnull().all():
        ax4.plot(val_epochs, df_val['val_acc_d'], label='Val Acc', color='indigo', alpha=0.6)

    ax4.set_title('Task 3: Diagnosis Classification', fontsize=14)
    ax4.set_xlabel('Epoch'); ax4.set_ylim(0, 1.0)
    if has_diag: ax4.legend(loc='lower right')
    ax4.grid(True)

    plt.tight_layout()
    plt.show()

    # --- Print Best Model Stats ---
    if 'combined_score' in df_val.columns:
        # Find the row with the max combined score
        best_idx = df_val['combined_score'].idxmax()
        best_row = df_val.loc[best_idx]

        print(f"\n🏆 BEST MODEL at Epoch {int(best_row['epoch'])}")
        print(f"Combined Score: {best_row['combined_score']:.4f}")

        if 'val_dice' in best_row:
            print(f"Seg Dice: {best_row['val_dice']:.4f}")

        if 'val_f1_g' in best_row:
            acc_str = f" (Acc: {best_row['val_acc_g']:.4f})" if 'val_acc_g' in best_row else ""
            print(f"Grade F1: {best_row['val_f1_g']:.4f}{acc_str}")

        # Robust check for Diagnosis
        if 'val_f1_d' in best_row and not pd.isna(best_row['val_f1_d']):
            acc_str = f" (Acc: {best_row['val_acc_d']:.4f})" if 'val_acc_d' in best_row else ""
            print(f"Diag F1:  {best_row['val_f1_d']:.4f}{acc_str}")
        else:
            print("Diag F1:  N/A (Column missing or NaN)")
else:
    print(f"❌ Error: CSV File not found at {csv_path}. Please run Step 5 training first.")

In [ ]:
# %% Βήμα 7 (FINAL TEST - FIXED): Full Evaluation (Metrics + Attention)
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
import numpy as np
import os
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# --- 1. Transformer Hook (Interceptor) ---
class TransformerInputCatcher:
    def __init__(self, model):
        self.tokens = None
        self.hook = model.fusion_transformer.register_forward_pre_hook(self.save_input)

    def save_input(self, module, inputs):
        self.tokens = inputs[0]

    def remove(self):
        self.hook.remove()

# --- 2. Attention Helper ---
def get_attention_maps(model, catcher):
    tokens = catcher.tokens
    B, N, D = tokens.shape
    attn_maps = []
    with torch.no_grad():
        curr_tokens = tokens.clone()
        for layer in model.fusion_transformer.layers:
            attn_layer = layer.self_attn
            qkv = F.linear(curr_tokens, attn_layer.in_proj_weight, attn_layer.in_proj_bias)
            qkv = qkv.reshape(B, N, 3, D).permute(2, 0, 1, 3)
            q, k, v = qkv[0], qkv[1], qkv[2]
            scores = torch.matmul(q, k.transpose(-2, -1)) / (D ** 0.5)
            attn_maps.append(F.softmax(scores, dim=-1))
            curr_tokens = layer(curr_tokens)
    return attn_maps

# --- 3. Load Best Model ---
model_path = os.path.join(runs_base_dir, "mtl_mask_guided_final", "best_mask_model.pt")
print(f"🔍 Φόρτωση Best Model: {model_path}")

try:
    model_test = AdvancedMaskGuidedModel(
        in_channels_img=IN_CHANNELS, out_channels_seg=OUT_CHANNELS_SEG,
        feature_size=FEATURE_SIZE, use_checkpoint=False,
        num_classes_grade=NUM_CLASSES_GRADE, num_classes_diag=NUM_CLASSES_DIAG,
        clinical_feature_size=CLINICAL_FEATURE_SIZE,
        finetuned_seg_model_path=None,
        d_model=128, n_heads=4, num_layers=2, dropout_rate=0.4
    ).to(DEVICE)
    model_test.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model_test.eval()
    print("✅ Μοντέλο έτοιμο.")
except Exception as e:
    print(f"❌ Σφάλμα: {e}"); raise e

# --- 4. Run Evaluation on TEST SET ---
print("\n🧪 Εκτέλεση Τελικού Ελέγχου στο TEST SET...")

catcher = TransformerInputCatcher(model_test)
total_L0, total_L1 = None, None
count = 0

all_preds_g, all_lbls_g = [], []
all_preds_d, all_lbls_d = [], []

with torch.no_grad():
    for batch in tqdm(test_loader_mtl, desc="Evaluating"):
        img = batch[image_key].to(DEVICE)
        clin = batch[clinical_key].to(DEVICE)
        lbl_g = batch[grade_key].to(DEVICE)
        lbl_d = batch[diag_key].to(DEVICE)

        outputs = model_test(img, clin)
        p_grade = outputs[1]
        p_diag = outputs[2]

        all_preds_g.extend(torch.argmax(p_grade, 1).cpu().numpy())
        all_lbls_g.extend(lbl_g.cpu().numpy())

        all_preds_d.extend(torch.argmax(p_diag, 1).cpu().numpy())
        all_lbls_d.extend(lbl_d.cpu().numpy())

        if catcher.tokens is not None:
            maps = get_attention_maps(model_test, catcher)

            curr_L0 = maps[0].sum(dim=0).cpu()
            if total_L0 is None: total_L0 = curr_L0
            else: total_L0 += curr_L0

            curr_L1 = maps[1].sum(dim=0).cpu()
            if total_L1 is None: total_L1 = curr_L1
            else: total_L1 += curr_L1

            count += img.shape[0]

catcher.remove()

# --- 5. Print Classification Reports (ROBUST FIX) ---
print("\n" + "="*60)
print("📊 FINAL TEST RESULTS")
print("="*60)

# Grade Report
print("\n--- 🧠 TUMOR GRADE REPORT ---")
# Ορίζουμε ρητά τα labels για να μην μπερδεύεται αν λείπει κάποια κλάση
labels_g = list(range(len(GRADE_CLASSES)))
print(classification_report(all_lbls_g, all_preds_g, labels=labels_g, target_names=[str(c) for c in GRADE_CLASSES], zero_division=0))
print("Confusion Matrix (Grade):")
print(confusion_matrix(all_lbls_g, all_preds_g))

# Diagnosis Report
print("\n--- 🧬 DIAGNOSIS REPORT ---")
labels_d = list(range(len(DIAG_CLASSES)))
print(classification_report(all_lbls_d, all_preds_d, labels=labels_d, target_names=[str(c) for c in DIAG_CLASSES], zero_division=0))
print("Confusion Matrix (Diagnosis):")
print(confusion_matrix(all_lbls_d, all_preds_d))

# --- 6. Plot Attention ---
if count > 0:
    mean_L0 = total_L0 / count
    mean_L1 = total_L1 / count
    labels = ["Grade", "Diag", "Enc (Img)", "Dec (Shp)", "Clin"]

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    fig.suptitle(f"Global Average Attention on TEST SET (N={count})", fontsize=18, fontweight='bold')

    sns.heatmap(mean_L0.numpy(), annot=True, fmt=".2f", xticklabels=labels, yticklabels=labels, cmap="Blues", cbar=False, ax=axes[0], vmin=0, vmax=1)
    axes[0].set_title("Layer 0 (Mixing)", fontsize=14)
    for i in range(len(labels)): axes[0].add_patch(plt.Rectangle((i, 0), 1, 1, fill=False, edgecolor='red', lw=3))

    sns.heatmap(mean_L1.numpy(), annot=True, fmt=".2f", xticklabels=labels, yticklabels=labels, cmap="Purples", cbar=False, ax=axes[1], vmin=0, vmax=1)
    axes[1].set_title("Layer 1 (Final Decision)", fontsize=14)
    for i in range(len(labels)): axes[1].add_patch(plt.Rectangle((i, 0), 1, 1, fill=False, edgecolor='red', lw=3))

    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Δεν βρέθηκαν δεδομένα Attention.")

In [ ]:
# %% Βήμα 7 (FINAL TEST - FIXED): Full Evaluation (Metrics + Attention)
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
import numpy as np
import os
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# --- 1. Transformer Hook (Interceptor) ---
class TransformerInputCatcher:
    def __init__(self, model):
        self.tokens = None
        self.hook = model.fusion_transformer.register_forward_pre_hook(self.save_input)

    def save_input(self, module, inputs):
        self.tokens = inputs[0]

    def remove(self):
        self.hook.remove()

# --- 2. Attention Helper ---
def get_attention_maps(model, catcher):
    tokens = catcher.tokens
    B, N, D = tokens.shape
    attn_maps = []
    with torch.no_grad():
        curr_tokens = tokens.clone()
        for layer in model.fusion_transformer.layers:
            attn_layer = layer.self_attn
            qkv = F.linear(curr_tokens, attn_layer.in_proj_weight, attn_layer.in_proj_bias)
            qkv = qkv.reshape(B, N, 3, D).permute(2, 0, 1, 3)
            q, k, v = qkv[0], qkv[1], qkv[2]
            scores = torch.matmul(q, k.transpose(-2, -1)) / (D ** 0.5)
            attn_maps.append(F.softmax(scores, dim=-1))
            curr_tokens = layer(curr_tokens)
    return attn_maps

# --- 3. Load Best Model ---
model_path = os.path.join(runs_base_dir, "mtl_mask_guided_final", "best_mask_model.pt")
print(f"🔍 Φόρτωση Best Model: {model_path}")

try:
    model_test = AdvancedMaskGuidedModel(
        in_channels_img=IN_CHANNELS, out_channels_seg=OUT_CHANNELS_SEG,
        feature_size=FEATURE_SIZE, use_checkpoint=False,
        num_classes_grade=NUM_CLASSES_GRADE, num_classes_diag=NUM_CLASSES_DIAG,
        clinical_feature_size=CLINICAL_FEATURE_SIZE,
        finetuned_seg_model_path=None,
        d_model=128, n_heads=4, num_layers=2, dropout_rate=0.4
    ).to(DEVICE)
    model_test.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model_test.eval()
    print("✅ Μοντέλο έτοιμο.")
except Exception as e:
    print(f"❌ Σφάλμα: {e}"); raise e

# --- 4. Run Evaluation on TEST SET ---
print("\n🧪 Εκτέλεση Τελικού Ελέγχου στο TEST SET...")

catcher = TransformerInputCatcher(model_test)
total_L0, total_L1 = None, None
count = 0

all_preds_g, all_lbls_g = [], []
all_preds_d, all_lbls_d = [], []

with torch.no_grad():
    for batch in tqdm(val_loader_mtl, desc="Evaluating"):
        img = batch[image_key].to(DEVICE)
        clin = batch[clinical_key].to(DEVICE)
        lbl_g = batch[grade_key].to(DEVICE)
        lbl_d = batch[diag_key].to(DEVICE)

        outputs = model_test(img, clin)
        p_grade = outputs[1]
        p_diag = outputs[2]

        all_preds_g.extend(torch.argmax(p_grade, 1).cpu().numpy())
        all_lbls_g.extend(lbl_g.cpu().numpy())

        all_preds_d.extend(torch.argmax(p_diag, 1).cpu().numpy())
        all_lbls_d.extend(lbl_d.cpu().numpy())

        if catcher.tokens is not None:
            maps = get_attention_maps(model_test, catcher)

            curr_L0 = maps[0].sum(dim=0).cpu()
            if total_L0 is None: total_L0 = curr_L0
            else: total_L0 += curr_L0

            curr_L1 = maps[1].sum(dim=0).cpu()
            if total_L1 is None: total_L1 = curr_L1
            else: total_L1 += curr_L1

            count += img.shape[0]

catcher.remove()

# --- 5. Print Classification Reports (ROBUST FIX) ---
print("\n" + "="*60)
print("📊 FINAL TEST RESULTS")
print("="*60)

# Grade Report
print("\n--- 🧠 TUMOR GRADE REPORT ---")
# Ορίζουμε ρητά τα labels για να μην μπερδεύεται αν λείπει κάποια κλάση
labels_g = list(range(len(GRADE_CLASSES)))
print(classification_report(all_lbls_g, all_preds_g, labels=labels_g, target_names=[str(c) for c in GRADE_CLASSES], zero_division=0))
print("Confusion Matrix (Grade):")
print(confusion_matrix(all_lbls_g, all_preds_g))

# Diagnosis Report
print("\n--- 🧬 DIAGNOSIS REPORT ---")
labels_d = list(range(len(DIAG_CLASSES)))
print(classification_report(all_lbls_d, all_preds_d, labels=labels_d, target_names=[str(c) for c in DIAG_CLASSES], zero_division=0))
print("Confusion Matrix (Diagnosis):")
print(confusion_matrix(all_lbls_d, all_preds_d))

# --- 6. Plot Attention ---
if count > 0:
    mean_L0 = total_L0 / count
    mean_L1 = total_L1 / count
    labels = ["Grade", "Diag", "Enc (Img)", "Dec (Shp)", "Clin"]

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    fig.suptitle(f"Global Average Attention on TEST SET (N={count})", fontsize=18, fontweight='bold')

    sns.heatmap(mean_L0.numpy(), annot=True, fmt=".2f", xticklabels=labels, yticklabels=labels, cmap="Blues", cbar=False, ax=axes[0], vmin=0, vmax=1)
    axes[0].set_title("Layer 0 (Mixing)", fontsize=14)
    for i in range(len(labels)): axes[0].add_patch(plt.Rectangle((i, 0), 1, 1, fill=False, edgecolor='red', lw=3))

    sns.heatmap(mean_L1.numpy(), annot=True, fmt=".2f", xticklabels=labels, yticklabels=labels, cmap="Purples", cbar=False, ax=axes[1], vmin=0, vmax=1)
    axes[1].set_title("Layer 1 (Final Decision)", fontsize=14)
    for i in range(len(labels)): axes[1].add_patch(plt.Rectangle((i, 0), 1, 1, fill=False, edgecolor='red', lw=3))

    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Δεν βρέθηκαν δεδομένα Attention.")

In [ ]:
# %% Βήμα 7 (ALL-IN-ONE): Dice, Classification, Attention & Visuals
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
import numpy as np
import os
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete, Activations, Compose
from monai.data import decollate_batch

# --- 1. Hooks & Helpers ---
class TransformerInputCatcher:
    def __init__(self, model):
        self.tokens = None
        self.hook = model.fusion_transformer.register_forward_pre_hook(self.save_input)
    def save_input(self, module, inputs): self.tokens = inputs[0]
    def remove(self): self.hook.remove()

def get_attention_maps(model, catcher):
    tokens = catcher.tokens
    B, N, D = tokens.shape
    attn_maps = []
    with torch.no_grad():
        curr_tokens = tokens.clone()
        for layer in model.fusion_transformer.layers:
            attn_layer = layer.self_attn
            qkv = F.linear(curr_tokens, attn_layer.in_proj_weight, attn_layer.in_proj_bias)
            qkv = qkv.reshape(B, N, 3, D).permute(2, 0, 1, 3)
            q, k, v = qkv[0], qkv[1], qkv[2]
            scores = torch.matmul(q, k.transpose(-2, -1)) / (D ** 0.5)
            attn_maps.append(F.softmax(scores, dim=-1))
            curr_tokens = layer(curr_tokens)
    return attn_maps

# --- 2. Load Model ---
model_path = os.path.join(runs_base_dir, "mtl_mask_guided_final", "best_mask_model.pt")
print(f"🔍 Φόρτωση Best Model: {model_path}")

try:
    model_eval = AdvancedMaskGuidedModel(
        in_channels_img=IN_CHANNELS, out_channels_seg=OUT_CHANNELS_SEG,
        feature_size=FEATURE_SIZE, use_checkpoint=False,
        num_classes_grade=NUM_CLASSES_GRADE, num_classes_diag=NUM_CLASSES_DIAG,
        clinical_feature_size=CLINICAL_FEATURE_SIZE,
        finetuned_seg_model_path=None,
        d_model=128, n_heads=4, num_layers=2, dropout_rate=0.4
    ).to(DEVICE)
    model_eval.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model_eval.eval()
except Exception as e: print(f"❌ Error: {e}"); raise e

# --- 3. Setup Metrics ---
# Dice Metric Setup
dice_metric = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)
post_pred = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
post_label = Compose([AsDiscrete(to_onehot=None)])

# --- 4. Run Evaluation (Combined Val + Test) ---
print("\n🧪 Εκτέλεση Πλήρους Ελέγχου (Val + Test)...")

catcher = TransformerInputCatcher(model_eval)
total_L0, total_L1 = None, None
count = 0

all_preds_g, all_lbls_g = [], []
all_preds_d, all_lbls_d = [], []

# Για αποθήκευση δειγμάτων για visualization
sample_imgs = []
sample_masks_true = []
sample_masks_pred = []

# Λίστα loaders (μπορείς να βάλεις μόνο test_loader_mtl αν θες)
loaders = [val_loader_mtl, test_loader_mtl]

with torch.no_grad():
    for loader in loaders:
        for batch in tqdm(loader, leave=False):
            img = batch[image_key].to(DEVICE)
            clin = batch[clinical_key].to(DEVICE)
            lbl_g = batch[grade_key].to(DEVICE)
            lbl_d = batch[diag_key].to(DEVICE)
            seg_true = batch[label_key].to(DEVICE)

            # Forward
            outputs = model_eval(img, clin)
            logits_seg, p_grade, p_diag = outputs[0], outputs[1], outputs[2]

            # --- A. Segmentation Dice ---
            val_outputs = [post_pred(i) for i in decollate_batch(logits_seg)]
            val_labels = [post_label(i) for i in decollate_batch(seg_true)]
            dice_metric(y_pred=val_outputs, y=val_labels)

            # Save visuals (middle slice) from first batch only
            if len(sample_imgs) < 3:
                mid_slice = img.shape[4] // 2
                sample_imgs.append(img[0, 0, :, :, mid_slice].cpu().numpy())
                sample_masks_true.append(seg_true[0, 0, :, :, mid_slice].cpu().numpy())
                sample_masks_pred.append(val_outputs[0][0, :, :, mid_slice].cpu().numpy())

            # --- B. Classification Metrics ---
            all_preds_g.extend(torch.argmax(p_grade, 1).cpu().numpy())
            all_lbls_g.extend(lbl_g.cpu().numpy())
            all_preds_d.extend(torch.argmax(p_diag, 1).cpu().numpy())
            all_lbls_d.extend(lbl_d.cpu().numpy())

            # --- C. Attention ---
            if catcher.tokens is not None:
                maps = get_attention_maps(model_eval, catcher)
                curr_L0 = maps[0].sum(dim=0).cpu()
                curr_L1 = maps[1].sum(dim=0).cpu()

                if total_L0 is None: total_L0 = curr_L0
                else: total_L0 += curr_L0

                if total_L1 is None: total_L1 = curr_L1
                else: total_L1 += curr_L1

                count += img.shape[0]

catcher.remove()

# --- 5. REPORTING ---
print("\n" + "="*60)
print(f"📊 FINAL COMBINED RESULTS (N={count})")
print("="*60)

# A. Dice Score
final_dice = dice_metric.aggregate().item()
print(f"\n🩺 SEGMENTATION DICE SCORE: {final_dice:.4f}")
if final_dice > 0.88: print("   ✅ SOTA Level Performance!")
elif final_dice > 0.80: print("   ✅ Excellent Performance")
else: print("   ⚠️ Needs Improvement")

# B. Classification
print("\n🧠 GRADE REPORT")
labels_g = list(range(len(GRADE_CLASSES)))
print(classification_report(all_lbls_g, all_preds_g, labels=labels_g, target_names=[str(c) for c in GRADE_CLASSES], zero_division=0))

print("\n🧬 DIAGNOSIS REPORT")
labels_d = list(range(len(DIAG_CLASSES)))
print(classification_report(all_lbls_d, all_preds_d, labels=labels_d, target_names=[str(c) for c in DIAG_CLASSES], zero_division=0))

# --- 6. VISUALIZATION ---
# A. Segmentation Visuals
print("\n👁️ Segmentation Check (Samples)")
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
for i in range(3):
    if i < len(sample_imgs):
        axes[i, 0].imshow(sample_imgs[i], cmap="gray"); axes[i, 0].set_title("MRI Image")
        axes[i, 0].axis('off')
        axes[i, 1].imshow(sample_masks_true[i], cmap="jet", interpolation='nearest'); axes[i, 1].set_title("True Mask")
        axes[i, 1].axis('off')
        axes[i, 2].imshow(sample_masks_pred[i], cmap="jet", interpolation='nearest'); axes[i, 2].set_title("Predicted Mask")
        axes[i, 2].axis('off')
plt.tight_layout(); plt.show()

# B. Attention Plot
if count > 0:
    mean_L0 = total_L0 / count
    mean_L1 = total_L1 / count
    labels = ["Grade", "Diag", "Enc", "Dec", "Clin"]

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    fig.suptitle(f"Global Attention (Avg over {count} patients)", fontsize=18, fontweight='bold')

    sns.heatmap(mean_L0.numpy(), annot=True, fmt=".2f", xticklabels=labels, yticklabels=labels, cmap="Blues", cbar=False, ax=axes[0], vmin=0, vmax=1)
    axes[0].set_title("Layer 0 (Mixing)", fontsize=14)
    for i in range(len(labels)): axes[0].add_patch(plt.Rectangle((i, 0), 1, 1, fill=False, edgecolor='red', lw=3))

    sns.heatmap(mean_L1.numpy(), annot=True, fmt=".2f", xticklabels=labels, yticklabels=labels, cmap="Purples", cbar=False, ax=axes[1], vmin=0, vmax=1)
    axes[1].set_title("Layer 1 (Decision)", fontsize=14)
    for i in range(len(labels)): axes[1].add_patch(plt.Rectangle((i, 0), 1, 1, fill=False, edgecolor='red', lw=3))

    plt.tight_layout()
    plt.show()

In [ ]:
# %% Βήμα 7 (FINAL TEST - FIXED): Full Evaluation (Metrics + Attention)
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
import numpy as np
import os
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# --- 1. Transformer Hook (Interceptor) ---
class TransformerInputCatcher:
    def __init__(self, model):
        self.tokens = None
        self.hook = model.fusion_transformer.register_forward_pre_hook(self.save_input)

    def save_input(self, module, inputs):
        self.tokens = inputs[0]

    def remove(self):
        self.hook.remove()

# --- 2. Attention Helper ---
def get_attention_maps(model, catcher):
    tokens = catcher.tokens
    B, N, D = tokens.shape
    attn_maps = []
    with torch.no_grad():
        curr_tokens = tokens.clone()
        for layer in model.fusion_transformer.layers:
            attn_layer = layer.self_attn
            qkv = F.linear(curr_tokens, attn_layer.in_proj_weight, attn_layer.in_proj_bias)
            qkv = qkv.reshape(B, N, 3, D).permute(2, 0, 1, 3)
            q, k, v = qkv[0], qkv[1], qkv[2]
            scores = torch.matmul(q, k.transpose(-2, -1)) / (D ** 0.5)
            attn_maps.append(F.softmax(scores, dim=-1))
            curr_tokens = layer(curr_tokens)
    return attn_maps

# --- 3. Load Best Model ---
model_path = os.path.join(runs_base_dir, "mtl_mask_guided_final", "best_mask_model.pt")
print(f"🔍 Φόρτωση Best Model: {model_path}")

try:
    model_test = AdvancedMaskGuidedModel(
        in_channels_img=IN_CHANNELS, out_channels_seg=OUT_CHANNELS_SEG,
        feature_size=FEATURE_SIZE, use_checkpoint=False,
        num_classes_grade=NUM_CLASSES_GRADE, num_classes_diag=NUM_CLASSES_DIAG,
        clinical_feature_size=CLINICAL_FEATURE_SIZE,
        finetuned_seg_model_path=None,
        d_model=128, n_heads=4, num_layers=2, dropout_rate=0.4
    ).to(DEVICE)
    model_test.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model_test.eval()
    print("✅ Μοντέλο έτοιμο.")
except Exception as e:
    print(f"❌ Σφάλμα: {e}"); raise e

# --- 4. Run Evaluation on TEST SET ---
print("\n🧪 Εκτέλεση Τελικού Ελέγχου στο TEST SET...")

catcher = TransformerInputCatcher(model_test)
total_L0, total_L1 = None, None
count = 0

all_preds_g, all_lbls_g = [], []
all_preds_d, all_lbls_d = [], []

with torch.no_grad():
    for batch in tqdm(train_loader_mtl, desc="Evaluating"):
        img = batch[image_key].to(DEVICE)
        clin = batch[clinical_key].to(DEVICE)
        lbl_g = batch[grade_key].to(DEVICE)
        lbl_d = batch[diag_key].to(DEVICE)

        outputs = model_test(img, clin)
        p_grade = outputs[1]
        p_diag = outputs[2]

        all_preds_g.extend(torch.argmax(p_grade, 1).cpu().numpy())
        all_lbls_g.extend(lbl_g.cpu().numpy())

        all_preds_d.extend(torch.argmax(p_diag, 1).cpu().numpy())
        all_lbls_d.extend(lbl_d.cpu().numpy())

        if catcher.tokens is not None:
            maps = get_attention_maps(model_test, catcher)

            curr_L0 = maps[0].sum(dim=0).cpu()
            if total_L0 is None: total_L0 = curr_L0
            else: total_L0 += curr_L0

            curr_L1 = maps[1].sum(dim=0).cpu()
            if total_L1 is None: total_L1 = curr_L1
            else: total_L1 += curr_L1

            count += img.shape[0]

catcher.remove()

# --- 5. Print Classification Reports (ROBUST FIX) ---
print("\n" + "="*60)
print("📊 FINAL TEST RESULTS")
print("="*60)

# Grade Report
print("\n--- 🧠 TUMOR GRADE REPORT ---")
# Ορίζουμε ρητά τα labels για να μην μπερδεύεται αν λείπει κάποια κλάση
labels_g = list(range(len(GRADE_CLASSES)))
print(classification_report(all_lbls_g, all_preds_g, labels=labels_g, target_names=[str(c) for c in GRADE_CLASSES], zero_division=0))
print("Confusion Matrix (Grade):")
print(confusion_matrix(all_lbls_g, all_preds_g))

# Diagnosis Report
print("\n--- 🧬 DIAGNOSIS REPORT ---")
labels_d = list(range(len(DIAG_CLASSES)))
print(classification_report(all_lbls_d, all_preds_d, labels=labels_d, target_names=[str(c) for c in DIAG_CLASSES], zero_division=0))
print("Confusion Matrix (Diagnosis):")
print(confusion_matrix(all_lbls_d, all_preds_d))

# --- 6. Plot Attention ---
if count > 0:
    mean_L0 = total_L0 / count
    mean_L1 = total_L1 / count
    labels = ["Grade", "Diag", "Enc (Img)", "Dec (Shp)", "Clin"]

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    fig.suptitle(f"Global Average Attention on TEST SET (N={count})", fontsize=18, fontweight='bold')

    sns.heatmap(mean_L0.numpy(), annot=True, fmt=".2f", xticklabels=labels, yticklabels=labels, cmap="Blues", cbar=False, ax=axes[0], vmin=0, vmax=1)
    axes[0].set_title("Layer 0 (Mixing)", fontsize=14)
    for i in range(len(labels)): axes[0].add_patch(plt.Rectangle((i, 0), 1, 1, fill=False, edgecolor='red', lw=3))

    sns.heatmap(mean_L1.numpy(), annot=True, fmt=".2f", xticklabels=labels, yticklabels=labels, cmap="Purples", cbar=False, ax=axes[1], vmin=0, vmax=1)
    axes[1].set_title("Layer 1 (Final Decision)", fontsize=14)
    for i in range(len(labels)): axes[1].add_patch(plt.Rectangle((i, 0), 1, 1, fill=False, edgecolor='red', lw=3))

    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Δεν βρέθηκαν δεδομένα Attention.")

In [ ]:
del model_mtl
torch.cuda.empty_cache()
import gc
gc.collect()

## Focal

In [ ]:
# %% Step 6: Visualization of Results (Advanced Full History Analysis)
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np

# --- Settings ---
# Ensure this matches the directory from Step 5
target_dir = os.path.join(runs_base_dir, "mtl_mask_guided_final")
# Ensure the CSV name matches what was saved in Step 5 (history_advanced_mask_detailed.csv)
csv_path = os.path.join(target_dir, "history_advanced_mask_detailed.csv")

print(f"🔍 Reading history file: {csv_path}")

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)

    # Separate rows that have validation data (not NaN in 'val_loss')
    # Using 'val_loss' as the anchor to find validation epochs
    df_val = df.dropna(subset=['val_loss'])

    epochs = df['epoch']
    val_epochs = df_val['epoch']

    # Plot Style
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(2, 2, figsize=(20, 12))
    fig.suptitle('Advanced Mask-Guided MTL Training Analysis', fontsize=20, weight='bold')

    # 1. LOSSES & COMBINED SCORE
    ax1 = axes[0, 0]
    if 'train_loss' in df.columns:
        ax1.plot(epochs, df['train_loss'], label='Train Loss', color='blue', alpha=0.6)
    if 'val_loss' in df_val.columns:
        ax1.plot(val_epochs, df_val['val_loss'], label='Val Loss', color='navy', linewidth=2, marker='o')

    # Combined Score (Target)
    if 'combined_score' in df_val.columns:
        ax1b = ax1.twinx()
        ax1b.plot(val_epochs, df_val['combined_score'], label='Combined Score', color='gold', linestyle='--', linewidth=2)
        ax1b.legend(loc='upper right')
        ax1b.set_ylabel('Score')

    ax1.set_title('Optimization Landscape', fontsize=14)
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.legend(loc='upper left')
    ax1.grid(True, linestyle='--', alpha=0.7)

    # 2. SEGMENTATION (DICE)
    ax2 = axes[0, 1]
    if 'train_dice' in df.columns:
        ax2.plot(epochs, df['train_dice'], label='Train Dice', color='gray', linestyle=':', alpha=0.8)
    if 'val_dice' in df_val.columns:
        ax2.plot(val_epochs, df_val['val_dice'], label='Val Dice', color='black', linewidth=2, marker='s')

    ax2.set_title('Task 1: Segmentation (Dice Score)', fontsize=14)
    ax2.set_xlabel('Epoch'); ax2.set_ylim(0, 1.0)
    ax2.legend(loc='lower right'); ax2.grid(True)

    # 3. GRADE CLASSIFICATION
    ax3 = axes[1, 0]
    if 'train_f1_g' in df.columns:
        ax3.plot(epochs, df['train_f1_g'], label='Train F1', color='lightgreen', linestyle='--')
    if 'val_f1_g' in df_val.columns:
        ax3.plot(val_epochs, df_val['val_f1_g'], label='Val F1', color='forestgreen', linewidth=2.5)

    # Accuracy (if available)
    if 'val_acc_g' in df_val.columns:
        ax3.plot(val_epochs, df_val['val_acc_g'], label='Val Acc', color='darkgreen', alpha=0.3, linestyle=':')

    ax3.set_title('Task 2: Tumor Grade Classification', fontsize=14)
    ax3.set_xlabel('Epoch'); ax3.set_ylim(0, 1.0)
    ax3.legend(loc='lower right'); ax3.grid(True)

    # 4. DIAGNOSIS CLASSIFICATION
    ax4 = axes[1, 1]
    has_diag = False

    # Check if Diagnosis columns exist
    if 'train_f1_d' in df.columns and not df['train_f1_d'].isnull().all():
        ax4.plot(epochs, df['train_f1_d'], label='Train F1', color='plum', linestyle='--')
        has_diag = True

    if 'val_f1_d' in df_val.columns and not df_val['val_f1_d'].isnull().all():
        ax4.plot(val_epochs, df_val['val_f1_d'], label='Val F1', color='darkmagenta', linewidth=2.5)
        has_diag = True

    # Accuracy (if available)
    if 'val_acc_d' in df_val.columns and not df_val['val_acc_d'].isnull().all():
        ax4.plot(val_epochs, df_val['val_acc_d'], label='Val Acc', color='indigo', alpha=0.6)

    ax4.set_title('Task 3: Diagnosis Classification', fontsize=14)
    ax4.set_xlabel('Epoch'); ax4.set_ylim(0, 1.0)
    if has_diag: ax4.legend(loc='lower right')
    ax4.grid(True)

    plt.tight_layout()
    plt.show()

    # --- Print Best Model Stats ---
    if 'combined_score' in df_val.columns:
        # Find the row with the max combined score
        best_idx = df_val['combined_score'].idxmax()
        best_row = df_val.loc[best_idx]

        print(f"\n🏆 BEST MODEL at Epoch {int(best_row['epoch'])}")
        print(f"Combined Score: {best_row['combined_score']:.4f}")

        if 'val_dice' in best_row:
            print(f"Seg Dice: {best_row['val_dice']:.4f}")

        if 'val_f1_g' in best_row:
            acc_str = f" (Acc: {best_row['val_acc_g']:.4f})" if 'val_acc_g' in best_row else ""
            print(f"Grade F1: {best_row['val_f1_g']:.4f}{acc_str}")

        # Robust check for Diagnosis
        if 'val_f1_d' in best_row and not pd.isna(best_row['val_f1_d']):
            acc_str = f" (Acc: {best_row['val_acc_d']:.4f})" if 'val_acc_d' in best_row else ""
            print(f"Diag F1:  {best_row['val_f1_d']:.4f}{acc_str}")
        else:
            print("Diag F1:  N/A (Column missing or NaN)")
else:
    print(f"❌ Error: CSV File not found at {csv_path}. Please run Step 5 training first.")

In [ ]:
# %% Βήμα 7 (FINAL TEST - FIXED): Full Evaluation (Metrics + Attention)
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
import numpy as np
import os
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# --- 1. Transformer Hook (Interceptor) ---
class TransformerInputCatcher:
    def __init__(self, model):
        self.tokens = None
        self.hook = model.fusion_transformer.register_forward_pre_hook(self.save_input)

    def save_input(self, module, inputs):
        self.tokens = inputs[0]

    def remove(self):
        self.hook.remove()

# --- 2. Attention Helper ---
def get_attention_maps(model, catcher):
    tokens = catcher.tokens
    B, N, D = tokens.shape
    attn_maps = []
    with torch.no_grad():
        curr_tokens = tokens.clone()
        for layer in model.fusion_transformer.layers:
            attn_layer = layer.self_attn
            qkv = F.linear(curr_tokens, attn_layer.in_proj_weight, attn_layer.in_proj_bias)
            qkv = qkv.reshape(B, N, 3, D).permute(2, 0, 1, 3)
            q, k, v = qkv[0], qkv[1], qkv[2]
            scores = torch.matmul(q, k.transpose(-2, -1)) / (D ** 0.5)
            attn_maps.append(F.softmax(scores, dim=-1))
            curr_tokens = layer(curr_tokens)
    return attn_maps

# --- 3. Load Best Model ---
model_path = os.path.join(runs_base_dir, "mtl_mask_guided_final", "best_mask_model.pt")
print(f"🔍 Φόρτωση Best Model: {model_path}")

try:
    model_test = AdvancedMaskGuidedModel(
        in_channels_img=IN_CHANNELS, out_channels_seg=OUT_CHANNELS_SEG,
        feature_size=FEATURE_SIZE, use_checkpoint=False,
        num_classes_grade=NUM_CLASSES_GRADE, num_classes_diag=NUM_CLASSES_DIAG,
        clinical_feature_size=CLINICAL_FEATURE_SIZE,
        finetuned_seg_model_path=None,
        d_model=128, n_heads=4, num_layers=2, dropout_rate=0.4
    ).to(DEVICE)
    model_test.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model_test.eval()
    print("✅ Μοντέλο έτοιμο.")
except Exception as e:
    print(f"❌ Σφάλμα: {e}"); raise e

# --- 4. Run Evaluation on TEST SET ---
print("\n🧪 Εκτέλεση Τελικού Ελέγχου στο TEST SET...")

catcher = TransformerInputCatcher(model_test)
total_L0, total_L1 = None, None
count = 0

all_preds_g, all_lbls_g = [], []
all_preds_d, all_lbls_d = [], []

with torch.no_grad():
    for batch in tqdm(train_loader_mtl, desc="Evaluating"):
        img = batch[image_key].to(DEVICE)
        clin = batch[clinical_key].to(DEVICE)
        lbl_g = batch[grade_key].to(DEVICE)
        lbl_d = batch[diag_key].to(DEVICE)

        outputs = model_test(img, clin)
        p_grade = outputs[1]
        p_diag = outputs[2]

        all_preds_g.extend(torch.argmax(p_grade, 1).cpu().numpy())
        all_lbls_g.extend(lbl_g.cpu().numpy())

        all_preds_d.extend(torch.argmax(p_diag, 1).cpu().numpy())
        all_lbls_d.extend(lbl_d.cpu().numpy())

        if catcher.tokens is not None:
            maps = get_attention_maps(model_test, catcher)

            curr_L0 = maps[0].sum(dim=0).cpu()
            if total_L0 is None: total_L0 = curr_L0
            else: total_L0 += curr_L0

            curr_L1 = maps[1].sum(dim=0).cpu()
            if total_L1 is None: total_L1 = curr_L1
            else: total_L1 += curr_L1

            count += img.shape[0]

catcher.remove()

# --- 5. Print Classification Reports (ROBUST FIX) ---
print("\n" + "="*60)
print("📊 FINAL TEST RESULTS")
print("="*60)

# Grade Report
print("\n--- 🧠 TUMOR GRADE REPORT ---")
# Ορίζουμε ρητά τα labels για να μην μπερδεύεται αν λείπει κάποια κλάση
labels_g = list(range(len(GRADE_CLASSES)))
print(classification_report(all_lbls_g, all_preds_g, labels=labels_g, target_names=[str(c) for c in GRADE_CLASSES], zero_division=0))
print("Confusion Matrix (Grade):")
print(confusion_matrix(all_lbls_g, all_preds_g))

# Diagnosis Report
print("\n--- 🧬 DIAGNOSIS REPORT ---")
labels_d = list(range(len(DIAG_CLASSES)))
print(classification_report(all_lbls_d, all_preds_d, labels=labels_d, target_names=[str(c) for c in DIAG_CLASSES], zero_division=0))
print("Confusion Matrix (Diagnosis):")
print(confusion_matrix(all_lbls_d, all_preds_d))

# --- 6. Plot Attention ---
if count > 0:
    mean_L0 = total_L0 / count
    mean_L1 = total_L1 / count
    labels = ["Grade", "Diag", "Enc (Img)", "Dec (Shp)", "Clin"]

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    fig.suptitle(f"Global Average Attention on TEST SET (N={count})", fontsize=18, fontweight='bold')

    sns.heatmap(mean_L0.numpy(), annot=True, fmt=".2f", xticklabels=labels, yticklabels=labels, cmap="Blues", cbar=False, ax=axes[0], vmin=0, vmax=1)
    axes[0].set_title("Layer 0 (Mixing)", fontsize=14)
    for i in range(len(labels)): axes[0].add_patch(plt.Rectangle((i, 0), 1, 1, fill=False, edgecolor='red', lw=3))

    sns.heatmap(mean_L1.numpy(), annot=True, fmt=".2f", xticklabels=labels, yticklabels=labels, cmap="Purples", cbar=False, ax=axes[1], vmin=0, vmax=1)
    axes[1].set_title("Layer 1 (Final Decision)", fontsize=14)
    for i in range(len(labels)): axes[1].add_patch(plt.Rectangle((i, 0), 1, 1, fill=False, edgecolor='red', lw=3))

    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Δεν βρέθηκαν δεδομένα Attention.")

In [ ]:
# %% Βήμα 7 (ALL-IN-ONE): Dice, Classification, Attention & Visuals
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
import numpy as np
import os
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete, Activations, Compose
from monai.data import decollate_batch

# --- 1. Hooks & Helpers ---
class TransformerInputCatcher:
    def __init__(self, model):
        self.tokens = None
        self.hook = model.fusion_transformer.register_forward_pre_hook(self.save_input)
    def save_input(self, module, inputs): self.tokens = inputs[0]
    def remove(self): self.hook.remove()

def get_attention_maps(model, catcher):
    tokens = catcher.tokens
    B, N, D = tokens.shape
    attn_maps = []
    with torch.no_grad():
        curr_tokens = tokens.clone()
        for layer in model.fusion_transformer.layers:
            attn_layer = layer.self_attn
            qkv = F.linear(curr_tokens, attn_layer.in_proj_weight, attn_layer.in_proj_bias)
            qkv = qkv.reshape(B, N, 3, D).permute(2, 0, 1, 3)
            q, k, v = qkv[0], qkv[1], qkv[2]
            scores = torch.matmul(q, k.transpose(-2, -1)) / (D ** 0.5)
            attn_maps.append(F.softmax(scores, dim=-1))
            curr_tokens = layer(curr_tokens)
    return attn_maps

# --- 2. Load Model ---
model_path = os.path.join(runs_base_dir, "mtl_mask_guided_final", "best_mask_model.pt")
print(f"🔍 Φόρτωση Best Model: {model_path}")

try:
    model_eval = AdvancedMaskGuidedModel(
        in_channels_img=IN_CHANNELS, out_channels_seg=OUT_CHANNELS_SEG,
        feature_size=FEATURE_SIZE, use_checkpoint=False,
        num_classes_grade=NUM_CLASSES_GRADE, num_classes_diag=NUM_CLASSES_DIAG,
        clinical_feature_size=CLINICAL_FEATURE_SIZE,
        finetuned_seg_model_path=None,
        d_model=128, n_heads=4, num_layers=2, dropout_rate=0.4
    ).to(DEVICE)
    model_eval.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model_eval.eval()
except Exception as e: print(f"❌ Error: {e}"); raise e

# --- 3. Setup Metrics ---
# Dice Metric Setup
dice_metric = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)
post_pred = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
post_label = Compose([AsDiscrete(to_onehot=None)])

# --- 4. Run Evaluation (Combined Val + Test) ---
print("\n🧪 Εκτέλεση Πλήρους Ελέγχου (Val + Test)...")

catcher = TransformerInputCatcher(model_eval)
total_L0, total_L1 = None, None
count = 0

all_preds_g, all_lbls_g = [], []
all_preds_d, all_lbls_d = [], []

# Για αποθήκευση δειγμάτων για visualization
sample_imgs = []
sample_masks_true = []
sample_masks_pred = []

# Λίστα loaders (μπορείς να βάλεις μόνο test_loader_mtl αν θες)
loaders = [val_loader_mtl, test_loader_mtl]

with torch.no_grad():
    for loader in loaders:
        for batch in tqdm(loader, leave=False):
            img = batch[image_key].to(DEVICE)
            clin = batch[clinical_key].to(DEVICE)
            lbl_g = batch[grade_key].to(DEVICE)
            lbl_d = batch[diag_key].to(DEVICE)
            seg_true = batch[label_key].to(DEVICE)

            # Forward
            outputs = model_eval(img, clin)
            logits_seg, p_grade, p_diag = outputs[0], outputs[1], outputs[2]

            # --- A. Segmentation Dice ---
            val_outputs = [post_pred(i) for i in decollate_batch(logits_seg)]
            val_labels = [post_label(i) for i in decollate_batch(seg_true)]
            dice_metric(y_pred=val_outputs, y=val_labels)

            # Save visuals (middle slice) from first batch only
            if len(sample_imgs) < 3:
                mid_slice = img.shape[4] // 2
                sample_imgs.append(img[0, 0, :, :, mid_slice].cpu().numpy())
                sample_masks_true.append(seg_true[0, 0, :, :, mid_slice].cpu().numpy())
                sample_masks_pred.append(val_outputs[0][0, :, :, mid_slice].cpu().numpy())

            # --- B. Classification Metrics ---
            all_preds_g.extend(torch.argmax(p_grade, 1).cpu().numpy())
            all_lbls_g.extend(lbl_g.cpu().numpy())
            all_preds_d.extend(torch.argmax(p_diag, 1).cpu().numpy())
            all_lbls_d.extend(lbl_d.cpu().numpy())

            # --- C. Attention ---
            if catcher.tokens is not None:
                maps = get_attention_maps(model_eval, catcher)
                curr_L0 = maps[0].sum(dim=0).cpu()
                curr_L1 = maps[1].sum(dim=0).cpu()

                if total_L0 is None: total_L0 = curr_L0
                else: total_L0 += curr_L0

                if total_L1 is None: total_L1 = curr_L1
                else: total_L1 += curr_L1

                count += img.shape[0]

catcher.remove()

# --- 5. REPORTING ---
print("\n" + "="*60)
print(f"📊 FINAL COMBINED RESULTS (N={count})")
print("="*60)

# A. Dice Score
final_dice = dice_metric.aggregate().item()
print(f"\n🩺 SEGMENTATION DICE SCORE: {final_dice:.4f}")
if final_dice > 0.88: print("   ✅ SOTA Level Performance!")
elif final_dice > 0.80: print("   ✅ Excellent Performance")
else: print("   ⚠️ Needs Improvement")

# B. Classification
print("\n🧠 GRADE REPORT")
labels_g = list(range(len(GRADE_CLASSES)))
print(classification_report(all_lbls_g, all_preds_g, labels=labels_g, target_names=[str(c) for c in GRADE_CLASSES], zero_division=0))

print("\n🧬 DIAGNOSIS REPORT")
labels_d = list(range(len(DIAG_CLASSES)))
print(classification_report(all_lbls_d, all_preds_d, labels=labels_d, target_names=[str(c) for c in DIAG_CLASSES], zero_division=0))

# --- 6. VISUALIZATION ---
# A. Segmentation Visuals
print("\n👁️ Segmentation Check (Samples)")
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
for i in range(3):
    if i < len(sample_imgs):
        axes[i, 0].imshow(sample_imgs[i], cmap="gray"); axes[i, 0].set_title("MRI Image")
        axes[i, 0].axis('off')
        axes[i, 1].imshow(sample_masks_true[i], cmap="jet", interpolation='nearest'); axes[i, 1].set_title("True Mask")
        axes[i, 1].axis('off')
        axes[i, 2].imshow(sample_masks_pred[i], cmap="jet", interpolation='nearest'); axes[i, 2].set_title("Predicted Mask")
        axes[i, 2].axis('off')
plt.tight_layout(); plt.show()

# B. Attention Plot
if count > 0:
    mean_L0 = total_L0 / count
    mean_L1 = total_L1 / count
    labels = ["Grade", "Diag", "Enc", "Dec", "Clin"]

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    fig.suptitle(f"Global Attention (Avg over {count} patients)", fontsize=18, fontweight='bold')

    sns.heatmap(mean_L0.numpy(), annot=True, fmt=".2f", xticklabels=labels, yticklabels=labels, cmap="Blues", cbar=False, ax=axes[0], vmin=0, vmax=1)
    axes[0].set_title("Layer 0 (Mixing)", fontsize=14)
    for i in range(len(labels)): axes[0].add_patch(plt.Rectangle((i, 0), 1, 1, fill=False, edgecolor='red', lw=3))

    sns.heatmap(mean_L1.numpy(), annot=True, fmt=".2f", xticklabels=labels, yticklabels=labels, cmap="Purples", cbar=False, ax=axes[1], vmin=0, vmax=1)
    axes[1].set_title("Layer 1 (Decision)", fontsize=14)
    for i in range(len(labels)): axes[1].add_patch(plt.Rectangle((i, 0), 1, 1, fill=False, edgecolor='red', lw=3))

    plt.tight_layout()
    plt.show()

# Multi Staged

In [ ]:
# %% Βήμα 4: MULTI-SCALE Mask-Guided Model (Features from Stage 2, 3, 4)
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.networks.nets import SwinUNETR
import os

class MultiScaleMaskGuidedModel(nn.Module):
    def __init__(self,
                 in_channels_img, out_channels_seg, feature_size,
                 num_classes_grade, num_classes_diag, clinical_feature_size,
                 d_model=128, n_heads=8, num_layers=2, dropout_rate=0.4,
                 use_checkpoint=True, finetuned_seg_model_path=None):
        super().__init__()

        # --- 1. SwinUNETR Backbone (UNFROZEN) ---
        self.swin_unetr = SwinUNETR(

            in_channels=in_channels_img,
            out_channels=out_channels_seg,
            feature_size=feature_size,
            use_checkpoint=use_checkpoint
        )

        if finetuned_seg_model_path and os.path.exists(finetuned_seg_model_path):
            print(f"🔍 Φόρτωση αρχικών βαρών από: {finetuned_seg_model_path}")
            try:
                checkpoint = torch.load(finetuned_seg_model_path, map_location=torch.device('cpu'))
                if "state_dict" in checkpoint: checkpoint = checkpoint["state_dict"]
                # Καθαρισμός prefix αν υπάρχει
                swin_dict = {k.replace("swin_unetr.", ""): v for k, v in checkpoint.items() if "swin_unetr" in k}
                if not swin_dict: swin_dict = checkpoint # Fallback

                self.swin_unetr.load_state_dict(swin_dict, strict=False)
                print("✅ Επιτυχής φόρτωση βαρών SwinUNETR.")
            except Exception as e:
                print(f"⚠️ Warning loading weights: {e}")

        # 2. Hooks Setup
        self.enc_feats_list = None # Θα κρατάει λίστα: [Stage0, Stage1, Stage2, Stage3, Stage4]
        self.dec_feat_captured = None

        # Hook στον Encoder (SwinViT) -> Πιάνει ΟΛΑ τα stages
        self.swin_unetr.swinViT.register_forward_hook(self._hook_encoder)
        # Hook στον Decoder -> Πιάνει το input στο τελικό layer
        self.swin_unetr.out.register_forward_hook(self._hook_decoder)

        # 3. Multi-Scale Projections
        # Channels βάσει feature_size=48:
        # Stage 2: 192 ch (High Res Texture)
        # Stage 3: 384 ch (Medium Semantic)
        # Stage 4: 768 ch (Bottleneck)

        self.proj_l2 = nn.Sequential(
            nn.Linear(192, d_model), nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(0.1)
        )
        self.proj_l3 = nn.Sequential(
            nn.Linear(384, d_model), nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(0.1)
        )
        self.proj_l4 = nn.Sequential(
            nn.Linear(768, d_model), nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(0.1)
        )

        # Decoder Projection (Shape)
        self.decoder_proj = nn.Linear(feature_size, d_model)

        # Clinical Projection
        self.clinical_proj = nn.Linear(clinical_feature_size, d_model)

        # 4. Tokens & Transformer
        self.grade_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.diag_token = nn.Parameter(torch.randn(1, 1, d_model))

        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dropout=dropout_rate, batch_first=True)
        self.fusion_transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 5. Heads
        self.grade_head = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Dropout(dropout_rate), nn.Linear(64, num_classes_grade))
        self.diagnosis_head = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Dropout(dropout_rate), nn.Linear(64, num_classes_diag))

        # 6. Aux Heads (Optional: Deep Supervision)
        self.aux_head_l4 = nn.Linear(d_model, num_classes_grade) # Βοηθητικό στο bottleneck

    def _hook_encoder(self, module, input, output):
        # Το output του SwinViT είναι ΛΙΣΤΑ με 5 tensors. Την κρατάμε όλη.
        self.enc_feats_list = output

    def _hook_decoder(self, module, input, output):
        self.dec_feat_captured = input[0]

    def forward(self, x_img, x_clinical):
        # A. Run Backbone (Generate Segmentation & Capture Features)
        logits_seg = self.swin_unetr(x_img)

        # Ανάκτηση features από τα hooks
        feats_list = self.enc_feats_list # [S0, S1, S2, S3, S4]
        feat_dec = self.dec_feat_captured

        # Επιλογή επιπέδων (Multi-Scale)
        raw_l2 = feats_list[2] # 192 ch
        raw_l3 = feats_list[3] # 384 ch
        raw_l4 = feats_list[4] # 768 ch (Bottleneck)

        # --- B. Mask Extraction ---
        seg_prob = torch.sigmoid(logits_seg)
        # Παίρνουμε το max probability map αν έχουμε πολλαπλά κανάλια ή το κανάλι 0
        mask = (seg_prob > 0.5).float() # Binary Mask για καθαρό filtering

        # --- C. Multi-Scale Masking & Tokenization ---

        # Level 2 (Texture)
        mask_l2 = F.interpolate(mask, size=raw_l2.shape[2:], mode='nearest')
        masked_l2 = raw_l2 * mask_l2
        vec_l2 = F.adaptive_avg_pool3d(masked_l2, 1).flatten(1)
        token_l2 = self.proj_l2(vec_l2).unsqueeze(1)

        # Level 3 (Mix)
        mask_l3 = F.interpolate(mask, size=raw_l3.shape[2:], mode='nearest')
        masked_l3 = raw_l3 * mask_l3
        vec_l3 = F.adaptive_avg_pool3d(masked_l3, 1).flatten(1)
        token_l3 = self.proj_l3(vec_l3).unsqueeze(1)

        # Level 4 (Semantic/Bottleneck)
        mask_l4 = F.interpolate(mask, size=raw_l4.shape[2:], mode='nearest')
        masked_l4 = raw_l4 * mask_l4
        vec_l4 = F.adaptive_avg_pool3d(masked_l4, 1).flatten(1)
        token_l4 = self.proj_l4(vec_l4).unsqueeze(1)

        # Decoder (Shape)
        mask_dec = F.interpolate(mask, size=feat_dec.shape[2:], mode='nearest')
        masked_dec = feat_dec * mask_dec
        vec_dec = F.adaptive_avg_pool3d(masked_dec, 1).flatten(1)
        token_dec = self.decoder_proj(vec_dec).unsqueeze(1)

        # --- D. Clinical & Fusion ---
        token_clin = self.clinical_proj(x_clinical).unsqueeze(1)

        if self.training:
            if torch.rand(1).item() < 0.4: # Modality Dropout
                token_clin = torch.zeros_like(token_clin)

        B = x_img.shape[0]
        grade_t = self.grade_token.expand(B, -1, -1)
        diag_t = self.diag_token.expand(B, -1, -1)

        # Concatenate ALL tokens: [Grade, Diag, L2, L3, L4, Dec, Clin]
        tokens = torch.cat([grade_t, diag_t, token_l2, token_l3, token_l4, token_dec, token_clin], dim=1)

        fused = self.fusion_transformer(tokens)

        # --- E. Predictions ---
        # Output 0 -> Grade, Output 1 -> Diagnosis
        pred_grade = self.grade_head(fused[:, 0, :])
        pred_diag = self.diagnosis_head(fused[:, 1, :])

        # Aux Output (Optional - using L4 token)
        aux_out = self.aux_head_l4(token_l4.squeeze(1))

        # Cleanup hooks refs to free memory
        self.enc_feats_list = None
        self.dec_feat_captured = None

        return logits_seg, pred_grade, pred_diag, aux_out

In [ ]:
# %% INITIALIZE MODEL (Run this before Training Loop)
import os

# Settings για το output
output_dir_mtl = os.path.join(runs_base_dir, "mtl_multiscale_final")
best_model_path = os.path.join(output_dir_mtl, "best_multiscale_model.pt")
os.makedirs(output_dir_mtl, exist_ok=True)

# Αρχικοποίηση του Μοντέλου
# Βεβαιώσου ότι έχεις τρέξει το κελί με την κλάση MultiScaleMaskGuidedModel πρώτα!
try:
    model_mtl = MultiScaleMaskGuidedModel(
        in_channels_img=IN_CHANNELS,
        out_channels_seg=OUT_CHANNELS_SEG,
        feature_size=FEATURE_SIZE,
        num_classes_grade=NUM_CLASSES_GRADE,
        num_classes_diag=NUM_CLASSES_DIAG,
        clinical_feature_size=CLINICAL_FEATURE_SIZE,
        d_model=128,            # Κρατάμε το 128 για ισορροπία
        n_heads=8,              # 8 κεφαλές για καλύτερο attention
        num_layers=2,           # 2 layers στον Transformer
        dropout_rate=0.4,       # Αυξημένο dropout για αποφυγή overfitting
        use_checkpoint=True,
        finetuned_seg_model_path=finetuned_model_path # Φόρτωση του Segmentation Backbone
    ).to(DEVICE)

    print("✅ Το μοντέλο 'model_mtl' (Multi-Scale) αρχικοποιήθηκε επιτυχώς!")
    print(f"   - Device: {DEVICE}")
    print(f"   - Output Dir: {output_dir_mtl}")

except NameError as e:
    print(f"❌ Σφάλμα: {e}")
    print("   -> Βεβαιώσου ότι έτρεξες το κελί με τον ορισμό της κλάσης 'MultiScaleMaskGuidedModel'.")

In [ ]:
# %% Step 5: Training Advanced Mask-Guided MTL (Final Loop with Reports & AUC)
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import pandas as pd
import numpy as np
import os
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete, Activations, Compose
from monai.losses import DiceFocalLoss, FocalLoss
from monai.data import decollate_batch
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# --- 1. Class Weights ---
print("Computing Class Weights...")
try:
    y_grade_all = df_processed['Grade_Label'].values
    y_diag_all = df_processed['Diagnosis_Label'].values
    weights_grade = compute_class_weight('balanced', classes=np.unique(y_grade_all), y=y_grade_all)
    weights_diag = compute_class_weight('balanced', classes=np.unique(y_diag_all), y=y_diag_all)
    weights_grade_tensor = torch.tensor(weights_grade, dtype=torch.float).to(DEVICE)
    weights_diag_tensor = torch.tensor(weights_diag, dtype=torch.float).to(DEVICE)
    print("Weights loaded.")
except:
    weights_grade_tensor = None; weights_diag_tensor = None
    print("Weights could not be computed (using uniform).")

# --- 2. Loss Functions (New Setup) ---
# Segmentation: DiceFocal (Το ισχυρό combo)
loss_seg_fn = DiceFocalLoss(
    include_background=False,
    sigmoid=True,
    gamma=2.0,
    lambda_dice=1.0,
    lambda_focal=1.0
)

# Classification: Focal Loss (Διατηρούμε το Focal που δούλεψε)
loss_cls_fn = FocalLoss(include_background=True, to_onehot_y=True, gamma=2.0, weight=weights_grade_tensor)
loss_diag_fn = FocalLoss(include_background=True, to_onehot_y=True, gamma=2.0, weight=weights_diag_tensor)

def advanced_loss_fn(outputs, labels_seg, labels_grade, labels_diag):
    # ΠΡΟΣΟΧΗ: Το νέο μοντέλο επιστρέφει 4 τιμές
    logits_seg, p_grade, p_diag, aux_l4 = outputs

    # Main Losses
    l_seg = loss_seg_fn(logits_seg, labels_seg)
    l_grade = loss_cls_fn(p_grade, labels_grade)
    l_diag = loss_diag_fn(p_diag, labels_diag)

    # Aux Loss (Deep Supervision στο Layer 4 για το Grade)
    l_aux = loss_cls_fn(aux_l4, labels_grade)

    # Weighted Sum
    total_loss = (
        1.0 * l_grade +
        0.5 * l_diag +
        1.0 * l_seg +
        0.2 * l_aux     # Μικρότερο βάρος στο aux
    )
    return total_loss

# --- 3. Settings ---
output_dir_mtl = os.path.join(runs_base_dir, "mtl_multiscale_final") # Νέος φάκελος
history_csv_path = os.path.join(output_dir_mtl, "history_multiscale_detailed.csv")
best_model_path = os.path.join(output_dir_mtl, "best_multiscale_model.pt")
os.makedirs(output_dir_mtl, exist_ok=True)

# --- 4. Check Model ---
if 'model_mtl' not in globals():
    raise ValueError("❌ Το μοντέλο 'model_mtl' δεν βρέθηκε! Τρέξε το προηγούμενο κελί (MultiScaleMaskGuidedModel) πρώτα.")
else:
    print("✅ Using pre-defined Multi-Scale Model.")

# --- 5. Optimizer & Scheduler ---
LR_BACKBONE = 2e-5  # Λίγο αυξημένο για να μάθει τα νέα features
LR_HEADS = 2e-4

params = [
    {'params': model_mtl.swin_unetr.parameters(), 'lr': LR_BACKBONE},
    {'params': model_mtl.proj_l2.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.proj_l3.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.proj_l4.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.decoder_proj.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.clinical_proj.parameters(), 'lr': LR_HEADS},
    {'params': [model_mtl.grade_token, model_mtl.diag_token], 'lr': LR_HEADS},
    {'params': model_mtl.fusion_transformer.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.grade_head.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.diagnosis_head.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.aux_head_l4.parameters(), 'lr': LR_HEADS},
]

optimizer = torch.optim.AdamW(params, weight_decay=0.05)

# Restart Strategy: T_0=20 epochs, T_mult=2 (Restarts @ 20, 60)
scheduler = CosineAnnealingWarmRestarts(
    optimizer,
    T_0=20,
    T_mult=2,
    eta_min=1e-6
)

# --- 6. Metrics Setup ---
dice_metric_train = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)
dice_metric_val = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)
post_pred_seg = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
post_label_seg = Compose([AsDiscrete(to_onehot=None)])

best_val_metric = -1.0
start_epoch = 0
GRAD_ACCUM_STEPS = 8

# Detailed History Storage
history = {
    "epoch": [],
    "train_loss": [], "train_dice": [],
    "train_f1_g": [], "train_acc_g": [], "train_auc_g": [],
    "train_f1_d": [], "train_acc_d": [], "train_auc_d": [],
    "val_loss": [], "val_dice": [],
    "val_f1_g": [], "val_acc_g": [], "val_auc_g": [],
    "val_f1_d": [], "val_acc_d": [], "val_auc_d": [],
    "combined_score": []
}

if os.path.exists(best_model_path): os.remove(best_model_path)

print(f"\n--- Starting Multi-Scale Training ({NUM_EPOCHS_MTL} Epochs) ---")

for epoch in range(start_epoch, NUM_EPOCHS_MTL):
    current_epoch = epoch + 1

    # === TRAIN ===
    model_mtl.train()
    dice_metric_train.reset()
    ep_loss = 0
    step = 0

    # Lists for Epoch Metrics
    tr_pred_g, tr_probs_g, tr_lbl_g = [], [], []
    tr_pred_d, tr_probs_d, tr_lbl_d = [], [], []

    optimizer.zero_grad()
    pbar = tqdm(train_loader_mtl, desc=f"Epoch {current_epoch} Train")

    for i, batch in enumerate(pbar):
        step += 1
        img = batch[image_key].to(DEVICE)
        seg = batch[label_key].to(DEVICE)
        clin = batch[clinical_key].to(DEVICE)
        lbl_g = batch[grade_key].to(DEVICE)
        lbl_d = batch[diag_key].to(DEVICE)

        outputs = model_mtl(img, clin)
        loss = advanced_loss_fn(outputs, seg, lbl_g, lbl_d)

        if not torch.isfinite(loss): continue

        (loss / GRAD_ACCUM_STEPS).backward()

        if (i + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model_mtl.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

        ep_loss += loss.item()

        # Unpack outputs for metrics
        logits_seg, p_grade, p_diag, _ = outputs

        # Segmentation Metric
        val_outputs = [post_pred_seg(i) for i in decollate_batch(logits_seg)]
        val_labels = [post_label_seg(i) for i in decollate_batch(seg)]
        dice_metric_train(y_pred=val_outputs, y=val_labels)

        # Classification Metrics Collection
        tr_pred_g.extend(torch.argmax(p_grade, 1).cpu().numpy())
        tr_probs_g.extend(torch.softmax(p_grade, 1).detach().cpu().numpy())
        tr_lbl_g.extend(lbl_g.cpu().numpy())

        tr_pred_d.extend(torch.argmax(p_diag, 1).cpu().numpy())
        tr_probs_d.extend(torch.softmax(p_diag, 1).detach().cpu().numpy())
        tr_lbl_d.extend(lbl_d.cpu().numpy())

        pbar.set_postfix(loss=f"{loss.item():.2f}")

    # --- Train Stats ---
    train_loss_avg = ep_loss / step
    train_dice = dice_metric_train.aggregate().item()

    # Grade
    train_f1_g = f1_score(tr_lbl_g, tr_pred_g, average='macro')
    train_acc_g = accuracy_score(tr_lbl_g, tr_pred_g)
    try: train_auc_g = roc_auc_score(tr_lbl_g, tr_probs_g, multi_class='ovr', average='macro')
    except: train_auc_g = 0.0

    # Diag
    train_f1_d = f1_score(tr_lbl_d, tr_pred_d, average='macro')
    train_acc_d = accuracy_score(tr_lbl_d, tr_pred_d)
    try: train_auc_d = roc_auc_score(tr_lbl_d, tr_probs_d, multi_class='ovr', average='macro')
    except: train_auc_d = 0.0

    # History
    history["epoch"].append(current_epoch)
    history["train_loss"].append(train_loss_avg); history["train_dice"].append(train_dice)
    history["train_f1_g"].append(train_f1_g); history["train_acc_g"].append(train_acc_g); history["train_auc_g"].append(train_auc_g)
    history["train_f1_d"].append(train_f1_d); history["train_acc_d"].append(train_acc_d); history["train_auc_d"].append(train_auc_d)

    print(f"Train | Loss: {train_loss_avg:.3f} | Dice: {train_dice:.3f} | G_F1: {train_f1_g:.3f} | G_AUC: {train_auc_g:.3f}")

    # === VALIDATION ===
    if current_epoch % VAL_INTERVAL == 0:
        model_mtl.eval()
        dice_metric_val.reset()
        val_loss_sum = 0
        val_steps = 0

        preds_g, probs_g, lbls_g = [], [], []
        preds_d, probs_d, lbls_d = [], [], []

        with torch.no_grad():
            for batch in tqdm(val_loader_mtl, desc="Validating"):
                val_steps += 1
                img = batch[image_key].to(DEVICE)
                seg = batch[label_key].to(DEVICE)
                clin = batch[clinical_key].to(DEVICE)
                lbl_g = batch[grade_key].to(DEVICE)
                lbl_d = batch[diag_key].to(DEVICE)

                outputs = model_mtl(img, clin)
                loss = advanced_loss_fn(outputs, seg, lbl_g, lbl_d)
                val_loss_sum += loss.item()

                logits_seg, p_grade, p_diag, _ = outputs

                # Seg Metric
                val_outputs = [post_pred_seg(i) for i in decollate_batch(logits_seg)]
                val_labels = [post_label_seg(i) for i in decollate_batch(seg)]
                dice_metric_val(y_pred=val_outputs, y=val_labels)

                # Class Metrics
                preds_g.extend(torch.argmax(p_grade, 1).cpu().numpy())
                probs_g.extend(torch.softmax(p_grade, 1).cpu().numpy())
                lbls_g.extend(lbl_g.cpu().numpy())

                preds_d.extend(torch.argmax(p_diag, 1).cpu().numpy())
                probs_d.extend(torch.softmax(p_diag, 1).cpu().numpy())
                lbls_d.extend(lbl_d.cpu().numpy())

        # --- Val Stats ---
        val_dice = dice_metric_val.aggregate().item()

        # Grade
        val_f1_g = f1_score(lbls_g, preds_g, average='macro')
        val_acc_g = accuracy_score(lbls_g, preds_g)
        try: val_auc_g = roc_auc_score(lbls_g, probs_g, multi_class='ovr', average='macro')
        except: val_auc_g = 0.0

        # Diag
        val_f1_d = f1_score(lbls_d, preds_d, average='macro')
        val_acc_d = accuracy_score(lbls_d, preds_d)
        try: val_auc_d = roc_auc_score(lbls_d, probs_d, multi_class='ovr', average='macro')
        except: val_auc_d = 0.0

        # Combined Score (Balanced)
        combined = (val_dice * 0.4) + (val_f1_g * 0.3) + (val_f1_d * 0.3)

        # Update History
        history["val_loss"].append(val_loss_sum/val_steps); history["val_dice"].append(val_dice)
        history["val_f1_g"].append(val_f1_g); history["val_acc_g"].append(val_acc_g); history["val_auc_g"].append(val_auc_g)
        history["val_f1_d"].append(val_f1_d); history["val_acc_d"].append(val_acc_d); history["val_auc_d"].append(val_auc_d)
        history["combined_score"].append(combined)

        print(f"--- Val Epoch {current_epoch} ---")
        print(f"Combined: {combined:.4f} | Loss: {val_loss_sum/val_steps:.4f} | Dice: {val_dice:.4f}")
        print(f"Grade -> F1: {val_f1_g:.3f} | Acc: {val_acc_g:.3f} | AUC: {val_auc_g:.3f}")
        print(f"Diag  -> F1: {val_f1_d:.3f} | Acc: {val_acc_d:.3f} | AUC: {val_auc_d:.3f}")

        # --- REPORTS ---
        print("\n=== 🧠 TUMOR GRADE REPORT ===")
        print(classification_report(lbls_g, preds_g, target_names=[str(c) for c in GRADE_CLASSES], zero_division=0))

        print("\n=== 🧬 DIAGNOSIS REPORT ===")
        print(classification_report(lbls_d, preds_d, target_names=[str(c) for c in DIAG_CLASSES], zero_division=0))

        if combined > best_val_metric:
            best_val_metric = combined
            torch.save(model_mtl.state_dict(), best_model_path)
            print("🏆 New Best Model Saved!")

        pd.DataFrame(history).to_csv(history_csv_path, index=False)

    else:
        # Pad val history for consistency
        for k in ["val_loss", "val_dice", "val_f1_g", "val_acc_g", "val_auc_g", "val_f1_d", "val_acc_d", "val_auc_d", "combined_score"]:
            history[k].append(np.nan)

    if scheduler: scheduler.step()

print(f"Done. Best Combined Score: {best_val_metric:.4f}")

In [ ]:
# %% Step 6: Visualization of Results (Advanced Full History Analysis)
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np

# --- Settings (UPDATED for Multi-Scale) ---
output_dir_mtl = os.path.join(runs_base_dir, "mtl_multiscale_final")
history_csv_path = os.path.join(output_dir_mtl, "history_multiscale_detailed.csv") # The correct filename

print(f"🔍 Reading history file: {history_csv_path}")

if os.path.exists(history_csv_path):
    df = pd.read_csv(history_csv_path)

    # Separate rows that have validation data (not NaN in 'val_loss')
    # Using 'val_loss' as the anchor to find validation epochs
    df_val = df.dropna(subset=['val_loss'])

    epochs = df['epoch']
    val_epochs = df_val['epoch']

    # Plot Style
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(2, 2, figsize=(20, 12))
    fig.suptitle('Multi-Scale Mask-Guided MTL Training Analysis', fontsize=20, weight='bold')

    # 1. LOSSES & COMBINED SCORE
    ax1 = axes[0, 0]
    if 'train_loss' in df.columns:
        ax1.plot(epochs, df['train_loss'], label='Train Loss', color='blue', alpha=0.6)
    if 'val_loss' in df_val.columns:
        ax1.plot(val_epochs, df_val['val_loss'], label='Val Loss', color='navy', linewidth=2, marker='o')

    # Combined Score (Target)
    if 'combined_score' in df_val.columns:
        ax1b = ax1.twinx()
        ax1b.plot(val_epochs, df_val['combined_score'], label='Combined Score', color='gold', linestyle='--', linewidth=2)
        ax1b.legend(loc='upper right')
        ax1b.set_ylabel('Score')

    ax1.set_title('Optimization Landscape', fontsize=14)
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.legend(loc='upper left')
    ax1.grid(True, linestyle='--', alpha=0.7)

    # 2. SEGMENTATION (DICE)
    ax2 = axes[0, 1]
    if 'train_dice' in df.columns:
        ax2.plot(epochs, df['train_dice'], label='Train Dice', color='gray', linestyle=':', alpha=0.8)
    if 'val_dice' in df_val.columns:
        ax2.plot(val_epochs, df_val['val_dice'], label='Val Dice', color='black', linewidth=2, marker='s')

    ax2.set_title('Task 1: Segmentation (Dice Score)', fontsize=14)
    ax2.set_xlabel('Epoch'); ax2.set_ylim(0, 1.0)
    ax2.legend(loc='lower right'); ax2.grid(True)

    # 3. GRADE CLASSIFICATION
    ax3 = axes[1, 0]
    if 'train_f1_g' in df.columns:
        ax3.plot(epochs, df['train_f1_g'], label='Train F1', color='lightgreen', linestyle='--')
    if 'val_f1_g' in df_val.columns:
        ax3.plot(val_epochs, df_val['val_f1_g'], label='Val F1', color='forestgreen', linewidth=2.5)

    # Accuracy (if available)
    if 'val_acc_g' in df_val.columns:
        ax3.plot(val_epochs, df_val['val_acc_g'], label='Val Acc', color='darkgreen', alpha=0.3, linestyle=':')

    ax3.set_title('Task 2: Tumor Grade Classification', fontsize=14)
    ax3.set_xlabel('Epoch'); ax3.set_ylim(0, 1.0)
    ax3.legend(loc='lower right'); ax3.grid(True)

    # 4. DIAGNOSIS CLASSIFICATION
    ax4 = axes[1, 1]
    has_diag = False

    # Check if Diagnosis columns exist
    if 'train_f1_d' in df.columns and not df['train_f1_d'].isnull().all():
        ax4.plot(epochs, df['train_f1_d'], label='Train F1', color='plum', linestyle='--')
        has_diag = True

    if 'val_f1_d' in df_val.columns and not df_val['val_f1_d'].isnull().all():
        ax4.plot(val_epochs, df_val['val_f1_d'], label='Val F1', color='darkmagenta', linewidth=2.5)
        has_diag = True

    # Accuracy (if available)
    if 'val_acc_d' in df_val.columns and not df_val['val_acc_d'].isnull().all():
        ax4.plot(val_epochs, df_val['val_acc_d'], label='Val Acc', color='indigo', alpha=0.6)

    ax4.set_title('Task 3: Diagnosis Classification', fontsize=14)
    ax4.set_xlabel('Epoch'); ax4.set_ylim(0, 1.0)
    if has_diag: ax4.legend(loc='lower right')
    ax4.grid(True)

    plt.tight_layout()
    plt.show()

    # --- Print Best Model Stats ---
    if 'combined_score' in df_val.columns:
        # Find the row with the max combined score
        best_idx = df_val['combined_score'].idxmax()
        best_row = df_val.loc[best_idx]

        print(f"\n🏆 BEST MODEL at Epoch {int(best_row['epoch'])}")
        print(f"Combined Score: {best_row['combined_score']:.4f}")

        if 'val_dice' in best_row:
            print(f"Seg Dice: {best_row['val_dice']:.4f}")

        if 'val_f1_g' in best_row:
            acc_str = f" (Acc: {best_row['val_acc_g']:.4f})" if 'val_acc_g' in best_row else ""
            print(f"Grade F1: {best_row['val_f1_g']:.4f}{acc_str}")

        # Robust check for Diagnosis
        if 'val_f1_d' in best_row and not pd.isna(best_row['val_f1_d']):
            acc_str = f" (Acc: {best_row['val_acc_d']:.4f})" if 'val_acc_d' in best_row else ""
            print(f"Diag F1:  {best_row['val_f1_d']:.4f}{acc_str}")
        else:
            print("Diag F1:  N/A (Column missing or NaN)")
else:
    print(f"❌ Error: CSV File not found at {history_csv_path}. Please run Step 5 training first.")

In [ ]:
# %% Step 7 (FINAL & COMPLETE): Multi-Scale Evaluation (Dice, Class, Dual Attention)
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
import numpy as np
import os
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete, Activations, Compose
from monai.data import decollate_batch

# --- 1. Hooks & Helpers ---
class TransformerInputCatcher:
    def __init__(self, model):
        self.tokens = None
        # Hook στο πρώτο layer για να πιάσουμε το input
        self.hook = model.fusion_transformer.layers[0].self_attn.register_forward_pre_hook(self.save_input)
    def save_input(self, module, inputs): self.tokens = inputs[0]
    def remove(self): self.hook.remove()

def get_attention_maps(model, catcher):
    tokens = catcher.tokens
    if tokens.dim() == 3: B, N, D = tokens.shape
    else: return []

    attn_maps = []
    with torch.no_grad():
        curr_tokens = tokens.clone()
        for i, layer in enumerate(model.fusion_transformer.layers):
            attn_layer = layer.self_attn
            if hasattr(attn_layer, 'in_proj_weight'):
                # Υπολογισμός Q, K
                qkv = F.linear(curr_tokens, attn_layer.in_proj_weight, attn_layer.in_proj_bias)
                qkv = qkv.reshape(B, N, 3, D).permute(2, 0, 1, 3)
                q, k = qkv[0], qkv[1]
                # Attention Score = Softmax(Q * K^T / sqrt(D))
                scores = torch.matmul(q, k.transpose(-2, -1)) / (D ** 0.5)
                attn_maps.append(F.softmax(scores, dim=-1))

            # Περνάμε το token από το layer για το επόμενο step
            curr_tokens = layer(curr_tokens)
    return attn_maps

# --- 2. Load Model ---
model_path = os.path.join(runs_base_dir, "mtl_multiscale_final", "best_multiscale_model.pt")
print(f"🔍 Φόρτωση Best Model: {model_path}")

try:
    model_eval = MultiScaleMaskGuidedModel(
        in_channels_img=IN_CHANNELS, out_channels_seg=OUT_CHANNELS_SEG,
        feature_size=FEATURE_SIZE, num_classes_grade=NUM_CLASSES_GRADE,
        num_classes_diag=NUM_CLASSES_DIAG, clinical_feature_size=CLINICAL_FEATURE_SIZE,
        d_model=128, n_heads=8, num_layers=2, dropout_rate=0.4,
        use_checkpoint=False, finetuned_seg_model_path=None
    ).to(DEVICE)
    model_eval.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model_eval.eval()
    print("✅ Model loaded successfully!")
except Exception as e:
    print(f"❌ Error: {e}"); raise e

# --- 3. Metrics Setup ---
dice_metric = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)
post_pred = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
post_label = Compose([AsDiscrete(to_onehot=None)])

# --- 4. Evaluation Loop ---
print("\n🧪 Start Evaluation (Val + Test)...")
try: catcher = TransformerInputCatcher(model_eval)
except: catcher = None

total_L0, total_L1 = None, None
count = 0
all_preds_g, all_lbls_g = [], []
all_preds_d, all_lbls_d = [], []
sample_imgs, sample_masks_true, sample_masks_pred = [], [], []

loaders_list = [val_loader_mtl, test_loader_mtl]

with torch.no_grad():
    for loader in loaders_list:
        for batch in tqdm(loader, leave=False):
            img = batch[image_key].to(DEVICE)
            clin = batch[clinical_key].to(DEVICE)
            lbl_g, lbl_d = batch[grade_key].to(DEVICE), batch[diag_key].to(DEVICE)
            seg_true = batch[label_key].to(DEVICE)

            outputs = model_eval(img, clin)
            logits_seg, p_grade, p_diag, _ = outputs

            # Metrics
            dice_metric(y_pred=[post_pred(i) for i in decollate_batch(logits_seg)],
                        y=[post_label(i) for i in decollate_batch(seg_true)])
            all_preds_g.extend(torch.argmax(p_grade, 1).cpu().numpy()); all_lbls_g.extend(lbl_g.cpu().numpy())
            all_preds_d.extend(torch.argmax(p_diag, 1).cpu().numpy()); all_lbls_d.extend(lbl_d.cpu().numpy())

            # Save Visuals (Max 3)
            if len(sample_imgs) < 3:
                mid = img.shape[4] // 2
                sample_imgs.append(img[0, 0, :, :, mid].cpu().numpy())
                sample_masks_true.append(seg_true[0, 0, :, :, mid].cpu().numpy())
                sample_masks_pred.append(logits_seg[0, 0, :, :, mid].sigmoid().cpu().numpy() > 0.5)

            # Attention Accumulation
            if catcher and catcher.tokens is not None:
                try:
                    maps = get_attention_maps(model_eval, catcher)
                    if len(maps) >= 2:
                        # Layer 0
                        l0_sum = maps[0].cpu().sum(dim=0) # Sum batch
                        total_L0 = l0_sum if total_L0 is None else total_L0 + l0_sum
                        # Layer 1
                        l1_sum = maps[1].cpu().sum(dim=0) # Sum batch
                        total_L1 = l1_sum if total_L1 is None else total_L1 + l1_sum

                        count += img.shape[0]
                except: pass

if catcher: catcher.remove()

# --- 5. Final Report ---
print(f"\n📊 RESULTS (N={len(all_lbls_g)})")
print(f"🩺 Dice Score: {dice_metric.aggregate().item():.4f}")
print("\n🧠 Grade Report:\n", classification_report(all_lbls_g, all_preds_g, target_names=[str(c) for c in GRADE_CLASSES], zero_division=0))
print("\n🧬 Diagnosis Report:\n", classification_report(all_lbls_d, all_preds_d, target_names=[str(c) for c in DIAG_CLASSES], zero_division=0))

# --- 6. Visualization ---
# A. Segmentation
fig, ax = plt.subplots(len(sample_imgs), 3, figsize=(10, 3*len(sample_imgs)))
for i in range(len(sample_imgs)):
    ax[i,0].imshow(sample_imgs[i], cmap="gray"); ax[i,0].set_title("MRI")
    ax[i,1].imshow(sample_masks_true[i], cmap="Greens"); ax[i,1].set_title("Truth")
    ax[i,2].imshow(sample_masks_pred[i], cmap="Reds"); ax[i,2].set_title("Pred")
    for j in range(3): ax[i,j].axis('off')
plt.tight_layout(); plt.show()

# B. Dual Attention Layers
if total_L0 is not None and count > 0:
    mean_L0 = total_L0 / count
    mean_L1 = total_L1 / count

    labels = ["Grade", "Diag", "L2", "L3", "L4", "Clin"]
    limit = min(mean_L0.shape[0], len(labels))

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    # Layer 0
    sns.heatmap(mean_L0[:limit, :limit].numpy(), annot=True, fmt=".2f",
                xticklabels=labels[:limit], yticklabels=labels[:limit], cmap="Blues", ax=axes[0])
    axes[0].set_title("Layer 0 (Information Mixing)", fontsize=16, weight='bold')
    axes[0].set_xlabel("Key (Source info)"); axes[0].set_ylabel("Query (Listener)")

    # Layer 1
    sns.heatmap(mean_L1[:limit, :limit].numpy(), annot=True, fmt=".2f",
                xticklabels=labels[:limit], yticklabels=labels[:limit], cmap="Purples", ax=axes[1])
    axes[1].set_title("Layer 1 (Final Decision)", fontsize=16, weight='bold')
    axes[1].set_xlabel("Key (Source info)"); axes[1].set_ylabel("Query (Listener)")

    plt.tight_layout()
    plt.show()



# TO DO:


## Radionomics
## SEGMAMBA/UMAMBA

In [ ]:
pip install git+https://github.com/AIM-Harvard/pyradiomics.git

In [ ]:
pip install SimpleITK

In [ ]:
import os
import glob
import re
import pandas as pd
import SimpleITK as sitk
from radiomics import featureextractor
import numpy as np

# --- CONFIGURATION (Based on your paths) ---
# Update 'base_drive_path' to your actual local path string
base_drive_path = "/content/drive/MyDrive/Διπλωματικη"
IMAGE_DIR = os.path.join(base_drive_path, "Brain_Tumor_Subset/volumes_subset")
LABEL_DIR = os.path.join(base_drive_path, "Brain_Tumor_Subset/segmentations_subset")
OUTPUT_CSV = "radiomics_features_raw.csv"

# --- RADIOMICS SETTINGS (MRI Optimized) ---
settings = {
    'normalize': True,
    'normalizeScale': 100,
    'resampledPixelSpacing': [1, 1, 1],
    'interpolator': sitk.sitkBSpline,
    'binWidth': 5,
    'voxelArrayShift': 300,
    'verbose': True
}

def get_extractor():
    extractor = featureextractor.RadiomicsFeatureExtractor(**settings)
    extractor.disableAllFeatures()
    extractor.enableFeatureClassByName('firstorder')
    extractor.enableFeatureClassByName('shape')
    extractor.enableFeatureClassByName('glcm')
    extractor.enableFeatureClassByName('glrlm')
    extractor.enableFeatureClassByName('glszm')
    extractor.enableFeatureClassByName('gldm')
    return extractor

def process_dataset():
    extractor = get_extractor()
    data_list = []

    # 1. Get List of Volume Files
    image_files = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.nii.gz")))
    print(f"Found {len(image_files)} images in {IMAGE_DIR}")

    for img_path in image_files:
        filename = os.path.basename(img_path)

        # 1. Extract the Patient ID by stripping the suffix (_T2, _T1c, _flair, etc.)
        # This regex looks for: (Everything) + "_" + (Suffix) + ".nii.gz"
        match = re.match(r"(.+)_[A-Za-z0-9]+\.nii\.gz", filename)

        if match:
            patient_id = match.group(1) # e.g., "UCSF-PDGM-0535"
        else:
            # Fallback if filename doesn't have an underscore suffix
            patient_id = filename.replace(".nii.gz", "")

        # 2. Construct the Expected Mask Filename
        # Adjust "_tumor_segmentation.nii.gz" if your files use a different ending
        mask_filename = f"{patient_id}_tumor_segmentation.nii.gz"
        mask_path = os.path.join(LABEL_DIR, mask_filename)

        # Debug print (Optional: enable if it still fails)
        # print(f"Checking: {img_path} -> Mask: {mask_path}")

        if not os.path.exists(mask_path):
            # Try alternative name just in case (some datasets use _segmentation.nii.gz)
            alt_mask_filename = f"{patient_id}_segmentation.nii.gz"
            alt_mask_path = os.path.join(LABEL_DIR, alt_mask_filename)

            if os.path.exists(alt_mask_path):
                mask_path = alt_mask_path
            else:
                print(f"⚠️ Mask not found for {patient_id} (Checked: {mask_filename})")
                continue

        try:
            print(f"Processing {patient_id}...", end="\r")

            # Run Extraction
            result = extractor.execute(img_path, mask_path)

            # Clean Data
            clean_row = {'patient_id': patient_id}
            for key, value in result.items():
                if "diagnostic" not in key:
                    if isinstance(value, (np.ndarray, np.generic)):
                        value = value.item()
                    clean_row[key] = value

            data_list.append(clean_row)

        except Exception as e:
            print(f"\n❌ Error on {patient_id}: {e}")

    # Save
    if data_list:
        df = pd.DataFrame(data_list)
        cols = ['patient_id'] + [c for c in df.columns if c != 'patient_id']
        df = df[cols]
        df.to_csv(OUTPUT_CSV, index=False)
        print(f"\n✅ Success! Saved {len(df)} patients to {OUTPUT_CSV}")
    else:
        print("\n❌ No features extracted.")

if __name__ == "__main__":
    process_dataset()

In [ ]:
import shutil
import os

source = "radiomics_features_raw.csv"
destination = os.path.join(base_drive_path, "Brain_Tumor_Subset/radiomics_features_raw.csv")

if os.path.exists(source):
    shutil.copy(source, destination)
    print(f"✅ File saved to Drive at: {destination}")
else:
    print("❌ File not found in current workspace. Did the script finish?")

In [ ]:
import pandas as pd

# Load both files
df_rad = pd.read_csv("radiomics_features_raw.csv")
df_meta = pd.read_csv(os.path.join(base_drive_path, "Brain_Tumor_Subset/balanced_metadata_gradeaware.csv"))

# Print the first 5 IDs from each
print("--- RADIOMICS IDs (First 5) ---")
print(df_rad['patient_id'].head().tolist())

print("\n--- METADATA IDs (First 5) ---")
print(df_meta['ID'].head().tolist())

# Check data types
print(f"\nRadiomics ID Type: {df_rad['patient_id'].dtype}")
print(f"Metadata ID Type: {df_meta['ID'].dtype}")

In [ ]:
import pandas as pd
import re

# 1. Load Data
df_rad = pd.read_csv("radiomics_features_raw.csv")
df_meta = pd.read_csv(os.path.join(base_drive_path, "Brain_Tumor_Subset/balanced_metadata_gradeaware.csv"))

# 2. Define Helper to extract the number
def get_id_number(id_str):
    # Finds the last group of digits in the string
    match = re.search(r'(\d+)$', str(id_str))
    if match:
        return int(match.group(1)) # Returns 14 for both "0014" and "014"
    return -1 # Fallback for errors

# 3. Create a temporary 'Merge_ID' column
df_rad['Merge_ID'] = df_rad['patient_id'].apply(get_id_number)
df_meta['Merge_ID'] = df_meta['ID'].apply(get_id_number)

# 4. Merge on the Number
df = pd.merge(df_rad, df_meta[['Merge_ID', 'Grade', 'ID']], on='Merge_ID', how='inner')

# 5. Clean up
# We keep the 'ID' from metadata as the official one and drop the temporary helper
df = df.rename(columns={'ID': 'Official_ID'})
df = df.drop(columns=['Merge_ID', 'patient_id'])

print(f"✅ Success! Merged {df.shape[0]} patients.")
print(f"Data Shape: {df.shape}")

# Check Class Balance
print(f"Class Balance:\n{df['Grade'].value_counts()}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np
import pandas as pd
import re

# --- 1. ROBUST LOAD & MERGE (Including Diagnosis) ---
df_rad = pd.read_csv("radiomics_features_raw.csv")
df_meta = pd.read_csv(os.path.join(base_drive_path, "Brain_Tumor_Subset/balanced_metadata_gradeaware.csv"))

# Helper to match IDs
def get_id_number(id_str):
    match = re.search(r'(\d+)$', str(id_str))
    return int(match.group(1)) if match else -1

df_rad['Merge_ID'] = df_rad['patient_id'].apply(get_id_number)
df_meta['Merge_ID'] = df_meta['ID'].apply(get_id_number)

# MERGE: Added 'Diagnosis' to the list of columns to keep
df = pd.merge(df_rad, df_meta[['Merge_ID', 'Grade', 'Diagnosis', 'ID']], on='Merge_ID', how='inner')
df = df.rename(columns={'ID': 'Official_ID'})
df = df.drop(columns=['Merge_ID', 'patient_id'])

print(f"✅ Merged. Data Shape: {df.shape}")
print(f"Diagnosis Counts:\n{df['Diagnosis'].value_counts()}")

# --- 2. PREPARE FEATURES ---
# Exclude all metadata columns
feature_cols = [c for c in df.columns if c not in ['Official_ID', 'Grade', 'Diagnosis', 'patient_id']]
X = df[feature_cols].fillna(0)

# Drop Constants
const_feats = [c for c in X.columns if X[c].nunique() <= 1]
X = X.drop(columns=const_feats)

# Scale for PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca = PCA(n_components=2)
components = pca.fit_transform(X_scaled)

# --- 3. PLOT: PCA COMPARISON (Grade vs Diagnosis) ---
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plot A: By Grade
sns.scatterplot(
    x=components[:,0], y=components[:,1],
    hue=df['Grade'], style=df['Grade'],
    palette='viridis', s=100, alpha=0.8, ax=axes[0]
)
axes[0].set_title(f'PCA by GRADE\n(Explained Var: {pca.explained_variance_ratio_.sum()*100:.1f}%)')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

# Plot B: By Diagnosis
# We use a distinct palette (Set2) to distinguish types clearly
sns.scatterplot(
    x=components[:,0], y=components[:,1],
    hue=df['Diagnosis'], style=df['Diagnosis'],
    palette='Set2', s=100, alpha=0.8, ax=axes[1]
)
axes[1].set_title(f'PCA by DIAGNOSIS\n(Same Features, Different Labels)')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')

plt.tight_layout()
plt.show()

# --- 4. PLOT: TOP FEATURES (Boxplots) ---
# We pick top 3 high-variance features and see how they split both targets
top_vars = X.var().nlargest(10).index

fig, axes = plt.subplots(3, 2, figsize=(16, 12))

for i, col in enumerate(top_vars):
    # Left Column: Grade
    sns.boxplot(data=df, x='Grade', y=col, palette='viridis', ax=axes[i, 0])
    axes[i, 0].set_title(f"{col} vs GRADE")

    # Right Column: Diagnosis
    sns.boxplot(data=df, x='Diagnosis', y=col, palette='Set2', ax=axes[i, 1])
    axes[i, 1].set_title(f"{col} vs DIAGNOSIS")
    # Rotate labels if diagnosis names are long
    axes[i, 1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- 1. PREPARE DATA ---
# Identify feature columns (exclude IDs and Targets)
feature_cols = [c for c in df.columns if c not in ['Official_ID', 'Grade', 'Diagnosis', 'patient_id', 'Merge_ID']]
X = df[feature_cols].fillna(0)

# --- 2. CALCULATE VARIANCE ---
# We calculate raw variance to see which values spread out the most
variances = X.var().sort_values(ascending=False)
top_10_features = variances.head(10).index.tolist()

print("--- Top 10 Highest Variance Features ---")
for i, feat in enumerate(top_10_features):
    print(f"{i+1}. {feat} (Var: {variances[feat]:.2f})")

# --- 3. PLOT A: MAGNITUDE OF VARIANCE ---
plt.figure(figsize=(10, 6))
sns.barplot(x=variances.head(10).values, y=top_10_features, palette="magma")
plt.xscale('log') # Log scale because Volume variance is usually massive compared to texture
plt.title('Top 10 Features by Variance (Log Scale)')
plt.xlabel('Variance (Log Scale)')
plt.show()

# --- 4. PLOT B: DISTRIBUTION BY DIAGNOSIS & GRADE ---
# We create a large grid of boxplots to see HOW these features vary across your classes
fig, axes = plt.subplots(5, 2, figsize=(16, 20)) # 5 rows, 2 columns for 10 features
axes = axes.flatten()

for i, col in enumerate(top_10_features):
    # We plot Diagnosis on X, and split colors (Hue) by Grade
    # This lets you see: "Does Glioblastoma (Diag) always have high Volume (Feat)?"
    sns.boxplot(
        data=df,
        x='Diagnosis',
        y=col,
        hue='Grade',
        palette='viridis',
        ax=axes[i],
        showfliers=False # Hide extreme outliers to keep the box readable
    )
    axes[i].set_title(col)
    axes[i].tick_params(axis='x', rotation=10) # Rotate labels slightly
    axes[i].set_xlabel('')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE, SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA

# --- 1. PREP DATA ---
# (Using the 'df' from your previous EDA step)
feature_cols = [c for c in df.columns if c not in ['Official_ID', 'Grade', 'Diagnosis', 'patient_id', 'Merge_ID']]
X = df[feature_cols].fillna(0)
y = df['Grade']

# Standardize (CRITICAL for LASSO and PCA)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# --- METHOD A: LASSO (L1 Regularization) ---
# We use Logistic Regression with L1 penalty.
# C is the inverse of regularization strength (Smaller C = More features killed)
lasso = LogisticRegression(penalty='l1', C=0.1, solver='liblinear', random_state=42)
lasso.fit(X_scaled, y)
model_l1 = SelectFromModel(lasso, prefit=True)
lasso_feats = X_scaled.columns[model_l1.get_support()]

print(f"LASSO selected {len(lasso_feats)} features:")
print(list(lasso_feats))

# --- METHOD B: Recursive Feature Elimination (RFE) ---
# Using Random Forest as the estimator
rf = RandomForestClassifier(n_estimators=50, random_state=42)
rfe = RFE(estimator=rf, n_features_to_select=10, step=1)
rfe.fit(X_scaled, y)
rfe_feats = X_scaled.columns[rfe.support_]

print(f"\nRandom Forest RFE selected {len(rfe_feats)} features:")
print(list(rfe_feats))

# --- METHOD C: PCA (Projection) ---
pca = PCA(n_components=10)
X_pca = pca.fit_transform(X_scaled)
print(f"\nPCA Component 1 is composed mostly of:")
# Look at the weights (loadings) of the first component
loading_scores = pd.Series(pca.components_[0], index=X_scaled.columns).abs().sort_values(ascending=False)
print(loading_scores.head(5))

# --- VISUAL COMPARISON ---
# Let's see feature overlap between LASSO and RFE
common = set(lasso_feats).intersection(set(rfe_feats))
print(f"\n✅ COMMON FEATURES (The 'Truth'): {list(common)}")

# Plotting the Importance of the RFE winners
plt.figure(figsize=(10, 6))
feat_importances = pd.Series(rfe.estimator_.feature_importances_, index=X_scaled.columns[rfe.support_])
feat_importances.nlargest(10).plot(kind='barh', color='teal')
plt.title('Top 10 Features (Wrapper Method - RFE)')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

print("--- Step 1: Feature Selection (Top 20 via RFE) ---")

# 1. Prepare Data (Assuming 'df' from EDA is loaded. If not, reload it)
# We exclude IDs and Targets to isolate the math features
feature_cols = [c for c in df.columns if c not in ['Official_ID', 'Grade', 'Diagnosis', 'patient_id', 'Merge_ID']]
X = df[feature_cols].fillna(0)
y = df['Grade']

# 2. Normalize (Crucial for RFE stability)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# 3. Run RFE for 20 Features
print("Running Recursive Feature Elimination (this may take a moment)...")
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rfe = RFE(estimator=rf, n_features_to_select=30, step=1)
rfe.fit(X_scaled, y)

# Get the list of 20 survivors
selected_20 = X_scaled.columns[rfe.support_].tolist()

print(f"\n✅ Selected 20 Features:")
print(selected_20)

# 4. Save to CSV
# Reconstruct DataFrame with IDs + Targets + Selected Features
output_cols = ['Official_ID', 'Grade', 'Diagnosis'] + selected_20
df_final_20 = df[['Official_ID', 'Grade', 'Diagnosis']].copy()
# Add the SCALED values (ready for MLP)
df_final_20[selected_20] = X_scaled[selected_20].values

output_path = os.path.join(base_drive_path, "Brain_Tumor_Subset/radiomics_selected_20.csv")
df_final_20.to_csv(output_path, index=False)
print(f"\nSaved training-ready data to: {output_path}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifwaier
from sklearn.metrics import classification_report, accuracy_score, ConfusionMatrixDisplay

print("--- Step 1: Data Loading & Preparation ---")

# 1. Load Data (20-feature version)
input_csv = os.path.join(base_drive_path, "Brain_Tumor_Subset/radiomics_selected_20.csv")
df_rad = pd.read_csv(input_csv)

# 2. Define Features (X)
feature_cols = [c for c in df_rad.columns if c not in ['Official_ID', 'Grade', 'Diagnosis']]
X = df_rad[feature_cols].values

# 3. Define Targets (Y) & Encode
le_grade = LabelEncoder()
y_grade = le_grade.fit_transform(df_rad['Grade'])
GRADE_CLASSES = le_grade.classes_

le_diag = LabelEncoder()
y_diag = le_diag.fit_transform(df_rad['Diagnosis'])
DIAG_CLASSES = le_diag.classes_

print(f"Total Samples: {len(X)}")
print(f"Grade Classes: {GRADE_CLASSES}")
print(f"Diagnosis Classes: {DIAG_CLASSES}")


# --- Step 2: Creating TWO SEPARATE STRATIFIED SPLITS ---
print("\n--- Creating Stratified Splits ---")

# ΑΛΛΑΓΗ ΕΔΩ: test_size=24
# SPLIT A: For Grading Task (Stratify by Grade)
X_train_g_raw, X_test_g_raw, y_train_g, y_test_g = train_test_split(
    X, y_grade, test_size=24, random_state=42, stratify=y_grade
)

# ΑΛΛΑΓΗ ΕΔΩ: test_size=24
# SPLIT B: For Diagnosis Task (Stratify by Diagnosis)
X_train_d_raw, X_test_d_raw, y_train_d, y_test_d = train_test_split(
    X, y_diag, test_size=24, random_state=42, stratify=y_diag
)

# Scaling (Fit on Train, Transform Test)
scaler_g = StandardScaler()
X_train_g = scaler_g.fit_transform(X_train_g_raw)
X_test_g = scaler_g.transform(X_test_g_raw)

scaler_d = StandardScaler()
X_train_d = scaler_d.fit_transform(X_train_d_raw)
X_test_d = scaler_d.transform(X_test_d_raw)

print(f"✅ Data Splits Ready & Scaled. Test Set Size: {len(X_test_g)}")

# Scaling (Fit on Train, Transform Test)
scaler_g = StandardScaler()
X_train_g = scaler_g.fit_transform(X_train_g_raw)
X_test_g = scaler_g.transform(X_test_g_raw)

scaler_d = StandardScaler()
X_train_d = scaler_d.fit_transform(X_train_d_raw)
X_test_d = scaler_d.transform(X_test_d_raw)

print("✅ Data Splits Ready & Scaled.")


# --- Step 3: Define Models ---
mlp_params = {
    'hidden_layer_sizes': [
        (50,50),
        (100,50),
        (150,100),
        (200,100),
        (200,150),
        (200,200),
        (100, 50, 25),   # Μικρό 3-layer
        (150, 100, 50),  # Μεσαίο 3-layer
        (200, 100, 50),
        (200, 150, 50),
        (200, 200, 50),
        (200, 200, 100),
        (200, 200, 150),
        (200, 200, 200),
        (300, 300, 300)# Μεγάλο 3-layer# Μεγάλο 3-layer
    ],
    'alpha': [0.0001, 0.001, 0.01],  # L2 Regularization (για overfitting)
    'learning_rate_init': [0.001, 0.0001],
    'max_iter' : [1000]# Αρχικός ρυθμός εκμάθησης
}

xgb_params = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0]
}
xgb_base = XGBClassifier(eval_metric='mlogloss', use_label_encoder=False, random_state=42)


# ==========================================
#  TASK 1: TUMOR GRADING (Stratified by Grade)
# ==========================================
print("\n" + "="*40)
print(" TASK 1: TUMOR GRADING ")
print("="*40)

# --- 1. MLP Grade ---
print("Training MLP (Grade)...")
grid_mlp_g = GridSearchCV(MLPClassifier(random_state=42), mlp_params, cv=5, scoring='f1_macro', n_jobs=-1)
grid_mlp_g.fit(X_train_g, y_train_g)
y_pred_mlp_g = grid_mlp_g.predict(X_test_g)
acc_mlp_g = accuracy_score(y_test_g, y_pred_mlp_g)
print(f"MLP Grade Accuracy: {acc_mlp_g:.4f}")

# --- 2. XGBoost Grade ---
print("Training XGBoost (Grade)...")
grid_xgb_g = GridSearchCV(xgb_base, xgb_params, cv=5, scoring='f1_macro', n_jobs=-1)
grid_xgb_g.fit(X_train_g, y_train_g)
y_pred_xgb_g = grid_xgb_g.predict(X_test_g)
acc_xgb_g = accuracy_score(y_test_g, y_pred_xgb_g)
print(f"XGBoost Grade Accuracy: {acc_xgb_g:.4f}")


# ==========================================
#  TASK 2: DIAGNOSIS (Stratified by Diagnosis)
# ==========================================
print("\n" + "="*40)
print(" TASK 2: DIAGNOSIS CLASSIFICATION ")
print("="*40)

# --- 3. MLP Diagnosis ---
print("Training MLP (Diagnosis)...")
grid_mlp_d = GridSearchCV(MLPClassifier(random_state=42), mlp_params, cv=5, scoring='f1_macro', n_jobs=-1)
grid_mlp_d.fit(X_train_d, y_train_d)
y_pred_mlp_d = grid_mlp_d.predict(X_test_d)
acc_mlp_d = accuracy_score(y_test_d, y_pred_mlp_d)
print(f"MLP Diagnosis Accuracy: {acc_mlp_d:.4f}")

# --- 4. XGBoost Diagnosis ---
print("Training XGBoost (Diagnosis)...")
grid_xgb_d = GridSearchCV(xgb_base, xgb_params, cv=5, scoring='f1_macro', n_jobs=-1)
grid_xgb_d.fit(X_train_d, y_train_d)
y_pred_xgb_d = grid_xgb_d.predict(X_test_d)
acc_xgb_d = accuracy_score(y_test_d, y_pred_xgb_d)
print(f"XGBoost Diagnosis Accuracy: {acc_xgb_d:.4f}")


# ==========================================
#  VISUALIZATION: CONFUSION MATRICES
# ==========================================
print("\n" + "="*40)
print(" VISUALIZING CONFUSION MATRICES ")
print("="*40)

# Plotting Grid: 2 Rows (Grade, Diagnosis) x 2 Columns (MLP, XGBoost)
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. MLP Grade
ConfusionMatrixDisplay.from_estimator(
    grid_mlp_g.best_estimator_, X_test_g, y_test_g,
    display_labels=GRADE_CLASSES.astype(str), cmap='Blues', ax=axes[0, 0]
)
axes[0, 0].set_title(f"MLP - Grade (Acc: {acc_mlp_g:.2f})")

# 2. XGBoost Grade
ConfusionMatrixDisplay.from_estimator(
    grid_xgb_g.best_estimator_, X_test_g, y_test_g,
    display_labels=GRADE_CLASSES.astype(str), cmap='Greens', ax=axes[0, 1]
)
axes[0, 1].set_title(f"XGBoost - Grade (Acc: {acc_xgb_g:.2f})")

# 3. MLP Diagnosis
ConfusionMatrixDisplay.from_estimator(
    grid_mlp_d.best_estimator_, X_test_d, y_test_d,
    display_labels=DIAG_CLASSES.astype(str), cmap='Blues', ax=axes[1, 0], xticks_rotation='vertical'
)
axes[1, 0].set_title(f"MLP - Diagnosis (Acc: {acc_mlp_d:.2f})")

# 4. XGBoost Diagnosis
ConfusionMatrixDisplay.from_estimator(
    grid_xgb_d.best_estimator_, X_test_d, y_test_d,
    display_labels=DIAG_CLASSES.astype(str), cmap='Greens', ax=axes[1, 1], xticks_rotation='vertical'
)
axes[1, 1].set_title(f"XGBoost - Diagnosis (Acc: {acc_xgb_d:.2f})")

plt.tight_layout()
plt.show()

# --- Summary Bar Plot ---
results_df = pd.DataFrame({
    'Task': ['Grade', 'Grade', 'Diagnosis', 'Diagnosis'],
    'Model': ['MLP', 'XGBoost', 'MLP', 'XGBoost'],
    'Accuracy': [acc_mlp_g, acc_xgb_g, acc_mlp_d, acc_xgb_d]
})

plt.figure(figsize=(8, 5))
sns.barplot(data=results_df, x='Task', y='Accuracy', hue='Model', palette='viridis')
plt.title("Summary: Radiomics Experiment 1")
plt.ylim(0, 1.05)
plt.ylabel("Test Accuracy")
plt.grid(axis='y', alpha=0.3)
for p in plt.gca().patches:
    if p.get_height() > 0:
        plt.gca().annotate(f'{p.get_height():.2f}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='center', xytext=(0, 9), textcoords='offset points')
plt.show()

# Radiomics + Clinical

# New Section

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, ConfusionMatrixDisplay

print("--- Step 1: Data Merging ---")

# 1. Load Clinical Data
meta_path = os.path.join(base_drive_path, "Brain_Tumor_Subset/balanced_metadata_gradeaware.csv")
df_meta = pd.read_csv(meta_path)
df_meta['ID'] = df_meta['ID'].astype(str)

# 2. Load Radiomics Data (20 selected features)
rad_path = os.path.join(base_drive_path, "Brain_Tumor_Subset/radiomics_selected_20.csv")
df_rad = pd.read_csv(rad_path)
df_rad['Official_ID'] = df_rad['Official_ID'].astype(str)

# 3. Merge
df = pd.merge(df_meta, df_rad, left_on='ID', right_on='Official_ID', how='inner')
print(f"✅ Merged {len(df)} patients.")

# --- Step 2: Auto-Feature Engineering ---
print("\n--- Step 2: Preparing ALL Features ---")

# 1. Identify Targets & IDs to Exclude
exclude_cols = [
    'ID', 'Official_ID', 'patient_id', 'Merge_ID', # IDs
    'Grade', 'Grade_x', 'Grade_y',                 # Targets (Raw)
    'Diagnosis', 'Diagnosis_x', 'Diagnosis_y',     # Targets (Raw)
    'Grade_Label', 'Diagnosis_Label'               # Targets (Encoded if present)
]

# 2. Select All Other Columns as Features
# We look for columns that are NOT in the exclude list
raw_feature_cols = [c for c in df.columns if c not in exclude_cols]

# 3. Create Feature Matrix (X)
X_raw = df[raw_feature_cols].copy()
print(f"Initial Features Found: {len(raw_feature_cols)}")
print(f"Columns: {raw_feature_cols}")
print("--- ΤΑ CLINICAL ΚΑΙ RAD FEATURES ΠΟΥ ΜΠΗΚΑΝ ΣΤΟ ΜΟΝΤΕΛΟ ---")
for i, col in enumerate(raw_feature_cols):
    print(f"{i}: {col}")
# 4. Handle Missing Values & Categoricals
# Numeric Columns: Fill NaN with Median
num_cols = X_raw.select_dtypes(include=[np.number]).columns
X_raw[num_cols] = X_raw[num_cols].fillna(X_raw[num_cols].median())

# Categorical Columns: One-Hot Encode (Convert 'Male' -> 1, 'Female' -> 0)
cat_cols = X_raw.select_dtypes(exclude=[np.number]).columns
if len(cat_cols) > 0:
    print(f"Encoding Categorical Features: {list(cat_cols)}")
    X_encoded = pd.get_dummies(X_raw, columns=cat_cols, drop_first=True)
else:
    X_encoded = X_raw

# Final Feature Matrix
X = X_encoded.values
print(f"Final Feature Count (after encoding): {X.shape[1]}")

# 5. Prepare Targets (Y)
le_grade = LabelEncoder()
y_grade = le_grade.fit_transform(df['Grade_y']) # Use Grade from Radiomics file (safer)
GRADE_CLASSES = le_grade.classes_

le_diag = LabelEncoder()
y_diag = le_diag.fit_transform(df['Diagnosis_y'])
DIAG_CLASSES = le_diag.classes_


# --- Step 3: Dual Stratified Splits ---
print("\n--- Step 3: Creating Splits ---")

# SPLIT A: For Grading (Stratified by Grade)
X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(
    X, y_grade, test_size=24, random_state=42, stratify=y_grade
)

# SPLIT B: For Diagnosis (Stratified by Diagnosis)
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X, y_diag, test_size=24, random_state=42, stratify=y_diag
)

# Scale Features (Important for MLP)
scaler_g = StandardScaler()
X_train_g = scaler_g.fit_transform(X_train_g)
X_test_g = scaler_g.transform(X_test_g)

scaler_d = StandardScaler()
X_train_d = scaler_d.fit_transform(X_train_d)
X_test_d = scaler_d.transform(X_test_d)

print("✅ Data Ready for Training.")


# --- Step 4: Run Models (MLP vs XGBoost) ---
print("\n" + "="*40)
print(" RUNNING EXPERIMENT 2 (ALL FEATURES) ")
print("="*40)

mlp_params = {
    'hidden_layer_sizes': [
        (50,50),
        (100,50),
        (150,100),
        (200,100),
        (200,150),
        (200,200),
        (100, 50, 25),   # Μικρό 3-layer
        (150, 100, 50),  # Μεσαίο 3-layer
        (200, 100, 50),
        (200, 150, 50),
        (200, 200, 50),
        (200, 200, 100),
        (200, 200, 150),
        (200, 200, 200),
        (300, 300, 300)# Μεγάλο 3-layer# Μεγάλο 3-layer
    ],
    'alpha': [0.0001, 0.001, 0.01],  # L2 Regularization (για overfitting)
    'learning_rate_init': [0.001, 0.0001],
    'max_iter' : [1000]# Αρχικός ρυθμός εκμάθησης
}
xgb_params = {'n_estimators': [200], 'max_depth': [3, 5], 'learning_rate': [0.1]}

# --- TASK 1: GRADE ---
print("\n[Grade] Training...")
# MLP
grid_mlp_g = GridSearchCV(MLPClassifier(random_state=42), mlp_params, scoring ='f1_macro', cv=5, n_jobs=-1)
grid_mlp_g.fit(X_train_g, y_train_g)
acc_mlp_g = accuracy_score(y_test_g, grid_mlp_g.predict(X_test_g))

# XGBoost
grid_xgb_g = GridSearchCV(XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42), xgb_params, cv=5, n_jobs=-1)
grid_xgb_g.fit(X_train_g, y_train_g)
acc_xgb_g = accuracy_score(y_test_g, grid_xgb_g.predict(X_test_g))

print(f"MLP Grade Acc: {acc_mlp_g:.4f}")
print(f"XGB Grade Acc: {acc_xgb_g:.4f}")

# --- TASK 2: DIAGNOSIS ---
print("\n[Diagnosis] Training...")
# MLP
grid_mlp_d = GridSearchCV(MLPClassifier(random_state=42), mlp_params, scoring ='f1_macro', cv=5, n_jobs=-1)
grid_mlp_d.fit(X_train_d, y_train_d)
acc_mlp_d = accuracy_score(y_test_d, grid_mlp_d.predict(X_test_d))

# XGBoost
grid_xgb_d = GridSearchCV(XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42), xgb_params, cv=5, n_jobs=-1)
grid_xgb_d.fit(X_train_d, y_train_d)
acc_xgb_d = accuracy_score(y_test_d, grid_xgb_d.predict(X_test_d))

print(f"MLP Diag Acc: {acc_mlp_d:.4f}")
print(f"XGB Diag Acc: {acc_xgb_d:.4f}")

# --- Step 5: Visualization ---
# Confusion Matrices
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Grade
ConfusionMatrixDisplay.from_estimator(grid_mlp_g.best_estimator_, X_test_g, y_test_g, display_labels=GRADE_CLASSES, cmap='Blues', ax=axes[0,0])
axes[0,0].set_title(f"MLP Fusion - Grade ({acc_mlp_g:.2f})")
ConfusionMatrixDisplay.from_estimator(grid_xgb_g.best_estimator_, X_test_g, y_test_g, display_labels=GRADE_CLASSES, cmap='Greens', ax=axes[0,1])
axes[0,1].set_title(f"XGB Fusion - Grade ({acc_xgb_g:.2f})")

# Diagnosis
ConfusionMatrixDisplay.from_estimator(grid_mlp_d.best_estimator_, X_test_d, y_test_d, display_labels=DIAG_CLASSES, cmap='Blues', ax=axes[1,0], xticks_rotation='vertical')
axes[1,0].set_title(f"MLP Fusion - Diag ({acc_mlp_d:.2f})")
ConfusionMatrixDisplay.from_estimator(grid_xgb_d.best_estimator_, X_test_d, y_test_d, display_labels=DIAG_CLASSES, cmap='Greens', ax=axes[1,1], xticks_rotation='vertical')
axes[1,1].set_title(f"XGB Fusion - Diag ({acc_xgb_d:.2f})")

plt.tight_layout()
plt.show()

print("\n--- Detailed Report: Best Model (Diagnosis) ---")
print(classification_report(y_test_d, grid_xgb_d.predict(X_test_d), target_names=DIAG_CLASSES.astype(str)))

In [ ]:
import pandas as pd
import numpy as np
import os

print("--- Data Loading & Encoding (No Training) ---")

# 1. Load Data
meta_path = os.path.join(base_drive_path, "Brain_Tumor_Subset/balanced_metadata_gradeaware.csv")
df_meta = pd.read_csv(meta_path)
df_meta['ID'] = df_meta['ID'].astype(str)

rad_path = os.path.join(base_drive_path, "Brain_Tumor_Subset/radiomics_selected_20.csv")
df_rad = pd.read_csv(rad_path)
df_rad['Official_ID'] = df_rad['Official_ID'].astype(str)

# 2. Merge
df = pd.merge(df_meta, df_rad, left_on='ID', right_on='Official_ID', how='inner')

# 3. Features & Exclusions
exclude_cols = [
    'ID', 'Official_ID', 'patient_id', 'Merge_ID',
    'Grade', 'Grade_x', 'Grade_y',
    'Diagnosis', 'Diagnosis_x', 'Diagnosis_y',
    'Grade_Label', 'Diagnosis_Label'
]
raw_feature_cols = [c for c in df.columns if c not in exclude_cols]
X_raw = df[raw_feature_cols].copy()

# 4. Missing Values & Encoding
num_cols = X_raw.select_dtypes(include=[np.number]).columns
X_raw[num_cols] = X_raw[num_cols].fillna(X_raw[num_cols].median())

cat_cols = X_raw.select_dtypes(exclude=[np.number]).columns
if len(cat_cols) > 0:
    X_encoded = pd.get_dummies(X_raw, columns=cat_cols, drop_first=True)
else:
    X_encoded = X_raw

# 5. Εκτύπωση της Λίστας!
print(f"✅ Final Feature Count: {X_encoded.shape[1]}")
print("\n--- ΑΝΤΙΣΤΟΙΧΙΑ 115 ΧΑΡΑΚΤΗΡΙΣΤΙΚΩΝ ---")
for i, col in enumerate(X_encoded.columns):
    print(f"Index {i} ---> {col}")

## Final Fusion (Deep + Clin + Rad)

In [ ]:
print("--- DIAGNOSTIC: ID FORMAT CHECK ---")

# 1. Check Image IDs
print(f"Image IDs (First 3): {list(image_dir_ft.keys())[:3]}")

# 2. Check Clinical IDs
print(f"Clinical IDs (First 3): {list(df_processed.index)[:3]}")

# 3. Check Radiomics IDs
print(f"Radiomics IDs (First 3): {list(rad_lookup.keys())[:3]}")

In [ ]:
# %% Βήμα 2 (Διορθωμένο): Προετοιμασία Metadata και Δημιουργία Data Dicts
import re
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split # Θα το χρησιμοποιήσουμε, αλλά με προσοχή
# --- ΝΕΟ IMPORT ---
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
# ---
from IPython.display import display
from glob import glob
import torch
metadata_path = os.path.join(base_drive_path, "Brain_Tumor_Subset/balanced_metadata_gradeaware.csv")
# --- 1. Φόρτωση του Ισορροπημένου Metadata ---
try:
    df_balanced = pd.read_csv(metadata_path)
    print(f"Φορτώθηκε το ισορροπημένο metadata. Σύνολο δειγμάτων: {len(df_balanced)}")
except FileNotFoundError:
    print(f"ΣΦΑΛΜΑ: Δεν βρέθηκε το αρχείο {metadata_path}.")
    # exit()

# --- 2. Επιλογή Βασικών Στηλών ---
columns_to_keep = [
    'ID', 'Grade', 'Diagnosis', 'Age', 'Sex', 'MGMT_Status', 'MGMT_Index',
    'Codeletion_1p19q', 'IDH', 'EOR', 'PreImaging_Biopsy'
]
df_processed = df_balanced[[col for col in columns_to_keep if col in df_balanced.columns]].copy()

# --- 3. Επεξεργασία 'MGMT_Index' ---
print("\n--- Επεξεργασία 'MGMT_Index' ---")
if 'MGMT_Index' in df_processed.columns:
    df_processed['MGMT_Index_Is_Unknown'] = (df_processed['MGMT_Index'] == 'unknown').astype(int)
    df_processed['MGMT_Index'] = pd.to_numeric(df_processed['MGMT_Index'], errors='coerce')
    mgmt_index_median = df_processed['MGMT_Index'].median()
    if pd.isna(mgmt_index_median): mgmt_index_median = 0
    df_processed['MGMT_Index'].fillna(mgmt_index_median, inplace=True)
    print(f"Η στήλη 'MGMT_Index' επεξεργάστηκε.")
else:
    print("Η στήλη 'MGMT_Index' δεν βρέθηκε.")

# --- 4. Διαχείριση Άλλων Ελλιπών Τιμών ---
print("\n--- Καθαρισμός Κατηγορικών Στηλών ---")
categorical_features_to_clean = ['Sex', 'MGMT_Status', 'Codeletion_1p19q', 'IDH', 'EOR', 'PreImaging_Biopsy']
for col in categorical_features_to_clean:
    if col in df_processed.columns:
        if df_processed[col].isnull().any():
            mode_val = df_processed[col].mode()[0]
            df_processed[col].fillna(mode_val, inplace=True)
        if (df_processed[col].astype(str) == 'unknown').any():
            try:
                mode_val = df_processed[col][df_processed[col] != 'unknown'].mode()[0]
            except IndexError:
                mode_val = "Missing"
            df_processed[col].replace('unknown', mode_val, inplace=True)

if 'Age' in df_processed.columns and df_processed['Age'].isnull().any():
    age_median = df_processed['Age'].median()
    df_processed['Age'].fillna(age_median, inplace=True)

# --- 5. Κωδικοποίηση Κατηγορικών Χαρακτηριστικών (One-Hot Encoding) ---
categorical_features_to_encode = ['Sex', 'Codeletion_1p19q', 'IDH', 'MGMT_Status', 'EOR', 'PreImaging_Biopsy']
existing_categorical_features = [col for col in categorical_features_to_encode if col in df_processed.columns]
if existing_categorical_features:
    print(f"Εφαρμογή One-Hot Encoding στις: {existing_categorical_features}")
    df_processed = pd.get_dummies(df_processed, columns=existing_categorical_features, dummy_na=False)
else:
    print("Δεν βρέθηκαν κατηγορικές στήλες για one-hot encoding.")

# --- 6. Κανονικοποίηση Συνεχών Χαρακτηριστικών (Scaling) ---
scaler = StandardScaler()
continuous_features_to_scale = ['Age', 'MGMT_Index']
existing_continuous_features = [col for col in continuous_features_to_scale if col in df_processed.columns]
if existing_continuous_features:
    print(f"Κανονικοποίηση (Standard Scaling) στις: {existing_continuous_features}")
    df_processed[existing_continuous_features] = scaler.fit_transform(df_processed[existing_continuous_features])
else:
    print("Δεν βρέθηκαν συνεχείς στήλες για scaling.")

# --- 7. Κωδικοποίηση Ετικετών (Label Encoding) ---
print("\n--- Κωδικοποίηση Ετικετών (Targets) ---")
grade_encoder = LabelEncoder()
df_processed['Grade_Label'] = grade_encoder.fit_transform(df_processed['Grade'])
GRADE_CLASSES = grade_encoder.classes_
NUM_CLASSES_GRADE = len(GRADE_CLASSES)
print(f"Κλάσεις Βαθμού (Grade): {list(GRADE_CLASSES)} -> {list(range(NUM_CLASSES_GRADE))}")

diag_encoder = LabelEncoder()
df_processed['Diagnosis_Label'] = diag_encoder.fit_transform(df_processed['Diagnosis'])
DIAG_CLASSES = diag_encoder.classes_
NUM_CLASSES_DIAG = len(DIAG_CLASSES)
print(f"Κλάσεις Διάγνωσης (Diagnosis): {list(DIAG_CLASSES)} -> {list(range(NUM_CLASSES_DIAG))}")

# --- 8. Ορισμός Χαρακτηριστικών (Clinical & Radiomics) ---
print("\n--- Βήμα 8: Προετοιμασία Χαρακτηριστικών ---")

# A. Clinical Features (Από το df_processed που φτιάξαμε στα βήματα 1-7)
non_feature_cols = [
    'ID', 'Grade', 'Diagnosis', 'Grade_Label', 'Diagnosis_Label',
    'OS', 'Survival_Status', 'BraTS21_ID', 'Segmentation_Cohort',
    'MGMT_Cohort', 'MGMT_Label', 'pid_formatted'
]
clinical_feature_columns = [col for col in df_processed.columns if col not in non_feature_cols]
CLINICAL_FEATURE_SIZE = len(clinical_feature_columns)
print(f"✅ Clinical Features ({CLINICAL_FEATURE_SIZE}): {clinical_feature_columns}")

# B. Radiomics Features (Φόρτωση από CSV)
rad_path = os.path.join(base_drive_path, "Brain_Tumor_Subset/radiomics_selected_20.csv")
try:
    df_rad = pd.read_csv(rad_path)
    df_rad['Official_ID'] = df_rad['Official_ID'].astype(str)

    # Identify Radiomics Columns (Exclude Meta)
    rad_cols = [c for c in df_rad.columns if c not in ['Official_ID', 'Grade', 'Diagnosis']]
    RADIOMICS_LEN = len(rad_cols)

    # Scale Radiomics (Essential for Transformer stability)
    rad_scaler = StandardScaler()
    df_rad[rad_cols] = rad_scaler.fit_transform(df_rad[rad_cols])

    # Create Lookup Dict: {'ID': [val1, val2...]}
    rad_lookup = df_rad.set_index('Official_ID')[rad_cols].T.to_dict('list')
    print(f"✅ Radiomics Features ({RADIOMICS_LEN}) loaded & scaled.")

except FileNotFoundError:
    print(f"❌ ΣΦΑΛΜΑ: Δεν βρέθηκε το {rad_path}")
    # Fallback για debugging (αν δεν υπάρχει το αρχείο)
    rad_lookup = {}
    RADIOMICS_LEN = 0

# %% Βήμα 9 (FIXED): Robust ID Normalization & Matching
def get_id_number(val):
    """
    Turns 'UCSF-PDGM-0004', '004', or 4 into integer 4.
    Returns None if no number found.
    """
    match = re.search(r'(\d+)', str(val))
    if match:
        return int(match.group(1))
    return None

print("--- 1. Locating Files & Normalizing IDs ---")

# --- 2. Find Files (Re-running this to be safe) ---
image_files_all = sorted(glob(os.path.join(image_dir_ft, "*.nii.gz")))
label_files_all = sorted(glob(os.path.join(label_dir_ft, "*_tumor_segmentation.nii.gz")))

# Map filenames to keys (Original String Keys)
# Assumes filename structure: "ID_suffix.nii.gz"
image_path_map = {os.path.basename(f).split('_')[0]: f for f in image_files_all}
label_path_map = {os.path.basename(f).replace("_tumor_segmentation.nii.gz", ""): f for f in label_files_all}

print(f"Found {len(image_path_map)} images and {len(label_path_map)} labels.")

# --- 3. Create Integer Maps for Images & Radiomics ---
# Image Map: {4: 'path/to/img.nii.gz'}
img_map_int = {get_id_number(k): v for k, v in image_path_map.items() if get_id_number(k) is not None}
lbl_map_int = {get_id_number(k): v for k, v in label_path_map.items() if get_id_number(k) is not None}

# Radiomics Map: {4: [vector...]}
# Ensure rad_lookup exists (from Step 8). If not, re-run Step 8.
try:
    rad_lookup_int = {get_id_number(k): v for k, v in rad_lookup.items() if get_id_number(k) is not None}
except NameError:
    print("⚠️ Warning: 'rad_lookup' not found. Make sure Step 8 ran successfully!")
    rad_lookup_int = {}

# --- 4. Fix Clinical IDs ---
# Ensure we use the 'ID' column
if 'ID' in df_processed.columns:
    df_processed['Int_ID'] = df_processed['ID'].apply(get_id_number)
elif 'pid_formatted' in df_processed.columns:
    df_processed['Int_ID'] = df_processed['pid_formatted'].apply(get_id_number)
else:
    # If ID is in the index
    df_processed.reset_index(inplace=True)
    if 'ID' in df_processed.columns:
        df_processed['Int_ID'] = df_processed['ID'].apply(get_id_number)
    else:
        # Fallback: Index is the ID
        df_processed['Int_ID'] = df_processed.index.map(get_id_number)

# Set the new Integer ID as index
df_clean = df_processed.dropna(subset=['Int_ID']).copy()
df_clean['Int_ID'] = df_clean['Int_ID'].astype(int)
df_clean.set_index('Int_ID', inplace=True)

print(f"Mapped Stats -> Images: {len(img_map_int)}, Clinical: {len(df_clean)}, Radiomics: {len(rad_lookup_int)}")

# --- 5. Execute Matching Loop ---
data_dicts_mtl_balanced = []
missing_log = {"mask": 0, "clinical": 0, "radiomics": 0}

for pid_num in img_map_int.keys():

    has_mask = pid_num in lbl_map_int
    has_clin = pid_num in df_clean.index
    has_rad = pid_num in rad_lookup_int

    if has_mask and has_clin and has_rad:
        patient_meta = df_clean.loc[pid_num]

        # Handle duplicates (take first)
        if isinstance(patient_meta, pd.DataFrame):
            patient_meta = patient_meta.iloc[0]

        data_dicts_mtl_balanced.append({
            "pid_num": pid_num,
            "label_2way": torch.tensor(1 if patient_meta['Grade'] == 4 else 0, dtype=torch.long),
            image_key: img_map_int[pid_num],
            label_key: lbl_map_int[pid_num],

            # Clinical
            clinical_key: torch.tensor(patient_meta[clinical_feature_columns].values.astype(np.float32), dtype=torch.float32),

            # Radiomics
            radiomics_key: torch.tensor(rad_lookup_int[pid_num], dtype=torch.float32),

            # Targets
            grade_key: torch.tensor(patient_meta['Grade_Label'], dtype=torch.long),
            diag_key: torch.tensor(patient_meta['Diagnosis_Label'], dtype=torch.long)
        })
    else:
        if not has_mask: missing_log["mask"] += 1
        if not has_clin: missing_log["clinical"] += 1
        if not has_rad: missing_log["radiomics"] += 1

print(f"\n✅ Final Match Count: {len(data_dicts_mtl_balanced)}")

if len(data_dicts_mtl_balanced) > 0:
    print("Ready for Step 10 (Split).")
else:
    print(f"❌ Matching Failed. Stats: {missing_log}")
# --- 10. Stratified Split ---
print("\n--- Βήμα 10: Stratified Split (Train/Val/Test) ---")

# Extract targets for stratification
Y_stratify = np.array([
    [d[grade_key].item(), d[diag_key].item()]
    for d in data_dicts_mtl_balanced
])

print("\n--- Βήμα 10: Stratified Split (Train/Val/Test) ---")

# 1. Extract targets for stratification
# We need a 2D array [N_samples, N_labels] for the stratifier
Y_stratify = np.array([
    [d[grade_key].item(), d[diag_key].item()]
    for d in data_dicts_mtl_balanced
])

# 2. Split 1: Train (80%) vs Rest (20%)
msss_train = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, rest_idx = next(msss_train.split(data_dicts_mtl_balanced, Y_stratify))

train_files_mtl = [data_dicts_mtl_balanced[i] for i in train_idx]
rest_files = [data_dicts_mtl_balanced[i] for i in rest_idx]
Y_rest = Y_stratify[rest_idx]

# 3. Split 2: Val (50% of Rest) vs Test (50% of Rest) -> 10% total each
msss_val = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx, test_idx = next(msss_val.split(rest_files, Y_rest))

val_files_mtl = [rest_files[i] for i in val_idx]
test_files_mtl = [rest_files[i] for i in test_idx]

print(f"Train Set: {len(train_files_mtl)} patients")
print(f"Val Set:   {len(val_files_mtl)} patients")
print(f"Test Set:  {len(test_files_mtl)} patients")

# --- DISTRIBUTION CHECKS (Requested) ---
print("\n--- 📊 Distribution Analysis ---")

def print_dist(name, dataset, key, class_names=None):
    labels = [d[key].item() for d in dataset]
    counts = pd.Series(labels).value_counts(normalize=True).sort_index()
    print(f"\n{name} - {key} Distribution:")
    for label, freq in counts.items():
        label_str = class_names[label] if class_names is not None else str(label)
        print(f"  Class {label_str}: {freq:.2%}")

# Diagnosis Distributions
print_dist("Validation", val_files_mtl, diag_key, DIAG_CLASSES)
print_dist("Test", test_files_mtl, diag_key, DIAG_CLASSES)

# Grade Distributions
print_dist("Validation", val_files_mtl, grade_key, GRADE_CLASSES)
print_dist("Test", test_files_mtl, grade_key, GRADE_CLASSES)

print(f"\nFinal Feature Sizes -> Clinical: {CLINICAL_FEATURE_SIZE}, Radiomics: {RADIOMICS_LEN}")

In [ ]:
# %% Βήμα 3 (Διορθωμένο): Transforms & DataLoaders (MTL)

# --- Βεβαιώσου ότι έχει γίνει import η list_data_collate ---
from monai.data import (
    DataLoader as MonaiDataLoader,
    PersistentDataset,
    partition_dataset,
    decollate_batch,
    list_data_collate # <-- ΣΗΜΑΝΤΙΚΟ
)
from monai.transforms import CenterSpatialCropd

# ----------------------------------------------------
# ⚠️ UPDATE HERE: Add radiomics_key
# ----------------------------------------------------
# Κλειδιά που θέλουμε ως float32 Tensors (Image + Mask + Clinical + Radiomics)
float_tensor_keys = [image_key, label_key, clinical_key, radiomics_key]

# Κλειδιά που θέλουμε ως long Tensors (Targets)
long_tensor_keys = [grade_key, diag_key]

# --- Transforms Εκπαίδευσης (Training) ---
train_transforms_mtl = Compose([
    LoadImaged(keys=spatial_keys, image_only=False),
    EnsureChannelFirstd(keys=spatial_keys),
    Orientationd(keys=spatial_keys, axcodes="RAS"),
    Spacingd(keys=spatial_keys, pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys=image_key, a_min=0, a_max=3000, b_min=0.0, b_max=1.0, clip=True),
    RandCropByPosNegLabeld(
        keys=spatial_keys,
        label_key=label_key,
        spatial_size=ROI_SIZE_FT,
        pos=1.0, neg=1.0,
        num_samples=2,
        image_key=image_key, image_threshold=0.0
    ),
    # ... Rand augmentations ...
    RandFlipd(keys=spatial_keys, prob=0.5, spatial_axis=0),
    RandFlipd(keys=spatial_keys, prob=0.5, spatial_axis=1),
    RandFlipd(keys=spatial_keys, prob=0.5, spatial_axis=2),
    RandAffined(keys=spatial_keys, prob=0.2, rotate_range=(0.05, 0.05, 0.05), scale_range=(0.1, 0.1, 0.1), mode=("bilinear", "nearest"), padding_mode="zeros"),
    RandGaussianNoised(keys=image_key, prob=0.2, mean=0.0, std=0.01),
    RandStdShiftIntensityd(keys=image_key, factors=0.1, prob=0.3),
    RandScaleIntensityd(keys=image_key, factors=0.1, prob=0.3),
    RandBiasFieldd(keys=image_key, prob=0.3),

    # --- ΔΙΟΡΘΩΣΗ 2: Εφάρμοσε το EnsureTyped σε ΟΛΑ τα κλειδιά ---
    # Τώρα περιλαμβάνει και το radiomics!
    EnsureTyped(keys=float_tensor_keys, dtype=torch.float32),
    EnsureTyped(keys=long_tensor_keys, dtype=torch.long),
])

# --- Transforms Επικύρωσης & Δοκιμής (Validation & Test) ---
val_transforms_mtl = Compose([
    LoadImaged(keys=spatial_keys, image_only=False),
    EnsureChannelFirstd(keys=spatial_keys),
    Orientationd(keys=spatial_keys, axcodes="RAS"),
    Spacingd(keys=spatial_keys, pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys=image_key, a_min=0, a_max=3000, b_min=0.0, b_max=1.0, clip=True),
    CenterSpatialCropd(keys=spatial_keys, roi_size=ROI_SIZE_FT),

    # --- ΔΙΟΡΘΩΣΗ 2 ---
    EnsureTyped(keys=float_tensor_keys, dtype=torch.float32),
    EnsureTyped(keys=long_tensor_keys, dtype=torch.long),
])

test_transforms_mtl = val_transforms_mtl

# --- DataLoaders ---
mtl_cache_dir = os.path.join(cache_base_dir, "mtl_run_v2_radiomics") # Changed folder name to avoid conflict
print(f"Δημιουργία cache στο: {mtl_cache_dir}")

print("Καθαρισμός παλιάς cache (αν υπάρχει)...")
shutil.rmtree(mtl_cache_dir, ignore_errors=True)

train_ds_mtl = PersistentDataset(
    data=train_files_mtl,
    transform=train_transforms_mtl,
    cache_dir=os.path.join(mtl_cache_dir, "train")
)
val_ds_mtl = PersistentDataset(
    data=val_files_mtl,
    transform=val_transforms_mtl,
    cache_dir=os.path.join(mtl_cache_dir, "val")
)
test_ds_mtl = PersistentDataset(
    data=test_files_mtl,
    transform=test_transforms_mtl,
    cache_dir=os.path.join(mtl_cache_dir, "test")
)

train_loader_mtl = MonaiDataLoader(
    train_ds_mtl,
    batch_size=BATCH_SIZE_FT,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=list_data_collate
)
val_loader_mtl = MonaiDataLoader(
    val_ds_mtl,
    batch_size=BATCH_SIZE_FT * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)
test_loader_mtl = MonaiDataLoader(
    test_ds_mtl,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("\nDataLoaders για V2 Fusion δημιουργήθηκαν.")
print(f"Training batches: {len(train_loader_mtl)}")
print(f"Validation batches: {len(val_loader_mtl)}")
print(f"Test batches: {len(test_loader_mtl)}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.networks.nets import SwinUNETR
import os

class AdvancedMaskGuidedModel_v2(nn.Module):
    def __init__(self,
                 in_channels_img, out_channels_seg, feature_size,
                 num_classes_grade, num_classes_diag,
                 clinical_len, radiomics_len,
                 d_model=128, n_heads=8, num_layers=2, dropout_rate=0.4,
                 use_checkpoint=True, finetuned_seg_model_path=None):
        super().__init__()

        # --- 1. SwinUNETR Backbone ---
        self.swin_unetr = SwinUNETR(
            in_channels=in_channels_img,
            out_channels=out_channels_seg,
            feature_size=feature_size,
            use_checkpoint=use_checkpoint
        )

        # Load weights if provided (Same as before)
        if finetuned_seg_model_path and os.path.exists(finetuned_seg_model_path):
            checkpoint = torch.load(finetuned_seg_model_path, map_location='cpu')
            try:
                self.swin_unetr.load_state_dict(checkpoint['model_state'], strict=False)
                print("✅ Weights loaded.")
            except: pass

        # --- 2. Hooks ---
        self.enc_feat_captured = None
        self.dec_feat_captured = None
        self.swin_unetr.swinViT.register_forward_hook(self._hook_encoder)
        self.swin_unetr.out.register_forward_hook(self._hook_decoder)

        # --- 3. Projections (NOW SEPARATE) ---
        # Image Projections
        self.encoder_proj = nn.Linear(768, d_model)
        self.decoder_proj = nn.Linear(feature_size, d_model)

        # Tabular Projections
        self.clinical_proj = nn.Sequential(
            nn.Linear(clinical_len, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU()
        )

        # NEW: Radiomics Projection
        self.radiomics_proj = nn.Sequential(
            nn.Linear(radiomics_len, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU()
        )

        # --- 4. Learnable Query Tokens ---
        self.grade_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.diag_token = nn.Parameter(torch.zeros(1, 1, d_model))

        # --- 5. Transformer Fusion ---
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dropout=dropout_rate, batch_first=True)
        self.fusion_transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # --- 6. Heads ---
        self.grade_head = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, num_classes_grade))
        self.diagnosis_head = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, num_classes_diag))

        # Aux Heads (Deep Supervision)
        self.aux_head_enc = nn.Linear(d_model, num_classes_grade)
        self.aux_head_dec = nn.Linear(d_model, num_classes_grade)
        # We can add aux heads for tabular data too, forcing them to be predictive
        self.aux_head_rad = nn.Linear(d_model, num_classes_grade)

    def _hook_encoder(self, module, input, output):
        if isinstance(output, (list, tuple)): self.enc_feat_captured = output[-1]
        else: self.enc_feat_captured = output

    def _hook_decoder(self, module, input, output):
        self.dec_feat_captured = input[0]

    def forward(self, x_img, x_clin, x_rad): # <--- NEW INPUT SIGNATURE
        # A. Backbone
        logits_seg = self.swin_unetr(x_img)

        # B. Masking & Visual Tokens
        seg_prob = torch.sigmoid(logits_seg)
        mask = torch.max(seg_prob, dim=1, keepdim=True)[0]

        # Encoder Token
        mask_enc = F.interpolate(mask, size=self.enc_feat_captured.shape[2:], mode='nearest')
        enc_vec = F.adaptive_avg_pool3d(self.enc_feat_captured * mask_enc, 1).flatten(1)
        enc_token = self.encoder_proj(enc_vec).unsqueeze(1)

        # Decoder Token
        mask_dec = F.interpolate(mask, size=self.dec_feat_captured.shape[2:], mode='nearest')
        dec_vec = F.adaptive_avg_pool3d(self.dec_feat_captured * mask_dec, 1).flatten(1)
        dec_token = self.decoder_proj(dec_vec).unsqueeze(1)

        # C. Tabular Tokens
        clin_token = self.clinical_proj(x_clin).unsqueeze(1)
        rad_token = self.radiomics_proj(x_rad).unsqueeze(1) # <--- NEW TOKEN

        # D. Modality Dropout (Robustness)
        if self.training:
            # 20% chance to drop Clinical, 20% chance to drop Radiomics
            # This forces the model not to rely on just one source
            if torch.rand(1).item() < 0.2: clin_token = torch.zeros_like(clin_token)
            if torch.rand(1).item() < 0.2: rad_token = torch.zeros_like(rad_token)

        # E. Fusion
        B = x_img.shape[0]
        tokens = torch.cat([
            self.grade_token.expand(B, -1, -1), # Index 0
            self.diag_token.expand(B, -1, -1),  # Index 1
            enc_token,                          # Index 2
            dec_token,                          # Index 3
            clin_token,                         # Index 4
            rad_token                           # Index 5
        ], dim=1)

        fused = self.fusion_transformer(tokens)

        # F. Heads
        # We pick the updated Query Tokens (Index 0 and 1)
        pred_grade = self.grade_head(fused[:, 0, :])
        pred_diag = self.diagnosis_head(fused[:, 1, :])

        # Aux Heads
        aux_enc = self.aux_head_enc(enc_token.squeeze(1))
        aux_dec = self.aux_head_dec(dec_token.squeeze(1))
        aux_rad = self.aux_head_rad(rad_token.squeeze(1))

        return logits_seg, pred_grade, pred_diag, aux_enc, aux_dec, aux_rad

In [ ]:
# %% Step 5: Training Advanced Mask-Guided MTL (Final V2 with Radiomics)
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import pandas as pd
import numpy as np
import os
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete, Activations, Compose
from monai.losses import DiceCELoss, FocalLoss
from monai.data import decollate_batch
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# --- 1. Class Weights ---
print("Computing Class Weights...")
try:
    y_grade_all = df_processed['Grade_Label'].values
    y_diag_all = df_processed['Diagnosis_Label'].values
    weights_grade = compute_class_weight('balanced', classes=np.unique(y_grade_all), y=y_grade_all)
    weights_diag = compute_class_weight('balanced', classes=np.unique(y_diag_all), y=y_diag_all)

    weights_grade_tensor = torch.tensor(weights_grade, dtype=torch.float).to(DEVICE)
    weights_diag_tensor = torch.tensor(weights_diag, dtype=torch.float).to(DEVICE)
    print("✅ Weights loaded.")
except Exception as e:
    weights_grade_tensor = None; weights_diag_tensor = None
    print(f"⚠️ Weights could not be computed (using uniform). Error: {e}")

# --- 2. Advanced Loss Function (6 Outputs) ---
class AdvancedFusionLoss(nn.Module):
    def __init__(self, weight_seg=1.0, weight_grade=1.0, weight_diag=0.5, weight_aux=0.2):
        super().__init__()
        self.w_seg = weight_seg
        self.w_grade = weight_grade
        self.w_diag = weight_diag
        self.w_aux = weight_aux

        # Losses
        self.seg_loss = DiceCELoss(include_background=False, sigmoid=True)
        # Focal Loss handles class imbalance better than CrossEntropy
        self.class_loss_grade = FocalLoss(include_background=True, to_onehot_y=True, gamma=2.0, weight=weights_grade_tensor)
        self.class_loss_diag = FocalLoss(include_background=True, to_onehot_y=True, gamma=2.0, weight=weights_diag_tensor)

    def forward(self, outputs, target_seg, target_grade, target_diag):
        # Unpack 6 outputs from V2 model
        logits_seg, p_grade, p_diag, aux_enc, aux_dec, aux_rad = outputs

        # A. Segmentation Loss (Dice)
        l_seg = self.seg_loss(logits_seg, target_seg)

        # B. Main Classification Losses
        l_grade = self.class_loss_grade(p_grade, target_grade)
        l_diag = self.class_loss_diag(p_diag, target_diag)

        # C. Auxiliary Losses (Deep Supervision)
        # We force intermediate tokens to predict Grade
        l_aux_enc = self.class_loss_grade(aux_enc, target_grade)
        l_aux_dec = self.class_loss_grade(aux_dec, target_grade)
        l_aux_rad = self.class_loss_grade(aux_rad, target_grade)

        # D. Weighted Sum
        total_loss = (
            (self.w_seg * l_seg) +
            (self.w_grade * l_grade) +
            (self.w_diag * l_diag) +
            (self.w_aux * (l_aux_enc + l_aux_dec + l_aux_rad))
        )
        return total_loss

criterion = AdvancedFusionLoss().to(DEVICE)

# --- 3. Settings ---
output_dir_mtl = os.path.join(runs_base_dir, "mtl_mask_guided_v2_final")
history_csv_path = os.path.join(output_dir_mtl, "history_v2_detailed.csv")
best_model_path = os.path.join(output_dir_mtl, "best_mask_model_v2.pt")
os.makedirs(output_dir_mtl, exist_ok=True)

# --- 4. Initialize Advanced Model V2 ---
# Ensure CLINICAL_FEATURE_SIZE and RADIOMICS_LEN are defined from Step 8
try:
    model_mtl = AdvancedMaskGuidedModel_v2(
        in_channels_img=IN_CHANNELS,
        out_channels_seg=OUT_CHANNELS_SEG,
        feature_size=FEATURE_SIZE,
        use_checkpoint=USE_CHECKPOINTING,
        num_classes_grade=NUM_CLASSES_GRADE,
        num_classes_diag=NUM_CLASSES_DIAG,
        clinical_len=CLINICAL_FEATURE_SIZE,
        radiomics_len=RADIOMICS_LEN,
        finetuned_seg_model_path=finetuned_model_path,
        d_model=128, n_heads=4, num_layers=2, dropout_rate=0.4
    ).to(DEVICE)
    print("✅ AdvancedMaskGuidedModel_v2 initialized.")
except NameError:
    print("❌ ERROR: AdvancedMaskGuidedModel_v2 class not found. Please run the model definition cell.")

# --- 5. Optimizer (Differential Learning Rates) ---
LR_BACKBONE = 1e-5
LR_HEADS = 2e-4

params = [
    {'params': model_mtl.swin_unetr.parameters(), 'lr': LR_BACKBONE},
    # Projections
    {'params': model_mtl.encoder_proj.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.decoder_proj.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.clinical_proj.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.radiomics_proj.parameters(), 'lr': LR_HEADS},
    # Transformer & Tokens
    {'params': [model_mtl.grade_token, model_mtl.diag_token], 'lr': LR_HEADS},
    {'params': model_mtl.fusion_transformer.parameters(), 'lr': LR_HEADS},
    # Heads
    {'params': model_mtl.grade_head.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.diagnosis_head.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.aux_head_enc.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.aux_head_dec.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.aux_head_rad.parameters(), 'lr': LR_HEADS},
]

optimizer = torch.optim.AdamW(params, weight_decay=WEIGHT_DECAY_MTL)

# Restart Strategy
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2, eta_min=1e-6)

# --- 6. Metrics & Loop Setup ---
dice_metric_train = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)
dice_metric_val = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)
post_pred_seg = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
post_label_seg = Compose([AsDiscrete(to_onehot=None)])

best_val_metric = -1.0
start_epoch = 0
GRAD_ACCUM_STEPS = 4 # Adjust based on GPU Memory

history = {
    "epoch": [],
    "train_loss": [], "train_dice": [],
    "train_f1_g": [], "train_acc_g": [], "train_auc_g": [],
    "train_f1_d": [], "train_acc_d": [], "train_auc_d": [],
    "val_loss": [], "val_dice": [],
    "val_f1_g": [], "val_acc_g": [], "val_auc_g": [],
    "val_f1_d": [], "val_acc_d": [], "val_auc_d": [],
    "combined_score": []
}

if os.path.exists(best_model_path): os.remove(best_model_path)

print(f"\n--- Starting V2 Training ({NUM_EPOCHS_MTL} Epochs) ---")

for epoch in range(start_epoch, NUM_EPOCHS_MTL):
    current_epoch = epoch + 1

    # === TRAIN ===
    model_mtl.train()
    dice_metric_train.reset()
    ep_loss = 0
    step = 0

    tr_pred_g, tr_probs_g, tr_lbl_g = [], [], []
    tr_pred_d, tr_probs_d, tr_lbl_d = [], [], []

    optimizer.zero_grad()
    pbar = tqdm(train_loader_mtl, desc=f"Epoch {current_epoch} Train")

    for i, batch in enumerate(pbar):
        step += 1
        img = batch[image_key].to(DEVICE)
        seg = batch[label_key].to(DEVICE)
        clin = batch[clinical_key].to(DEVICE)
        rad = batch[radiomics_key].to(DEVICE) # <--- RADIOMICS INPUT
        lbl_g = batch[grade_key].to(DEVICE)
        lbl_d = batch[diag_key].to(DEVICE)

        # Forward (3 inputs -> 6 outputs)
        outputs = model_mtl(img, clin, rad)

        # Loss
        loss = criterion(outputs, seg, lbl_g, lbl_d)

        if not torch.isfinite(loss): continue

        (loss / GRAD_ACCUM_STEPS).backward()

        if (i + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model_mtl.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

        ep_loss += loss.item()

        # Metrics Collection
        logits_seg, p_grade, p_diag, _, _, _ = outputs

        # Dice
        val_outputs = [post_pred_seg(i) for i in decollate_batch(logits_seg)]
        val_labels = [post_label_seg(i) for i in decollate_batch(seg)]
        dice_metric_train(y_pred=val_outputs, y=val_labels)

        # Predictions & Probs (for Metrics)
        tr_pred_g.extend(torch.argmax(p_grade, 1).cpu().numpy())
        tr_probs_g.extend(torch.softmax(p_grade, 1).detach().cpu().numpy())
        tr_lbl_g.extend(lbl_g.cpu().numpy())

        tr_pred_d.extend(torch.argmax(p_diag, 1).cpu().numpy())
        tr_probs_d.extend(torch.softmax(p_diag, 1).detach().cpu().numpy())
        tr_lbl_d.extend(lbl_d.cpu().numpy())

        pbar.set_postfix(loss=f"{loss.item():.2f}")

    # --- Train Stats ---
    train_loss_avg = ep_loss / step
    train_dice = dice_metric_train.aggregate().item()

    # Grade Metrics
    train_f1_g = f1_score(tr_lbl_g, tr_pred_g, average='macro')
    train_acc_g = accuracy_score(tr_lbl_g, tr_pred_g)
    try: train_auc_g = roc_auc_score(tr_lbl_g, tr_probs_g, multi_class='ovr', average='macro')
    except: train_auc_g = 0.0

    # Diag Metrics
    train_f1_d = f1_score(tr_lbl_d, tr_pred_d, average='macro')
    train_acc_d = accuracy_score(tr_lbl_d, tr_pred_d)
    try: train_auc_d = roc_auc_score(tr_lbl_d, tr_probs_d, multi_class='ovr', average='macro')
    except: train_auc_d = 0.0

    history["epoch"].append(current_epoch)
    history["train_loss"].append(train_loss_avg)
    history["train_dice"].append(train_dice)
    history["train_f1_g"].append(train_f1_g); history["train_acc_g"].append(train_acc_g); history["train_auc_g"].append(train_auc_g)
    history["train_f1_d"].append(train_f1_d); history["train_acc_d"].append(train_acc_d); history["train_auc_d"].append(train_auc_d)

    print(f"Train | Loss: {train_loss_avg:.3f} | Dice: {train_dice:.3f} | G_F1: {train_f1_g:.3f} | G_AUC: {train_auc_g:.3f}")

    # === VALIDATION ===
    if current_epoch % VAL_INTERVAL == 0:
        model_mtl.eval()
        dice_metric_val.reset()
        val_loss_sum = 0
        val_steps = 0

        preds_g, probs_g, lbls_g = [], [], []
        preds_d, probs_d, lbls_d = [], [], []

        with torch.no_grad():
            for batch in tqdm(val_loader_mtl, desc="Validating"):
                val_steps += 1
                img = batch[image_key].to(DEVICE)
                seg = batch[label_key].to(DEVICE)
                clin = batch[clinical_key].to(DEVICE)
                rad = batch[radiomics_key].to(DEVICE) # <--- RADIOMICS INPUT
                lbl_g = batch[grade_key].to(DEVICE)
                lbl_d = batch[diag_key].to(DEVICE)

                # Forward
                outputs = model_mtl(img, clin, rad)

                # Loss
                loss = criterion(outputs, seg, lbl_g, lbl_d)
                val_loss_sum += loss.item()

                logits_seg, p_grade, p_diag, _, _, _ = outputs

                # Dice
                val_outputs = [post_pred_seg(i) for i in decollate_batch(logits_seg)]
                val_labels = [post_label_seg(i) for i in decollate_batch(seg)]
                dice_metric_val(y_pred=val_outputs, y=val_labels)

                # Collect Metrics
                preds_g.extend(torch.argmax(p_grade, 1).cpu().numpy())
                probs_g.extend(torch.softmax(p_grade, 1).cpu().numpy())
                lbls_g.extend(lbl_g.cpu().numpy())

                preds_d.extend(torch.argmax(p_diag, 1).cpu().numpy())
                probs_d.extend(torch.softmax(p_diag, 1).cpu().numpy())
                lbls_d.extend(lbl_d.cpu().numpy())

        # --- Val Stats ---
        val_dice = dice_metric_val.aggregate().item()

        # Grade
        val_f1_g = f1_score(lbls_g, preds_g, average='macro')
        val_acc_g = accuracy_score(lbls_g, preds_g)
        try: val_auc_g = roc_auc_score(lbls_g, probs_g, multi_class='ovr', average='macro')
        except: val_auc_g = 0.0

        # Diag
        val_f1_d = f1_score(lbls_d, preds_d, average='macro')
        val_acc_d = accuracy_score(lbls_d, preds_d)
        try: val_auc_d = roc_auc_score(lbls_d, probs_d, multi_class='ovr', average='macro')
        except: val_auc_d = 0.0

        # Combined Score (Prioritize Grade Accuracy & F1)
        # Weighting: 40% Grade F1, 30% Dice, 30% Diag
        combined = (val_dice * 0.3) + (val_f1_g * 0.4) + (val_f1_d * 0.3)

        history["val_loss"].append(val_loss_sum/val_steps)
        history["val_dice"].append(val_dice)
        history["val_f1_g"].append(val_f1_g); history["val_acc_g"].append(val_acc_g); history["val_auc_g"].append(val_auc_g)
        history["val_f1_d"].append(val_f1_d); history["val_acc_d"].append(val_acc_d); history["val_auc_d"].append(val_auc_d)
        history["combined_score"].append(combined)

        print(f"--- Val Epoch {current_epoch} ---")
        print(f"Combined: {combined:.4f} | Loss: {val_loss_sum/val_steps:.4f} | Dice: {val_dice:.4f}")
        print(f"Grade -> F1: {val_f1_g:.3f} | Acc: {val_acc_g:.3f} | AUC: {val_auc_g:.3f}")
        print(f"Diag  -> F1: {val_f1_d:.3f} | Acc: {val_acc_d:.3f} | AUC: {val_auc_d:.3f}")

        # Reports
        print("\n=== 🧠 TUMOR GRADE REPORT ===")
        print(classification_report(lbls_g, preds_g, target_names=[str(c) for c in GRADE_CLASSES], zero_division=0))
        print("\n=== 🧬 DIAGNOSIS REPORT ===")
        print(classification_report(lbls_d, preds_d, target_names=[str(c) for c in DIAG_CLASSES], zero_division=0))

        if combined > best_val_metric:
            best_val_metric = combined
            torch.save(model_mtl.state_dict(), best_model_path)
            print("🏆 New Best Model Saved!")

        pd.DataFrame(history).to_csv(history_csv_path, index=False)

    else:
        # Pad history for non-val epochs
        for k in ["val_loss", "val_dice", "val_f1_g", "val_acc_g", "val_auc_g", "val_f1_d", "val_acc_d", "val_auc_d", "combined_score"]:
            history[k].append(np.nan)

    if scheduler: scheduler.step()

print(f"Done. Best Combined Score: {best_val_metric:.4f}")

In [ ]:
# %% Step 6: Visualization of Results (Advanced Full History Analysis)
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np

# --- Settings ---
# Ensure this matches the directory from Step 5
target_dir = os.path.join(runs_base_dir, "mtl_mask_guided_v2_final")
# Ensure the CSV name matches what was saved in Step 5 (history_advanced_mask_detailed.csv)
csv_path = os.path.join(target_dir, "history_v2_detailed.csv")

print(f"🔍 Reading history file: {csv_path}")

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)

    # Separate rows that have validation data (not NaN in 'val_loss')
    # Using 'val_loss' as the anchor to find validation epochs
    df_val = df.dropna(subset=['val_loss'])

    epochs = df['epoch']
    val_epochs = df_val['epoch']

    # Plot Style
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(2, 2, figsize=(20, 12))
    fig.suptitle('Advanced Mask-Guided MTL Training Analysis', fontsize=20, weight='bold')

    # 1. LOSSES & COMBINED SCORE
    ax1 = axes[0, 0]
    if 'train_loss' in df.columns:
        ax1.plot(epochs, df['train_loss'], label='Train Loss', color='blue', alpha=0.6)
    if 'val_loss' in df_val.columns:
        ax1.plot(val_epochs, df_val['val_loss'], label='Val Loss', color='navy', linewidth=2, marker='o')

    # Combined Score (Target)
    if 'combined_score' in df_val.columns:
        ax1b = ax1.twinx()
        ax1b.plot(val_epochs, df_val['combined_score'], label='Combined Score', color='gold', linestyle='--', linewidth=2)
        ax1b.legend(loc='upper right')
        ax1b.set_ylabel('Score')

    ax1.set_title('Optimization Landscape', fontsize=14)
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.legend(loc='upper left')
    ax1.grid(True, linestyle='--', alpha=0.7)

    # 2. SEGMENTATION (DICE)
    ax2 = axes[0, 1]
    if 'train_dice' in df.columns:
        ax2.plot(epochs, df['train_dice'], label='Train Dice', color='gray', linestyle=':', alpha=0.8)
    if 'val_dice' in df_val.columns:
        ax2.plot(val_epochs, df_val['val_dice'], label='Val Dice', color='black', linewidth=2, marker='s')

    ax2.set_title('Task 1: Segmentation (Dice Score)', fontsize=14)
    ax2.set_xlabel('Epoch'); ax2.set_ylim(0, 1.0)
    ax2.legend(loc='lower right'); ax2.grid(True)

    # 3. GRADE CLASSIFICATION
    ax3 = axes[1, 0]
    if 'train_f1_g' in df.columns:
        ax3.plot(epochs, df['train_f1_g'], label='Train F1', color='lightgreen', linestyle='--')
    if 'val_f1_g' in df_val.columns:
        ax3.plot(val_epochs, df_val['val_f1_g'], label='Val F1', color='forestgreen', linewidth=2.5)

    # Accuracy (if available)
    if 'val_acc_g' in df_val.columns:
        ax3.plot(val_epochs, df_val['val_acc_g'], label='Val Acc', color='darkgreen', alpha=0.3, linestyle=':')

    ax3.set_title('Task 2: Tumor Grade Classification', fontsize=14)
    ax3.set_xlabel('Epoch'); ax3.set_ylim(0, 1.0)
    ax3.legend(loc='lower right'); ax3.grid(True)

    # 4. DIAGNOSIS CLASSIFICATION
    ax4 = axes[1, 1]
    has_diag = False

    # Check if Diagnosis columns exist
    if 'train_f1_d' in df.columns and not df['train_f1_d'].isnull().all():
        ax4.plot(epochs, df['train_f1_d'], label='Train F1', color='plum', linestyle='--')
        has_diag = True

    if 'val_f1_d' in df_val.columns and not df_val['val_f1_d'].isnull().all():
        ax4.plot(val_epochs, df_val['val_f1_d'], label='Val F1', color='darkmagenta', linewidth=2.5)
        has_diag = True

    # Accuracy (if available)
    if 'val_acc_d' in df_val.columns and not df_val['val_acc_d'].isnull().all():
        ax4.plot(val_epochs, df_val['val_acc_d'], label='Val Acc', color='indigo', alpha=0.6)

    ax4.set_title('Task 3: Diagnosis Classification', fontsize=14)
    ax4.set_xlabel('Epoch'); ax4.set_ylim(0, 1.0)
    if has_diag: ax4.legend(loc='lower right')
    ax4.grid(True)

    plt.tight_layout()
    plt.show()

    # --- Print Best Model Stats ---
    if 'combined_score' in df_val.columns:
        # Find the row with the max combined score
        best_idx = df_val['combined_score'].idxmax()
        best_row = df_val.loc[best_idx]

        print(f"\n🏆 BEST MODEL at Epoch {int(best_row['epoch'])}")
        print(f"Combined Score: {best_row['combined_score']:.4f}")

        if 'val_dice' in best_row:
            print(f"Seg Dice: {best_row['val_dice']:.4f}")

        if 'val_f1_g' in best_row:
            acc_str = f" (Acc: {best_row['val_acc_g']:.4f})" if 'val_acc_g' in best_row else ""
            print(f"Grade F1: {best_row['val_f1_g']:.4f}{acc_str}")

        # Robust check for Diagnosis
        if 'val_f1_d' in best_row and not pd.isna(best_row['val_f1_d']):
            acc_str = f" (Acc: {best_row['val_acc_d']:.4f})" if 'val_acc_d' in best_row else ""
            print(f"Diag F1:  {best_row['val_f1_d']:.4f}{acc_str}")
        else:
            print("Diag F1:  N/A (Column missing or NaN)")
else:
    print(f"❌ Error: CSV File not found at {csv_path}. Please run Step 5 training first.")

In [ ]:
# %% Βήμα 7 (ALL-IN-ONE): Dice, Classification, Attention & Visuals (V2 Compatible)
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
import numpy as np
import os
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete, Activations, Compose
from monai.data import decollate_batch

# --- 1. Hooks & Helpers ---
class TransformerInputCatcher:
    def __init__(self, model):
        self.tokens = None
        # Hook into the fusion transformer to capture input tokens
        self.hook = model.fusion_transformer.register_forward_pre_hook(self.save_input)
    def save_input(self, module, inputs):
        # Inputs to transformer are (src, src_mask...), we want src [Batch, Seq, Dim]
        self.tokens = inputs[0]
    def remove(self):
        self.hook.remove()

def get_attention_maps(model, catcher):
    tokens = catcher.tokens
    B, N, D = tokens.shape
    attn_maps = []

    with torch.no_grad():
        curr_tokens = tokens.clone()
        for layer in model.fusion_transformer.layers:
            attn_layer = layer.self_attn
            # Calculate Q, K, V manually to extract attention scores
            qkv = F.linear(curr_tokens, attn_layer.in_proj_weight, attn_layer.in_proj_bias)
            qkv = qkv.reshape(B, N, 3, D).permute(2, 0, 1, 3)
            q, k, v = qkv[0], qkv[1], qkv[2]

            # Scaled Dot-Product Attention
            scores = torch.matmul(q, k.transpose(-2, -1)) / (D ** 0.5)
            attn_maps.append(F.softmax(scores, dim=-1))

            # Pass through layer for next iteration
            curr_tokens = layer(curr_tokens)

    return attn_maps

# --- 2. Load Model V2 ---
model_path = os.path.join(runs_base_dir, "mtl_mask_guided_v2_final", "best_mask_model_v2.pt")
print(f"🔍 Φόρτωση Best Model V2: {model_path}")

try:
    # Ensure these variables exist from previous steps
    model_eval = AdvancedMaskGuidedModel_v2(
        in_channels_img=IN_CHANNELS,
        out_channels_seg=OUT_CHANNELS_SEG,
        feature_size=FEATURE_SIZE,
        use_checkpoint=False,
        num_classes_grade=NUM_CLASSES_GRADE,
        num_classes_diag=NUM_CLASSES_DIAG,
        clinical_len=CLINICAL_FEATURE_SIZE,
        radiomics_len=RADIOMICS_LEN,
        finetuned_seg_model_path=None, # No need to reload weights for backbone
        d_model=128, n_heads=4, num_layers=2, dropout_rate=0.4
    ).to(DEVICE)

    model_eval.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model_eval.eval()
    print("✅ Model loaded successfully.")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    raise e

# --- 3. Setup Metrics ---
dice_metric = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)
post_pred = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
post_label = Compose([AsDiscrete(to_onehot=None)])

# --- 4. Run Evaluation (Combined Val + Test) ---
print("\n🧪 Εκτέλεση Πλήρους Ελέγχου (Val + Test)...")

catcher = TransformerInputCatcher(model_eval)
total_L0, total_L1 = None, None
count = 0

all_preds_g, all_lbls_g = [], []
all_preds_d, all_lbls_d = [], []

sample_imgs = []
sample_masks_true = []
sample_masks_pred = []

# Use both sets for robust evaluation
loaders = [val_loader_mtl, test_loader_mtl]

with torch.no_grad():
    for loader in loaders:
        for batch in tqdm(loader, leave=False):
            img = batch[image_key].to(DEVICE)
            clin = batch[clinical_key].to(DEVICE)
            rad = batch[radiomics_key].to(DEVICE) # <--- NEW INPUT
            lbl_g = batch[grade_key].to(DEVICE)
            lbl_d = batch[diag_key].to(DEVICE)
            seg_true = batch[label_key].to(DEVICE)

            # Forward (Pass all 3 modalities)
            outputs = model_eval(img, clin, rad)
            logits_seg, p_grade, p_diag = outputs[0], outputs[1], outputs[2]

            # --- A. Segmentation Dice ---
            val_outputs = [post_pred(i) for i in decollate_batch(logits_seg)]
            val_labels = [post_label(i) for i in decollate_batch(seg_true)]
            dice_metric(y_pred=val_outputs, y=val_labels)

            # Save visuals
            if len(sample_imgs) < 3:
                mid_slice = img.shape[4] // 2
                sample_imgs.append(img[0, 0, :, :, mid_slice].cpu().numpy())
                sample_masks_true.append(seg_true[0, 0, :, :, mid_slice].cpu().numpy())
                sample_masks_pred.append(val_outputs[0][0, :, :, mid_slice].cpu().numpy())

            # --- B. Classification Metrics ---
            all_preds_g.extend(torch.argmax(p_grade, 1).cpu().numpy())
            all_lbls_g.extend(lbl_g.cpu().numpy())
            all_preds_d.extend(torch.argmax(p_diag, 1).cpu().numpy())
            all_lbls_d.extend(lbl_d.cpu().numpy())

            # --- C. Attention ---
            if catcher.tokens is not None:
                maps = get_attention_maps(model_eval, catcher)
                # Map 0 = Layer 1 (Mixing), Map 1 = Layer 2 (Refining)
                curr_L0 = maps[0].sum(dim=0).cpu()
                curr_L1 = maps[1].sum(dim=0).cpu()

                if total_L0 is None: total_L0 = curr_L0
                else: total_L0 += curr_L0

                if total_L1 is None: total_L1 = curr_L1
                else: total_L1 += curr_L1

                count += img.shape[0]

catcher.remove()

# --- 5. REPORTING ---
print("\n" + "="*60)
print(f"📊 FINAL COMBINED RESULTS (N={count})")
print("="*60)

# A. Dice Score
final_dice = dice_metric.aggregate().item()
print(f"\n🩺 SEGMENTATION DICE SCORE: {final_dice:.4f}")

# B. Classification
print("\n🧠 GRADE REPORT")
# labels must match GRADE_CLASSES indices
labels_g = list(range(len(GRADE_CLASSES)))
print(classification_report(all_lbls_g, all_preds_g, labels=labels_g, target_names=[str(c) for c in GRADE_CLASSES], zero_division=0))

print("\n🧬 DIAGNOSIS REPORT")
labels_d = list(range(len(DIAG_CLASSES)))
print(classification_report(all_lbls_d, all_preds_d, labels=labels_d, target_names=[str(c) for c in DIAG_CLASSES], zero_division=0))

# --- 6. VISUALIZATION ---

# A. Segmentation
print("\n👁️ Segmentation Check (Samples)")
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
for i in range(3):
    if i < len(sample_imgs):
        axes[i, 0].imshow(sample_imgs[i], cmap="gray"); axes[i, 0].set_title("MRI Image")
        axes[i, 0].axis('off')
        axes[i, 1].imshow(sample_masks_true[i], cmap="jet", interpolation='nearest'); axes[i, 1].set_title("True Mask")
        axes[i, 1].axis('off')
        axes[i, 2].imshow(sample_masks_pred[i], cmap="jet", interpolation='nearest'); axes[i, 2].set_title("Predicted Mask")
        axes[i, 2].axis('off')
plt.tight_layout(); plt.show()

# B. Attention Heatmap (The "Explainability" Plot)
if count > 0:
    mean_L0 = total_L0 / count
    mean_L1 = total_L1 / count

    # Updated Labels for 6 Tokens
    labels = ["Grade", "Diag", "Enc", "Dec", "Clin", "Rad"]

    fig, axes = plt.subplots(1, 2, figsize=(22, 9))
    fig.suptitle(f"Global Cross-Modal Attention (Avg over {count} patients)", fontsize=20, fontweight='bold')

    # Layer 0
    sns.heatmap(mean_L0.numpy(), annot=True, fmt=".2f", xticklabels=labels, yticklabels=labels, cmap="Blues", cbar=True, ax=axes[0], vmin=0)
    axes[0].set_title("Layer 1: Feature Mixing", fontsize=16)
    # Highlight the Grade Row (Row 0)
    axes[0].add_patch(plt.Rectangle((0, 0), 6, 1, fill=False, edgecolor='red', lw=3))

    # Layer 1
    sns.heatmap(mean_L1.numpy(), annot=True, fmt=".2f", xticklabels=labels, yticklabels=labels, cmap="Purples", cbar=True, ax=axes[1], vmin=0)
    axes[1].set_title("Layer 2: Final Decision", fontsize=16)
    # Highlight the Grade Row (Row 0)
    axes[1].add_patch(plt.Rectangle((0, 0), 6, 1, fill=False, edgecolor='red', lw=3))

    plt.tight_layout()
    plt.show()

    print("\n💡 INTERPRETATION GUIDE:")
    print("1. Look at the 'Grade' Row (Row 0) in Layer 2.")
    print("2. Which column is darker? (Enc/Dec = Image, Clin = Clinical, Rad = Radiomics)")
    print("3. If 'Rad' is dark, your Radiomics features are driving the decision!")

## Soft masking + higher Modality Dropout + weight decay + w/e

In [ ]:
# %% Step 5 (FIXED V3): Training with Soft Masking & Robust Deep Supervision
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import pandas as pd
import numpy as np
import os
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete, Activations, Compose
from monai.losses import DiceCELoss, FocalLoss
from monai.data import decollate_batch
from monai.networks.nets import SwinUNETR
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# --- 1. Class Weights ---
print("Computing Class Weights...")
try:
    y_grade_all = df_processed['Grade_Label'].values
    y_diag_all = df_processed['Diagnosis_Label'].values
    weights_grade = compute_class_weight('balanced', classes=np.unique(y_grade_all), y=y_grade_all)
    weights_diag = compute_class_weight('balanced', classes=np.unique(y_diag_all), y=y_diag_all)

    weights_grade_tensor = torch.tensor(weights_grade, dtype=torch.float).to(DEVICE)
    weights_diag_tensor = torch.tensor(weights_diag, dtype=torch.float).to(DEVICE)
    print("✅ Weights loaded.")
except Exception as e:
    weights_grade_tensor = None; weights_diag_tensor = None
    print(f"⚠️ Weights could not be computed (using uniform). Error: {e}")

# --- 2. Define Soft-Masking Model (V3) ---
class AdvancedMaskGuidedModel_v3_Soft(nn.Module):
    def __init__(self,
                 in_channels_img, out_channels_seg, feature_size,
                 num_classes_grade, num_classes_diag,
                 clinical_len, radiomics_len,
                 d_model=128, n_heads=4, num_layers=2, dropout_rate=0.5,
                 use_checkpoint=True,  # <--- FIXED: Added this argument
                 finetuned_seg_model_path=None):
        super().__init__()

        # Backbone
        self.swin_unetr = SwinUNETR(
            in_channels=in_channels_img,
            out_channels=out_channels_seg,
            feature_size=feature_size,
            use_checkpoint=use_checkpoint # <--- FIXED: Passed argument here
        )

        # Load Weights
        if finetuned_seg_model_path:
            try:
                weights = torch.load(finetuned_seg_model_path, map_location='cpu')
                if 'state_dict' in weights: weights = weights['state_dict']
                elif 'model_state' in weights: weights = weights['model_state']
                self.swin_unetr.load_state_dict(weights, strict=False)
                print("✅ Backbone Weights Loaded.")
            except Exception as e: print(f"⚠️ Weight loading skipped: {e}")

        # Hooks
        self.enc_feat_captured = None
        self.dec_feat_captured = None
        self.swin_unetr.swinViT.register_forward_hook(self._hook_encoder)
        self.swin_unetr.out.register_forward_hook(self._hook_decoder)

        # Projections
        self.encoder_proj = nn.Linear(768, d_model)
        self.decoder_proj = nn.Linear(feature_size, d_model)

        # Robust Tabular Projections (Dropout included)
        self.clinical_proj = nn.Sequential(
            nn.Linear(clinical_len, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.radiomics_proj = nn.Sequential(
            nn.Linear(radiomics_len, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # Transformer
        self.grade_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.diag_token = nn.Parameter(torch.zeros(1, 1, d_model))
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dropout=dropout_rate, batch_first=True)
        self.fusion_transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Heads
        self.grade_head = nn.Linear(d_model, num_classes_grade)
        self.diagnosis_head = nn.Linear(d_model, num_classes_diag)
        self.aux_head_enc = nn.Linear(d_model, num_classes_grade)
        self.aux_head_dec = nn.Linear(d_model, num_classes_grade)
        self.aux_head_rad = nn.Linear(d_model, num_classes_grade)

    def _hook_encoder(self, module, input, output):
        if isinstance(output, (list, tuple)): self.enc_feat_captured = output[-1]
        else: self.enc_feat_captured = output

    def _hook_decoder(self, module, input, output):
        self.dec_feat_captured = input[0]

    def forward(self, x_img, x_clin, x_rad):
        # A. Backbone
        logits_seg = self.swin_unetr(x_img)
        seg_prob = torch.sigmoid(logits_seg)

        # B. Soft Feature Masking (Residual: Feat * (1+Mask))
        raw_enc = self.enc_feat_captured
        raw_dec = self.dec_feat_captured

        # Interpolate mask to feature size
        mask_enc = F.interpolate(seg_prob, size=raw_enc.shape[2:], mode='trilinear', align_corners=False)
        mask_dec = F.interpolate(seg_prob, size=raw_dec.shape[2:], mode='trilinear', align_corners=False)

        # Apply Residual Masking
        enc_weighted = raw_enc * (1.0 + mask_enc)
        dec_weighted = raw_dec * (1.0 + mask_dec)

        # Pool to tokens
        enc_vec = F.adaptive_avg_pool3d(enc_weighted, 1).flatten(1)
        dec_vec = F.adaptive_avg_pool3d(dec_weighted, 1).flatten(1)

        enc_token = self.encoder_proj(enc_vec).unsqueeze(1)
        dec_token = self.decoder_proj(dec_vec).unsqueeze(1)

        # C. Tabular Tokens
        clin_token = self.clinical_proj(x_clin).unsqueeze(1)
        rad_token = self.radiomics_proj(x_rad).unsqueeze(1)

        # D. Modality Dropout
        if self.training:
            if torch.rand(1).item() < 0.5: rad_token = torch.zeros_like(rad_token)
            if torch.rand(1).item() < 0.4: clin_token = torch.zeros_like(clin_token)

        # E. Fusion
        B = x_img.shape[0]
        tokens = torch.cat([
            self.grade_token.expand(B, -1, -1),
            self.diag_token.expand(B, -1, -1),
            enc_token, dec_token, clin_token, rad_token
        ], dim=1)

        fused = self.fusion_transformer(tokens)

        # F. Outputs
        return (
            logits_seg,
            self.grade_head(fused[:, 0, :]),
            self.diagnosis_head(fused[:, 1, :]),
            self.aux_head_enc(enc_token.squeeze(1)),
            self.aux_head_dec(dec_token.squeeze(1)),
            self.aux_head_rad(rad_token.squeeze(1))
        )

# --- 3. Robust Loss Function ---
class RobustFusionLoss(nn.Module):
    def __init__(self, weights_grade, weights_diag):
        super().__init__()
        self.seg_loss = DiceCELoss(include_background=False, sigmoid=True)
        self.class_loss_grade = FocalLoss(include_background=True, to_onehot_y=True, gamma=2.0, weight=weights_grade)
        self.class_loss_diag = FocalLoss(include_background=True, to_onehot_y=True, gamma=2.0, weight=weights_diag)

    def forward(self, outputs, target_seg, target_grade, target_diag):
        logits_seg, p_grade, p_diag, aux_enc, aux_dec, aux_rad = outputs

        l_seg = self.seg_loss(logits_seg, target_seg)
        l_grade = self.class_loss_grade(p_grade, target_grade)
        l_diag = self.class_loss_diag(p_diag, target_diag)

        # Robust Deep Supervision
        l_aux_enc = self.class_loss_grade(aux_enc, target_grade) * 1.0
        l_aux_dec = self.class_loss_grade(aux_dec, target_grade) * 1.0
        l_aux_rad = self.class_loss_grade(aux_rad, target_grade) * 0.3

        total_loss = (l_seg + l_grade + 0.5*l_diag + l_aux_enc + l_aux_dec + l_aux_rad)
        return total_loss

criterion = RobustFusionLoss(weights_grade_tensor, weights_diag_tensor).to(DEVICE)

# --- 4. Settings ---
output_dir_mtl = os.path.join(runs_base_dir, "mtl_mask_guided_v3_soft")
history_csv_path = os.path.join(output_dir_mtl, "history_v3_detailed.csv")
best_model_path = os.path.join(output_dir_mtl, "best_mask_model_v3.pt")
os.makedirs(output_dir_mtl, exist_ok=True)

# --- 5. Initialize Model V3 ---
try:
    model_mtl = AdvancedMaskGuidedModel_v3_Soft(
        in_channels_img=IN_CHANNELS,
        out_channels_seg=OUT_CHANNELS_SEG,
        feature_size=FEATURE_SIZE,
        # Correctly passing use_checkpoint now:
        use_checkpoint=USE_CHECKPOINTING,
        num_classes_grade=NUM_CLASSES_GRADE,
        num_classes_diag=NUM_CLASSES_DIAG,
        clinical_len=CLINICAL_FEATURE_SIZE,
        radiomics_len=RADIOMICS_LEN,
        finetuned_seg_model_path=finetuned_model_path,
        d_model=128, n_heads=4, num_layers=2, dropout_rate=0.5
    ).to(DEVICE)
    print("✅ AdvancedMaskGuidedModel_v3_Soft initialized.")
except NameError as e:
    print(f"❌ ERROR: Model class initialization failed: {e}")

# --- 6. Optimizer (Increased Regularization) ---
LR_BACKBONE = 1e-5
LR_HEADS = 2e-4

optimizer = torch.optim.AdamW(model_mtl.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2, eta_min=1e-6)

# --- 7. Metrics & Loop Setup ---
dice_metric_train = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)
dice_metric_val = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)
post_pred_seg = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
post_label_seg = Compose([AsDiscrete(to_onehot=None)])

best_val_metric = -1.0
start_epoch = 0
GRAD_ACCUM_STEPS = 4

history = {
    "epoch": [],
    "train_loss": [], "train_dice": [],
    "train_f1_g": [], "train_acc_g": [], "train_auc_g": [],
    "train_f1_d": [], "train_acc_d": [], "train_auc_d": [],
    "val_loss": [], "val_dice": [],
    "val_f1_g": [], "val_acc_g": [], "val_auc_g": [],
    "val_f1_d": [], "val_acc_d": [], "val_auc_d": [],
    "combined_score": []
}

if os.path.exists(best_model_path): os.remove(best_model_path)

print(f"\n--- Starting V3 (Soft Mask + Robust) Training ({NUM_EPOCHS_MTL} Epochs) ---")

for epoch in range(start_epoch, NUM_EPOCHS_MTL):
    current_epoch = epoch + 1

    # === TRAIN ===
    model_mtl.train()
    dice_metric_train.reset()
    ep_loss = 0
    step = 0

    tr_pred_g, tr_probs_g, tr_lbl_g = [], [], []
    tr_pred_d, tr_probs_d, tr_lbl_d = [], [], []

    optimizer.zero_grad()
    pbar = tqdm(train_loader_mtl, desc=f"Epoch {current_epoch} Train")

    for i, batch in enumerate(pbar):
        step += 1
        img = batch[image_key].to(DEVICE)
        seg = batch[label_key].to(DEVICE)
        clin = batch[clinical_key].to(DEVICE)
        rad = batch[radiomics_key].to(DEVICE)
        lbl_g = batch[grade_key].to(DEVICE)
        lbl_d = batch[diag_key].to(DEVICE)

        outputs = model_mtl(img, clin, rad)
        loss = criterion(outputs, seg, lbl_g, lbl_d)

        if not torch.isfinite(loss): continue

        (loss / GRAD_ACCUM_STEPS).backward()

        if (i + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model_mtl.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

        ep_loss += loss.item()

        # Metrics
        logits_seg, p_grade, p_diag, _, _, _ = outputs
        val_outputs = [post_pred_seg(i) for i in decollate_batch(logits_seg)]
        val_labels = [post_label_seg(i) for i in decollate_batch(seg)]
        dice_metric_train(y_pred=val_outputs, y=val_labels)

        tr_pred_g.extend(torch.argmax(p_grade, 1).cpu().numpy())
        tr_probs_g.extend(torch.softmax(p_grade, 1).detach().cpu().numpy())
        tr_lbl_g.extend(lbl_g.cpu().numpy())

        tr_pred_d.extend(torch.argmax(p_diag, 1).cpu().numpy())
        tr_probs_d.extend(torch.softmax(p_diag, 1).detach().cpu().numpy())
        tr_lbl_d.extend(lbl_d.cpu().numpy())

        pbar.set_postfix(loss=f"{loss.item():.2f}")

    # --- Train Stats ---
    train_loss_avg = ep_loss / step
    train_dice = dice_metric_train.aggregate().item()

    train_f1_g = f1_score(tr_lbl_g, tr_pred_g, average='macro')
    train_acc_g = accuracy_score(tr_lbl_g, tr_pred_g)
    try: train_auc_g = roc_auc_score(tr_lbl_g, tr_probs_g, multi_class='ovr', average='macro')
    except: train_auc_g = 0.0

    train_f1_d = f1_score(tr_lbl_d, tr_pred_d, average='macro')
    train_acc_d = accuracy_score(tr_lbl_d, tr_pred_d)
    try: train_auc_d = roc_auc_score(tr_lbl_d, tr_probs_d, multi_class='ovr', average='macro')
    except: train_auc_d = 0.0

    history["epoch"].append(current_epoch)
    history["train_loss"].append(train_loss_avg)
    history["train_dice"].append(train_dice)
    history["train_f1_g"].append(train_f1_g); history["train_acc_g"].append(train_acc_g); history["train_auc_g"].append(train_auc_g)
    history["train_f1_d"].append(train_f1_d); history["train_acc_d"].append(train_acc_d); history["train_auc_d"].append(train_auc_d)

    print(f"Train | Loss: {train_loss_avg:.3f} | Dice: {train_dice:.3f} | G_F1: {train_f1_g:.3f} | G_AUC: {train_auc_g:.3f}")

    # === VALIDATION ===
    if current_epoch % VAL_INTERVAL == 0:
        model_mtl.eval()
        dice_metric_val.reset()
        val_loss_sum = 0
        val_steps = 0

        preds_g, probs_g, lbls_g = [], [], []
        preds_d, probs_d, lbls_d = [], [], []

        with torch.no_grad():
            for batch in tqdm(val_loader_mtl, desc="Validating"):
                val_steps += 1
                img = batch[image_key].to(DEVICE)
                seg = batch[label_key].to(DEVICE)
                clin = batch[clinical_key].to(DEVICE)
                rad = batch[radiomics_key].to(DEVICE)
                lbl_g = batch[grade_key].to(DEVICE)
                lbl_d = batch[diag_key].to(DEVICE)

                outputs = model_mtl(img, clin, rad)
                loss = criterion(outputs, seg, lbl_g, lbl_d)
                val_loss_sum += loss.item()

                logits_seg, p_grade, p_diag, _, _, _ = outputs

                val_outputs = [post_pred_seg(i) for i in decollate_batch(logits_seg)]
                val_labels = [post_label_seg(i) for i in decollate_batch(seg)]
                dice_metric_val(y_pred=val_outputs, y=val_labels)

                preds_g.extend(torch.argmax(p_grade, 1).cpu().numpy())
                probs_g.extend(torch.softmax(p_grade, 1).cpu().numpy())
                lbls_g.extend(lbl_g.cpu().numpy())

                preds_d.extend(torch.argmax(p_diag, 1).cpu().numpy())
                probs_d.extend(torch.softmax(p_diag, 1).cpu().numpy())
                lbls_d.extend(lbl_d.cpu().numpy())

        val_dice = dice_metric_val.aggregate().item()

        val_f1_g = f1_score(lbls_g, preds_g, average='macro')
        val_acc_g = accuracy_score(lbls_g, preds_g)
        try: val_auc_g = roc_auc_score(lbls_g, probs_g, multi_class='ovr', average='macro')
        except: val_auc_g = 0.0

        val_f1_d = f1_score(lbls_d, preds_d, average='macro')
        val_acc_d = accuracy_score(lbls_d, preds_d)
        try: val_auc_d = roc_auc_score(lbls_d, probs_d, multi_class='ovr', average='macro')
        except: val_auc_d = 0.0

        combined = (val_dice * 0.3) + (val_f1_g * 0.4) + (val_f1_d * 0.3)

        history["val_loss"].append(val_loss_sum/val_steps)
        history["val_dice"].append(val_dice)
        history["val_f1_g"].append(val_f1_g); history["val_acc_g"].append(val_acc_g); history["val_auc_g"].append(val_auc_g)
        history["val_f1_d"].append(val_f1_d); history["val_acc_d"].append(val_acc_d); history["val_auc_d"].append(val_auc_d)
        history["combined_score"].append(combined)

        print(f"--- Val Epoch {current_epoch} ---")
        print(f"Combined: {combined:.4f} | Loss: {val_loss_sum/val_steps:.4f} | Dice: {val_dice:.4f}")
        print(f"Grade -> F1: {val_f1_g:.3f} | Acc: {val_acc_g:.3f} | AUC: {val_auc_g:.3f}")
        print(f"Diag  -> F1: {val_f1_d:.3f} | Acc: {val_acc_d:.3f} | AUC: {val_auc_d:.3f}")

        print("\n=== 🧠 TUMOR GRADE REPORT ===")
        print(classification_report(lbls_g, preds_g, target_names=[str(c) for c in GRADE_CLASSES], zero_division=0))
        print("\n=== 🧬 DIAGNOSIS REPORT ===")
        print(classification_report(lbls_d, preds_d, target_names=[str(c) for c in DIAG_CLASSES], zero_division=0))

        if combined > best_val_metric:
            best_val_metric = combined
            torch.save(model_mtl.state_dict(), best_model_path)
            print("🏆 New Best Model Saved!")

        pd.DataFrame(history).to_csv(history_csv_path, index=False)

    else:
        for k in ["val_loss", "val_dice", "val_f1_g", "val_acc_g", "val_auc_g", "val_f1_d", "val_acc_d", "val_auc_d", "combined_score"]:
            history[k].append(np.nan)

    if scheduler: scheduler.step()

print(f"Done. Best Combined Score: {best_val_metric:.4f}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import pandas as pd
import numpy as np
import os
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete, Activations, Compose
from monai.losses import DiceCELoss, FocalLoss
from monai.data import decollate_batch
from monai.networks.nets import SwinUNETR
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# --- 1. Class Weights ---
print("Computing Class Weights...")
try:
    y_grade_all = df_processed['Grade_Label'].values
    y_diag_all = df_processed['Diagnosis_Label'].values
    weights_grade = compute_class_weight('balanced', classes=np.unique(y_grade_all), y=y_grade_all)
    weights_diag = compute_class_weight('balanced', classes=np.unique(y_diag_all), y=y_diag_all)

    weights_grade_tensor = torch.tensor(weights_grade, dtype=torch.float).to(DEVICE)
    weights_diag_tensor = torch.tensor(weights_diag, dtype=torch.float).to(DEVICE)
    print("✅ Weights loaded.")
except Exception as e:
    weights_grade_tensor = None; weights_diag_tensor = None
    print(f"⚠️ Weights could not be computed (using uniform). Error: {e}")

# --- 2. Define Soft-Masking Model (V3) ---
class AdvancedMaskGuidedModel_v3_Soft(nn.Module):
    def __init__(self,
                 in_channels_img, out_channels_seg, feature_size,
                 num_classes_grade, num_classes_diag,
                 clinical_len, radiomics_len,
                 d_model=128, n_heads=4, num_layers=2, dropout_rate=0.5,
                 use_checkpoint=True,  # <--- FIXED: Added this argument
                 finetuned_seg_model_path=None):
        super().__init__()

        # Backbone
        self.swin_unetr = SwinUNETR(
            in_channels=in_channels_img,
            out_channels=out_channels_seg,
            feature_size=feature_size,
            use_checkpoint=use_checkpoint # <--- FIXED: Passed argument here
        )

        # Load Weights
        if finetuned_seg_model_path:
            try:
                weights = torch.load(finetuned_seg_model_path, map_location='cpu')
                if 'state_dict' in weights: weights = weights['state_dict']
                elif 'model_state' in weights: weights = weights['model_state']
                self.swin_unetr.load_state_dict(weights, strict=False)
                print("✅ Backbone Weights Loaded.")
            except Exception as e: print(f"⚠️ Weight loading skipped: {e}")

        # Hooks
        self.enc_feat_captured = None
        self.dec_feat_captured = None
        self.swin_unetr.swinViT.register_forward_hook(self._hook_encoder)
        self.swin_unetr.out.register_forward_hook(self._hook_decoder)

        # Projections
        self.encoder_proj = nn.Linear(768, d_model)
        self.decoder_proj = nn.Linear(feature_size, d_model)

        # Robust Tabular Projections (Dropout included)
        self.clinical_proj = nn.Sequential(
            nn.Linear(clinical_len, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.radiomics_proj = nn.Sequential(
            nn.Linear(radiomics_len, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # Transformer
        self.grade_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.diag_token = nn.Parameter(torch.zeros(1, 1, d_model))
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dropout=dropout_rate, batch_first=True)
        self.fusion_transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Heads
        self.grade_head = nn.Linear(d_model, num_classes_grade)
        self.diagnosis_head = nn.Linear(d_model, num_classes_diag)
        self.aux_head_enc = nn.Linear(d_model, num_classes_grade)
        self.aux_head_dec = nn.Linear(d_model, num_classes_grade)
        self.aux_head_rad = nn.Linear(d_model, num_classes_grade)

    def _hook_encoder(self, module, input, output):
        if isinstance(output, (list, tuple)): self.enc_feat_captured = output[-1]
        else: self.enc_feat_captured = output

    def _hook_decoder(self, module, input, output):
        self.dec_feat_captured = input[0]

    def forward(self, x_img, x_clin, x_rad):
        # A. Backbone
        logits_seg = self.swin_unetr(x_img)
        seg_prob = torch.sigmoid(logits_seg)

        # B. Soft Feature Masking (Residual: Feat * (1+Mask))
        raw_enc = self.enc_feat_captured
        raw_dec = self.dec_feat_captured

        # Interpolate mask to feature size
        mask_enc = F.interpolate(seg_prob, size=raw_enc.shape[2:], mode='trilinear', align_corners=False)
        mask_dec = F.interpolate(seg_prob, size=raw_dec.shape[2:], mode='trilinear', align_corners=False)

        # Apply Residual Masking
        enc_weighted = raw_enc * (1.0 + mask_enc)
        dec_weighted = raw_dec * (1.0 + mask_dec)

        # Pool to tokens
        enc_vec = F.adaptive_avg_pool3d(enc_weighted, 1).flatten(1)
        dec_vec = F.adaptive_avg_pool3d(dec_weighted, 1).flatten(1)

        enc_token = self.encoder_proj(enc_vec).unsqueeze(1)
        dec_token = self.decoder_proj(dec_vec).unsqueeze(1)

        # C. Tabular Tokens
        clin_token = self.clinical_proj(x_clin).unsqueeze(1)
        rad_token = self.radiomics_proj(x_rad).unsqueeze(1)

        # D. Modality Dropout
        if self.training:
            if torch.rand(1).item() < 0.5: rad_token = torch.zeros_like(rad_token)
            if torch.rand(1).item() < 0.4: clin_token = torch.zeros_like(clin_token)

        # E. Fusion
        B = x_img.shape[0]
        tokens = torch.cat([
            self.grade_token.expand(B, -1, -1),
            self.diag_token.expand(B, -1, -1),
            enc_token, dec_token, clin_token, rad_token
        ], dim=1)

        fused = self.fusion_transformer(tokens)

        # F. Outputs
        return (
            logits_seg,
            self.grade_head(fused[:, 0, :]),
            self.diagnosis_head(fused[:, 1, :]),
            self.aux_head_enc(enc_token.squeeze(1)),
            self.aux_head_dec(dec_token.squeeze(1)),
            self.aux_head_rad(rad_token.squeeze(1))
        )

# --- 3. Robust Loss Function ---
class RobustFusionLoss(nn.Module):
    def __init__(self, weights_grade, weights_diag):
        super().__init__()
        self.seg_loss = DiceCELoss(include_background=False, sigmoid=True)
        self.class_loss_grade = FocalLoss(include_background=True, to_onehot_y=True, gamma=2.0, weight=weights_grade)
        self.class_loss_diag = FocalLoss(include_background=True, to_onehot_y=True, gamma=2.0, weight=weights_diag)

    def forward(self, outputs, target_seg, target_grade, target_diag):
        logits_seg, p_grade, p_diag, aux_enc, aux_dec, aux_rad = outputs

        l_seg = self.seg_loss(logits_seg, target_seg)
        l_grade = self.class_loss_grade(p_grade, target_grade)
        l_diag = self.class_loss_diag(p_diag, target_diag)

        # Robust Deep Supervision
        l_aux_enc = self.class_loss_grade(aux_enc, target_grade) * 1.0
        l_aux_dec = self.class_loss_grade(aux_dec, target_grade) * 1.0
        l_aux_rad = self.class_loss_grade(aux_rad, target_grade) * 0.3

        total_loss = (l_seg + l_grade + 0.5*l_diag + l_aux_enc + l_aux_dec + l_aux_rad)
        return total_loss

criterion = RobustFusionLoss(weights_grade_tensor, weights_diag_tensor).to(DEVICE)

In [ ]:
# %% Step 6: Visualization of Results (Advanced Full History Analysis)
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np

# --- Settings ---
# Ensure this matches the directory from Step 5
target_dir = os.path.join(runs_base_dir, "mtl_mask_guided_v3_soft")
# Ensure the CSV name matches what was saved in Step 5 (history_advanced_mask_detailed.csv)
csv_path = os.path.join(target_dir, "history_v3_detailed.csv")

print(f"🔍 Reading history file: {csv_path}")

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)

    # Separate rows that have validation data (not NaN in 'val_loss')
    # Using 'val_loss' as the anchor to find validation epochs
    df_val = df.dropna(subset=['val_loss'])

    epochs = df['epoch']
    val_epochs = df_val['epoch']

    # Plot Style
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(2, 2, figsize=(20, 12))
    fig.suptitle('Advanced Mask-Guided MTL Training Analysis', fontsize=20, weight='bold')

    # 1. LOSSES & COMBINED SCORE
    ax1 = axes[0, 0]
    if 'train_loss' in df.columns:
        ax1.plot(epochs, df['train_loss'], label='Train Loss', color='blue', alpha=0.6)
    if 'val_loss' in df_val.columns:
        ax1.plot(val_epochs, df_val['val_loss'], label='Val Loss', color='navy', linewidth=2, marker='o')

    # Combined Score (Target)
    if 'combined_score' in df_val.columns:
        ax1b = ax1.twinx()
        ax1b.plot(val_epochs, df_val['combined_score'], label='Combined Score', color='gold', linestyle='--', linewidth=2)
        ax1b.legend(loc='upper right')
        ax1b.set_ylabel('Score')

    ax1.set_title('Optimization Landscape', fontsize=14)
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.legend(loc='upper left')
    ax1.grid(True, linestyle='--', alpha=0.7)

    # 2. SEGMENTATION (DICE)
    ax2 = axes[0, 1]
    if 'train_dice' in df.columns:
        ax2.plot(epochs, df['train_dice'], label='Train Dice', color='gray', linestyle=':', alpha=0.8)
    if 'val_dice' in df_val.columns:
        ax2.plot(val_epochs, df_val['val_dice'], label='Val Dice', color='black', linewidth=2, marker='s')

    ax2.set_title('Task 1: Segmentation (Dice Score)', fontsize=14)
    ax2.set_xlabel('Epoch'); ax2.set_ylim(0, 1.0)
    ax2.legend(loc='lower right'); ax2.grid(True)

    # 3. GRADE CLASSIFICATION
    ax3 = axes[1, 0]
    if 'train_f1_g' in df.columns:
        ax3.plot(epochs, df['train_f1_g'], label='Train F1', color='lightgreen', linestyle='--')
    if 'val_f1_g' in df_val.columns:
        ax3.plot(val_epochs, df_val['val_f1_g'], label='Val F1', color='forestgreen', linewidth=2.5)

    # Accuracy (if available)
    if 'val_acc_g' in df_val.columns:
        ax3.plot(val_epochs, df_val['val_acc_g'], label='Val Acc', color='darkgreen', alpha=0.3, linestyle=':')

    ax3.set_title('Task 2: Tumor Grade Classification', fontsize=14)
    ax3.set_xlabel('Epoch'); ax3.set_ylim(0, 1.0)
    ax3.legend(loc='lower right'); ax3.grid(True)

    # 4. DIAGNOSIS CLASSIFICATION
    ax4 = axes[1, 1]
    has_diag = False

    # Check if Diagnosis columns exist
    if 'train_f1_d' in df.columns and not df['train_f1_d'].isnull().all():
        ax4.plot(epochs, df['train_f1_d'], label='Train F1', color='plum', linestyle='--')
        has_diag = True

    if 'val_f1_d' in df_val.columns and not df_val['val_f1_d'].isnull().all():
        ax4.plot(val_epochs, df_val['val_f1_d'], label='Val F1', color='darkmagenta', linewidth=2.5)
        has_diag = True

    # Accuracy (if available)
    if 'val_acc_d' in df_val.columns and not df_val['val_acc_d'].isnull().all():
        ax4.plot(val_epochs, df_val['val_acc_d'], label='Val Acc', color='indigo', alpha=0.6)

    ax4.set_title('Task 3: Diagnosis Classification', fontsize=14)
    ax4.set_xlabel('Epoch'); ax4.set_ylim(0, 1.0)
    if has_diag: ax4.legend(loc='lower right')
    ax4.grid(True)

    plt.tight_layout()
    plt.show()

    # --- Print Best Model Stats ---
    if 'combined_score' in df_val.columns:
        # Find the row with the max combined score
        best_idx = df_val['combined_score'].idxmax()
        best_row = df_val.loc[best_idx]

        print(f"\n🏆 BEST MODEL at Epoch {int(best_row['epoch'])}")
        print(f"Combined Score: {best_row['combined_score']:.4f}")

        if 'val_dice' in best_row:
            print(f"Seg Dice: {best_row['val_dice']:.4f}")

        if 'val_f1_g' in best_row:
            acc_str = f" (Acc: {best_row['val_acc_g']:.4f})" if 'val_acc_g' in best_row else ""
            print(f"Grade F1: {best_row['val_f1_g']:.4f}{acc_str}")

        # Robust check for Diagnosis
        if 'val_f1_d' in best_row and not pd.isna(best_row['val_f1_d']):
            acc_str = f" (Acc: {best_row['val_acc_d']:.4f})" if 'val_acc_d' in best_row else ""
            print(f"Diag F1:  {best_row['val_f1_d']:.4f}{acc_str}")
        else:
            print("Diag F1:  N/A (Column missing or NaN)")
else:
    print(f"❌ Error: CSV File not found at {csv_path}. Please run Step 5 training first.")

In [ ]:
# %% Βήμα 7 (ALL-IN-ONE): Dice, Classification, Attention & Visuals (V2 Compatible)
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
import numpy as np
import os
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete, Activations, Compose
from monai.data import decollate_batch

# --- 1. Hooks & Helpers ---
class TransformerInputCatcher:
    def __init__(self, model):
        self.tokens = None
        # Hook into the fusion transformer to capture input tokens
        self.hook = model.fusion_transformer.register_forward_pre_hook(self.save_input)
    def save_input(self, module, inputs):
        # Inputs to transformer are (src, src_mask...), we want src [Batch, Seq, Dim]
        self.tokens = inputs[0]
    def remove(self):
        self.hook.remove()

def get_attention_maps(model, catcher):
    tokens = catcher.tokens
    B, N, D = tokens.shape
    attn_maps = []

    with torch.no_grad():
        curr_tokens = tokens.clone()
        for layer in model.fusion_transformer.layers:
            attn_layer = layer.self_attn
            # Calculate Q, K, V manually to extract attention scores
            qkv = F.linear(curr_tokens, attn_layer.in_proj_weight, attn_layer.in_proj_bias)
            qkv = qkv.reshape(B, N, 3, D).permute(2, 0, 1, 3)
            q, k, v = qkv[0], qkv[1], qkv[2]

            # Scaled Dot-Product Attention
            scores = torch.matmul(q, k.transpose(-2, -1)) / (D ** 0.5)
            attn_maps.append(F.softmax(scores, dim=-1))

            # Pass through layer for next iteration
            curr_tokens = layer(curr_tokens)

    return attn_maps

# --- 2. Load Model V2 ---
model_path = os.path.join(runs_base_dir, "mtl_mask_guided_v3_soft", "best_mask_model_v3.pt")
print(f"🔍 Φόρτωση Best Model V2: {model_path}")

try:
    # Ensure these variables exist from previous steps
    model_eval = AdvancedMaskGuidedModel_v3_Soft(
        in_channels_img=IN_CHANNELS,
        out_channels_seg=OUT_CHANNELS_SEG,
        feature_size=FEATURE_SIZE,
        use_checkpoint=False,
        num_classes_grade=NUM_CLASSES_GRADE,
        num_classes_diag=NUM_CLASSES_DIAG,
        clinical_len=CLINICAL_FEATURE_SIZE,
        radiomics_len=RADIOMICS_LEN,
        finetuned_seg_model_path=None, # No need to reload weights for backbone
        d_model=128, n_heads=4, num_layers=2, dropout_rate=0.4
    ).to(DEVICE)

    model_eval.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model_eval.eval()
    print("✅ Model loaded successfully.")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    raise e

# --- 3. Setup Metrics ---
dice_metric = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)
post_pred = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
post_label = Compose([AsDiscrete(to_onehot=None)])

# --- 4. Run Evaluation (Combined Val + Test) ---
print("\n🧪 Εκτέλεση Πλήρους Ελέγχου (Val + Test)...")

catcher = TransformerInputCatcher(model_eval)
total_L0, total_L1 = None, None
count = 0

all_preds_g, all_lbls_g = [], []
all_preds_d, all_lbls_d = [], []

sample_imgs = []
sample_masks_true = []
sample_masks_pred = []

# Use both sets for robust evaluation
loaders = [test_loader_mtl]

with torch.no_grad():
    for loader in loaders:
        for batch in tqdm(loader, leave=False):
            img = batch[image_key].to(DEVICE)
            clin = batch[clinical_key].to(DEVICE)
            rad = batch[radiomics_key].to(DEVICE) # <--- NEW INPUT
            lbl_g = batch[grade_key].to(DEVICE)
            lbl_d = batch[diag_key].to(DEVICE)
            seg_true = batch[label_key].to(DEVICE)

            # Forward (Pass all 3 modalities)
            outputs = model_eval(img, clin, rad)
            logits_seg, p_grade, p_diag = outputs[0], outputs[1], outputs[2]

            # --- A. Segmentation Dice ---
            val_outputs = [post_pred(i) for i in decollate_batch(logits_seg)]
            val_labels = [post_label(i) for i in decollate_batch(seg_true)]
            dice_metric(y_pred=val_outputs, y=val_labels)

            # Save visuals
            if len(sample_imgs) < 3:
                mid_slice = img.shape[4] // 2
                sample_imgs.append(img[0, 0, :, :, mid_slice].cpu().numpy())
                sample_masks_true.append(seg_true[0, 0, :, :, mid_slice].cpu().numpy())
                sample_masks_pred.append(val_outputs[0][0, :, :, mid_slice].cpu().numpy())

            # --- B. Classification Metrics ---
            all_preds_g.extend(torch.argmax(p_grade, 1).cpu().numpy())
            all_lbls_g.extend(lbl_g.cpu().numpy())
            all_preds_d.extend(torch.argmax(p_diag, 1).cpu().numpy())
            all_lbls_d.extend(lbl_d.cpu().numpy())

            # --- C. Attention ---
            if catcher.tokens is not None:
                maps = get_attention_maps(model_eval, catcher)
                # Map 0 = Layer 1 (Mixing), Map 1 = Layer 2 (Refining)
                curr_L0 = maps[0].sum(dim=0).cpu()
                curr_L1 = maps[1].sum(dim=0).cpu()

                if total_L0 is None: total_L0 = curr_L0
                else: total_L0 += curr_L0

                if total_L1 is None: total_L1 = curr_L1
                else: total_L1 += curr_L1

                count += img.shape[0]

catcher.remove()

# --- 5. REPORTING ---
print("\n" + "="*60)
print(f"📊 FINAL COMBINED RESULTS (N={count})")
print("="*60)

# A. Dice Score
final_dice = dice_metric.aggregate().item()
print(f"\n🩺 SEGMENTATION DICE SCORE: {final_dice:.4f}")

# B. Classification
print("\n🧠 GRADE REPORT")
# labels must match GRADE_CLASSES indices
labels_g = list(range(len(GRADE_CLASSES)))
print(classification_report(all_lbls_g, all_preds_g, labels=labels_g, target_names=[str(c) for c in GRADE_CLASSES], zero_division=0))

print("\n🧬 DIAGNOSIS REPORT")
labels_d = list(range(len(DIAG_CLASSES)))
print(classification_report(all_lbls_d, all_preds_d, labels=labels_d, target_names=[str(c) for c in DIAG_CLASSES], zero_division=0))

# --- 6. VISUALIZATION ---

# A. Segmentation
print("\n👁️ Segmentation Check (Samples)")
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
for i in range(3):
    if i < len(sample_imgs):
        axes[i, 0].imshow(sample_imgs[i], cmap="gray"); axes[i, 0].set_title("MRI Image")
        axes[i, 0].axis('off')
        axes[i, 1].imshow(sample_masks_true[i], cmap="jet", interpolation='nearest'); axes[i, 1].set_title("True Mask")
        axes[i, 1].axis('off')
        axes[i, 2].imshow(sample_masks_pred[i], cmap="jet", interpolation='nearest'); axes[i, 2].set_title("Predicted Mask")
        axes[i, 2].axis('off')
plt.tight_layout(); plt.show()

# B. Attention Heatmap (The "Explainability" Plot)
if count > 0:
    mean_L0 = total_L0 / count
    mean_L1 = total_L1 / count

    # Updated Labels for 6 Tokens
    labels = ["Grade", "Diag", "Enc", "Dec", "Clin", "Rad"]

    fig, axes = plt.subplots(1, 2, figsize=(22, 9))
    fig.suptitle(f"Global Cross-Modal Attention (Avg over {count} patients)", fontsize=20, fontweight='bold')

    # Layer 0
    sns.heatmap(mean_L0.numpy(), annot=True, fmt=".2f", xticklabels=labels, yticklabels=labels, cmap="Blues", cbar=True, ax=axes[0], vmin=0)
    axes[0].set_title("Layer 1: Feature Mixing", fontsize=16)
    # Highlight the Grade Row (Row 0)
    axes[0].add_patch(plt.Rectangle((0, 0), 6, 1, fill=False, edgecolor='red', lw=3))

    # Layer 1
    sns.heatmap(mean_L1.numpy(), annot=True, fmt=".2f", xticklabels=labels, yticklabels=labels, cmap="Purples", cbar=True, ax=axes[1], vmin=0)
    axes[1].set_title("Layer 2: Final Decision", fontsize=16)
    # Highlight the Grade Row (Row 0)
    axes[1].add_patch(plt.Rectangle((0, 0), 6, 1, fill=False, edgecolor='red', lw=3))

    plt.tight_layout()
    plt.show()

    print("\n💡 INTERPRETATION GUIDE:")
    print("1. Look at the 'Grade' Row (Row 0) in Layer 2.")
    print("2. Which column is darker? (Enc/Dec = Image, Clin = Clinical, Rad = Radiomics)")
    print("3. If 'Rad' is dark, your Radiomics features are driving the decision!")

In [ ]:
# %% Βήμα 7 (ALL-IN-ONE): Dice, Classification, Attention & Visuals (V2 Compatible)
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
import numpy as np
import os
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete, Activations, Compose
from monai.data import decollate_batch

# --- 1. Hooks & Helpers ---
class TransformerInputCatcher:
    def __init__(self, model):
        self.tokens = None
        # Hook into the fusion transformer to capture input tokens
        self.hook = model.fusion_transformer.register_forward_pre_hook(self.save_input)
    def save_input(self, module, inputs):
        # Inputs to transformer are (src, src_mask...), we want src [Batch, Seq, Dim]
        self.tokens = inputs[0]
    def remove(self):
        self.hook.remove()

def get_attention_maps(model, catcher):
    tokens = catcher.tokens
    B, N, D = tokens.shape
    attn_maps = []

    with torch.no_grad():
        curr_tokens = tokens.clone()
        for layer in model.fusion_transformer.layers:
            attn_layer = layer.self_attn
            # Calculate Q, K, V manually to extract attention scores
            qkv = F.linear(curr_tokens, attn_layer.in_proj_weight, attn_layer.in_proj_bias)
            qkv = qkv.reshape(B, N, 3, D).permute(2, 0, 1, 3)
            q, k, v = qkv[0], qkv[1], qkv[2]

            # Scaled Dot-Product Attention
            scores = torch.matmul(q, k.transpose(-2, -1)) / (D ** 0.5)
            attn_maps.append(F.softmax(scores, dim=-1))

            # Pass through layer for next iteration
            curr_tokens = layer(curr_tokens)

    return attn_maps

# --- 2. Load Model V2 ---
model_path = os.path.join(runs_base_dir, "mtl_mask_guided_v3_soft", "best_mask_model_v3.pt")
print(f"🔍 Φόρτωση Best Model V2: {model_path}")

try:
    # Ensure these variables exist from previous steps
    model_eval = AdvancedMaskGuidedModel_v3_Soft(
        in_channels_img=IN_CHANNELS,
        out_channels_seg=OUT_CHANNELS_SEG,
        feature_size=FEATURE_SIZE,
        use_checkpoint=False,
        num_classes_grade=NUM_CLASSES_GRADE,
        num_classes_diag=NUM_CLASSES_DIAG,
        clinical_len=CLINICAL_FEATURE_SIZE,
        radiomics_len=RADIOMICS_LEN,
        finetuned_seg_model_path=None, # No need to reload weights for backbone
        d_model=128, n_heads=4, num_layers=2, dropout_rate=0.4
    ).to(DEVICE)

    model_eval.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model_eval.eval()
    print("✅ Model loaded successfully.")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    raise e

# --- 3. Setup Metrics ---
dice_metric = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)
post_pred = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
post_label = Compose([AsDiscrete(to_onehot=None)])

# --- 4. Run Evaluation (Combined Val + Test) ---
print("\n🧪 Εκτέλεση Πλήρους Ελέγχου (Val + Test)...")

catcher = TransformerInputCatcher(model_eval)
total_L0, total_L1 = None, None
count = 0

all_preds_g, all_lbls_g = [], []
all_preds_d, all_lbls_d = [], []

sample_imgs = []
sample_masks_true = []
sample_masks_pred = []

# Use both sets for robust evaluation
loaders = [val_loader_mtl]

with torch.no_grad():
    for loader in loaders:
        for batch in tqdm(loader, leave=False):
            img = batch[image_key].to(DEVICE)
            clin = batch[clinical_key].to(DEVICE)
            rad = batch[radiomics_key].to(DEVICE) # <--- NEW INPUT
            lbl_g = batch[grade_key].to(DEVICE)
            lbl_d = batch[diag_key].to(DEVICE)
            seg_true = batch[label_key].to(DEVICE)

            # Forward (Pass all 3 modalities)
            outputs = model_eval(img, clin, rad)
            logits_seg, p_grade, p_diag = outputs[0], outputs[1], outputs[2]

            # --- A. Segmentation Dice ---
            val_outputs = [post_pred(i) for i in decollate_batch(logits_seg)]
            val_labels = [post_label(i) for i in decollate_batch(seg_true)]
            dice_metric(y_pred=val_outputs, y=val_labels)

            # Save visuals
            if len(sample_imgs) < 3:
                mid_slice = img.shape[4] // 2
                sample_imgs.append(img[0, 0, :, :, mid_slice].cpu().numpy())
                sample_masks_true.append(seg_true[0, 0, :, :, mid_slice].cpu().numpy())
                sample_masks_pred.append(val_outputs[0][0, :, :, mid_slice].cpu().numpy())

            # --- B. Classification Metrics ---
            all_preds_g.extend(torch.argmax(p_grade, 1).cpu().numpy())
            all_lbls_g.extend(lbl_g.cpu().numpy())
            all_preds_d.extend(torch.argmax(p_diag, 1).cpu().numpy())
            all_lbls_d.extend(lbl_d.cpu().numpy())

            # --- C. Attention ---
            if catcher.tokens is not None:
                maps = get_attention_maps(model_eval, catcher)
                # Map 0 = Layer 1 (Mixing), Map 1 = Layer 2 (Refining)
                curr_L0 = maps[0].sum(dim=0).cpu()
                curr_L1 = maps[1].sum(dim=0).cpu()

                if total_L0 is None: total_L0 = curr_L0
                else: total_L0 += curr_L0

                if total_L1 is None: total_L1 = curr_L1
                else: total_L1 += curr_L1

                count += img.shape[0]

catcher.remove()

# --- 5. REPORTING ---
print("\n" + "="*60)
print(f"📊 FINAL COMBINED RESULTS (N={count})")
print("="*60)

# A. Dice Score
final_dice = dice_metric.aggregate().item()
print(f"\n🩺 SEGMENTATION DICE SCORE: {final_dice:.4f}")

# B. Classification
print("\n🧠 GRADE REPORT")
# labels must match GRADE_CLASSES indices
labels_g = list(range(len(GRADE_CLASSES)))
print(classification_report(all_lbls_g, all_preds_g, labels=labels_g, target_names=[str(c) for c in GRADE_CLASSES], zero_division=0))

print("\n🧬 DIAGNOSIS REPORT")
labels_d = list(range(len(DIAG_CLASSES)))
print(classification_report(all_lbls_d, all_preds_d, labels=labels_d, target_names=[str(c) for c in DIAG_CLASSES], zero_division=0))

# --- 6. VISUALIZATION ---

# A. Segmentation
print("\n👁️ Segmentation Check (Samples)")
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
for i in range(3):
    if i < len(sample_imgs):
        axes[i, 0].imshow(sample_imgs[i], cmap="gray"); axes[i, 0].set_title("MRI Image")
        axes[i, 0].axis('off')
        axes[i, 1].imshow(sample_masks_true[i], cmap="jet", interpolation='nearest'); axes[i, 1].set_title("True Mask")
        axes[i, 1].axis('off')
        axes[i, 2].imshow(sample_masks_pred[i], cmap="jet", interpolation='nearest'); axes[i, 2].set_title("Predicted Mask")
        axes[i, 2].axis('off')
plt.tight_layout(); plt.show()

# B. Attention Heatmap (The "Explainability" Plot)
if count > 0:
    mean_L0 = total_L0 / count
    mean_L1 = total_L1 / count

    # Updated Labels for 6 Tokens
    labels = ["Grade", "Diag", "Enc", "Dec", "Clin", "Rad"]

    fig, axes = plt.subplots(1, 2, figsize=(22, 9))
    fig.suptitle(f"Global Cross-Modal Attention (Avg over {count} patients)", fontsize=20, fontweight='bold')

    # Layer 0
    sns.heatmap(mean_L0.numpy(), annot=True, fmt=".2f", xticklabels=labels, yticklabels=labels, cmap="Blues", cbar=True, ax=axes[0], vmin=0)
    axes[0].set_title("Layer 1: Feature Mixing", fontsize=16)
    # Highlight the Grade Row (Row 0)
    axes[0].add_patch(plt.Rectangle((0, 0), 6, 1, fill=False, edgecolor='red', lw=3))

    # Layer 1
    sns.heatmap(mean_L1.numpy(), annot=True, fmt=".2f", xticklabels=labels, yticklabels=labels, cmap="Purples", cbar=True, ax=axes[1], vmin=0)
    axes[1].set_title("Layer 2: Final Decision", fontsize=16)
    # Highlight the Grade Row (Row 0)
    axes[1].add_patch(plt.Rectangle((0, 0), 6, 1, fill=False, edgecolor='red', lw=3))

    plt.tight_layout()
    plt.show()

    print("\n💡 INTERPRETATION GUIDE:")
    print("1. Look at the 'Grade' Row (Row 0) in Layer 2.")
    print("2. Which column is darker? (Enc/Dec = Image, Clin = Clinical, Rad = Radiomics)")
    print("3. If 'Rad' is dark, your Radiomics features are driving the decision!")

# Shit at the wall

In [ ]:
# %% Step 6: Late Fusion Ensemble (Deep Features + XGBoost)
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
import numpy as np

# --- 1. Helper to Extract Features ---
def extract_deep_features(loader, model, device):
    """
    Runs the dataset through the V3 model and captures:
    1. Deep Image Features (Encoder Output)
    2. Original Clinical Data
    3. Original Radiomics Data
    4. Labels
    """
    model.eval()
    deep_feats = []
    clin_feats = []
    rad_feats = []
    labels_grade = []
    labels_diag = []

    # Hook to capture the Encoder Token (before the head)
    # We want the output of 'encoder_proj' which is [Batch, 1, d_model]
    captured_token = {}
    def hook_fn(module, input, output):
        # output shape is [Batch, 1, d_model] -> flatten to [Batch, d_model]
        captured_token['feat'] = output.detach().flatten(1).cpu().numpy()

    # Register hook on the projection layer
    handle = model.encoder_proj.register_forward_hook(hook_fn)

    with torch.no_grad():
        for batch in tqdm(loader, desc="Extracting Deep Features"):
            img = batch[image_key].to(device)
            clin = batch[clinical_key].to(device)
            rad = batch[radiomics_key].to(device)

            # Run Forward Pass (Triggers the hook)
            _ = model(img, clin, rad)

            # Collect Data
            deep_feats.append(captured_token['feat'])
            clin_feats.append(clin.cpu().numpy())
            rad_feats.append(rad.cpu().numpy())
            labels_grade.append(batch[grade_key].numpy())
            labels_diag.append(batch[diag_key].numpy())

    handle.remove() # Cleanup

    # Concatenate all batches
    X_deep = np.concatenate(deep_feats, axis=0)
    X_clin = np.concatenate(clin_feats, axis=0)
    X_rad = np.concatenate(rad_feats, axis=0)
    y_grade = np.concatenate(labels_grade, axis=0)
    y_diag = np.concatenate(labels_diag, axis=0)

    # Combine into one Super-Vector
    # Shape: [N, d_model + n_clinical + n_radiomics]
    X_combined = np.hstack([X_deep, X_clin, X_rad])

    return X_combined, y_grade, y_diag

# --- 2. Extract Data ---
print("Extracting features from Training Set...")
X_train_full, y_train_g, y_train_d = extract_deep_features(train_loader_mtl, model_mtl, DEVICE)

print("Extracting features from Validation/Test Set...")
X_val_full, y_val_g, y_val_d = extract_deep_features(val_loader_mtl, model_mtl, DEVICE)

print(f"✅ Feature Extraction Complete.")
print(f"Feature Vector Size: {X_train_full.shape[1]} dimensions")
print(f"(Deep: {model_mtl.encoder_proj.out_features} + Clin: {CLINICAL_FEATURE_SIZE} + Rad: {RADIOMICS_LEN})")

# --- 3. Train XGBoost (Grade) ---
print("\n=== 🧠 Training XGBoost for GRADE ===")
xgb_grade = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    use_label_encoder=False,
    eval_metric='mlogloss'
)

xgb_grade.fit(X_train_full, y_train_g)
y_pred_g = xgb_grade.predict(X_val_full)
y_prob_g = xgb_grade.predict_proba(X_val_full)

print(classification_report(y_val_g, y_pred_g, target_names=[str(c) for c in GRADE_CLASSES]))
print(f"XGB Grade Accuracy: {accuracy_score(y_val_g, y_pred_g):.4f}")

# --- 4. Train XGBoost (Diagnosis) ---
print("\n=== 🧬 Training XGBoost for DIAGNOSIS ===")
xgb_diag = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    use_label_encoder=False,
    eval_metric='mlogloss'
)

xgb_diag.fit(X_train_full, y_train_d)
y_pred_d = xgb_diag.predict(X_val_full)
y_prob_d = xgb_diag.predict_proba(X_val_full)

print(classification_report(y_val_d, y_pred_d, target_names=[str(c) for c in DIAG_CLASSES]))
print(f"XGB Diagnosis Accuracy: {accuracy_score(y_val_d, y_pred_d):.4f}")

# --- 5. Feature Importance Analysis ---
# Lets see if the model used the Deep Features or just the Radiomics!
import matplotlib.pyplot as plt

def plot_importance(model, title):
    gain = model.feature_importances_
    # Create labels
    n_deep = model_mtl.encoder_proj.out_features
    labels = ([f"Deep_{i}" for i in range(n_deep)] +
              [f"Clin_{i}" for i in range(CLINICAL_FEATURE_SIZE)] +
              [f"Rad_{i}" for i in range(RADIOMICS_LEN)])

    # Sort and Plot Top 20
    indices = np.argsort(gain)[::-1][:20]
    plt.figure(figsize=(10, 5))
    plt.title(f"Top 20 Features: {title}")
    plt.bar(range(20), gain[indices], align="center")
    plt.xticks(range(20), [labels[i] for i in indices], rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

plot_importance(xgb_grade, "Tumor Grade")
plot_importance(xgb_diag, "Diagnosis")

In [ ]:
# %% Step 6: Late Fusion Ensemble (Deep Features + XGBoost)
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
import numpy as np

try:
    model_mtl = AdvancedMaskGuidedModel_v3_Soft(
        in_channels_img=IN_CHANNELS,
        out_channels_seg=OUT_CHANNELS_SEG,
        feature_size=FEATURE_SIZE,
        # Correctly passing use_checkpoint now:
        use_checkpoint=USE_CHECKPOINTING,
        num_classes_grade=NUM_CLASSES_GRADE,
        num_classes_diag=NUM_CLASSES_DIAG,
        clinical_len=CLINICAL_FEATURE_SIZE,
        radiomics_len=RADIOMICS_LEN,
        finetuned_seg_model_path=finetuned_model_path,
        d_model=128, n_heads=4, num_layers=2, dropout_rate=0.5
    ).to(DEVICE)
    print("✅ AdvancedMaskGuidedModel_v3_Soft initialized.")
except NameError as e:
    print(f"❌ ERROR: Model class initialization failed: {e}")
# --- 1. Helper to Extract Features ---
def extract_deep_features(loader, model, device):
    """
    Runs the dataset through the V3 model and captures:
    1. Deep Image Features (Encoder Output)
    2. Original Clinical Data
    3. Original Radiomics Data
    4. Labels
    """
    model.eval()
    deep_feats = []
    clin_feats = []
    rad_feats = []
    labels_grade = []
    labels_diag = []

    # Hook to capture the Encoder Token (before the head)
    # We want the output of 'encoder_proj' which is [Batch, 1, d_model]
    captured_token = {}
    def hook_fn(module, input, output):
        # output shape is [Batch, 1, d_model] -> flatten to [Batch, d_model]
        captured_token['feat'] = output.detach().flatten(1).cpu().numpy()

    # Register hook on the projection layer
    handle = model.encoder_proj.register_forward_hook(hook_fn)

    with torch.no_grad():
        for batch in tqdm(loader, desc="Extracting Deep Features"):
            img = batch[image_key].to(device)
            clin = batch[clinical_key].to(device)
            rad = batch[radiomics_key].to(device)

            # Run Forward Pass (Triggers the hook)
            _ = model(img, clin, rad)

            # Collect Data
            deep_feats.append(captured_token['feat'])
            clin_feats.append(clin.cpu().numpy())
            rad_feats.append(rad.cpu().numpy())
            labels_grade.append(batch[grade_key].numpy())
            labels_diag.append(batch[diag_key].numpy())

    handle.remove() # Cleanup

    # Concatenate all batches
    X_deep = np.concatenate(deep_feats, axis=0)
    X_clin = np.concatenate(clin_feats, axis=0)
    X_rad = np.concatenate(rad_feats, axis=0)
    y_grade = np.concatenate(labels_grade, axis=0)
    y_diag = np.concatenate(labels_diag, axis=0)

    # Combine into one Super-Vector
    # Shape: [N, d_model + n_clinical + n_radiomics]
    X_combined = np.hstack([X_deep, X_clin, X_rad])

    return X_combined, y_grade, y_diag

# --- 2. Extract Data ---
print("Extracting features from Training Set...")
X_train_full, y_train_g, y_train_d = extract_deep_features(train_loader_mtl, model_mtl, DEVICE)

print("Extracting features from Validation/Test Set...")
X_val_full, y_val_g, y_val_d = extract_deep_features(val_loader_mtl, model_mtl, DEVICE)

print(f"✅ Feature Extraction Complete.")
print(f"Feature Vector Size: {X_train_full.shape[1]} dimensions")
print(f"(Deep: {model_mtl.encoder_proj.out_features} + Clin: {CLINICAL_FEATURE_SIZE} + Rad: {RADIOMICS_LEN})")

# --- 3. Train XGBoost (Grade) ---
print("\n=== 🧠 Training XGBoost for GRADE ===")
xgb_grade = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    use_label_encoder=False,
    eval_metric='mlogloss'
)

xgb_grade.fit(X_train_full, y_train_g)
y_pred_g = xgb_grade.predict(X_val_full)
y_prob_g = xgb_grade.predict_proba(X_val_full)

print(classification_report(y_val_g, y_pred_g, target_names=[str(c) for c in GRADE_CLASSES]))
print(f"XGB Grade Accuracy: {accuracy_score(y_val_g, y_pred_g):.4f}")

# --- 4. Train XGBoost (Diagnosis) ---
print("\n=== 🧬 Training XGBoost for DIAGNOSIS ===")
xgb_diag = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    use_label_encoder=False,
    eval_metric='mlogloss'
)

xgb_diag.fit(X_train_full, y_train_d)
y_pred_d = xgb_diag.predict(X_val_full)
y_prob_d = xgb_diag.predict_proba(X_val_full)

print(classification_report(y_val_d, y_pred_d, target_names=[str(c) for c in DIAG_CLASSES]))
print(f"XGB Diagnosis Accuracy: {accuracy_score(y_val_d, y_pred_d):.4f}")

# --- 5. Feature Importance Analysis ---
# Lets see if the model used the Deep Features or just the Radiomics!
import matplotlib.pyplot as plt

def plot_importance(model, title):
    gain = model.feature_importances_
    # Create labels
    n_deep = model_mtl.encoder_proj.out_features
    labels = ([f"Deep_{i}" for i in range(n_deep)] +
              [f"Clin_{i}" for i in range(CLINICAL_FEATURE_SIZE)] +
              [f"Rad_{i}" for i in range(RADIOMICS_LEN)])

    # Sort and Plot Top 20
    indices = np.argsort(gain)[::-1][:20]
    plt.figure(figsize=(10, 5))
    plt.title(f"Top 20 Features: {title}")
    plt.bar(range(20), gain[indices], align="center")
    plt.xticks(range(20), [labels[i] for i in indices], rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

plot_importance(xgb_grade, "Tumor Grade")
plot_importance(xgb_diag, "Diagnosis")

In [ ]:
# %% Step 6: Late Fusion Ensemble (Deep Features + XGBoost)
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
import numpy as np

# --- 1. Helper to Extract Features ---
def extract_deep_features(loader, model, device):
    """
    Runs the dataset through the V3 model and captures:
    1. Deep Image Features (Encoder Output)
    2. Original Clinical Data
    3. Original Radiomics Data
    4. Labels
    """
    model.eval()
    deep_feats = []
    clin_feats = []
    rad_feats = []
    labels_grade = []
    labels_diag = []

    # Hook to capture the Encoder Token (before the head)
    # We want the output of 'encoder_proj' which is [Batch, 1, d_model]
    captured_token = {}
    def hook_fn(module, input, output):
        # output shape is [Batch, 1, d_model] -> flatten to [Batch, d_model]
        captured_token['feat'] = output.detach().flatten(1).cpu().numpy()

    # Register hook on the projection layer
    handle = model.encoder_proj.register_forward_hook(hook_fn)

    with torch.no_grad():
        for batch in tqdm(loader, desc="Extracting Deep Features"):
            img = batch[image_key].to(device)
            clin = batch[clinical_key].to(device)
            rad = batch[radiomics_key].to(device)

            # Run Forward Pass (Triggers the hook)
            _ = model(img, clin, rad)

            # Collect Data
            deep_feats.append(captured_token['feat'])
            clin_feats.append(clin.cpu().numpy())
            rad_feats.append(rad.cpu().numpy())
            labels_grade.append(batch[grade_key].numpy())
            labels_diag.append(batch[diag_key].numpy())

    handle.remove() # Cleanup

    # Concatenate all batches
    X_deep = np.concatenate(deep_feats, axis=0)
    X_clin = np.concatenate(clin_feats, axis=0)
    X_rad = np.concatenate(rad_feats, axis=0)
    y_grade = np.concatenate(labels_grade, axis=0)
    y_diag = np.concatenate(labels_diag, axis=0)

    # Combine into one Super-Vector
    # Shape: [N, d_model + n_clinical + n_radiomics]
    X_combined = np.hstack([X_deep, X_clin, X_rad])

    return X_combined, y_grade, y_diag

# --- 2. Extract Data ---
print("Extracting features from Training Set...")
X_train_full, y_train_g, y_train_d = extract_deep_features(train_loader_mtl, model_mtl, DEVICE)

print("Extracting features from Validation/Test Set...")
X_val_full, y_val_g, y_val_d = extract_deep_features(val_loader_mtl, model_mtl, DEVICE)

print(f"✅ Feature Extraction Complete.")
print(f"Feature Vector Size: {X_train_full.shape[1]} dimensions")
print(f"(Deep: {model_mtl.encoder_proj.out_features} + Clin: {CLINICAL_FEATURE_SIZE} + Rad: {RADIOMICS_LEN})")

# --- 3. Train XGBoost (Grade) ---
print("\n=== 🧠 Training XGBoost for GRADE ===")
xgb_grade = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    use_label_encoder=False,
    eval_metric='mlogloss'
)

xgb_grade.fit(X_train_full, y_train_g)
y_pred_g = xgb_grade.predict(X_val_full)
y_prob_g = xgb_grade.predict_proba(X_val_full)

print(classification_report(y_val_g, y_pred_g, target_names=[str(c) for c in GRADE_CLASSES]))
print(f"XGB Grade Accuracy: {accuracy_score(y_val_g, y_pred_g):.4f}")

# --- 4. Train XGBoost (Diagnosis) ---
print("\n=== 🧬 Training XGBoost for DIAGNOSIS ===")
xgb_diag = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    use_label_encoder=False,
    eval_metric='mlogloss'
)

xgb_diag.fit(X_train_full, y_train_d)
y_pred_d = xgb_diag.predict(X_val_full)
y_prob_d = xgb_diag.predict_proba(X_val_full)

print(classification_report(y_val_d, y_pred_d, target_names=[str(c) for c in DIAG_CLASSES]))
print(f"XGB Diagnosis Accuracy: {accuracy_score(y_val_d, y_pred_d):.4f}")

# --- 5. Feature Importance Analysis ---
# Lets see if the model used the Deep Features or just the Radiomics!
import matplotlib.pyplot as plt

def plot_importance(model, title):
    gain = model.feature_importances_
    # Create labels
    n_deep = model_mtl.encoder_proj.out_features
    labels = ([f"Deep_{i}" for i in range(n_deep)] +
              [f"Clin_{i}" for i in range(CLINICAL_FEATURE_SIZE)] +
              [f"Rad_{i}" for i in range(RADIOMICS_LEN)])

    # Sort and Plot Top 20
    indices = np.argsort(gain)[::-1][:20]
    plt.figure(figsize=(10, 5))
    plt.title(f"Top 20 Features: {title}")
    plt.bar(range(20), gain[indices], align="center")
    plt.xticks(range(20), [labels[i] for i in indices], rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

plot_importance(xgb_grade, "Tumor Grade")
plot_importance(xgb_diag, "Diagnosis")

In [ ]:
# %% Step 6: Late Fusion Ensemble (Deep Features + XGBoost) - FINAL CORRECTED
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# --- 1. Helper to Extract Features ---
def extract_deep_features(loader, model, device):
    """
    Runs the dataset through the V3 model and captures:
    1. Deep Image Features (Encoder Output) - The 'Visual Embedding'
    2. Original Clinical Data
    3. Original Radiomics Data
    4. Labels (Grade & Diagnosis)
    """
    model.eval()
    deep_feats = []
    clin_feats = []
    rad_feats = []
    labels_grade = []
    labels_diag = []

    # Hook to capture the Encoder Token (before the head)
    # We want the output of 'encoder_proj' which is [Batch, 1, d_model]
    captured_token = {}
    def hook_fn(module, input, output):
        # output shape is [Batch, 1, d_model] -> flatten to [Batch, d_model]
        captured_token['feat'] = output.detach().flatten(1).cpu().numpy()

    # Register hook on the projection layer of the Image Encoder
    handle = model.encoder_proj.register_forward_hook(hook_fn)

    with torch.no_grad():
        for batch in tqdm(loader, desc="Extracting Deep Features"):
            img = batch[image_key].to(device)
            clin = batch[clinical_key].to(device)
            rad = batch[radiomics_key].to(device)

            # Run Forward Pass (Triggers the hook)
            _ = model(img, clin, rad)

            # Collect Data
            deep_feats.append(captured_token['feat'])
            clin_feats.append(clin.cpu().numpy())
            rad_feats.append(rad.cpu().numpy())
            labels_grade.append(batch[grade_key].numpy())
            labels_diag.append(batch[diag_key].numpy())

    handle.remove() # Cleanup hook

    # Concatenate all batches
    X_deep = np.concatenate(deep_feats, axis=0)
    X_clin = np.concatenate(clin_feats, axis=0)
    X_rad = np.concatenate(rad_feats, axis=0)
    y_grade = np.concatenate(labels_grade, axis=0)
    y_diag = np.concatenate(labels_diag, axis=0)

    # Combine into one Super-Vector
    # Shape: [N, (d_model + n_clinical + n_radiomics)]
    X_combined = np.hstack([X_deep, X_clin, X_rad])

    return X_combined, y_grade, y_diag

# --- 2. Extract Data (Train, Val, AND Test) ---
print("--- 1. Extracting Features from Training Set ---")
X_train, y_train_g, y_train_d = extract_deep_features(train_loader_mtl, model_mtl, DEVICE)

print("\n--- 2. Extracting Features from Validation Set ---")
X_val, y_val_g, y_val_d = extract_deep_features(val_loader_mtl, model_mtl, DEVICE)

print("\n--- 3. Extracting Features from Test Set (Holdout) ---")
try:
    X_test, y_test_g, y_test_d = extract_deep_features(test_loader_mtl, model_mtl, DEVICE)
    print(f"Test Set Loaded: {X_test.shape[0]} samples")
except NameError:
    print("⚠️ Warning: 'test_loader_mtl' not found. Skipping Test extraction.")
    X_test = None

# Create feature names for importance plotting
n_deep = model_mtl.encoder_proj.out_features
try:
    # Use real names if available from Step 8
    feat_names = ([f"Deep_Img_{i}" for i in range(n_deep)] +
                  clinical_feature_columns +
                  list(radiomics_feature_names))
except:
    # Fallback to generic names
    feat_names = ([f"Deep_{i}" for i in range(n_deep)] +
                  [f"Clin_{i}" for i in range(CLINICAL_FEATURE_SIZE)] +
                  [f"Rad_{i}" for i in range(RADIOMICS_LEN)])

print(f"\n✅ Feature Extraction Complete.")
print(f"Vector Dimensions: {X_train.shape[1]}")

# --- 3. Train XGBoost (Grade) ---
print("\n=============================================")
print("🧠 XGBOOST: TUMOR GRADE CLASSIFICATION")
print("=============================================")

xgb_grade = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    use_label_encoder=False,
    eval_metric='mlogloss'
)

xgb_grade.fit(X_train, y_train_g)

# Validation
print("\n--- Validation Set Results (Grade) ---")
y_pred_val_g = xgb_grade.predict(X_val)
print(classification_report(y_val_g, y_pred_val_g,
                            target_names=[str(c) for c in GRADE_CLASSES],
                            labels=range(len(GRADE_CLASSES)), zero_division=0))

# Test
if X_test is not None:
    print("\n--- 🏆 TEST SET Results (Grade) ---")
    y_pred_test_g = xgb_grade.predict(X_test)
    print(classification_report(y_test_g, y_pred_test_g,
                                target_names=[str(c) for c in GRADE_CLASSES],
                                labels=range(len(GRADE_CLASSES)), zero_division=0))
    print(f"Test Accuracy: {accuracy_score(y_test_g, y_pred_test_g):.4f}")

# --- 4. Train XGBoost (Diagnosis) ---
print("\n=============================================")
print("🧬 XGBOOST: DIAGNOSIS CLASSIFICATION")
print("=============================================")

xgb_diag = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42
)

xgb_diag.fit(X_train, y_train_d)

# Validation
print("\n--- Validation Set Results (Diagnosis) ---")
y_pred_val_d = xgb_diag.predict(X_val)
print(classification_report(y_val_d, y_pred_val_d,
                            target_names=[str(c) for c in DIAG_CLASSES],
                            labels=range(len(DIAG_CLASSES)), zero_division=0))

# Test
if X_test is not None:
    print("\n--- 🏆 TEST SET Results (Diagnosis) ---")
    y_pred_test_d = xgb_diag.predict(X_test)

    # FIXED: We use 'labels=range(4)' to tell it there ARE 4 classes,
    # even if the test set only has 3.
    print(classification_report(y_test_d, y_pred_test_d,
                                target_names=[str(c) for c in DIAG_CLASSES],
                                labels=range(len(DIAG_CLASSES)),
                                zero_division=0))
    print(f"Test Accuracy: {accuracy_score(y_test_d, y_pred_test_d):.4f}")

# --- 5. Feature Importance Plot ---
def plot_importance(model, feature_names, title, top_n=20):
    gain = model.feature_importances_
    indices = np.argsort(gain)[::-1][:top_n]

    plt.figure(figsize=(12, 6))
    plt.title(f"Top {top_n} Features Driving {title}")

    # Color code bars: Blue=Deep(Image), Green=Clinical, Orange=Radiomics
    colors = []
    for idx in indices:
        name = feature_names[idx]
        if "Deep" in name: colors.append('royalblue')
        elif "Rad" in name or "original" in name: colors.append('darkorange')
        else: colors.append('forestgreen')

    plt.bar(range(top_n), gain[indices], align="center", color=colors)
    plt.xticks(range(top_n), [feature_names[i] for i in indices], rotation=45, ha='right', fontsize=9)
    plt.ylabel("Importance (Gain)")
    plt.tight_layout()
    plt.show()

print("\n🔍 Analyzing Decision Drivers...")
try:
    plot_importance(xgb_grade, feat_names, "Tumor Grade")
    plot_importance(xgb_diag, feat_names, "Diagnosis")
except Exception as e:
    print(f"Could not plot importance: {e}")

In [ ]:
# %% Step 7: Thesis Visualization Suite (Confusion Matrix, ROC, Saliency)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from itertools import cycle
import numpy as np
import torch
import cv2

# Settings
output_dir_viz = os.path.join(runs_base_dir, "thesis_visualizations")
os.makedirs(output_dir_viz, exist_ok=True)

# --- 1. Load Data & Model (If not already loaded) ---
# Ensure model is in eval mode
model_mtl.eval()

# Get predictions from Validation/Test set (Using V3 Deep Learning Model)
# Note: For the thesis, you can report XGBoost numbers in text,
# but visualize the Deep Learning Saliency maps for the "Qualitative" section.
y_true_g = []
y_pred_g = []
y_prob_g = []
y_true_d = []
y_pred_d = []
y_prob_d = []

print("Generating predictions for visualization...")
with torch.no_grad():
    for batch in tqdm(val_loader_mtl):
        img = batch[image_key].to(DEVICE)
        clin = batch[clinical_key].to(DEVICE)
        rad = batch[radiomics_key].to(DEVICE)

        # Turn OFF dropout for final eval
        model_mtl.training = False

        outputs = model_mtl(img, clin, rad)
        _, p_grade, p_diag, _, _, _ = outputs

        # Grade
        y_true_g.extend(batch[grade_key].numpy())
        y_pred_g.extend(torch.argmax(p_grade, 1).cpu().numpy())
        y_prob_g.extend(torch.softmax(p_grade, 1).cpu().numpy())

        # Diagnosis
        y_true_d.extend(batch[diag_key].numpy())
        y_pred_d.extend(torch.argmax(p_diag, 1).cpu().numpy())
        y_prob_d.extend(torch.softmax(p_diag, 1).cpu().numpy())

# --- 2. Plot Confusion Matrices ---
def plot_cm(y_true, y_pred, classes, title, filename):
    cm = confusion_matrix(y_true, y_pred)
    # Normalize
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=classes, yticklabels=classes)
    plt.title(f"{title} (Normalized)")
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir_viz, filename))
    plt.show()

print("\n📊 Generating Confusion Matrices...")
plot_cm(y_true_g, y_pred_g, [str(c) for c in GRADE_CLASSES], "Tumor Grade Confusion Matrix", "cm_grade.png")
plot_cm(y_true_d, y_pred_d, [str(c) for c in DIAG_CLASSES], "Diagnosis Confusion Matrix", "cm_diag.png")

# --- 3. Plot Multiclass ROC Curves ---
def plot_roc(y_true, y_prob, classes, title, filename):
    y_true_bin = label_binarize(y_true, classes=range(len(classes)))
    n_classes = len(classes)

    fpr = dict()
    tpr = dict()
    roc_auc = dict()

    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    plt.figure(figsize=(10, 8))
    colors = cycle(['blue', 'red', 'green', 'orange'])
    for i, color in zip(range(n_classes), colors):
        plt.plot(fpr[i], tpr[i], color=color, lw=2,
                 label=f'{classes[i]} (AUC = {roc_auc[i]:.2f})')

    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{title} ROC Curves')
    plt.legend(loc="lower right")
    plt.savefig(os.path.join(output_dir_viz, filename))
    plt.show()

print("\n📈 Generating ROC Curves...")
try:
    plot_roc(y_true_g, np.array(y_prob_g), GRADE_CLASSES, "Tumor Grade", "roc_grade.png")
    plot_roc(y_true_d, np.array(y_prob_d), DIAG_CLASSES, "Diagnosis", "roc_diag.png")
except Exception as e:
    print(f"Skipping ROC plot (Error: {e})")

# --- 4. Qualitative Saliency Map (The "Soft Masking" Proof) ---
# We visualize one patient to show the model focusing on the tumor.
print("\n👁️ Generating Saliency Maps...")

# Pick a patient from the batch
sample_idx = 0
img_tensor = batch[image_key][sample_idx].unsqueeze(0).to(DEVICE)
clin_tensor = batch[clinical_key][sample_idx].unsqueeze(0).to(DEVICE)
rad_tensor = batch[radiomics_key][sample_idx].unsqueeze(0).to(DEVICE)

# Enable gradients for the image
img_tensor.requires_grad_()

# Forward pass
outputs = model_mtl(img_tensor, clin_tensor, rad_tensor)
logits_grade = outputs[1]

# Backward pass (Gradient of the top predicted class w.r.t input image)
score, predicted_class = torch.max(logits_grade, 1)
score.backward()

# Get gradients
gradients = img_tensor.grad.data.cpu().numpy()[0, 0] # [D, H, W]
original_img = img_tensor.detach().cpu().numpy()[0, 0]

# Pick the middle slice
mid_slice = gradients.shape[0] // 2
grad_slice = np.abs(gradients[mid_slice])
img_slice = original_img[mid_slice]

# Plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 3, 1)
plt.imshow(img_slice, cmap='gray')
plt.title("Original MRI Slice")
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(grad_slice, cmap='jet')
plt.title("Saliency (Attention Map)")
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(img_slice, cmap='gray')
plt.imshow(grad_slice, cmap='jet', alpha=0.5) # Overlay
plt.title("Overlay")
plt.axis('off')

plt.suptitle(f"Visual Evidence: Where the Model Looks (Grade {GRADE_CLASSES[predicted_class]})")
plt.savefig(os.path.join(output_dir_viz, "saliency_map.png"))
plt.show()

print(f"✅ All visualizations saved to: {output_dir_viz}")

# Trinity

In [ ]:
pip install iterative-stratification

In [ ]:
# [ΚΕΛΙ 2 + 2.5] Data Preprocessing, Matching & Stratified Split
import os
import re
import pandas as pd
import numpy as np
import torch
from glob import glob
from sklearn.preprocessing import StandardScaler, LabelEncoder
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# --- 0. Global Settings & Paths ---
# Βεβαιώσου ότι τα paths είναι σωστά για το δικό σου σύστημα
metadata_path = os.path.join(base_drive_path, "Brain_Tumor_Subset/balanced_metadata_gradeaware.csv")
rad_path = os.path.join(base_drive_path, "Brain_Tumor_Subset/radiomics_selected_20.csv")

# Ορισμός Κλειδιών (Keys) για το λεξικό δεδομένων
image_key = "image"
label_key = "label"
clinical_key = "clinical"
radiomics_key = "radiomics" # Ξεχωριστό κλειδί!
grade_key = "grade"
diag_key = "diagnosis"
spatial_keys = [image_key, label_key]

# ==============================================================================
# ΜΕΡΟΣ Α: Επεξεργασία Metadata (Κλινικά)
# ==============================================================================
print("--- 1. Φόρτωση & Καθαρισμός Metadata ---")
try:
    df_balanced = pd.read_csv(metadata_path)
    print(f"✅ Metadata loaded. Samples: {len(df_balanced)}")
except FileNotFoundError:
    raise FileNotFoundError(f"❌ Δεν βρέθηκε το αρχείο: {metadata_path}")

# Επιλογή στηλών
columns_to_keep = [
    'ID', 'Grade', 'Diagnosis', 'Age', 'Sex', 'MGMT_Status', 'MGMT_Index',
    'Codeletion_1p19q', 'IDH', 'EOR', 'PreImaging_Biopsy', 'pid_formatted'
]
df_processed = df_balanced[[col for col in columns_to_keep if col in df_balanced.columns]].copy()

# Καθαρισμός MGMT_Index (Continuous)
if 'MGMT_Index' in df_processed.columns:
    df_processed['MGMT_Index'] = pd.to_numeric(df_processed['MGMT_Index'], errors='coerce')
    df_processed['MGMT_Index'].fillna(df_processed['MGMT_Index'].median(), inplace=True)

# Καθαρισμός Κατηγορικών & Age
categorical_cols = ['Sex', 'MGMT_Status', 'Codeletion_1p19q', 'IDH', 'EOR', 'PreImaging_Biopsy']
for col in categorical_cols:
    if col in df_processed.columns:
        # Fill NA with mode
        if df_processed[col].isnull().any():
            df_processed[col].fillna(df_processed[col].mode()[0], inplace=True)
        # Handle 'unknown' string
        if df_processed[col].dtype == object:
             df_processed[col] = df_processed[col].replace('unknown', df_processed[col].mode()[0])

if 'Age' in df_processed.columns:
    df_processed['Age'].fillna(df_processed['Age'].median(), inplace=True)

# One-Hot Encoding
existing_cat_cols = [c for c in categorical_cols if c in df_processed.columns]
if existing_cat_cols:
    df_processed = pd.get_dummies(df_processed, columns=existing_cat_cols, dummy_na=False)

# Scaling Continuous (Age, MGMT_Index)
scaler_clin = StandardScaler()
cont_cols = [c for c in ['Age', 'MGMT_Index'] if c in df_processed.columns]
if cont_cols:
    df_processed[cont_cols] = scaler_clin.fit_transform(df_processed[cont_cols])

# Label Encoding (Targets)
grade_encoder = LabelEncoder()
df_processed['Grade_Label'] = grade_encoder.fit_transform(df_processed['Grade'])
GRADE_CLASSES = grade_encoder.classes_
NUM_CLASSES_GRADE = len(GRADE_CLASSES)

diag_encoder = LabelEncoder()
df_processed['Diagnosis_Label'] = diag_encoder.fit_transform(df_processed['Diagnosis'])
DIAG_CLASSES = diag_encoder.classes_
NUM_CLASSES_DIAG = len(DIAG_CLASSES)

print(f"Classes -> Grade: {NUM_CLASSES_GRADE}, Diagnosis: {NUM_CLASSES_DIAG}")

# Ορισμός Τελικών Clinical Features
non_feat = ['ID', 'Grade', 'Diagnosis', 'Grade_Label', 'Diagnosis_Label', 'pid_formatted']
clinical_feature_columns = [c for c in df_processed.columns if c not in non_feat]
CLINICAL_FEATURE_SIZE = len(clinical_feature_columns)
print(f"✅ Clinical Feature Size: {CLINICAL_FEATURE_SIZE}")

# ==============================================================================
# ΜΕΡΟΣ Β: Επεξεργασία Radiomics
# ==============================================================================
print("\n--- 2. Φόρτωση & Scaling Radiomics ---")
try:
    df_rad = pd.read_csv(rad_path)
    df_rad['Official_ID'] = df_rad['Official_ID'].astype(str) # Ensure ID is string for matching

    # Identify Radiomics Columns (Exclude Meta)
    rad_cols = [c for c in df_rad.columns if c not in ['Official_ID', 'Grade', 'Diagnosis', 'ID']]
    RADIOMICS_LEN = len(rad_cols)

    # Scale Radiomics (Κρίσιμο για Transformer!)
    rad_scaler = StandardScaler()
    df_rad[rad_cols] = rad_scaler.fit_transform(df_rad[rad_cols])

    # Δημιουργία Lookup Dictionary: {'ID_String': [values...]}
    # Χρησιμοποιούμε το Official_ID ως κλειδί
    rad_lookup_str = df_rad.set_index('Official_ID')[rad_cols].T.to_dict('list')
    print(f"✅ Radiomics loaded. Features: {RADIOMICS_LEN}")

except FileNotFoundError:
    print(f"⚠️ Warning: Radiomics file not found at {rad_path}. Using Zeros.")
    rad_lookup_str = {}
    RADIOMICS_LEN = 0

# ==============================================================================
# ΜΕΡΟΣ Γ: Robust ID Matching
# ==============================================================================
print("\n--- 3. Matching IDs (Images <-> Metadata <-> Radiomics) ---")

def get_id_number(val):
    """Εξάγει τον ακέραιο από string μορφής 'UCSF-PDGM-0004' -> 4"""
    match = re.search(r'(\d+)', str(val))
    return int(match.group(1)) if match else None

# 1. Εύρεση Αρχείων Εικόνας/Μάσκας
image_files = sorted(glob(os.path.join(image_dir_ft, "*.nii.gz")))
label_files = sorted(glob(os.path.join(label_dir_ft, "*_tumor_segmentation.nii.gz")))

# Map Int_ID -> Path
# Υποθέτουμε filename format: "ID_suffix.nii.gz"
img_map = {get_id_number(os.path.basename(f).split('_')[0]): f for f in image_files if get_id_number(os.path.basename(f)) is not None}
lbl_map = {get_id_number(os.path.basename(f).split('_')[0]): f for f in label_files if get_id_number(os.path.basename(f)) is not None}

# 2. Map Int_ID -> Clinical Data
# Προσπαθούμε να βρούμε το ID από διάφορες στήλες
if 'ID' in df_processed.columns:
    df_processed['Int_ID'] = df_processed['ID'].apply(get_id_number)
elif 'pid_formatted' in df_processed.columns:
    df_processed['Int_ID'] = df_processed['pid_formatted'].apply(get_id_number)
else:
    df_processed['Int_ID'] = df_processed.index.to_series().apply(get_id_number)

df_clean = df_processed.dropna(subset=['Int_ID']).copy()
df_clean['Int_ID'] = df_clean['Int_ID'].astype(int)
df_clean.set_index('Int_ID', inplace=True)

# 3. Map Int_ID -> Radiomics
# Το rad_lookup_str έχει κλειδιά string. Τα μετατρέπουμε σε int.
rad_lookup_int = {get_id_number(k): v for k, v in rad_lookup_str.items() if get_id_number(k) is not None}

# 4. The Loop
data_dicts_mtl_balanced = []
missing_stats = {"mask": 0, "clin": 0, "rad": 0}

for pid in img_map.keys():
    has_mask = pid in lbl_map
    has_clin = pid in df_clean.index
    has_rad = pid in rad_lookup_int

    if has_mask and has_clin and has_rad:
        # Get Data
        row = df_clean.loc[pid]
        if isinstance(row, pd.DataFrame): row = row.iloc[0] # Handle duplicates

        data_dicts_mtl_balanced.append({
            "pid_str": str(pid),
            image_key: img_map[pid],
            label_key: lbl_map[pid],
            # Clinical Tensor
            clinical_key: torch.tensor(row[clinical_feature_columns].values.astype(np.float32), dtype=torch.float32),
            # Radiomics Tensor
            radiomics_key: torch.tensor(rad_lookup_int[pid], dtype=torch.float32),
            # Targets
            grade_key: torch.tensor(row['Grade_Label'], dtype=torch.long),
            diag_key: torch.tensor(row['Diagnosis_Label'], dtype=torch.long)
        })
    else:
        if not has_mask: missing_stats["mask"] += 1
        if not has_clin: missing_stats["clin"] += 1
        if not has_rad: missing_stats["rad"] += 1

print(f"✅ Final Matched Dataset: {len(data_dicts_mtl_balanced)} patients")
if len(data_dicts_mtl_balanced) == 0:
    print(f"❌ CRITICAL: No matches found! Check ID formats. Stats: {missing_stats}")
    # raise ValueError("Dataset is empty.")

# ==============================================================================
# ΜΕΡΟΣ Δ: Multilabel Stratified Split (Βήμα 2.5)
# ==============================================================================
print("\n--- 4. Stratified Split (Balanced Grade & Diagnosis) ---")

print("--- ✂️ Εκτέλεση Smart Stratified Split ---")

# Στόχος: Train 70%, Temp 30% (που θα γίνει Val 15% / Test 15%)
TEST_SIZE_INITIAL = 0.30

print(f"--- ✂️ Re-Splitting Data (Target: Train {1-TEST_SIZE_INITIAL:.0%}, Val/Test {TEST_SIZE_INITIAL:.0%}) ---")

# 1. Prepare Targets
Y_stratify = np.array([
    [d[grade_key].item(), d[diag_key].item()]
    for d in data_dicts_mtl_balanced
])

best_seed = -1
found_good_split = False

# Seed Search Loop
for seed in range(100):
    # Split 1: Train (70%) vs Rest (30%)
    msss_train = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=TEST_SIZE_INITIAL, random_state=seed)
    train_idx, temp_idx = next(msss_train.split(data_dicts_mtl_balanced, Y_stratify))

    Y_temp = Y_stratify[temp_idx]

    # Split 2: Val (50% of Rest) vs Test (50% of Rest) -> 15% Total each
    msss_val = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=seed)
    val_idx_rel, test_idx_rel = next(msss_val.split(temp_idx, Y_temp))

    # Check rare class coverage
    test_labels = Y_temp[test_idx_rel, 1]
    val_labels = Y_temp[val_idx_rel, 1]

    if len(np.unique(test_labels)) == NUM_CLASSES_DIAG and len(np.unique(val_labels)) == NUM_CLASSES_DIAG:
        print(f"✅ Found Good Split with Random Seed: {seed}")
        best_seed = seed

        # Apply Split
        train_files_mtl = [data_dicts_mtl_balanced[i] for i in train_idx]
        temp_files = [data_dicts_mtl_balanced[i] for i in temp_idx]
        val_files_mtl = [temp_files[i] for i in val_idx_rel]
        test_files_mtl = [temp_files[i] for i in test_idx_rel]

        found_good_split = True
        break

if not found_good_split:
    print("⚠️ Warning: Could not find perfect split. Using fallback seed 42.")
    # Fallback logic... (omitted for brevity, usually finds one)

print(f"Stats: Train={len(train_files_mtl)}, Val={len(val_files_mtl)}, Test={len(test_files_mtl)}")

print(f"Train: {len(train_files_mtl)}, Val: {len(val_files_mtl)}, Test: {len(test_files_mtl)}")

# --- ΕΛΕΓΧΟΣ (Updated Display) ---
def show_distribution_table(datasets, set_names, key, class_labels, title):
    print(f"\n📊 === {title} DISTRIBUTION (Seed {best_seed}) ===")
    combined_counts = {}
    for ds, name in zip(datasets, set_names):
        labels = [d[key].item() for d in ds]
        counts = pd.Series(labels).value_counts(normalize=True).sort_index()
        if class_labels is not None:
            label_map = {i: label for i, label in enumerate(class_labels)}
            counts.index = counts.index.map(label_map)
        combined_counts[name] = counts

    df_dist = pd.DataFrame(combined_counts).fillna(0)
    # Fix for deprecation warning: use .map instead of .applymap
    print(df_dist.map(lambda x: f"{x:.2%}"))

show_distribution_table([train_files_mtl, val_files_mtl, test_files_mtl], ["Train", "Val", "Test"], grade_key, GRADE_CLASSES, "GRADE")
show_distribution_table([train_files_mtl, val_files_mtl, test_files_mtl], ["Train", "Val", "Test"], diag_key, DIAG_CLASSES, "DIAGNOSIS")

In [ ]:
# %% Step 3: Transforms & DataLoaders (Using CacheDataset)
import torch
from monai.data import DataLoader as MonaiDataLoader, CacheDataset, list_data_collate
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    ScaleIntensityRanged, RandCropByPosNegLabeld, RandFlipd, RandAffined,
    RandGaussianNoised, RandStdShiftIntensityd, RandScaleIntensityd,
    RandBiasFieldd, EnsureTyped, CenterSpatialCropd
)

# 1. Define Keys
# Note: We keep Clinical & Radiomics separate (no concatenation here)
float_tensor_keys = [image_key, label_key, clinical_key, radiomics_key]
long_tensor_keys = [grade_key, diag_key]

# 2. Training Transforms
train_transforms_mtl = Compose([
    LoadImaged(keys=spatial_keys, image_only=False),
    EnsureChannelFirstd(keys=spatial_keys),
    Orientationd(keys=spatial_keys, axcodes="RAS"),
    Spacingd(keys=spatial_keys, pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys=image_key, a_min=0, a_max=3000, b_min=0.0, b_max=1.0, clip=True),

    # Augmentations
    RandCropByPosNegLabeld(
        keys=spatial_keys, label_key=label_key, spatial_size=ROI_SIZE_FT,
        pos=1.0, neg=1.0, num_samples=2, image_key=image_key, image_threshold=0.0
    ),
    RandFlipd(keys=spatial_keys, prob=0.5, spatial_axis=0),
    RandFlipd(keys=spatial_keys, prob=0.5, spatial_axis=1),
    RandFlipd(keys=spatial_keys, prob=0.5, spatial_axis=2),
    RandAffined(keys=spatial_keys, prob=0.2, rotate_range=(0.05, 0.05, 0.05), scale_range=(0.1, 0.1, 0.1)),
    RandGaussianNoised(keys=image_key, prob=0.2, mean=0.0, std=0.01),
    RandStdShiftIntensityd(keys=image_key, factors=0.1, prob=0.3),
    RandBiasFieldd(keys=image_key, prob=0.3),

    # Type Casting (Critical for Trinity Model)
    EnsureTyped(keys=float_tensor_keys, dtype=torch.float32),
    EnsureTyped(keys=long_tensor_keys, dtype=torch.long),
])

# 3. Validation/Test Transforms
val_transforms_mtl = Compose([
    LoadImaged(keys=spatial_keys, image_only=False),
    EnsureChannelFirstd(keys=spatial_keys),
    Orientationd(keys=spatial_keys, axcodes="RAS"),
    Spacingd(keys=spatial_keys, pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys=image_key, a_min=0, a_max=3000, b_min=0.0, b_max=1.0, clip=True),
    CenterSpatialCropd(keys=spatial_keys, roi_size=ROI_SIZE_FT),

    EnsureTyped(keys=float_tensor_keys, dtype=torch.float32),
    EnsureTyped(keys=long_tensor_keys, dtype=torch.long),
])

test_transforms_mtl = val_transforms_mtl

# 4. Create CacheDatasets (RAM)
print("🚀 Initializing CacheDatasets (Loading data into RAM)...")

# cache_rate=1.0 means cache 100% of the data.
# Reduce to 0.5 if you run out of RAM.
train_ds_mtl = CacheDataset(
    data=train_files_mtl,
    transform=train_transforms_mtl,
    cache_rate=1.0,
    num_workers=NUM_WORKERS
)

val_ds_mtl = CacheDataset(
    data=val_files_mtl,
    transform=val_transforms_mtl,
    cache_rate=1.0,
    num_workers=NUM_WORKERS
)

test_ds_mtl = CacheDataset(
    data=test_files_mtl,
    transform=test_transforms_mtl,
    cache_rate=1.0,
    num_workers=NUM_WORKERS
)

# 5. DataLoaders
print("🔄 Creating DataLoaders...")

train_loader_mtl = MonaiDataLoader(
    train_ds_mtl,
    batch_size=BATCH_SIZE_FT,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=list_data_collate
)

val_loader_mtl = MonaiDataLoader(
    val_ds_mtl,
    batch_size=BATCH_SIZE_FT, # Validation doesn't need smaller batches usually
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader_mtl = MonaiDataLoader(
    test_ds_mtl,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print(f"✅ DataLoaders Ready (Cached in RAM)!")
print(f"Train Batches: {len(train_loader_mtl)}")
print(f"Val Batches:   {len(val_loader_mtl)}")

In [ ]:
# %% Script 1: Trinity Model & Losses
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.networks.nets import SwinUNETR
from monai.losses import DiceCELoss, FocalLoss
import os

# ==============================================================================
# 1. CUSTOM LOSS FUNCTIONS
# ==============================================================================

class SupConLoss(nn.Module):
    """
    Supervised Contrastive Loss.
    Encourages features of the same class (e.g., Grade 4) to be close,
    and different classes to be far apart.
    """
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        device = features.device
        batch_size = features.shape[0]
        labels = labels.view(-1, 1)

        # Create mask for positive pairs (same label)
        mask = torch.eq(labels, labels.T).float().to(device)

        # Mask out self-contrast (diagonal)
        logits_mask = torch.scatter(torch.ones_like(mask), 1, torch.arange(batch_size).view(-1, 1).to(device), 0)
        mask = mask * logits_mask

        # Compute similarity
        sim_matrix = torch.matmul(features, features.T) / self.temperature
        logits_max, _ = torch.max(sim_matrix, dim=1, keepdim=True)
        sim_matrix = sim_matrix - logits_max.detach()

        # Log-Sum-Exp trick for numerical stability
        exp_logits = torch.exp(sim_matrix) * logits_mask
        log_prob = sim_matrix - torch.log(exp_logits.sum(1, keepdim=True) + 1e-6)

        # Mean log-likelihood
        mean_log_prob_pos = (mask * log_prob).sum(1) / (mask.sum(1) + 1e-6)
        return -mean_log_prob_pos.mean()

class MedicalConsistencyLoss(nn.Module):
    """
    Penalizes biologically impossible or rare combinations.
    Example: Glioblastoma (Diag) should NOT be Grade 2.
    """
    def __init__(self, device):
        super().__init__()
        # Matrix Rows: Diagnosis (0:AstroMut, 1:AstroWild, 2:GlioWild, 3:Oligo)
        # Matrix Cols: Grade (0:G2, 1:G3, 2:G4)
        # Values: 1.0 = High Penalty, 0.5 = Medium, 0.0 = No Penalty
        self.penalty_matrix = torch.tensor([
            [0.0, 0.0, 1.0], # Astro Mutant -> G2/G3 common, G4 rare.
            [1.0, 0.5, 0.0], # Astro Wild -> Mostly G4.
            [1.0, 1.0, 0.0], # Glioblastoma -> G4 ONLY.
            [0.0, 0.0, 1.0]  # Oligo -> G2/G3 common.
        ]).to(device)

    def forward(self, logits_diag, logits_grade):
        p_d = F.softmax(logits_diag, dim=1)
        p_g = F.softmax(logits_grade, dim=1)
        # Joint Probability P(Diag, Grade)
        joint = torch.bmm(p_d.unsqueeze(2), p_g.unsqueeze(1))
        return (joint * self.penalty_matrix).sum()

class SemiRelaxedTrinityLoss(nn.Module):
    def __init__(self, weight_grade, weight_diag, device):
        super().__init__()
        self.seg_loss = DiceCELoss(include_background=False, sigmoid=True)
        self.cls_grade = FocalLoss(include_background=True, to_onehot_y=True, gamma=2.0, weight=weight_grade)
        self.cls_diag = FocalLoss(include_background=True, to_onehot_y=True, gamma=2.0, weight=weight_diag)

        # ✅ ΚΡΑΤΑΜΕ ΤΟ SUPCON (Contrastive)
        self.supcon = SupConLoss().to(device)

        # ❌ ΑΦΑΙΡΟΥΜΕ ΤΟ MEDICAL CONSISTENCY (No Penalty Matrix)
        # self.know = MedicalConsistencyLoss(device)

    def forward(self, outputs, t_seg, t_grade, t_diag):
        o_seg, o_g, o_d, o_con, a_g2, a_g3, a_d2, a_d3 = outputs

        # A. Main Task Losses
        L_main = self.seg_loss(o_seg, t_seg) + self.cls_grade(o_g, t_grade) + self.cls_diag(o_d, t_diag)

        # B. Aux Losses (Deep Supervision)
        L_aux = 0.2 * (self.cls_grade(a_g2, t_grade) + self.cls_grade(a_g3, t_grade) +
                       self.cls_diag(a_d2, t_diag) + self.cls_diag(a_d3, t_diag))

        # C. Specialized Losses (ΜΟΝΟ SUPCON)
        # Κρατάμε το weight 0.1 για να μην καπελώσει τα άλλα
        L_spec = 0.1 * self.supcon(o_con, t_grade)

        return L_main + L_aux + L_spec

# ==============================================================================
# 2. TRINITY MODEL ARCHITECTURE
# ==============================================================================

class TrinityMTLModel(nn.Module):
    def __init__(self, in_channels_img, out_channels_seg, feature_size, use_checkpoint,
                 num_classes_grade, num_classes_diag,
                 clinical_dim, radiomics_dim,
                 finetuned_seg_model_path=None, d_model=128, n_heads=8, dropout_rate=0.4):
        super().__init__()

        # 1. Backbone (SwinUNETR)
        self.swin_unetr = SwinUNETR(
            in_channels=in_channels_img, out_channels=out_channels_seg,
            feature_size=feature_size, use_checkpoint=use_checkpoint
        )

        # Load Pretrained Weights
        if finetuned_seg_model_path and os.path.exists(finetuned_seg_model_path):
            try:
                print(f"🔄 Loading backbone from: {finetuned_seg_model_path}")
                ckpt = torch.load(finetuned_seg_model_path, map_location='cpu')
                # Handle dictionary nesting
                if 'state_dict' in ckpt: ckpt = ckpt['state_dict']
                elif 'model' in ckpt: ckpt = ckpt['model']
                self.swin_unetr.load_state_dict(ckpt, strict=False)
            except Exception as e:
                print(f"⚠️ Failed to load backbone weights: {e}")

        # 2. Robust Hooking Strategy
        self.features = {}
        def get_act(name):
            def hook(model, input, output): self.features[name] = output
            return hook

        # Find BasicLayers (Stages) dynamically
        # Indices: 0=48ch, 1=96ch, 2=192ch, 3=384ch (assuming feature_size=48)
        layers = [m for m in self.swin_unetr.swinViT.modules() if m.__class__.__name__ == 'BasicLayer']

        if len(layers) >= 4:
            print(f"✅ Hooking Swin Layers: Indices 1 (96ch), 2 (192ch), 3 (384ch)")
            layers[1].register_forward_hook(get_act('l1'))
            layers[2].register_forward_hook(get_act('l2'))
            layers[3].register_forward_hook(get_act('l3'))
        else:
            print("⚠️ Warning: Dynamic hooking failed. Using fallback indexing (might fail on some versions).")
            self.swin_unetr.swinViT.layers[1].register_forward_hook(get_act('l1'))
            self.swin_unetr.swinViT.layers[2].register_forward_hook(get_act('l2'))
            self.swin_unetr.swinViT.layers[3].register_forward_hook(get_act('l3'))

        # 3. Projections (Matched to Swin Feature Sizes)
        # Stage 1: feature_size * 2^1 = 48 * 2 = 96
        self.proj_l1 = nn.Linear(feature_size * 2, d_model)
        # Stage 2: feature_size * 2^2 = 48 * 4 = 192
        self.proj_l2 = nn.Linear(feature_size * 4, d_model)
        # Stage 3: feature_size * 2^3 = 48 * 8 = 384
        self.proj_l3 = nn.Linear(feature_size * 8, d_model)

        # Separate Projections for Metadata
        self.proj_clin = nn.Sequential(nn.Linear(clinical_dim, d_model), nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout_rate))
        self.proj_rad = nn.Sequential(nn.Linear(radiomics_dim, d_model), nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout_rate))
        self.norm = nn.LayerNorm(d_model)

        # 4. Cross-Attention
        self.cross_attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, batch_first=True)

        # 5. Heads
        # Aux Heads (attached to earlier image layers)
        self.aux_g2 = nn.Linear(d_model, num_classes_grade); self.aux_g3 = nn.Linear(d_model, num_classes_grade)
        self.aux_d2 = nn.Linear(d_model, num_classes_diag); self.aux_d3 = nn.Linear(d_model, num_classes_diag)

        # Main Heads
        self.contrastive = nn.Sequential(nn.Linear(d_model, d_model), nn.ReLU(), nn.Linear(d_model, 64))
        self.head_g = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Dropout(dropout_rate), nn.Linear(64, num_classes_grade))
        self.head_d = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Dropout(dropout_rate), nn.Linear(64, num_classes_diag))

    def process_scale(self, feat_name, proj_layer, attn_mask):
        if feat_name not in self.features:
            return torch.zeros(attn_mask.shape[0], 1, 128).to(attn_mask.device) # Dummy

        feat = self.features[feat_name]

        # Ensure channel last for projection: [B, C, H, W, D] -> [B, H, W, D, C]
        if feat.dim() == 5 and feat.shape[1] == proj_layer.in_features:
            feat = feat.permute(0, 2, 3, 4, 1) # Permute only if C is in dim 1
        elif feat.dim() == 5 and feat.shape[-1] != proj_layer.in_features:
             # Try permuting if channels mismatch last dim
             feat = feat.permute(0, 2, 3, 4, 1)

        # Safe Projection
        if feat.shape[-1] != proj_layer.in_features:
             # Fail silently or print debug here
             return torch.zeros(attn_mask.shape[0], 1, 128).to(attn_mask.device)

        # Masking
        mask = F.interpolate(attn_mask, size=feat.shape[1:4], mode='nearest')
        # Apply mask (broadcasting mask [B,1,H,W,D] -> [B,H,W,D,1] to match feat)
        mask = mask.permute(0, 2, 3, 4, 1)

        masked_feat = feat * mask
        pooled = masked_feat.view(feat.shape[0], -1, feat.shape[-1]).mean(dim=1) # GAP
        return proj_layer(pooled).unsqueeze(1)

    def forward(self, x_img, x_clin, x_rad):
        # A. Segmentation
        l_seg = self.swin_unetr(x_img)
        mask = torch.max(torch.sigmoid(l_seg), dim=1, keepdim=True)[0]

        # B. Image Features (Keys)
        t_l1 = self.process_scale('l1', self.proj_l1, mask)
        t_l2 = self.process_scale('l2', self.proj_l2, mask)
        t_l3 = self.process_scale('l3', self.proj_l3, mask)
        img_toks = torch.cat([t_l1, t_l2, t_l3], dim=1)

        # C. Metadata (Queries)
        t_c = self.proj_clin(x_clin).unsqueeze(1)
        t_r = self.proj_rad(x_rad).unsqueeze(1)
        queries = torch.cat([t_c, t_r], dim=1)

        # D. Fusion
        fused_seq, _ = self.cross_attn(query=queries, key=img_toks, value=img_toks)
        final_feat = self.norm(fused_seq + queries).mean(dim=1)

        # E. Outputs
        o_g = self.head_g(final_feat); o_d = self.head_d(final_feat)
        o_con = F.normalize(self.contrastive(final_feat), dim=1)

        a_g2 = self.aux_g2(t_l1.squeeze(1)); a_g3 = self.aux_g3(t_l2.squeeze(1))
        a_d2 = self.aux_d2(t_l1.squeeze(1)); a_d3 = self.aux_d3(t_l2.squeeze(1))

        return l_seg, o_g, o_d, o_con, a_g2, a_g3, a_d2, a_d3

print("✅ Script 1 Loaded: TrinityMTLModel, Losses, and Wrappers defined.")

In [ ]:
# %% DIAGNOSTIC: INSPECT CHECKPOINT KEYS
import torch
import os

# 1. Load the checkpoint
ckpt_path = os.path.join(runs_base_dir, "segmentation_finetune_custom/best_metric_model_custom.pt")
print(f"📂 Reading: {ckpt_path}")

checkpoint = torch.load(ckpt_path, map_location='cpu')

# 2. Extract State Dict
if 'state_dict' in checkpoint:
    state_dict = checkpoint['state_dict']
    print("ℹ️ Found 'state_dict' key.")
elif 'model' in checkpoint:
    state_dict = checkpoint['model']
    print("ℹ️ Found 'model' key.")
else:
    state_dict = checkpoint
    print("ℹ️ Loaded raw state_dict.")

# 3. Print First 10 Keys of CHECKPOINT
print("\n--- 🔑 First 10 Keys in CHECKPOINT file ---")
for i, k in enumerate(list(state_dict.keys())[:10]):
    print(f"{i}: {k}")

# 4. Print First 10 Keys of CURRENT MODEL
print("\n--- 🤖 First 10 Keys in NEW TRINITY MODEL ---")
model_temp = TrinityMTLModel(
    in_channels_img=IN_CHANNELS, out_channels_seg=OUT_CHANNELS_SEG,
    feature_size=FEATURE_SIZE, use_checkpoint=True,
    num_classes_grade=NUM_CLASSES_GRADE, num_classes_diag=NUM_CLASSES_DIAG,
    clinical_dim=CLINICAL_FEATURE_SIZE, radiomics_dim=RADIOMICS_LEN,
    finetuned_seg_model_path=None, d_model=128
)
for i, k in enumerate(list(model_temp.state_dict().keys())[:10]):
    print(f"{i}: {k}")

In [ ]:
# %% Step 6: Trinity Training Loop (FIXED Weight Loading)
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete, Activations, Compose
from monai.data import decollate_batch
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from tqdm import tqdm
import pandas as pd
import numpy as np
import os
import sys

# 1. SETUP PATHS
output_dir_mtl = os.path.join(runs_base_dir, "mtl_trinity_final")
os.makedirs(output_dir_mtl, exist_ok=True)
best_model_path = os.path.join(output_dir_mtl, "best_trinity_model.pt")
history_csv_path = os.path.join(output_dir_mtl, "history_trinity.csv")

# Compute Weights
try:
    y_grade = df_processed['Grade_Label'].values
    y_diag = df_processed['Diagnosis_Label'].values
    w_g = compute_class_weight('balanced', classes=np.unique(y_grade), y=y_grade)
    w_d = compute_class_weight('balanced', classes=np.unique(y_diag), y=y_diag)
    w_g_tensor = torch.tensor(w_g, dtype=torch.float).to(DEVICE)
    w_d_tensor = torch.tensor(w_d, dtype=torch.float).to(DEVICE)
except:
    w_g_tensor = None; w_d_tensor = None
    print("⚠️ Warning: Weights failed. Using uniform.")

# 2. INITIALIZE MODEL
# ===================
model_mtl = TrinityMTLModel(
    in_channels_img=IN_CHANNELS, out_channels_seg=OUT_CHANNELS_SEG,
    feature_size=FEATURE_SIZE, use_checkpoint=True,
    num_classes_grade=NUM_CLASSES_GRADE, num_classes_diag=NUM_CLASSES_DIAG,
    clinical_dim=CLINICAL_FEATURE_SIZE, radiomics_dim=RADIOMICS_LEN,
    finetuned_seg_model_path=None, # Manual loading below
    d_model=128
).to(DEVICE)

# --- CORRECT WEIGHT LOADING LOGIC ---
finetuned_model_path = os.path.join(runs_base_dir, "segmentation_finetune_custom/best_metric_model_custom.pt")

if os.path.exists(finetuned_model_path):
    print(f"🔄 Loading weights from: {finetuned_model_path}")
    checkpoint = torch.load(finetuned_model_path, map_location='cpu')

    # Εδώ ήταν το λάθος: Τα βάρη είναι στο 'model_state'
    if 'model_state' in checkpoint:
        state_dict = checkpoint['model_state']
        print("ℹ️ Found weights in 'model_state'.")
    elif 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    else:
        state_dict = checkpoint

    new_state_dict = {}
    matched_keys = 0
    for k, v in state_dict.items():
        k = k.replace("module.", "") # Καθαρισμός DistributedDataParallel prefix

        # Αντιστοίχιση: Το saved model έχει 'swinViT...', το νέο θέλει 'swin_unetr.swinViT...'
        if k.startswith("swin_unetr"):
            new_key = k # Already has prefix
        else:
            new_key = f"swin_unetr.{k}" # Add prefix

        # Φιλτράρισμα: Κρατάμε μόνο τα βάρη του backbone
        # (Αποφεύγουμε ασυμβατότητες σε output heads αν υπάρχουν)
        if new_key in model_mtl.state_dict():
            if model_mtl.state_dict()[new_key].shape == v.shape:
                new_state_dict[new_key] = v
                matched_keys += 1

    msg = model_mtl.load_state_dict(new_state_dict, strict=False)
    print(f"✅ SUCCESSFULLY LOADED {matched_keys} LAYERS!")
    if matched_keys < 100:
        print("⚠️ WARNING: Very few layers loaded. Check keys again.")
else:
    print("⚠️ Checkpoint not found. Training from scratch.")

# 3. SETUP TRAINING
optimizer = torch.optim.AdamW(model_mtl.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2, eta_min=1e-6)
criterion = SemiRelaxedTrinityLoss(w_g_tensor, w_d_tensor, DEVICE)

dice_metric_train = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)
dice_metric_val = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)
post_pred_seg = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
post_label_seg = Compose([AsDiscrete(to_onehot=None)])

history = {k: [] for k in ["epoch", "train_loss", "train_dice", "train_f1_g", "train_f1_d",
                           "val_loss", "val_dice", "val_f1_g", "val_acc_g", "val_auc_g",
                           "val_f1_d", "val_acc_d", "val_auc_d", "combined_score"]}

best_val_metric = -1.0
GRAD_ACCUM_STEPS = 4
WARMUP_EPOCHS = 5

# Freeze Backbone
print(f"❄️ Warmup: Freezing Backbone for {WARMUP_EPOCHS} epochs.")
for param in model_mtl.swin_unetr.parameters(): param.requires_grad = False
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model_mtl.parameters()), lr=2e-4)

# 4. TRAINING LOOP
print(f"\n🚀 Starting Trinity Training ({NUM_EPOCHS_MTL} Epochs)...")

for epoch in range(0, NUM_EPOCHS_MTL):
    current_epoch = epoch + 1

    # Unfreeze
    if current_epoch == (WARMUP_EPOCHS + 1):
        print("\n🔥 Unfreezing Backbone & Reducing LR...")
        for param in model_mtl.parameters(): param.requires_grad = True
        optimizer = torch.optim.AdamW([
            {'params': model_mtl.swin_unetr.parameters(), 'lr': 1e-5},
            {'params': model_mtl.head_g.parameters(), 'lr': 1e-4},
            {'params': model_mtl.head_d.parameters(), 'lr': 1e-4},
            {'params': model_mtl.proj_l2.parameters(), 'lr': 1e-4},
            {'params': model_mtl.proj_clin.parameters(), 'lr': 1e-4}
        ], weight_decay=1e-2)
        scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2, eta_min=1e-6)

    model_mtl.train()
    dice_metric_train.reset()
    ep_loss = 0; step = 0
    tr_pred_g, tr_lbl_g = [], []; tr_pred_d, tr_lbl_d = [], []

    pbar = tqdm(train_loader_mtl, desc=f"Ep {current_epoch} Train")
    for i, batch in enumerate(pbar):
        step += 1
        img = batch[image_key].to(DEVICE)
        seg = batch[label_key].to(DEVICE)
        clin = batch[clinical_key].to(DEVICE)
        rad = batch[radiomics_key].to(DEVICE)
        lbl_g = batch[grade_key].to(DEVICE); lbl_d = batch[diag_key].to(DEVICE)

        if current_epoch > WARMUP_EPOCHS:
            p = torch.rand(1).item()
            if p < 0.15: img = torch.zeros_like(img)
            elif p < 0.30: rad = torch.zeros_like(rad)
            elif p < 0.45: clin = torch.zeros_like(clin)

        outputs = model_mtl(img, clin, rad)
        loss = criterion(outputs, seg, lbl_g, lbl_d)

        if not torch.isfinite(loss): continue
        (loss / GRAD_ACCUM_STEPS).backward()
        if (i+1)%GRAD_ACCUM_STEPS == 0: optimizer.step(); optimizer.zero_grad()
        ep_loss += loss.item()

        o_s, o_g, o_d, _, _, _, _, _ = outputs
        dice_metric_train(y_pred=[post_pred_seg(i) for i in decollate_batch(o_s)], y=[post_label_seg(i) for i in decollate_batch(seg)])
        tr_pred_g.extend(torch.argmax(o_g, 1).cpu().numpy()); tr_lbl_g.extend(lbl_g.cpu().numpy())
        tr_pred_d.extend(torch.argmax(o_d, 1).cpu().numpy()); tr_lbl_d.extend(lbl_d.cpu().numpy())
        pbar.set_postfix(loss=f"{loss.item():.2f}")

    # Train Stats
    tr_loss = ep_loss/step; tr_dice = dice_metric_train.aggregate().item()
    tr_f1_g = f1_score(tr_lbl_g, tr_pred_g, average='macro'); tr_f1_d = f1_score(tr_lbl_d, tr_pred_d, average='macro')

    # Validation
    if current_epoch % VAL_INTERVAL == 0:
        model_mtl.eval()
        dice_metric_val.reset()
        val_loss=0; val_steps=0
        preds_g, probs_g, lbls_g = [], [], []
        preds_d, probs_d, lbls_d = [], [], []

        with torch.no_grad():
            for batch in tqdm(val_loader_mtl, desc="Validating"):
                val_steps += 1
                img = batch[image_key].to(DEVICE); seg = batch[label_key].to(DEVICE)
                clin = batch[clinical_key].to(DEVICE); rad = batch[radiomics_key].to(DEVICE)
                lbl_g = batch[grade_key].to(DEVICE); lbl_d = batch[diag_key].to(DEVICE)

                outputs = model_mtl(img, clin, rad)
                val_loss += criterion(outputs, seg, lbl_g, lbl_d).item()
                o_s, o_g, o_d, _, _, _, _, _ = outputs

                dice_metric_val(y_pred=[post_pred_seg(i) for i in decollate_batch(o_s)], y=[post_label_seg(i) for i in decollate_batch(seg)])
                preds_g.extend(torch.argmax(o_g, 1).cpu().numpy()); probs_g.extend(torch.softmax(o_g, 1).cpu().numpy()); lbls_g.extend(lbl_g.cpu().numpy())
                preds_d.extend(torch.argmax(o_d, 1).cpu().numpy()); probs_d.extend(torch.softmax(o_d, 1).cpu().numpy()); lbls_d.extend(lbl_d.cpu().numpy())

        val_dice = dice_metric_val.aggregate().item()
        val_f1_g = f1_score(lbls_g, preds_g, average='macro'); val_acc_g = accuracy_score(lbls_g, preds_g)
        try: val_auc_g = roc_auc_score(lbls_g, probs_g, multi_class='ovr', average='macro')
        except: val_auc_g = 0.0
        val_f1_d = f1_score(lbls_d, preds_d, average='macro'); val_acc_d = accuracy_score(lbls_d, preds_d)
        try: val_auc_d = roc_auc_score(lbls_d, probs_d, multi_class='ovr', average='macro')
        except: val_auc_d = 0.0

        combined = (val_dice * 0.3) + (val_f1_g * 0.4) + (val_f1_d * 0.3)

        # Print Reports
        print(f"--- Val Epoch {current_epoch} ---")
        print(f"Combined: {combined:.4f} | Dice: {val_dice:.4f}")
        print(f"Grade F1: {val_f1_g:.3f} | Diag F1: {val_f1_d:.3f}")

        print("\n=== 🧠 GRADE REPORT ===")
        print(classification_report(lbls_g, preds_g, target_names=[str(c) for c in GRADE_CLASSES], zero_division=0))

        print("\n=== 🧬 DIAGNOSIS REPORT ===")
        print(classification_report(lbls_d, preds_d, target_names=[str(c) for c in DIAG_CLASSES], zero_division=0))

        if combined > best_val_metric:
            best_val_metric = combined
            torch.save(model_mtl.state_dict(), best_model_path)
            print("🏆 New Best Model Saved!")

        # Update History
        history["epoch"].append(current_epoch); history["train_loss"].append(tr_loss); history["train_dice"].append(tr_dice)
        history["train_f1_g"].append(tr_f1_g); history["train_f1_d"].append(tr_f1_d)
        history["val_loss"].append(val_loss/val_steps); history["val_dice"].append(val_dice)
        history["val_f1_g"].append(val_f1_g); history["val_acc_g"].append(val_acc_g); history["val_auc_g"].append(val_auc_g)
        history["val_f1_d"].append(val_f1_d); history["val_acc_d"].append(val_acc_d); history["val_auc_d"].append(val_auc_d)
        history["combined_score"].append(combined)

        pd.DataFrame(history).to_csv(history_csv_path, index=False)
    else:
        # Append stats for train, NaNs for val
        history["epoch"].append(current_epoch); history["train_loss"].append(tr_loss); history["train_dice"].append(tr_dice)
        history["train_f1_g"].append(tr_f1_g); history["train_f1_d"].append(tr_f1_d)
        for k in ["val_loss", "val_dice", "val_f1_g", "val_acc_g", "val_auc_g", "val_f1_d", "val_acc_d", "val_auc_d", "combined_score"]:
            history[k].append(np.nan)

    if scheduler: scheduler.step()

# Mamba

In [ ]:
!git clone https://github.com/ge-xing/SegMamba.git

In [ ]:
!pip uninstall -y transformers

# 2. Εγκατάσταση της "χρυσής" έκδοσης για SegMamba
!pip install transformers==4.39.3

In [ ]:
# %% CORRECT INSTALLATION (Using bundled source)
import os

# 1. Καθαρισμός από προηγούμενες (λάθος) εγκαταστάσεις
print("🗑️ Uninstalling incorrect versions...")
!pip uninstall -y mamba-ssm causal-conv1d

# 2. Εγκατάσταση του bundled Causal-Conv1d
# Μπαίνουμε στον φάκελο του repo και χτίζουμε το δικό τους πακέτο
print("⏳ Installing bundled Causal-Conv1d...")
if os.path.exists("SegMamba/causal-conv1d"):
    !cd SegMamba/causal-conv1d && pip install .
else:
    print("❌ Error: Folder SegMamba/causal-conv1d not found!")

# 3. Εγκατάσταση του bundled Mamba
# Αυτό είναι το κρίσιμο βήμα που έδειξε το link σου
print("⏳ Installing bundled Mamba (Takes ~5 mins)...")
if os.path.exists("SegMamba/mamba"):
    !cd SegMamba/mamba && pip install .
else:
    print("❌ Error: Folder SegMamba/mamba not found!")

print("✅ Installation complete. Now try importing.")

In [ ]:
# %% Verify SegMamba Import
import sys
import os
import torch

# Προσθήκη path
repo_path = os.path.abspath("SegMamba")
if repo_path not in sys.path:
    sys.path.append(repo_path)

try:
    from model_segmamba.segmamba import SegMamba

    # Δοκιμή στο GPU
    model = SegMamba(
        in_chans=1,
        out_chans=1,
        depths=[2, 2, 2, 2],
        feat_size=[48, 96, 192, 384]
    ).to('cuda')

    print("✅ SegMamba initialized successfully on GPU!")
    del model

except Exception as e:
    print(f"❌ Error: {e}")
    print("💡 TIP: If it still fails, go to 'Runtime' -> 'Restart Session' and try ONLY this cell again.")

In [ ]:
finetuned_model_path_mamba = os.path.join(runs_base_dir, "segmentation_segmamba_scratch/best_model_segmamba_scratch.pt")

In [ ]:
class AdvancedMaskGuidedModel_v3_Mamba(nn.Module):
    def __init__(self, clinical_len, radiomics_len, d_model=128, pretrained_weights_path=None):
        super().__init__()

        # 1. SegMamba Backbone
        self.segmamba = SegMamba(
            in_chans=1, # Ή 4, ανάλογα με το τι εκπαίδευσες
            out_chans=1,
            feat_size=[48, 96, 192, 384],
            hidden_size=768
        )

        # Φόρτωση βαρών
        if pretrained_weights_path and os.path.exists(pretrained_weights_path):
            checkpoint = torch.load(pretrained_weights_path, map_location='cpu')
            pretrained_dict = checkpoint['model_state'] if 'model_state' in checkpoint else checkpoint
            self.segmamba.load_state_dict(pretrained_dict, strict=True)
            print("✅ Success: Pre-trained SegMamba weights loaded perfectly!")

        # Hooks
        self.captured_bottleneck = None
        self.segmamba.encoder5.register_forward_hook(self._hook_bottleneck)
        self.attn_weights = None
        # --- ΕΔΩ ΕΙΝΑΙ ΤΑ ΟΝΟΜΑΤΑ ΠΟΥ ΨΑΧΝΕΙ Ο OPTIMIZER ---
        self.image_proj = nn.Linear(768, d_model)
        self.clinical_proj = nn.Sequential(
            nn.Linear(clinical_len, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU()
        )
        self.radiomics_proj = nn.Sequential(
            nn.Linear(radiomics_len, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU()
        )

        self.grade_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.diag_token = nn.Parameter(torch.zeros(1, 1, d_model))

        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=4, batch_first=True)
        self.fusion_transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        self.grade_3way_head = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, 3))
        self.grade_2way_head = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, 2))

        self.diagnosis_head = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, 4))

        # Auxiliary Heads
        self.aux_head_enc = nn.Linear(d_model, 3)  # Αντιστοιχεί στο aux_grade_core
        self.aux_head_dec = nn.Linear(d_model, 3)  # Αντιστοιχεί στο aux_grade_peri
        self.aux_head_rad = nn.Linear(d_model, 3)  # Αντιστοιχεί στο aux_diag_rad (για το grade)


    def _hook_bottleneck(self, module, input, output):
        self.captured_bottleneck = output

    def forward(self, x_img, x_clin, x_rad):
        logits_seg = self.segmamba(x_img)

        # Masking Logic
        seg_prob = torch.sigmoid(logits_seg)
        mask_core = (torch.max(seg_prob, dim=1, keepdim=True)[0] > 0.5).float()
        mask_dilated = F.max_pool3d(mask_core, kernel_size=5, stride=1, padding=2)
        mask_peri = torch.clamp(mask_dilated - mask_core, min=0.0)

        # Feature Pooling
        spatial_size = self.captured_bottleneck.shape[2:]
        m_core = F.interpolate(mask_core, size=spatial_size, mode='nearest')
        m_peri = F.interpolate(mask_peri, size=spatial_size, mode='nearest')

        f_core = torch.sum(self.captured_bottleneck * m_core, dim=(2,3,4)) / (torch.sum(m_core) + 1e-6)
        f_peri = torch.sum(self.captured_bottleneck * m_peri, dim=(2,3,4)) / (torch.sum(m_peri) + 1e-6)

        # Projection
        t_core = self.image_proj(f_core).unsqueeze(1)
        t_peri = self.image_proj(f_peri).unsqueeze(1)
        t_clin = self.clinical_proj(x_clin).unsqueeze(1)
        t_rad = self.radiomics_proj(x_rad).unsqueeze(1)

        # Fusion
        B = x_img.shape[0]
        tokens = torch.cat([
            self.grade_token.expand(B, -1, -1),   # Index 0
            self.diag_token.expand(B, -1, -1),    # Index 1
            t_core,                               # Index 2
            t_peri,                               # Index 3
            t_clin,                               # Index 4
            t_rad                                 # Index 5
        ], dim=1)

        # Για να πάρουμε το attention, θα χρησιμοποιήσουμε μια custom προσέγγιση
        # ή θα αναλύσουμε το βάρος του τελευταίου layer.
        # Εδώ είναι ένας τρόπος να το κάνεις extract:
        fused = self.fusion_transformer(tokens)
        self.token_names = ["Grade_Tok", "Diag_Tok", "Core_Img", "Peri_Img", "Clinical", "Radiomics"]

        grade_feature = fused[:, 0, :]

        # Το ίδιο feature δίνει δύο προβλέψεις:
        p_grade_3way = self.grade_3way_head(grade_feature) # Πρόβλεψη G2, G3, G4
        p_grade_2way = self.grade_2way_head(grade_feature) # Πρόβλεψη LGG vs HGG

        # Παίρνουμε το fused token στο Index 1 (Diagnosis Token)
        p_diag = self.diagnosis_head(fused[:, 1, :])

        # Aux (Deep Supervision)
        aux_enc = self.aux_head_enc(t_core.squeeze(1))
        aux_dec = self.aux_head_dec(t_peri.squeeze(1))
        aux_rad = self.aux_head_rad(t_rad.squeeze(1))

        return logits_seg, p_grade_3way, p_grade_2way, p_diag, aux_enc, aux_dec, aux_rad

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete, Activations, Compose
from monai.losses import DiceCELoss, FocalLoss
from monai.data import decollate_batch
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# --- 1. Class Weights ---
print("Computing Class Weights for Grade and Diagnosis...")
try:
    y_grade_all = df_processed['Grade_Label'].values
    y_diag_all = df_processed['Diagnosis_Label'].values
    weights_grade = compute_class_weight('balanced', classes=np.unique(y_grade_all), y=y_grade_all)
    weights_diag = compute_class_weight('balanced', classes=np.unique(y_diag_all), y=y_diag_all)

    # Ειδικά για το Grade 3, αν θες να το ενισχύσεις παραπάνω:
    # weights_grade[1] *= 1.2

    weights_grade_tensor = torch.tensor(weights_grade, dtype=torch.float).to(DEVICE)
    weights_diag_tensor = torch.tensor(weights_diag, dtype=torch.float).to(DEVICE)
    print("✅ Weights loaded and moved to DEVICE.")
except Exception as e:
    weights_grade_tensor = None; weights_diag_tensor = None
    print(f"⚠️ Weights could not be computed. Error: {e}")

# --- Helper: Save Confusion Matrices ---
def save_cm_plot(lbls_3w, preds_3w, lbls_2w, preds_2w, epoch, path):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    sns.heatmap(confusion_matrix(lbls_3w, preds_3w), annot=True, fmt='d', cmap='Blues', ax=ax1,
                xticklabels=GRADE_CLASSES, yticklabels=GRADE_CLASSES)
    ax1.set_title(f'3-way Grade CM (Epoch {epoch})')
    sns.heatmap(confusion_matrix(lbls_2w, preds_2w), annot=True, fmt='d', cmap='Greens', ax=ax2,
                xticklabels=["LGG", "HGG"], yticklabels=["LGG", "HGG"])
    ax2.set_title(f'2-way Grade CM (LGG vs HGG)')
    plt.tight_layout()
    plt.savefig(os.path.join(path, f"cm_epoch_{epoch}.png"))
    plt.close()

In [ ]:
class AdvancedFusionLoss_V4(nn.Module):
    def __init__(self, weights_3w, weights_2w, weights_diag,
                 w_seg=1.0, w_3w=1.0, w_2w=1.0, w_diag=0.7, w_aux=0.3):
        super().__init__()
        self.w_seg = w_seg
        self.w_3w = w_3w
        self.w_2w = w_2w
        self.w_diag = w_diag
        self.w_aux = w_aux

        # A. Segmentation Loss
        self.seg_loss = DiceCELoss(include_background=False, sigmoid=True)

        # B. Classification Losses (Focal Loss για ανθεκτικότητα σε imbalances)
        self.class_loss_3w = FocalLoss(include_background=True, to_onehot_y=True, gamma=2.0, weight=weights_3w)
        self.class_loss_2w = FocalLoss(include_background=True, to_onehot_y=True, gamma=2.0, weight=weights_2w)
        self.class_loss_diag = FocalLoss(include_background=True, to_onehot_y=True, gamma=2.0, weight=weights_diag)

    def forward(self, outputs, target_seg, target_3w, target_diag):
        """
        outputs: (l_seg, p_3w, p_2w, p_diag, ag_core, ag_peri, ag_rad) -> 7 items
        target_3w: Labels 0, 1, 2 (Grade 2, 3, 4)
        """
        l_seg, p_3w, p_2w, p_diag, ag_core, ag_peri, ag_rad = outputs

        # 1. Δυναμική δημιουργία Binary Target: LGG (0,1) -> 0, HGG (2) -> 1
        # Προσοχή: Το .long() είναι απαραίτητο για το CrossEntropy/Focal
        target_2w = (target_3w == 2).long()

        # 2. Υπολογισμός επιμέρους Losses
        loss_seg = self.seg_loss(l_seg, target_seg)
        loss_3w = self.class_loss_3w(p_3w, target_3w)
        loss_2w = self.class_loss_2w(p_2w, target_2w)
        loss_diag = self.class_loss_diag(p_diag, target_diag)

        # 3. Auxiliary Losses (Deep Supervision πάντα στο 3-way για μέγιστη λεπτομέρεια)
        loss_aux = (self.class_loss_3w(ag_core, target_3w) +
                    self.class_loss_3w(ag_peri, target_3w) +
                    self.class_loss_3w(ag_rad, target_3w))

        # 4. Τελικό Σταθμισμένο Άθροισμα
        # Η εξίσωση: L_total = w_s*L_s + w_3w*L_3w + w_2w*L_2w + w_d*L_d + w_a*L_a
        total_loss = (
            (self.w_seg * loss_seg) +
            (self.w_3w * loss_3w) +
            (self.w_2w * loss_2w) +
            (self.w_diag * loss_diag) +
            (self.w_aux * loss_aux)
        )

        return total_loss

criterion = AdvancedFusionLoss_V4(
    weights_3w=weights_grade_tensor,     # [G2, G3, G4]
    weights_2w=torch.tensor([1.0, 1.8]).to(DEVICE), # Ενίσχυση του HGG (κλάση 1)
    weights_diag=weights_diag_tensor,
    w_seg=1.0,   # Σταθερή βάση το Segmentation
    w_3w=0.6,    # Χαμηλότερο βάρος στο δύσκολο task στην αρχή
    w_2w=1.4,    # Υψηλό βάρος στο 2-way για να "κλειδώσει" την κακοήθεια
    w_diag=0.8,  # Σημαντικό για τον υποτύπο
    w_aux=0.3    # Βοηθητικό βάρος
).to(DEVICE)

In [ ]:
# --- Settings & Model Init ---
output_dir_mtl = os.path.join(runs_base_dir, "mtl_mask_guided_mamba_v3")
os.makedirs(output_dir_mtl, exist_ok=True)
best_model_path = os.path.join(output_dir_mtl, "best_mamba_fusion.pt")
history_csv_path = os.path.join(output_dir_mtl, "history_mamba.csv")
model_mtl = AdvancedMaskGuidedModel_v3_Mamba(
    clinical_len=CLINICAL_FEATURE_SIZE,
    radiomics_len=RADIOMICS_LEN,
    d_model=128,
    pretrained_weights_path=finetuned_model_path_mamba # Το path από το segmentation
).to(DEVICE)

# --- 5. Optimizer με διαφορετικά LR ---
LR_BACKBONE = 1e-6 # Ξεκινάμε πολύ χαμηλά για το pre-trained
LR_HEADS = 1e-4

# Παγώνουμε το backbone για το Warm-up (Stage 1)
# Πάγωμα για Stage 1
for param in model_mtl.segmamba.parameters():
    param.requires_grad = False

# --- ΔΙΟΡΘΩΜΕΝΗ ΛΙΣΤΑ ΠΑΡΑΜΕΤΡΩΝ ---
# Προσθέτουμε τα νέα heads και ενημερώνουμε τα ονόματα
params = [
    {'params': model_mtl.segmamba.parameters(), 'lr': LR_BACKBONE},
    {'params': model_mtl.image_proj.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.clinical_proj.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.radiomics_proj.parameters(), 'lr': LR_HEADS},
    {'params': [model_mtl.grade_token, model_mtl.diag_token], 'lr': LR_HEADS},
    {'params': model_mtl.fusion_transformer.parameters(), 'lr': LR_HEADS},

    # ΕΔΩ Η ΑΛΛΑΓΗ: Δύο heads αντί για ένα για το Grade
    {'params': model_mtl.grade_3way_head.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.grade_2way_head.parameters(), 'lr': LR_HEADS},

    {'params': model_mtl.diagnosis_head.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.aux_head_enc.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.aux_head_dec.parameters(), 'lr': LR_HEADS},
    {'params': model_mtl.aux_head_rad.parameters(), 'lr': LR_HEADS},
]

optimizer = torch.optim.AdamW(params, weight_decay=1e-3)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=15, T_mult=2, eta_min=1e-6)

In [ ]:
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete, Activations, Compose
from monai.data import decollate_batch
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score

# 1. Metrics για Segmentation
dice_metric_train = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)
dice_metric_val = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)

# 2. Post-processing για τις μάσκες
post_pred_seg = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
post_label_seg = Compose([AsDiscrete(to_onehot=None)])

# 3. Helper για τη μετατροπή των 3-way labels σε 2-way (LGG/HGG)
# Αυτό το χρησιμοποιούμε για να συγκρίνουμε το output του 2-way head με την αλήθεια
def get_binary_labels(lbls_3w):
    """
    Μετατρέπει τα labels:
    Grade 2 (0) & Grade 3 (1) -> 0 (LGG)
    Grade 4 (2) -> 1 (HGG)
    """
    return (np.array(lbls_3w) == 2).astype(int).flatten()

# 4. (Προαιρετικό) Helper για έλεγχο συνέπειας (Consistency Check)
def compute_derived_2way(probs_3w):
    """
    Υπολογίζει το 'μαθηματικό' 2-way από το 3-way head για να δούμε αν συμφωνούν.
    """
    probs_3w = np.array(probs_3w)
    probs_2w_derived = np.stack([probs_3w[:, 0] + probs_3w[:, 1], probs_3w[:, 2]], axis=1)
    preds_2w_derived = np.argmax(probs_2w_derived, axis=1).flatten()
    return preds_2w_derived

print("✅ Dual-Metrics system initialized for 3-way and 2-way evaluation.")

In [ ]:
# --- 1. INITIAL STATE & OPTIMIZER ---
best_val_metric = -1.0
GRAD_ACCUM_STEPS = 4
WARMUP_EPOCHS = 10
VAL_INTERVAL = 1

# Πάγωμα Backbone για το Stage 1
for param in model_mtl.segmamba.parameters():
    param.requires_grad = False

# Αρχικός Optimizer (Stage 1) - ADAMW για τα Heads & Transformer
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model_mtl.parameters()), lr=2e-4)

# Αρχικοποίηση History (Full Version)
history = {
    "epoch": [],
    "train_loss": [], "train_dice": [],
    "train_f1_g": [], "train_f1_g2": [], "train_f1_d": [],
    "val_loss": [], "val_dice": [],
    "val_f1_g": [], "val_acc_g": [], "val_auc_g": [],
    "val_f1_g2": [], "val_acc_g2": [], "val_auc_g2": [],
    "val_f1_d": [], "val_acc_d": [], "val_auc_d": [],
    "combined_score": []
}

print(f"\n🚀 Starting Trinity V4 Dual-Grade Training ({NUM_EPOCHS_MTL} Epochs)...")

# ---------------------------------------------------------
# ΚΥΡΙΟΣ ΒΡΟΧΟΣ ΕΚΠΑΙΔΕΥΣΗΣ
# ---------------------------------------------------------
for epoch in range(0, NUM_EPOCHS_MTL):
    current_epoch = epoch + 1

    # === UNFREEZE STRATEGY (Stage 2) ===
    if current_epoch == (WARMUP_EPOCHS + 1):
        print("\n" + "!"*60 + "\n🔥 Stage 2: Unfreezing Backbone & Applying Differential LRs...\n" + "!"*60)
        for param in model_mtl.parameters():
            param.requires_grad = True

        optimizer = torch.optim.AdamW([
            # Backbone: Το κρατάμε σταθερό, είναι η βάση μας
            {'params': model_mtl.segmamba.parameters(), 'lr': 1e-5},

            # Projections & Transformer: Εδώ γίνεται το μεγάλο overfitting
            # Μειώνουμε κι άλλο το LR και αυξάνουμε το τοπικό Weight Decay
            {'params': model_mtl.image_proj.parameters(), 'lr': 5e-5},
            {'params': model_mtl.clinical_proj.parameters(), 'lr': 5e-5},
            {'params': model_mtl.radiomics_proj.parameters(), 'lr': 5e-5},
            {'params': model_mtl.fusion_transformer.parameters(), 'lr': 5e-5},

            # Classification Heads: Τα κάνουμε ακόμα πιο "αργά" για να ακολουθούν το backbone
            {'params': model_mtl.grade_3way_head.parameters(), 'lr': 5e-5},
            {'params': model_mtl.grade_2way_head.parameters(), 'lr': 5e-5},
            {'params': model_mtl.diagnosis_head.parameters(), 'lr': 5e-5}
        ], weight_decay=5e-2)
        scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2, eta_min=1e-6)

    model_mtl.train()
    dice_metric_train.reset()
    ep_loss, step = 0, 0
    tr_pred_g3, tr_pred_g2, tr_lbl_g3 = [], [], []
    tr_pred_d, tr_lbl_d = [], []

    pbar = tqdm(train_loader_mtl, desc=f"Ep {current_epoch} [Train]")
    for i, batch in enumerate(pbar):
        step += 1
        img, seg = batch[image_key].to(DEVICE), batch[label_key].to(DEVICE)
        clin, rad = batch[clinical_key].to(DEVICE), batch[radiomics_key].to(DEVICE)
        lbl_g = batch[grade_key].to(DEVICE) # 3-way label (0,1,2)
        lbl_d = batch[diag_key].to(DEVICE)

        # === MODALITY DROPOUT ===
        if current_epoch > WARMUP_EPOCHS:
            p = torch.rand(1).item()
            if p < 0.25: img = torch.zeros_like(img)
            elif p < 0.20: rad = torch.zeros_like(rad)

        # Forward (V4 outputs: seg, p_g3, p_g2, p_diag, aux1, aux2, aux3)
        outputs = model_mtl(img, clin, rad)
        loss = criterion(outputs, seg, lbl_g, lbl_d)

        if not torch.isfinite(loss): continue
        (loss / GRAD_ACCUM_STEPS).backward()

        if (i+1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model_mtl.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

        ep_loss += loss.item()

        # Unpack for Stats
        o_s, o_g3, o_g2, o_d = outputs[0], outputs[1], outputs[2], outputs[3]

        # Segmentation Stats
        dice_metric_train(y_pred=[post_pred_seg(idx) for idx in decollate_batch(o_s)],
                          y=[post_label_seg(idx) for idx in decollate_batch(seg)])

        # Classification Stats
        tr_pred_g3.extend(torch.argmax(o_g3, 1).cpu().numpy())
        tr_pred_g2.extend(torch.argmax(o_g2, 1).cpu().numpy())
        tr_lbl_g3.extend(lbl_g.cpu().numpy())
        tr_pred_d.extend(torch.argmax(o_d, 1).cpu().numpy())
        tr_lbl_d.extend(lbl_d.cpu().numpy())

        pbar.set_postfix(loss=f"{loss.item():.2f}")

    # --- Train Metrics Calculation ---
    tr_loss = ep_loss/step
    tr_dice = dice_metric_train.aggregate().item()
    tr_f1_g3 = f1_score(tr_lbl_g3, tr_pred_g3, average='macro', zero_division=0)
    tr_f1_g2 = f1_score((np.array(tr_lbl_g3)==2).astype(int), tr_pred_g2, average='binary', zero_division=0)
    tr_f1_d = f1_score(tr_lbl_d, tr_pred_d, average='macro', zero_division=0)

    # === VALIDATION PHASE ===
    if current_epoch % VAL_INTERVAL == 0:
        model_mtl.eval()
        dice_metric_val.reset()
        val_loss_sum, val_steps = 0, 0
        preds_g3, probs_g3, lbls_g3 = [], [], []
        preds_g2, probs_g2, lbls_g2 = [], [], []
        preds_d, probs_d, lbls_d = [], [], []

        with torch.no_grad():
            for batch in tqdm(val_loader_mtl, desc="Validating"):
                val_steps += 1
                img, seg = batch[image_key].to(DEVICE), batch[label_key].to(DEVICE)
                clin, rad = batch[clinical_key].to(DEVICE), batch[radiomics_key].to(DEVICE)
                lbl_g, lbl_diag = batch[grade_key].to(DEVICE), batch[diag_key].to(DEVICE)

                outputs = model_mtl(img, clin, rad)
                val_loss_sum += criterion(outputs, seg, lbl_g, lbl_diag).item()

                o_s, o_g3, o_g2, o_d = outputs[0], outputs[1], outputs[2], outputs[3]

                dice_metric_val(y_pred=[post_pred_seg(idx) for idx in decollate_batch(o_s)],
                                y=[post_label_seg(idx) for idx in decollate_batch(seg)])

                # Grade 3-way
                preds_g3.extend(torch.argmax(o_g3, 1).cpu().numpy())
                probs_g3.extend(torch.softmax(o_g3, 1).cpu().numpy())
                lbls_g3.extend(lbl_g.cpu().numpy())

                # Grade 2-way (Direct from Head)
                preds_g2.extend(torch.argmax(o_g2, 1).cpu().numpy())
                probs_g2.extend(torch.softmax(o_g2, 1).cpu().numpy())
                lbls_g2.extend((lbl_g.cpu().numpy() == 2).astype(int))

                # Diagnosis
                preds_d.extend(torch.argmax(o_d, 1).cpu().numpy())
                probs_d.extend(torch.softmax(o_d, 1).cpu().numpy())
                lbls_d.extend(lbl_diag.cpu().numpy())

        # --- Metric Summaries ---
        val_dice = dice_metric_val.aggregate().item()
        val_f1_g3 = f1_score(lbls_g3, preds_g3, average='macro', zero_division=0)
        val_acc_g3 = accuracy_score(lbls_g3, preds_g3)
        val_f1_g2 = f1_score(lbls_g2, preds_g2, average='binary', zero_division=0)
        val_acc_g2 = accuracy_score(lbls_g2, preds_g2)
        val_f1_d = f1_score(lbls_d, preds_d, average='macro', zero_division=0)
        val_acc_d = accuracy_score(lbls_d, preds_d)

        try: val_auc_g3 = roc_auc_score(lbls_g3, probs_g3, multi_class='ovr', average='macro')
        except: val_auc_g3 = 0.0
        try: val_auc_g2 = roc_auc_score(lbls_g2, np.array(probs_g2)[:, 1])
        except: val_auc_g2 = 0.0
        try: val_auc_d = roc_auc_score(lbls_d, probs_d, multi_class='ovr', average='macro')
        except: val_auc_d = 0.0

        # Optimization Target
        combined = (val_dice * 0.3) + (val_f1_g2 * 0.3) + (val_f1_g3 * 0.2) + (val_f1_d * 0.2)

        # --- FULL LOGGING REPORTS ---
        print(f"\n--- Val Epoch {current_epoch} ---")
        print(f"Combined Score: {combined:.4f} | Dice: {val_dice:.4f} | Train Loss: {tr_loss:.4f} | Val Loss: {val_loss_sum/val_steps:.4f}")
        print(f"Grade 3-way F1: {val_f1_g3:.3f} | Grade 2-way F1: {val_f1_g2:.3f}")

        print("\n=== 🧠 3-WAY GRADE REPORT ===")
        print(classification_report(lbls_g3, preds_g3, target_names=["G2", "G3", "G4"], zero_division=0))

        print("\n=== ⚖️ 2-WAY GRADE REPORT (LGG vs HGG) ===")
        print(classification_report(lbls_g2, preds_g2, target_names=["LGG", "HGG"], zero_division=0))

        print("\n=== 🧬 DIAGNOSIS REPORT ===")
        print(classification_report(lbls_d, preds_d, target_names=[str(c)[:25] for c in DIAG_CLASSES], zero_division=0))

        # Save Best Model
        if combined > best_val_metric:
            best_val_metric = combined
            torch.save(model_mtl.state_dict(), best_model_path)
            save_cm_plot(lbls_g3, preds_g3, lbls_g2, preds_g2, current_epoch, output_dir_mtl)
            print("🏆 New Best Model & Confusion Matrices Saved!")

        # --- UPDATE HISTORY ---
        history["epoch"].append(current_epoch)
        history["train_loss"].append(tr_loss); history["train_dice"].append(tr_dice)
        history["train_f1_g"].append(tr_f1_g3); history["train_f1_g2"].append(tr_f1_g2); history["train_f1_d"].append(tr_f1_d)
        history["val_loss"].append(val_loss_sum/val_steps); history["val_dice"].append(val_dice)
        history["val_f1_g"].append(val_f1_g3); history["val_acc_g"].append(val_acc_g3); history["val_auc_g"].append(val_auc_g3)
        history["val_f1_g2"].append(val_f1_g2); history["val_acc_g2"].append(val_acc_g2); history["val_auc_g2"].append(val_auc_g2)
        history["val_f1_d"].append(val_f1_d); history["val_acc_d"].append(val_acc_d); history["val_auc_d"].append(val_auc_d)
        history["combined_score"].append(combined)

        pd.DataFrame(history).to_csv(history_csv_path, index=False)

    else:
        # Padding for non-validation epochs
        for k in history.keys():
            if k == "epoch": history[k].append(current_epoch)
            elif k.startswith("train"): # fill what we have
                if k == "train_loss": history[k].append(tr_loss)
                elif k == "train_dice": history[k].append(tr_dice)
                elif k == "train_f1_g": history[k].append(tr_f1_g3)
                elif k == "train_f1_g2": history[k].append(tr_f1_g2)
                elif k == "train_f1_d": history[k].append(tr_f1_d)
                else: history[k].append(np.nan)
            else: history[k].append(np.nan)

    if current_epoch > WARMUP_EPOCHS:
        scheduler.step()

print(f"\n✅ Training Complete. Best Combined Score: {best_val_metric:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
import numpy as np
import os
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete, Activations, Compose
from monai.data import decollate_batch

# --- 1. Hook & Attention Extraction Helpers ---
class TransformerInputCatcher:
    def __init__(self, model):
        self.tokens = None
        self.hook = model.fusion_transformer.register_forward_pre_hook(self.save_input)
    def save_input(self, module, inputs):
        self.tokens = inputs[0] # Capture [Batch, Seq, Dim]
    def remove(self):
        self.hook.remove()

def get_attention_maps(model, catcher):
    tokens = catcher.tokens
    B, N, D = tokens.shape
    attn_maps = []
    with torch.no_grad():
        curr_tokens = tokens.clone()
        for layer in model.fusion_transformer.layers:
            attn_layer = layer.self_attn
            qkv = F.linear(curr_tokens, attn_layer.in_proj_weight, attn_layer.in_proj_bias)
            qkv = qkv.reshape(B, N, 3, D).permute(2, 0, 1, 3)
            q, k = qkv[0], qkv[1]
            scores = torch.matmul(q, k.transpose(-2, -1)) / (D ** 0.5)
            attn_maps.append(F.softmax(scores, dim=-1))
            curr_tokens = layer(curr_tokens)
    return attn_maps

# --- 2. Initialization & Model Loading ---
model_path = os.path.join(output_dir_mtl, "best_mamba_fusion.pt") # Το path του V3
print(f"🔍 Loading Final V3 Mamba Model: {model_path}")

model_eval = AdvancedMaskGuidedModel_v3_Mamba(
    clinical_len=CLINICAL_FEATURE_SIZE,
    radiomics_len=RADIOMICS_LEN,
    d_model=128
).to(DEVICE)

model_eval.load_state_dict(torch.load(model_path, map_location=DEVICE))
model_eval.eval()

# --- 3. Metrics Setup ---
dice_metric = DiceMetric(include_background=False, reduction="mean")
catcher = TransformerInputCatcher(model_eval)
all_probs_g, all_preds_g, all_lbls_g = [], [], []
all_preds_d, all_lbls_d = [], []
total_L1, count = None, 0

# --- 4. Main Evaluation Loop ---
print("\n🧪 Running Trinity Evaluation (Segmentation + 3-way + 2-way + Attention)...")

with torch.no_grad():
    for loader in [val_loader_mtl, test_loader_mtl]:
        for batch in tqdm(loader, leave=False):
            img, clin, rad = batch[image_key].to(DEVICE), batch[clinical_key].to(DEVICE), batch[radiomics_key].to(DEVICE)
            lbl_g, lbl_d, seg_true = batch[grade_key].to(DEVICE), batch[diag_key].to(DEVICE), batch[label_key].to(DEVICE)

            outputs = model_eval(img, clin, rad)
            l_seg, p_grade, p_diag = outputs[0], outputs[1], outputs[2] # Correct V3 Unpacking

            # A. Dice
            v_out = [Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])(i) for i in decollate_batch(l_seg)]
            v_lab = [AsDiscrete(to_onehot=None)(i) for i in decollate_batch(seg_true)]
            dice_metric(y_pred=v_out, y=v_lab)

            # B. Classification
            all_probs_g.extend(torch.softmax(p_grade, 1).cpu().numpy())
            all_preds_g.extend(torch.argmax(p_grade, 1).cpu().numpy())
            all_lbls_g.extend(lbl_g.cpu().numpy())
            all_preds_d.extend(torch.argmax(p_diag, 1).cpu().numpy())
            all_lbls_d.extend(lbl_d.cpu().numpy())

            # C. Global Attention (Layer 2)
            if catcher.tokens is not None:
                maps = get_attention_maps(model_eval, catcher)
                curr_L1 = maps[1].sum(dim=0).cpu() # Focus on Layer 2 (Decision)
                if total_L1 is None: total_L1 = curr_L1
                else: total_L1 += curr_L1
                count += img.shape[0]

catcher.remove()

# --- 5. LOGICAL 2-WAY MAPPING & AGREEMENT ---
all_probs_g = np.array(all_probs_g)
# LGG = P(G2) + P(G3)
all_probs_g2 = np.stack([all_probs_g[:, 0] + all_probs_g[:, 1], all_probs_g[:, 2]], axis=1)
all_preds_g2 = np.argmax(all_probs_g2, axis=1)
all_lbls_g2 = (np.array(all_lbls_g) == 2).astype(int)

# Consistency Check: Πόσο συχνά η 3-way πρόβλεψη συμφωνεί με τη 2-way
hard_3w_as_2w = (np.array(all_preds_g) == 2).astype(int)
agreement = np.mean(hard_3w_as_2w == all_preds_g2)

# --- 6. FINAL REPORTS ---
print("\n" + "="*50)
print(f"📊 FINAL RESULTS (Agreement Rate: {agreement:.2%})")
print("="*50)
print(f"🩺 Dice Score: {dice_metric.aggregate().item():.4f}")

print("\n🧠 3-WAY GRADE REPORT")
print(classification_report(all_lbls_g, all_preds_g, target_names=[str(c) for c in GRADE_CLASSES], zero_division=0))

print("\n⚖️ 2-WAY GRADE REPORT (LGG vs HGG)")
print(classification_report(all_lbls_g2, all_preds_g2, target_names=["LGG", "HGG"], zero_division=0))

# --- 7. ATTENTION HEATMAP VISUALIZATION ---
if count > 0:
    mean_L1 = (total_L1 / count).numpy()
    labels = ["Grade_T", "Diag_T", "Core_Img", "Peri_Img", "Clinical", "Radiomics"]

    plt.figure(figsize=(12, 9))
    sns.heatmap(mean_L1, annot=True, fmt=".2f", xticklabels=labels, yticklabels=labels, cmap="magma", vmin=0, vmax=0.4)
    plt.title(f"Global Cross-Modal Attention (Decision Layer)\nAvg over {count} Patients", fontsize=14)
    # Highlight Grade Decision Row
    plt.gca().add_patch(plt.Rectangle((0, 0), 6, 1, fill=False, edgecolor='cyan', lw=4))
    plt.savefig(os.path.join(output_dir_mtl, "final_attention_xai.png"), dpi=300)
    plt.show()

In [ ]:
del model_mtl
torch.cuda.empty_cache()
gc.collect()

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# --- ΡΥΘΜΙΣΕΙΣ ---
BASE_DIR = '/content/drive/MyDrive/Διπλωματικη/runs'
SAVE_DIR = '/content/drive/MyDrive/Διπλωματικη/THESIS_FINAL_VISUALS'
os.makedirs(SAVE_DIR, exist_ok=True)

# Ονόματα στηλών
COL_EPOCH = 'epoch'
COL_TRAIN_LOSS = 'train_loss'
COL_VAL_DICE = 'val_dice'

# Λεξικό με τα paths των CSVs
models_data = {
    'U-Net': os.path.join(BASE_DIR, 'segmentation_unet_scratch', 'history_unet_scratch.csv'),
    'SegResNet': os.path.join(BASE_DIR, 'segmentation_segresnet_scratch', 'history_segresnet_scratch.csv'),
    'SegMamba': os.path.join(BASE_DIR, 'segmentation_segmamba_scratch', 'history_segmamba_scratch.csv'),
    'SwinUNETR': os.path.join(BASE_DIR, 'segmentation_swin_unetr_scratch', 'history_swin_unetr_scratch.csv'),
    'SwinUNETR(SSL)': os.path.join(BASE_DIR, 'segmentation_finetune_custom', 'training_history.csv')
}

# Ακαδημαϊκή αισθητική γραφημάτων
sns.set_context("paper", font_scale=1.4)
sns.set_style("whitegrid")
plt.rcParams['font.family'] = 'serif'

# Λεξικό Χρωμάτων
colors = {
    'U-Net': 'red',
    'SegResNet': 'blue',
    'SegMamba': 'green',
    'SwinUNETR': 'purple',
    'SwinUNETR(SSL)': 'orange'
}

# --- 2. ΦΟΡΤΩΣΗ ΚΑΙ ΔΙΟΡΘΩΣΗ ΔΕΔΟΜΕΝΩΝ ---
dfs = {}
for name, path in models_data.items():
    if os.path.exists(path):
        # Διαβάζουμε το CSV. (Αν τα δεδομένα σου χωρίζονται με κενά, βάλε sep='\s+')
        try:
            df = pd.read_csv(path)
        except Exception:
            df = pd.read_csv(path, sep='\s+') # Fallback αν δεν έχει κόμματα

        # ΔΙΟΡΘΩΣΗ ΓΙΑ ΤΟ BUGGY CSV:
        if name == 'SwinUNETR(SSL)':
            # Πετάμε όποια στήλη είναι εντελώς άδεια (π.χ. το ψεύτικο val_dice)
            df = df.dropna(axis=1, how='all')
            # Μετονομάζουμε βίαια τις 3 στήλες που έμειναν στα σωστά ονόματα!
            # Η 1η είναι epoch, η 2η είναι loss, η 3η είναι dice.
            df.columns = [COL_EPOCH, COL_TRAIN_LOSS, COL_VAL_DICE]

        dfs[name] = df
    else:
        print(f"⚠️ Δεν βρέθηκε το αρχείο: {path}")

# ==================================================
# ΓΡΑΦΗΜΑ 1: Training Loss Comparison
# ==================================================
plt.figure(figsize=(10, 6))
for name, df in dfs.items():
    if COL_TRAIN_LOSS in df.columns:
        plot_color = colors.get(name, 'black')
        plt.plot(df[COL_EPOCH], df[COL_TRAIN_LOSS], label=name, color=plot_color, linewidth=2, alpha=0.8)

plt.title('Training Loss Convergence Across Architectures', weight='bold', pad=15)
plt.xlabel('Epochs')
plt.xlim(0, 60)
plt.ylabel('Training Loss (Dice-CE)')
plt.legend(title='Architectures', frameon=True, shadow=True)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'combined_training_loss.png'), dpi=300)
plt.close()

# ==================================================
# ΓΡΑΦΗΜΑ 2: Validation Dice Comparison
# ==================================================
plt.figure(figsize=(10, 6))
for name, df in dfs.items():
    if COL_VAL_DICE in df.columns:
        plot_color = colors.get(name, 'black')
        # Υπολογισμός κυλιόμενου μέσου όρου (rolling mean) για εξομάλυνση της καμπύλης
        smoothed_dice = df[COL_VAL_DICE].rolling(window=3, min_periods=1).mean()
        plt.plot(df[COL_EPOCH], smoothed_dice, label=name, color=plot_color, linewidth=2.5, alpha=0.9)

plt.title('Validation Dice Score Progression', weight='bold', pad=15)
plt.xlabel('Epochs')
plt.xlim(0, 60)
plt.ylabel('Mean Dice Coefficient')
plt.ylim(0, 1.0)
plt.legend(title='Architectures', loc='lower right', frameon=True, shadow=True)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'combined_val_dice.png'), dpi=300)
plt.close()

print(f"✅ Τα γραφήματα αποθηκεύτηκαν επιτυχώς στο: {SAVE_DIR}")